# DME Express — Operations Summary Dashboard

**Version v1.6.0 — 2026-08-15**

Produces, for the **company**, each **VP**, each **metro** and each **warehouse**:
technician productivity, **census per technician**, **overtime**, **lost equipment**,
redeliveries and **stock outs** — all **weekly, with a trailing 4-week average** — and
three new document deliverables: an **editable Word metric dictionary**, an **editable Word
recommendations document per VP**, and **one consolidated publishable PDF**.

**Changelog v1.6.0**

*Version note: the repo convention in `CLAUDE.md` is that every revision bumps the version,
so this is v1.6.0 rather than an edit in place on v1.5.0. v1.5.0 is left on disk unchanged
so the two can be diffed.*

- **FIX — Cell 13.5 reported the census mapping gap as one number when it is two, and
  warned about the wrong one.** v1.5.0 counted every unmapped patient-day together and told
  the analyst to *"check the warehouse vocabulary in `SERP_APC_DAILY` against
  `SERP_WAREHOUSES`"*. That warning fired on v1.5.0's own change: v1.5.0 started **keeping**
  virtual (`Z%`) census rather than dropping it, and a virtual code can never map at this
  join — Cell 8.1b re-attributes virtual *tickets* to physical warehouses, so no `Z%` value
  survives as a `tech_warehouse` for census to join against. So 100% of virtual census landed
  in the "vocabulary gap" bucket, inflated it, and pointed at a warehouse-naming problem for
  something that is a deliberate design decision two cells earlier. The two causes are now
  separate numbers with separate meanings — *virtual* (expected, unresolvable without a new
  key on the census feed) and *vocabulary gap* (a real, actionable mismatch between two name
  lists) — and only the second can trip the warning. Both are in the new
  `Census_Unmapped_Split` sheet.
- **FIX — census trailing ratios mixed a four-week numerator with a two-week denominator.**
  `techs_distinct` and `avg_active_techs` were classified as *rates*, so they were averaged
  over the weeks an entity happened to appear in, while `pt_days` and `census_days` were
  classified as *counts* and zero-filled across the whole window. A site with census in four
  weeks but technicians in only two therefore divided a four-week census by a two-week
  headcount, and the trailing ratio read **high for precisely the sites that had a quiet
  week**. Both are counts — a week in which a site had no technicians on the road genuinely
  had zero of them — and every census trailing ratio is now rebuilt from the pooled count
  columns. `attendance_rate_pct` is pooled too; v1.5.0 left that one averaged.
- **NEW — editable Word metric dictionary** (`OpsDashboard_MetricDictionary_*.docx`, Cell 21).
  Every published metric with its output column, its calculation, the **specific source
  fields** behind it, its grain and denominator, its direction, and the assumptions a reader
  needs in order to interpret it — plus the cross-cutting assumptions that apply to all of
  them and the restatements in this release. **The dictionary is validated against the code
  on every run:** a documented metric that the notebook does not produce raises and fails the
  build, and any published column with no definition is listed as a gap. A dictionary that
  drifts from the notebook is worse than none, because it is trusted.
- **NEW — editable Word recommendations document per VP**
  (`OpsDashboard_Recommendations_<VP>_*.docx`, Cell 22). The top 5 actions for each VP,
  ranked by how far that VP sits from the company figure in the adverse direction, each with
  the figure it rests on, why it matters, concrete actions, and — where the data supports
  naming a person — the technicians involved.
  **Naming individuals is deliberately narrow.** Technicians are named for **tickets per
  active day** and **overtime**, which is what the request asked for and what the data can
  actually attach to a person: tickets per active day is denominator-honest (PTO and
  part-time do not distort it), carries a presence floor, and is compared to the technician's
  **own VP median** rather than a company one, so route density is not mistaken for effort;
  overtime is computed on the person from de-duplicated payroll hours before any site
  allocation. Everywhere else the documents name **sites**: lost equipment because
  `CLAUDE.md` is explicit that attribution is proximity and not fault, attendance because the
  feed cannot separate approved PTO from an unfilled schedule, and stock outs because they
  are supply-side. `COACHING_CAVEAT` is attached to every row of `tech_perf_vp` as well as to
  every list in the documents, so a name cannot be copied out of a table without it.
- **NEW — one consolidated publishable PDF** (`OpsDashboard_Publishable_*.pdf`, Cell 23):
  cover, executive summary, metric dictionary, company page, then **each VP's dashboard page
  followed immediately by that VP's recommendations**, then metros, warehouses, and an
  appendix of limitations and open questions. Bookmarked by section. The **restatement notice
  sits on page 2, not in the appendix** — two figures move against the last pack, one by
  ~40%, and one of those has already been circulated at ~120.
- **The Word documents and the PDF are rendered from ONE content model** (Cell 20), so a
  metric definition cannot say two different things in two places. The Word files are
  genuinely editable — real styles, real tables, no images or text boxes — so an analyst can
  change a definition or delete a recommendation and re-circulate without re-running the
  notebook.
- **The recommendation evidence is published, not just the conclusions** — new `VP_Scorecard`
  and `Tech_Detail_By_VP` sheets in the workbook. A VP who disputes a recommendation can be
  shown the row it came from.
- **Narrative pages match the dashboard page size** (23×15in) rather than Letter, because a
  distributed PDF that alternates page size makes every reader fight their viewer, and
  scaling twelve-panel pages down to Letter makes the panels unreadable. Type is scaled to
  suit the page; the text column is held to an 11-inch measure and the surplus width becomes
  margin, because a 23-inch line length is unreadable.
- New dependencies: `python-docx`, `reportlab`, `pypdf` (Cell 1). Each is imported behind a
  guard — if one is missing the corresponding deliverable is **skipped with a message**
  naming what to install, and the dashboard PDF and Excel workbook are unaffected.

**Changelog v1.5.0**

Released in response to an analyst's sanity check: company ADC ≈ 25,000 over just under
300 technicians is ≈ 85 patients per technician, but the v1.4.0 pack printed ≈ 120. The
audit that followed is written up in `insights/metric-audit-adc-per-tech-2026-08-15.md`.
The ≈ 120 was arithmetically correct and wrong as labelled; the audit also turned up one
hard bug in a different metric.

- **CENSUS PER TECHNICIAN IS RESTATED — the denominator was average attendance, not
  headcount.** v1.4.0 divided ADC by `weekday tech-days ÷ weekdays in month`, i.e. by the
  average number of technicians *on the road on a working day*, and titled it "Census per
  Active Technician". Using this notebook's own Jul-2026 inputs: 4,112 apportioned
  tech-days ÷ 23 weekdays = **178.8**, against **251** distinct technicians on tickets and
  **~300** on payroll. ADC 23,356 therefore reads **130.6 / 93.1 / 77.9** depending purely
  on which denominator you pick. The entire gap is an average weekday attendance rate of
  **71–74%** — PTO, sick, training, part-time, and weekend work, which the weekday filter
  discards.
  **Why this is a definitional error and not a labelling quibble.** ADC is a **stock** —
  patients on service on an average day. Tech-days are a **flow** — attendance events. A
  patient does not leave service because their technician took PTO, so the caseload is
  carried by the whole roster every day, and the honest caseload denominator is
  **headcount**. Read the other way the v1.4.0 figure is a real quantity — *field coverage
  intensity* — but it is not caseload and must never sit beside a per-technician staffing
  target.
  **Three measures now travel together** on every census output: `census_per_tech_headcount`
  (**headline**), `census_per_active_tech_weekday` (the v1.4.0 basis, retained so the
  restatement is measurable line by line) and `attendance_rate_pct` (the wedge). Cell 13.5
  prints the month-by-month bridge on every run.
- **RETIRED — v1.4.0's stated census limitation was pointing at the wrong effect.** Every
  panel and sheet warned that "ADC is a 7-day calendar average while tech-days are weekdays
  only". Census is a *stock*, so a 7-day mean and a 5-day mean differ only by the weekend
  admit/discharge rhythm — low single digits — while the attendance wedge it did *not* warn
  about is ~40%, and was written up as a deliberate design choice.
- **PRODUCTIVITY IS UNAFFECTED, and that is a finding too.**
  `tickets_per_active_day` is flow ÷ flow, and its denominator is asserted every run to
  reconcile to distinct (tech, date). Overtime, redeliveries and stock outs never divided
  by census at all.
- **BUG FIX — VP and metro lost-equipment rates were divided by the *whole company's*
  census.** Cell 14 branched on `if 'tech_warehouse' in gcols`, merging per-warehouse ADC
  in that branch and the company total in the `else`. Both the metro and the VP grain took
  the `else`, so every VP row and every metro row divided its lost cost by the entire
  company's patient-days. `lost_cost_per_1k_pt_days` on a VP page was understated by
  roughly (company census ÷ that VP's census) — order of **5–10× for a VP**, far more for a
  metro. Company and warehouse grains were correct, which is exactly why it survived
  review: the error is invisible unless you compare two grains. Every grain now joins census
  at its own grain, and the cell **proves** it.
- **CENSUS SOURCE — three defects in one query.** Cell 7.5 built *two different definitions
  of census* from `SERP_APC_DAILY`: the APC snapshot excluded facility `(F)`,
  inpatient-unit `(IPU)` and `Contract Test` customers, and the ADC series that every
  census metric and the lost-equipment rate actually consumed excluded **none** of them.
  ADC was also a **sum of per-warehouse monthly averages taken over different day sets**,
  which corresponds to no actual day when site coverage differs — live right now for the
  partial current month and for the five sites the insights log records going silent after
  Jun-2026. ADC is now **pooled patient-days ÷ distinct dates observed** at every grain.
  And **virtual (`Z%`) census is no longer dropped from the numerator while its technicians
  stay in the denominator** — the same asymmetry that produced the false July productivity
  drop, re-introduced in a new panel.
- **Suppression floor corrected.** v1.4.0 tested `tech_days < 20`, but the ratio's
  denominator is `tech_days ÷ weekdays` — at the floor it was suppressing at ~0.95
  technicians, i.e. barely at all. The test is now on the denominators themselves.
- **EVERY METRIC IS NOW WEEKLY WITH A TRAILING 4-WEEK AVERAGE.** New **Cell 4.3** owns one
  calendar week spine and the trailing-average helper.
  **The trailing average is computed on the spine, not on the rows present.** A rolling mean
  over "the last four rows" is not a trailing 4-week average when an entity has a quiet
  week — it silently reaches back six calendar weeks and reports it as four. Counts treat a
  missing week as **zero** inside the entity's span; rates leave it **missing**. Partial
  (window-clipped) weeks are excluded from trailing windows and drawn hatched.
  **Rate trailing columns are pooled** — trailing numerator ÷ trailing denominator, not the
  mean of four weekly rates.
- **Overtime keeps its own Thu–Wed payroll spine** and is *not* alignable week-for-week
  with the Sun–Sat panels. OT site attribution apportions by each technician's ticket share
  **over that same payroll week**, not their monthly share.
- **PDF layout 2×4 → 3×4, and no panel has two y-axes any more.** A twin-axis chart has no
  defensible crossing point: where the line appears to overtake the bars is an artifact of
  two independently auto-ranged scales. Volumes are rows 0–1; the four normalised rates have
  their own panels on row 2. The v1.2.0 re-attribution overlay lost its second axis too and
  is now a rug of ticks along the top of the productivity panel.
- **Two family colours were replaced because they failed a machine check**, not a taste
  argument: overtime `#8c564b` and stock outs `#7f7f7f` fell below the chroma floor, and
  redeliveries `#e377c2` sat under 3:1 against the surface.

**Still open** — see `insights/questions-for-cfo.md`:

- Which denominator does a per-technician caseload *target* use — payroll headcount,
  distinct technicians on tickets, or FTE? The three differ by ~20% among themselves.
- Should facility `(F)` and inpatient-unit `(IPU)` census count toward technician caseload?
- `NOT LIKE` is NULL-unsafe, so a NULL `customer` falls out of the filtered census.
- ~300 payroll technicians versus ~250 attributed to tickets.
---

**Changelog v1.4.0**

- **THE PAYROLL WEEK IS NOW THURSDAY–WEDNESDAY, NOT SUN–SAT.** v1.3.0 inferred overtime
  over an FLSA-textbook Sun–Sat week. That was a default, not an observed fact about this
  employer: Paylocity (the `PLC_*` feeds) runs a **Thursday-to-Wednesday** week, so every
  technician whose hours straddled a Wednesday had their overtime attributed to the wrong
  week — and any week where the 40-hour line was crossed only because two payroll weeks
  were fused reported overtime that payroll never paid. `OT_WEEK_START_DOW` (Cell 3.1) now
  carries the boundary; set it to `6` to reproduce v1.3.0 exactly. **The change restates
  published overtime** — the size of the restatement is printed by Cell 15.5 on every run
  rather than left for the reader to work out, and the Sun–Sat basis is retained as
  `ot_hours_sunsat` in `OT_Weekly_Detail`.
  *This is not a free-floating assumption:* `sql_explorer_2026-08-15` Cell 8 determines
  the boundary empirically from the feed (pay-period run detection on the repeated
  `Reg_Hrs` value, plus a recorded-OT fit across all seven candidate start days).
  **Run it and confirm Thursday before circulating restated overtime.**
- **NOTE — the productivity week is unchanged.** Cell 13 still uses Sun–Sat. An
  operational reporting week and a payroll week are different objects; forcing them to
  match would silently restate every published productivity figure. Outputs label which
  week basis they use.
- **NEW SECTION — census per active technician** (Cell 13.5, new panel on every page).
  Average daily census (ADC, from `SERP_APC_DAILY`) over the average number of
  technicians active per weekday, monthly, at company / VP / metro / warehouse grain.
  The denominator is the *same apportioned tech-day count* the productivity metric uses,
  so the two are directly comparable. **Limitation, carried on every output:** ADC is a
  7-day calendar average while active tech-days are weekdays only, so the ratio reads
  "patients on service per technician-weekday". Months below `CENSUS_MIN_TECH_DAYS`
  tech-days are suppressed rather than plotted — a thin denominator makes the ratio
  explode.
- **Top/bottom technicians are now legibly highlighted** — green for top, red for bottom,
  in both the PDF table and the Excel sheets. The previous PDF tints (`#eaf4ea` /
  `#fdecec`) were close to invisible in print. Green-versus-red is exactly the pair a
  red–green colour-blind reader cannot separate, so **colour is never the only channel**:
  every row keeps its `Top`/`Bottom` word and gains a ▲/▼ glyph. Contrast was computed,
  not eyeballed — body text sits at 17.2:1 (top) and 16.3:1 (bottom) on its fill, and the
  status-coloured rank label at 6.2:1 / 6.3:1. The saturated status hues are deliberately
  *not* used behind text: white on `#0ca30c` measures 3.35:1 and fails.
- **Page layout 2×3 → 2×4** to seat the census panel; the top/bottom table now spans two
  cells and is readable at print size for the first time.

**Changelog v1.3.0**

*Sections populated*
- **LOST EQUIPMENT — source was dead, now rebuilt.** `SERP_ACTIVE_TAGGED_INV.Lost`
  is NULL on **all 583,530 rows**, so `WHERE ATI.Lost IS NOT NULL` could never
  return anything and the panel had been structurally empty. v1.0.1's probe found
  the zero and blamed the feed, but the register had moved. Audited every
  candidate: `ATI_HISTORY` froze 2025-03-30; `SERP_LOST_EQUIPMENT_MANUAL` stops in
  2025; `Quarterly_Lost_Equipment` holds one month, one warehouse and one asset
  repeated 3,047 times. The live source is **`SERP_LOST_EQUIPMENT`** (91,023 rows,
  2025-01 → 2026-05, warehouse names matching the master 71 of 72), and it adds
  Lost Reason, Resolution, Resolved Date and Unit Cost.
  Two traps handled: `[Lost Date]` is not a date but an HTML fragment
  (`01/02/2025<br/>(147)` — `TRY_CONVERT` fails on **100%** of rows, which is how a
  naive port would silently produce the same empty panel), and `[Unit Cost]` is a
  currency string. The parse failure rate is now **asserted**, not printed.
  **The feed ends 2026-05, so every page and sheet carries a staleness banner** and
  a >45-day freshness warning: a final-month decline is missing data, not recovery.
  The **2025-08-01 bulk event (30,902 rows, $3.06M, 99.9% of it in the virtual
  buckets `Z Equipment Collections` / `Z CS`) is excluded from every metric and
  trend** per the analyst's instruction, and reported in `Lost_Bulk_Events`. New
  spikes are surveilled and warned about but never auto-excluded — a date has to be
  added to `LOST_BULK_EVENT_DATES` deliberately. `recovery_rate_pct` is computed
  only over cohorts ≥90 days old, because a raw rate falls every month by
  construction when recent losses have had no time to be found.
- **STOCK OUTS — reframed from "open backlog" to incidence + outcome + time-to-fulfil.**
  `Status='EnRoute'` is not an open backlog: 19,891 of those 21,438 orders carry a
  completion date, so rows are retained after fulfilment and v1.1.0's caveat that
  fulfilled stock-outs *"disappear from this query entirely"* was wrong. Every other
  status value is frozen at a single 2026-01-20 load, so "oldest open 1,021 days" was a
  stale row rather than an aging crisis. EnRoute by creation month is therefore **true
  incidence**.
  Orders resolve **three** ways, and the middle one is the find: **fulfilled 13,856
  (64.7%)**, **canceled/abandoned 6,035 (28.2%)** — where the only completion evidence
  sits on a *canceled* ticket — and **genuinely open 1,518 (7.1%)**, median 115 days old.
  Counting canceled tickets as deliveries inflates fulfilment by ~28 points to a false
  93%, so the three outcomes are reported separately and the open and abandoned sets get
  their own sheets: chasing an open order and explaining a dropped one are different
  jobs. Time-to-fulfil is P25 3 / median 6 / P75 9 / P90 17 days — a distribution, not a
  mean. Stock-out events are now bounded by the dashboard window like every other panel
  (27 pre-2025 orders excluded and reported).
- **STOCK OUTS — warehouse and metro pages now carry numbers.** v1.1.0 asserted no
  physical warehouse exists on stock-out rows and printed an n/a notice. It does:
  5,726 EnRoute rows carry one of 65 real warehouses, and the order join (via the new
  shared `order_wh_map`, then widened to tickets under any reason) resolves **76.0% of
  orders** to a real site.
- **OVERTIME — new section.** v1.0.0 skipped payroll on purpose; this needs it. Four
  landmines in `PLC_EMPLOYEE_HOURS`, all load-bearing:
  (a) **the feed re-loads overlapping windows, so rows accumulate** — Jun-2026 holds
  13,697 Patient-Care-Technician rows for only 5,729 distinct employee-days, one
  employee-day appearing 30 times. Un-deduplicated, inferred June OT is 77,957 hours
  on 118,747 (**66% OT**); after dedup, 7,383 on 46,694.
  (b) **the recorded OT columns are unusable** — `Reg_Hrs`/`OT1_Hrs`/`OT2_Hrs`/
  `Paid_Hrs`/`Est_*` are pay-period values repeated on every daily row (~14× daily
  `Hours`), `OT1_Hrs` is 0 for every month from 2026-02 on and `OT2_Hrs` is 0
  throughout. OT is inferred FLSA-style from daily `Hours`; a diagnostic re-proves
  this on live data every run so it is not re-litigated.
  (c) 709 rows carry NULL department/employee and 2,600–3,100 hours each (1.05M
  hours, more than every real row combined) — explicitly guarded.
  (d) the feed holds **future-dated rows** (to 2026-08-19 against an as-of of
  2026-08-13) — dropped by a half-open bound.
  **`Pay_Type` separates PTO and Holiday, so paid leave no longer inflates the
  40-hour threshold** — this retires the standing v1.34.0 caveat and is worth 18%
  (Jul-2026: 10,264 hours including leave vs 8,384 excluding). Both are emitted.
  Scope: warehouse/metro/VP panels cover Patient Care Technicians only, so overtime
  and productivity share one population; company pages add an all-departments line.
  Payroll has no usable warehouse key (**2 of 139** `Location_Name` values match the
  master), so hours reach a site through the technician name match and are then
  apportioned by ticket share — the same rule Cell 13 uses for active days.
- **NEW Cell 11.3 — `order_wh_map`**, one shared order → physical warehouse (+ VP,
  state, metro, completion date) map, used by both stock outs and lost equipment
  instead of each inventing its own attribution.
- Renderer went **2×2 → 2×3**: productivity, overtime, lost equipment,
  redeliveries, stock outs, top/bottom. Excel grows to ~42 sheets.

*Audit fixes (see `insights/code-audit-ops-dashboard-2026-08-14.md`)*
- `Redel_Unlinked_Monthly` is now actually written. Cell 11.2 has been printing
  *"exported to Redel_Unlinked_Monthly"* since v1.0.0 for a sheet that never
  existed, leaving 10,476 excluded redelivery events undocumented.
- The two conflicting redelivery denominators are unified on **attributed**
  tickets, and the column is renamed to state its basis.
- Key Vault login no longer discards the working server string — the bare hostname
  it substituted produced `[08001] TCP Provider: Timeout error [258]`;
  `tcp:{host},1433` connects first try.
- `display(df_tx.head(3))` removed: it rendered `record_id` and `order_num` into
  notebook output, against rule 2. Replaced with a shape/dtype/null summary.
- `plt.show()` now fires only for the company and VP pages. It previously ran for
  all ~85, embedding 110 figures and inflating the saved notebook to 33 MB.
- `run_query` **enforces read-only in-process** (SELECT/WITH only), which rule 4
  describes but nothing implemented.
- The H3 unparseable-date probe is windowed instead of scanning the whole table.
- `resolve_state_series`'s docstring no longer claims to be vectorized.
- Dead weight removed: `df_master`, and the unused `OUTLIER_Z_THRESH`,
  `DATA_FRESHNESS_THRESHOLD`, `MIN_HOURS_MONTH`, `STATE_PREFIX_LEN` constants plus
  the `scipy.stats` / `statsmodels` imports. `df_inventory_total` and `df_adc` were
  extracted and abandoned — they are now live as lost-equipment denominators.
  `DASH_MIN_ACTIVE_DAYS` is derived from `MIN_ACTIVE_DAYS_RANK` rather than
  duplicating the number. The Cell 0 claim that payroll and census "are
  deliberately NOT pulled" is gone — both are pulled and used.
- Still standing, deliberately: Cell 10's visit-deduplication output remains unused
  by this notebook (it is lifted from `tech_workload`, where it is used). Left in
  place rather than removed so the two notebooks stay comparable, but it is not a
  dependency of anything here.

*Known limits carried into this version*
- Lost equipment ends 2026-05; company/VP/state stock-out figures are sound while
  warehouse-level ones rest on the 76% that the order join resolves; overtime site
  rows sum to less than the company row by the unattributable payroll share; and
  `tech_workload` v1.35.0 still carries the v1.1.0 Z-warehouse exclusion and
  whole-day denominator, so it will not reconcile to this notebook until re-synced.

**Version v1.2.0 — 2026-08-14**

**Changelog v1.2.0**
- **ROOT CAUSE FOUND for the "July productivity drop": it is a measurement
  artifact, not an operations slowdown.** From 2026-06 a virtual warehouse
  (`Z CS`) began carrying ordinary completed field tickets — Additional
  Equipment (D), New Admit (D), Hospital Discharge (D), Respiratory Distress (D).
  v1.1.0's extraction filter `Tech_Warehouse NOT LIKE 'Z%'` deleted those tickets
  from the **numerator** while the technician's calendar day still counted in full
  in the **denominator**. Virtual-warehouse share of eligible weekday tickets:
  0.7% (May-2026) → 9.3% (Jun) → **24.3% (Jul)**. Company tickets per active
  tech-day, Jun→Jul: **−13.2% as reported vs −2.1% corrected**. Raw ticket volume
  actually *rose* 6.4% in July and census (ADC) hit an all-time high of 23,356 —
  there was no work slowdown.
- EXTRACTION (Cell 7.1): the blanket `Z%` exclusion is removed. Virtual-warehouse
  rows now arrive tagged (`_is_virtual_wh`, `_wh_ticket_stamped`).
- NEW Cell 8.1b — VIRTUAL WAREHOUSE RESOLUTION: re-attributes a virtual-warehouse
  ticket to the physical warehouse the same technician actually worked that
  **day**, else that **week**, else that **month**; anything left becomes
  `Virtual - Unresolved` (never silently dropped). Jul-2026 resolution: 4,019
  same-day + 1,885 same-week + 53 same-month, **2 unresolved**. Runs before the
  matcher, so state, VP, metro, redelivery linking and Excel all see a physical
  warehouse. Config-gated by `INCLUDE_VIRTUAL_WH_IN_PRODUCTIVITY` (False
  reproduces v1.1.0 exactly).
- **SECOND, INDEPENDENT DEFECT — warehouse-grain active days were double-counted.**
  A technician whose day spanned several warehouses contributed a *whole* active
  day to *each* of them while their tickets were split, deflating every site's
  ratio by the fragmentation factor. Warehouses per technician per month: 1.29
  (May-2026) → 2.22 (Jun) → **2.86 (Jul)**, so per-warehouse active-day sums ran
  **34.9% above** the honest company total in July (≈3% historically). Cell 13 now
  apportions each technician-day across warehouses by that day's ticket share
  (`active_day_equiv`), reconciling exactly to distinct (tech, date). Jun→Jul
  warehouse median: **−14.2% as reported → −4.3% corrected**; warehouses declining
  43/50 → 35/50. Config-gated by `APPORTION_TECH_DAYS`.
- NEW Cell 13.0 — ROUTING-BREAK DIAGNOSTIC (D1–D5): monthly filter funnel,
  virtual-warehouse volume by code, the mechanism with as-reported vs corrected
  ratios, warehouse-grain denominator health, and entity churn. Re-run whenever a
  series steps: it separates "the feed moved" from "operations moved". Adds four
  `Diag_*` Excel sheets.
- Rankings (Cell 17) rank on the apportioned ratio; the eligibility floor stays on
  whole active weekdays (a presence test, not a productivity one).
- Weekly panels mark `ROUTING_CHANGE_DATE` with a structural-break line and plot
  the virtual-warehouse ticket share, so future numerator leakage shows up on the
  page instead of reading as a productivity decline.
- STILL OPEN (`insights/questions-for-cfo.md`): what `Z CS` is and who owns it,
  and whether the five warehouses that went silent after Jun-2026 (R14 San Antonio
  WH 2, R14 Harlingen, Mailouts - Texas, R01 Walker, R15 T Storage) closed or were
  renamed. Until answered, treat Jun-2026-onward **warehouse-level** comparisons as
  provisional; company/VP/state levels are sound.
- Outputs stripped from this file (never committed — CLAUDE.md rule 3).

**Version v1.1.0 — 2026-08-14**

**Changelog v1.1.0**
- STOCK OUTS WIRED: `SERP_ORDER_MANAGEMENT_STOCK_OUTS`, Status='EnRoute' (source
  confirmed by owner). PII-minimized select (no patient/caregiver columns; address
  reduced to state code and dropped). Creation Date parsed Python-side (MM/DD/YYYY
  VARCHAR — locked lesson). Trend = OPEN BACKLOG by creation month (aging view),
  not historical incidence — the table is a snapshot of unfulfilled orders.
- ATTRIBUTION: company / state / VP only. Every stock-out row sits in the virtual
  warehouse 'Z Equipment Needed', so no physical warehouse/metro exists on the
  data; those pages show a notice instead of fabricated numbers. States served by
  multiple VPs are flagged shared_state and appear on each matching VP page.
- New Excel sheets: Co/State/VP_Stockout_Monthly, StockOut_Product_Backlog,
  StockOut_Aging_Detail. Renderer overlay now aligns stock-out points to the
  redelivery month axis by period (was positional).

**Version v1.0.1 — 2026-08-14**

**Changelog v1.0.1**
- FIX (live-run KeyError 'tx_metro'): Cell 15 rewritten against the FINAL redel_linked
  schema — the lifted linking cell renames tx_* columns back to plain names and drops
  the prefixed versions at its tail; v1.0.0 referenced the mid-cell names. Rates now
  key numerator and denominator on identical plain-name grains.
- NEW: ATI diagnostic probe in Cell 14 — fires only when the lost-equipment window is
  empty (as on the 2026-08-14 run) and reports whether lost-flagged rows exist at all,
  how many have unparseable Lost_Date, and the parseable date range, to separate a dead
  feed from a raw-string date-filter miss (the H3 pattern).
- Base: the user's executed v1.0.0 upload (their USER_ROOT=aenew edit preserved).

**Version v1.0.0 — 2026-08-14** | companion to `tech_workload` v1.35.0

Produces, for the **company**, each **VP**, each **metro**, and each **warehouse**:
technician productivity over time (weekly, rolling 4-week average), lost equipment
over time (monthly), redeliveries over time (monthly, with per-100-ticket rates),
stock outs over time (**pending source table — config-gated**), and disjoint
top/bottom technician lists with statistics.

**Outputs:** `OpsDashboard_Pages_{date}.pdf` (one page per entity, board reading
order: company → VPs → metros → warehouses, ~85 pages), per-VP/company PNGs, and
`OpsDashboard_Report_{date}.xlsx` (17+ sheets).

**Architecture (decided 2026-08-14):** self-contained — re-queries Azure SQL via the
Key Vault login and duplicates the v1.35.0 extraction/matching machinery. Duplicated
cells are marked LIFTED VERBATIM; when the main notebook's matching changes, re-sync
them or the two reports will diverge. _(v1.3.0: payroll and census ARE now pulled — see the v1.3.0 overtime and
lost-equipment notes above. The original v1.0.0 claim that they were deliberately
omitted no longer holds.)_

## Cell 1 — INSTALL DEPENDENCIES

In [1]:
# v1.6.0 adds python-docx (editable Word deliverables), reportlab (narrative PDF
# pages) and pypdf (merging those pages with the dashboard pages). Each is imported
# behind a guard in Cell 20 — a missing one skips its deliverable with a message
# rather than failing the run.
%pip install rapidfuzz matplotlib python-dotenv pypyodbc openpyxl python-dateutil azure-identity azure-keyvault-secrets python-docx reportlab pypdf

Note: you may need to restart the kernel to use updated packages.


## Cell 2 — IMPORTS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [2]:
# Standard library + analysis stack. azure.identity/azure.keyvault power the Key
# Vault login in Cell 4. rapidfuzz is optional: the _RAPIDFUZZ_OK flag
# lets later cells degrade gracefully instead of crashing on import.
import os, re, io, time, warnings, calendar as _cal
from datetime import datetime, date, timedelta
from collections import defaultdict
from functools import reduce
from dateutil.relativedelta import relativedelta
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from dotenv import load_dotenv
import pypyodbc as odbc
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

try:
    from rapidfuzz.distance import DamerauLevenshtein as _DL
    _RAPIDFUZZ_OK = True
except ImportError:
    _RAPIDFUZZ_OK = False
    print('rapidfuzz not installed — falling back to pure-Python Levenshtein.')
# v1.3.0: scipy.stats and statsmodels were imported behind a _SM_OK guard and never
# called anywhere in this notebook — removed. Re-add if a model is actually fitted.
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
CHART_DPI = 150
matplotlib.rcParams['figure.dpi'] = CHART_DPI
print(f'pandas {pd.__version__}  |  numpy {np.__version__}  |  rapidfuzz: {_RAPIDFUZZ_OK}')

pandas 2.3.3  |  numpy 2.3.4  |  rapidfuzz: True


## Cell 3.1 — RUN WINDOW, PATHS & THRESHOLDS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [3]:
RUN_DATE     = datetime.now().strftime('%Y-%m-%d')
FILTER_START = '2025-01-01'
AS_OF_DATE   = datetime.now().date() - timedelta(days=1)
FILTER_END   = AS_OF_DATE.strftime('%Y-%m-%d')
print(f'AS_OF_DATE (last full day of data): {AS_OF_DATE.isoformat()}')
EXCLUDE_CURRENT_MONTH = False
if EXCLUDE_CURRENT_MONTH:
    _last = datetime.now().replace(day=1) - relativedelta(days=1)
    FILTER_END = _last.strftime('%Y-%m-%d')

USER_ROOT = '/./Users/aenew'
#USER_ROOT = '/./Users/AlexNewton'
REPORT_ROOT = f'{USER_ROOT}/OneDrive - DME Express/Reports/TechWorkload'
ENV_FILE = f'{USER_ROOT}/OneDrive - DME Express/Documents/IT/Python/MSAIKey.env'  # _LEGACY v1.35.0: login now via Key Vault (Cell 4); delete with the commented .env login after 2026-Q4
DRIVER_NAME = 'ODBC Driver 18 for SQL Server'
SERVER_NAME = 'tcp:dmeexpress.database.windows.net,1433'
DATABASE_NAME = 'DMEEXPRESS'

FUZZY_EDIT_DIST  = 1
# v1.3.0: OUTLIER_Z_THRESH, DATA_FRESHNESS_THRESHOLD and MIN_HOURS_MONTH were lifted
# from tech_workload and never read by this notebook — removed rather than left to look
# load-bearing. OT_* below DO go live in v1.3.0 (Cell 15.5).

# ── v1.33.0 (N1): Overtime configuration ────────────────────────────────────
# FLSA-style inference from PLC daily hours: any hours over OT_WEEKLY_THRESHOLD
# in a Sun–Sat workweek count as overtime. Computed on RAW per-person hours
# (before any warehouse pro-rating) because the threshold applies to the
# employee, not the warehouse. CAVEAT: if PLC 'Hours' includes PTO/holiday pay,
# OT is overstated (PTO does not count toward the 40-hr threshold). Confirm the
# feed with payroll; if an earnings-code column exists, prefer it over inference.
OT_WEEKLY_THRESHOLD = 40.0   # hours per workweek before OT begins

# ── v1.4.0: THE PAYROLL WEEK IS THURSDAY -> WEDNESDAY ────────────────────────
# Monday=0 .. Sunday=6, matching dow_mon0 from the PLC extract. 3 = Thursday, so the
# workweek runs Thu -> Wed and the 40-hour threshold is applied across that span.
# This matches Paylocity, the payroll system behind the PLC_* tables. v1.3.0 used
# Sun-Sat (6), which is the FLSA textbook default and was never checked against the
# payroll system; it mis-attributed OT for anyone whose hours straddled a Wednesday.
#
# VERIFY, DO NOT ASSUME. sql_explorer_2026-08-15 Cell 8 derives this from the feed
# itself — pay-period run detection on the repeated Reg_Hrs value, corroborated by a
# recorded-OT fit over all seven candidate start days. Re-run it if payroll changes
# processor or period. Set to 6 to reproduce v1.3.0's numbers exactly.
OT_WEEK_START_DOW = 3
_DOW_ABBR = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
OT_WEEK_LABEL = f'{_DOW_ABBR[OT_WEEK_START_DOW]}-{_DOW_ABBR[(OT_WEEK_START_DOW + 6) % 7]}'
# The PRODUCTIVITY week (Cell 13) is deliberately NOT tied to this. An operational
# reporting week and a payroll week are different objects, and collapsing them would
# restate every published productivity figure as a side effect of a payroll fix.
PROD_WEEK_LABEL = 'Sun-Sat' 
MIN_HOURS_FOR_OT_RATE = 80.0 # total-hours floor before a tech appears in OT-rate rankings
MIN_ACTIVE_DAYS_RANK  = 30   # active-day floor for tech top/bottom ranking (M1)
PCT_DEPT = 'Patient Care Technician'


AS_OF_DATE (last full day of data): 2026-08-14


## Cell 3.2 — ROLE CLASSIFICATION & INCLUDED REASONS *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [4]:
# Role classification vocabularies: a matched employee whose title/dept hits the
# FIELD_TECH sets is countable workload; INTERNAL_OPS hits (dispatchers, CSRs) are
# excluded from productivity and reported separately. INCLUDED_REASONS is the
# locked ticket-reason whitelist — SQL filters on it, so adding a reason code
# upstream requires adding it here or those tickets silently never arrive.
FIELD_TECH_TITLES = {'patient care technician','service technician','lead technician','lead tech',
                     'warehouse technician','warehouse tech','warehouse manager','site manager','area manager',}
FIELD_TECH_DEPTS = {'field operations','warehouse'}
INTERNAL_OPS_TITLES = {'customer service','csr','dispatcher','dispatch','routing','call center','intake','scheduler','scheduling'}
INTERNAL_OPS_DEPTS = {'customer service','dispatch','routing','call center','intake','scheduling'}

INCLUDED_REASONS = [
    'Priority 1 - Hospital Discharge (D)',
    'Priority 1 - Respiratory Distress (D)',
    'Priority 1 - Respiratory Service/Exchange (D)',
    'Priority 1 - Respiratory Service/Exchange (P)',
    'Priority 1 - Respiratory Service/Exchange (S)',
    'Priority 1 - Service Correction (D)',
    'Priority 1 - Service Correction (P)',
    'Priority 2 - Exchange (S)',
    'Priority 2 - New Admit (D)',
    'Priority 2 - Respiratory Equipment (D)',
    'Priority 2 - Service/Exchange (D)',
    'Priority 2 - Service/Exchange (P)',
    'Priority 2 - Service/Repair (S)',
    'Priority 2 - Swap Out (Equipment Provider)',
    'Priority 3 - Additional Equipment (D)',
    'Priority 3 - Change Address (D)',
    'Priority 3 - Change Address (P)',
    'Priority 3 - Customer/Patient Request (P)',
    'Priority 3 - Exchange (S)',
    'Priority 3 - Inservice (S)',
    'Priority 3 - Live Discharge (P)',
    'Priority 3 - O2 Refill (D)',
    'Priority 3 - O2 Refill (P)',
    'Priority 3 - Patient Expired (P)',
    'Priority 3 - Respite Stay (D)',
    'Priority 3 - Respite Stay (P)',
    'Priority 3 - Service/Exchange (D)',
    'Priority 3 - Service/Exchange (P)',
    'Priority 3 - Swap Out (Equipment Provider)',
    'Split Order',
]


## Cell 3.3 — GEOGRAPHY: METROS, PALETTE, OUTPUT FOLDER *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [5]:
METRO_GROUPS = {
    'DFW':         ['irving','garland','fort worth','txs garland'],
    'Houston':     ['houston','league city','south houston'],
    'San Antonio': ['san antonio'],
}

PALETTE = {
    'attributed'   : '#1f77b4',
    'dark_unattrib': '#d62728',
    'unmatched'    : '#ff7f0e',
    'blank'        : '#9467bd',
    'redelivery'   : '#e377c2',
    'internal_ops' : '#8c564b',
    'company_avg'  : '#000000',
}

OUT_DIR = os.path.join(REPORT_ROOT, RUN_DATE)
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output: {OUT_DIR}')
print(f'Window: {FILTER_START} -> {FILTER_END}')
print(f'Metro groups: {list(METRO_GROUPS.keys())}')

Output: /./Users/aenew/OneDrive - DME Express/Reports/TechWorkload\2026-08-15
Window: 2025-01-01 -> 2026-08-14
Metro groups: ['DFW', 'Houston', 'San Antonio']


## Cell 3.4 — WAREHOUSE → STATE OVERRIDES *(LIFTED VERBATIM from tech_workload v1.35.0 — re-sync if the main notebook changes)*

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# WAREHOUSE_STATE_OVERRIDES
# Manual state mapping for warehouses where SERP_WAREHOUSES.[State Province]
# is NULL AND no sibling warehouse (same city, different region prefix) has a
# populated state. The resolution pipeline in Cell 9 tries (1) master,
# (2) sibling-city lookup from df_hier, (3) this override map, (4) 'Unknown'.
# Add new entries here as new orphan warehouses appear.
# ─────────────────────────────────────────────────────────────────────────────

# Manual state map for warehouses whose SERP_WAREHOUSES.[State Province] is NULL
# and that have no sibling-city row to borrow from. Layer 3 of the resolution
# pipeline in Cell 9.4 — add new orphan warehouses here as they appear.
WAREHOUSE_STATE_OVERRIDES = {
    'Distribution Center - Alabama':   'AL',
    'Distribution Center - Louisiana': 'LA',
    'R03 Hot Springs':       'AR',
    'R04 Camden':            'AR',
    'R04 Fort Smith':        'AR',
    'R05 Natchez Storage':   'MS',
    'R05 Tuscaloosa':        'AL',
    'R08 Alexander City':    'AL',
    'R09 Calhoun':           'GA',
    'R10 Hunt Valley':       'MD',
    'R10 Beltsville - MD':   'MD',
    'R11 Columbia - SC':     'SC',
    'R12 Garland':           'TX',
    'R13 Akron':             'OH',
    'R14 Hendersonville':    'TN',
    'R14 Jackson, TN':       'TN',
    'R15 Chantilly - VA':    'VA',
    'R15 Fredericksburg':    'VA',
    'R16 H3S':               'TX',
    'R16 Houston':           'TX',
    'RNW Austin':            'TX',
    'RNW Garland':           'TX',
}
print(f'WAREHOUSE_STATE_OVERRIDES: {len(WAREHOUSE_STATE_OVERRIDES)} entries')

WAREHOUSE_STATE_OVERRIDES: 22 entries


## Cell 3.5 — DASHBOARD CONFIGURATION

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# DASHBOARD CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DASH_TOP_N          = 5      # top/bottom technicians listed per grouping
DASH_MIN_ACTIVE_DAYS = MIN_ACTIVE_DAYS_RANK  # v1.3.0: derived, not a second copy of 30
DASH_ROLL_WEEKS     = 4      # rolling window on weekly productivity

# ── v1.2.0: VIRTUAL ("Z") WAREHOUSE HANDLING ─────────────────────────────────
# 'Z%' warehouse codes are virtual/administrative holding buckets, not buildings.
# Until 2026-05 they carried only a trickle of housekeeping rows, so v1.1.0 dropped
# them in SQL. From 2026-06 'Z CS' began carrying REAL completed field tickets
# (24.3% of eligible weekday tickets by Jul-2026). Dropping them removed numerator
# while the technician's active day stayed in the denominator — that, and nothing
# operational, is the "July drop". Cell 8.1b re-attributes them. Set this False to
# reproduce v1.1.0's numbers exactly.
VIRTUAL_WH_PREFIX  = 'Z'
INCLUDE_VIRTUAL_WH_IN_PRODUCTIVITY = True
VIRTUAL_UNRESOLVED_LABEL = 'Virtual - Unresolved'   # labelled, never silently dropped
WAREHOUSE_STATE_OVERRIDES[VIRTUAL_UNRESOLVED_LABEL] = 'Unknown'  # keeps the Cell 9.4 audit quiet

# Apportion a technician's calendar day across the warehouses they worked that day,
# weighted by that day's ticket share, instead of crediting a whole active day to
# each. Without this the warehouse-grain denominator sums ~35% above the honest
# company total (Jul-2026) and every site page reads low. Company totals are
# unaffected — apportioned days reconcile exactly to distinct (tech, date).
APPORTION_TECH_DAYS = True

# Known structural break in ticket routing — marked on every weekly panel.
ROUTING_CHANGE_DATE = '2026-06-01'
RUN_ROUTING_DIAGNOSTIC = True   # Cell 13.0; costs one extra aggregate query

# ── v1.3.0: LOST EQUIPMENT ───────────────────────────────────────────────────
# SOURCE CHANGE. v1.2.0 and earlier read SERP_ACTIVE_TAGGED_INV with
# "WHERE ATI.Lost IS NOT NULL" — that column is NULL on all 583,530 rows, so the
# panel could never populate (the v1.0.1 probe found 0 rows and correctly blamed
# the feed, but the data had moved). Audited alternatives:
#   SERP_ACTIVE_TAGGED_INV.Lost ....... 0 non-null            dead column
#   SERP_ACTIVE_TAGGED_INV_HISTORY .... 537,335 rows, ends 2025-03-30   frozen
#   SERP_LOST_EQUIPMENT_MANUAL ........ 53,424 rows, ends 2025          stale
#   Quarterly_Lost_Equipment .......... 2024-06 only, 1 wh, 1 asset     abandoned
#   SERP_LOST_EQUIPMENT ............... 91,023 rows, 2025-01 -> 2026-05  LIVE  <-
# SERP_LOST_EQUIPMENT also carries Lost Reason, Resolution, Resolved Date and
# Unit Cost, and its Warehouse values match SERP_WAREHOUSES 71 of 72.
LOST_TABLE = 'SERP_LOST_EQUIPMENT'

# TRAP 1: [Lost Date] is NOT a date. Values look like '01/02/2025<br/>(147)' — an
# HTML fragment with a trailing counter — so TRY_CONVERT fails on 100% of rows. We
# strip the tail and parse in Python (locked lesson: VARCHAR dates are parsed
# Python-side, never filtered in SQL).
# TRAP 2: [Unit Cost] is a currency string ('$11.88') — clean_numbers() handles it.
LOST_DATE_TAIL_RE = r'<br\s*/?>.*$'        # strips '<br/>(147)' and anything after
LOST_DATE_FMT     = '%m/%d/%Y'

# KNOWN BULK EVENTS — excluded from every metric and trend, reported separately.
# 2025-08-01: 30,902 rows / $3.06M unit cost against a 50-185/day baseline, with
# 30,141 of them parked in 'Z Equipment Collections' and 632 in 'Z CS' — i.e. 99.9%
# in virtual buckets, not at physical sites. Confirmed by the analyst as a known
# bulk discard/reconciliation, not operational loss. Add dates here to exclude them;
# nothing is ever excluded that is not listed.
LOST_BULK_EVENT_DATES = ['2025-08-01']
LOST_SPIKE_FACTOR = 8.0    # warn when a single lost date exceeds N x the window median
LOST_STALENESS_WARN_DAYS = 45   # feed ends 2026-05: banner every page that uses it
# Recovery rate is CENSORED for recent cohorts — an asset lost last week has had no
# time to be found. Only cohorts at least this old get a recovery rate.
LOST_RECOVERY_MATURITY_DAYS = 90
LOST_RESOLUTION_RECOVERED = ['Recover no bill', 'Recover and bill']
LOST_RESOLUTION_DISCARDED = ['Discard no bill']

# ── v1.3.0: STOCK OUTS — REFRAMED FROM "OPEN BACKLOG" TO INCIDENCE ───────────
# v1.1.0 read Status='EnRoute' as the open backlog and trended it as an aging view.
# The data does not support that: 19,891 of the 21,438 'EnRoute' orders carry a
# completion date, and every other status value is frozen at a single 2026-01-20 load.
# Status simply is not maintained. So:
#   * EnRoute by creation month IS true stock-out incidence (rows are retained after
#     fulfilment — the v1.1.0 caveat that fulfilled rows "disappear" was wrong);
#   * time-to-fulfil is computable by joining [Order] to SERP TRANSACTIONS
#     (P25 3 / median 6 / P75 9 / P90 17 days);
#   * an order resolves THREE ways, not two. 6,035 of those completion dates sit only
#     on a CANCELED ticket, i.e. the order was abandoned rather than delivered, so
#     genuine fulfilment is 64.7% and NOT the 93% a naive "has a completion date"
#     test returns. Canceled/abandoned is 28.2%; the real backlog is the remaining
#     7.1% (1,518 orders, median age 115 days).
STOCKOUT_TABLE = 'SERP_ORDER_MANAGEMENT_STOCK_OUTS'
STOCKOUT_EVENT_STATUS = 'EnRoute'     # the only maintained status = the event log
STOCKOUT_FROZEN_LOAD_DATE = '2026-01-20'  # every other status stops here; reconciliation only
# Warehouse attribution: v1.1.0 claimed no physical warehouse exists on stock-out
# rows. It does — 5,726 EnRoute rows carry one of 65 real warehouses, and joining
# [Order] to the ticket feed resolves 76.0% of orders to a real site.
STOCKOUT_ATTRIBUTION_MIN_COVERAGE = 0.50   # warn if the order join resolves less than this

# ── v1.3.0: OVERTIME (PLC_EMPLOYEE_HOURS) ────────────────────────────────────
# v1.0.0 deliberately skipped payroll ("no metric here needs hours"). v1.3.0 needs it.
#
# TRAP 1 — THE FEED RE-LOADS OVERLAPPING WINDOWS, so rows accumulate. SystemUpdatedDate
# shows loads on 2026-08-11, 08-12 and 08-14 all covering WorkDate 2026-08-06..08-19.
# Jun-2026 has 13,697 PCT rows for only 5,729 distinct (employee, workday) pairs — one
# employee-day appeared 30 times. Un-deduplicated, inferred June OT is 77,957 hours on
# 118,747 total (66% OT). After dedup: 7,383 on 46,694. ALWAYS DEDUPLICATE.
PLC_TABLE = 'PLC_EMPLOYEE_HOURS'
PLC_DEDUP_KEYS = ['plc_id', 'employee_name', 'workdate', 'pay_type', 'location_name', 'hours']
#
# TRAP 2 — THE RECORDED OT COLUMNS ARE UNUSABLE. Reg_Hrs/OT1_Hrs/OT2_Hrs/Paid_Hrs/Est_*
# are pay-period values repeated on every daily row (Pay_Type='Work': Hours 958,685 vs
# Reg_Hrs 13,626,530 — ~14x), OT1_Hrs is 0 for every month from 2026-02 on, and OT2_Hrs
# is 0 everywhere. So OT is INFERRED FLSA-style from daily [Hours], as tech_workload
# v1.35.0 does. OT_WEEKLY_THRESHOLD lives in Cell 3.1.
#
# TRAP 3 — 709 rows have Department_Name/Employee_Name NULL and 2,600-3,100 hours each
# (1,051,659 hours total, more than every real row combined). Aggregate garbage rows;
# excluded by the department filter and by an explicit guard.
#
# TRAP 4 — PLC contains FUTURE-DATED rows (through 2026-08-19 vs AS_OF_DATE 2026-08-13).
# The half-open upper bound in the extract drops them.
#
# PAY TYPES: Pay_Type separates Work / On Call Hours / PTO / Holiday. Only worked time
# counts toward the 40-hour threshold — paid leave does not (FLSA). This retires the
# standing v1.34.0 caveat "if PLC Hours includes PTO, OT is overstated": Jul-2026
# inferred OT is 10,264 hours including leave vs 8,384 excluding it (18% overstated).
# The all-pay-types figure is still emitted as a reconciliation column.
OT_PAY_TYPES_WORKED = ['Work', 'On Call Hours']
OT_PAY_TYPES_LEAVE  = ['PTO', 'Holiday']
#
# SCOPE: per-technician and per-warehouse OT uses the Patient Care Technician
# population only, so it shares the exact denominator as the productivity metric and
# the ratios are comparable. Company and VP pages additionally carry an
# all-departments OT line for total labour cost.
OT_DEPTS_TECH = ['Patient Care Technician']
OT_DEPTS_ALL  = ['Patient Care Technician', 'Internal Operations', 'Dispatch',
                 'Distribution Center']
# PLC has no usable warehouse key — only 2 of 139 Location_Name values match
# SERP_WAREHOUSES ('San Antonio TX' vs 'R14 San Antonio', 'Lafayette (Scott), LA', ...).
# Hours are therefore attributed to a site through the technician name match, then
# apportioned across that technician's warehouses by ticket share — the same rule
# Cell 13 uses for active days, so OT and productivity share one attribution model.
OT_APPORTION_BY_TICKET_SHARE = True

# ── v1.5.0: ONE REPORTING GRAIN — WEEKLY, WITH A TRAILING 4-WEEK AVERAGE ─────
# Every metric family was on a different grain: productivity weekly, census /
# overtime / lost equipment / redeliveries / stock outs monthly. A reader comparing
# two panels on one page was comparing two different time bases, and a monthly
# series gives 19 points over the window where a weekly one gives ~85 — month-end
# effects and single-week breaks (the Jun-2026 routing change, a feed outage) are
# invisible at monthly grain.
#
# So: WEEKLY IS THE PUBLISHED GRAIN FOR EVERY METRIC, each with a trailing
# ROLL_WEEKS average. Monthly frames are retained where they are the bridge back to
# a previously circulated pack (census, lost equipment, redeliveries) — they are not
# plotted, they exist so a moved number can be reconciled.
#
# THE TRAILING AVERAGE IS COMPUTED ON A CALENDAR SPINE, NOT ON OBSERVED ROWS. A
# rolling mean over "the last 4 rows present" is not a trailing 4-week average when
# an entity has no rows in a week — it silently reaches further back and reports a
# 4-week window that spans six weeks. Cell 12.9 reindexes every entity onto the full
# week spine first, so an absent week is a zero (for counts) or a gap (for rates),
# never a skipped row.
PROD_WEEK_START_DOW  = 6      # Sunday=6 -> Sun-Sat, matching PROD_WEEK_LABEL. NOT the
                              # payroll week (OT_WEEK_START_DOW) — see Cell 3.1.
ROLL_WEEKS           = DASH_ROLL_WEEKS   # 4; single source of truth for the window
TRAILING_MIN_WEEKS   = 2      # weeks required before a trailing average is emitted
TRAILING_SUFFIX      = '_t4w'
# A partial week — the first or last week of the window, clipped by its boundary — is
# excluded from every trailing window, counts and rates alike. One rule, because the
# alternative was tested and produced artifacts: a truncated week's COUNT is low for
# calendar reasons, and its RATE is not trustworthy either whenever the numerator carries a
# weekly threshold or the denominator is small. A 2-day payroll week shows 0% overtime
# because nobody reached 40 hours, not because overtime stopped; a 1-day ticket week put
# stock-outs at 32 per 100 against a 27 baseline. Partial weeks are still PLOTTED, drawn
# hollow or hatched so they are visible and identifiable, and they are still in every sheet
# — they just do not move a trailing average.
TRAILING_EXCLUDE_PARTIAL_WEEKS = True
# v1.2.0's data-integrity overlay (the share of tickets re-attributed from a virtual
# warehouse) used to ride a second y-axis on the productivity panel. It is now a rug of
# ticks along the top — same warning, no invented crossing point. A week is marked when its
# re-attributed share clears this threshold.
VIRTUAL_RUG_THRESHOLD_PCT = 5.0

# ── v1.5.0: CENSUS PER TECHNICIAN — DENOMINATOR CORRECTED ────────────────────
# v1.4.0 published ADC / (weekday tech-days / weekdays in month) and labelled it
# "Census per Active Technician". That number is ~40% above ADC / technician count and
# the gap is not an error in the arithmetic — it is the average weekday ATTENDANCE rate
# (71-74%: PTO, sick, training, part-time, and weekend work, which the weekday filter
# discards). Using the notebook's own Jul-2026 inputs: 4,112 tech-days / 23 weekdays =
# 178.8 "average technicians on the road", against 251 distinct technicians on tickets
# and ~300 on payroll. ADC 23,356 therefore reads 130.6 per "active tech", 93.1 per
# distinct tech, 77.9 per payroll head.
#
# WHY THIS IS A DEFINITIONAL ERROR AND NOT A LABELLING QUIBBLE. ADC is a STOCK — patients
# on service on an average day. Tech-days are a FLOW — attendance events. A patient does
# not leave service because their technician took PTO, so the caseload is carried by the
# whole roster every day; the honest caseload denominator is HEADCOUNT. Read the other
# way, ADC / avg-techs-on-the-road is a real quantity (coverage intensity: "are we thin
# in the field?") but it is not caseload and must not sit next to a per-technician
# staffing target.
#
# The productivity metric is unaffected and stays as it is: tickets / tech-days is
# flow / flow, which is the case the shared denominator was right for.
#
# THREE MEASURES ARE PUBLISHED SIDE BY SIDE so the denominator is a visible choice:
#   census_per_tech_headcount        ADC / distinct technicians in the period  [HEADLINE]
#   census_per_active_tech_weekday   the v1.4.0 number, retained so the restatement is
#                                    measurable and the last pack reconciles
#   attendance_rate_pct              the wedge between them — the whole story
# v1.4.0's stated caveat ("ADC is a 7-day average, tech-days are weekdays only") is
# RETIRED as the headline limitation: census is a stock, so a 7-day mean and a 5-day mean
# differ by the weekend admit/discharge rhythm — low single digits — while the attendance
# wedge is ~40%. Pointing a reader at the first mis-sizes the second.
CENSUS_HEADLINE_METRIC   = 'census_per_tech_headcount'
CENSUS_MIN_ACTIVE_TECHS  = 1.0    # suppress the ratio below this many average technicians
CENSUS_MIN_TECHS         = 2      # ...and below this many distinct technicians in the week
CENSUS_WARN_UNMAPPED_PCT = 10.0   # warn if more than this share of census has no entity mapping
PALETTE['census']        = '#17becf'   # headline series
PALETTE['census_alt']    = '#9edae5'   # the retained v1.4.0 basis, drawn subordinate

# CENSUS SOURCE FILTERS (v1.5.0). Cell 7.5 built TWO different definitions of census from
# one table: the APC snapshot excluded facility '(F)', inpatient-unit '(IPU)' and
# 'Contract Test' customers, and the ADC series — the one every census metric and the
# lost-equipment rate actually consume — excluded none of them. So ADC carried census the
# company's own APC definition does not. One filter now, applied to both, and the size of
# the change is printed on every run.
CENSUS_CUSTOMER_EXCLUDE_LIKE = ['(F)%', '(IPU)%', '%Contract Test%']
# Virtual ('Z%') warehouses are NO LONGER dropped from census. v1.4.0 dropped them from
# the numerator while the denominator kept the technicians who serve them (Cell 8.1b
# re-attributes their tickets) — the same asymmetry that produced the false July
# "productivity drop", re-introduced in a new panel. Census cannot be re-attributed the
# way tickets can (a census row has no technician), so virtual census stays in the COMPANY
# total, is labelled, and is reported separately rather than deleted.
CENSUS_KEEP_VIRTUAL_WH = True

# ── v1.6.0: DOCUMENT DELIVERABLES ────────────────────────────────────────────
# Three new outputs alongside the dashboard PDF and the Excel workbook:
#   * an EDITABLE Word metric dictionary — every metric, its data elements, its calculation
#     and the assumptions needed to interpret it;
#   * an EDITABLE Word recommendations document per VP, with the top RECS_TOP_N actions and,
#     where the data supports naming a person, the technicians involved;
#   * one consolidated publishable PDF for distribution.
# The Word files and the PDF are rendered from ONE content model (Cell 20), so a definition
# cannot say two different things in two places.
DASH_PAGE_INCHES = (23, 15)      # dashboard figure size; the narrative pages match it
DOC_PAGE_INCHES = DASH_PAGE_INCHES
# The page is wide because a twelve-panel dashboard needs to be. Text does NOT get wider
# with it — a 23-inch line length is unreadable — so the column is held to a normal measure
# and the surplus becomes margin.
DOC_TEXT_WIDTH_IN = 11.0         # single-column measure (used by the cover page)
DOC_FONT_SCALE = 1.55            # 9pt on a 23-inch page reads like ~6pt on Letter
# Two columns, because one column at a readable measure fills a third of a 23-inch page and
# leaves the rest blank. Content flows column 1 -> column 2 -> next page.
DOC_COLUMNS = 2
DOC_COL_GUTTER_IN = 0.9

REPORT_DOCX_DICTIONARY = f'OpsDashboard_MetricDictionary_{RUN_DATE}.docx'
REPORT_DOCX_VP_RECS = 'OpsDashboard_Recommendations_{vp}_' + f'{RUN_DATE}.docx'
REPORT_PUBLISH_PDF = f'OpsDashboard_Publishable_{RUN_DATE}.pdf'
NB_VERSION = '1.6.0'

# ── RECOMMENDATION ENGINE ────────────────────────────────────────────────────
# A recommendation is generated only where a VP is adverse to the COMPANY figure on a metric
# with a usable base. Nothing is hand-written per VP; Cell 21 ranks what Cell 17.5 measured.
RECS_TOP_N = 5              # recommendations per VP
RECS_LOOKBACK_WEEKS = 13    # one quarter — long enough to be stable, short enough to act on
RECS_MIN_WEEKS = 4          # a metric needs this many observed weeks to carry a recommendation
RECS_NAME_N = 5             # technicians named per recommendation
RECS_MIN_ACTIVE_DAYS = 20   # presence floor before a technician can be named, IN THE WINDOW.
# Deliberately lower than MIN_ACTIVE_DAYS_RANK (30), which applies to the full run window:
# 30 active days inside a 13-week window would exclude anyone part-time, and the point of a
# denominator-honest metric is that part-time technicians can be measured fairly.
RECS_EFF_GAP_PCT = 15.0     # only name a technician this far below their own VP median
# Materiality floor. A VP sitting 0.03 percentage points off the company figure does not need
# a formal recommendation, and padding the list with rounding is how a reader learns to
# ignore it. Immaterial gaps stay visible in the scorecard; they just do not become actions.
RECS_MIN_SEVERITY_PCT = 5.0   # gap must be at least this % of the company figure

# Attached to every row of tech_perf_vp and printed with every named list. It rides on the
# DATA, not just the document, so a name cannot be copied out of a table without it.
COACHING_CAVEAT = (
    'These names are prompts for a coaching conversation, not a performance finding, and '
    'must not be used for discipline without ticket-level review. Tickets per active day '
    'divides by days actually worked, so PTO and part-time schedules do not push anyone onto '
    'this list — but route density, ticket mix, vehicle reliability and the share of complex '
    'set-ups all move the number, and none of them is visible here. Comparison is against '
    'this VP\'s own median, not the company\'s.')

DASH_PDF_NAME  = f'OpsDashboard_Pages_{RUN_DATE}.pdf'
DASH_XLSX_NAME = f'OpsDashboard_Report_{RUN_DATE}.xlsx'
print(f'Dashboard config loaded. Stock-outs configured: {STOCKOUT_TABLE is not None}')
print(f'Reporting grain: WEEKLY ({PROD_WEEK_LABEL}) + trailing {ROLL_WEEKS}-week average '
      f'on every metric. Overtime keeps its own {OT_WEEK_LABEL} payroll week.')

Dashboard config loaded. Stock-outs configured: True
Reporting grain: WEEKLY (Sun-Sat) + trailing 4-week average on every metric. Overtime keeps its own Thu-Wed payroll week.


## Cell 4.1 — SECURE SQL CONNECTION: AZURE KEY VAULT

In [8]:
# load_dotenv(ENV_FILE)
# sql_conn = odbc.connect(
#    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
#    f'UID={os.getenv("SQL_USERNAME")};PWD={os.getenv("SQL_PASSWORD")};'
#    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
#)

KEY_VAULT_URI   = 'https://dmee-keyvault.vault.azure.net/'
HOSTNAME_SECRET = 'DataWarehouseHostname'
SQL_USERNAME    = 'anewton-ro'   # secret name is the login; its value is the password

kv = SecretClient(vault_url=KEY_VAULT_URI, credential=DefaultAzureCredential())
# v1.3.0 (audit B4): the vault stores a BARE hostname. Substituting it for the
# 'tcp:host,1433' form from Cell 3.1 produced '[08001] TCP Provider: Timeout error [258]'
# on a cold connect; wrapping it connects first try. Keep the prefix and the port.
SERVER_NAME  = f'tcp:{kv.get_secret(HOSTNAME_SECRET).value},1433'
SQL_PASSWORD = kv.get_secret(SQL_USERNAME).value
print('Secrets loaded from Key Vault.')

sql_conn = odbc.connect(
    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
    f'UID={SQL_USERNAME};PWD={SQL_PASSWORD};'
    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
)

print('DB connection established.')

Secrets loaded from Key Vault.
DB connection established.


## Cell 4.2 — QUERY RUNNER & UTILITIES

In [9]:
_READ_ONLY_STARTS = ('select', 'with')
def run_query(query, label='', verbose=False):
    """Run a READ-ONLY query and return a DataFrame.

    v1.3.0 (audit C2): CLAUDE.md rule 4 says analysis queries are SELECT/WITH only and
    that this helper enforces it in-process. It did not — it executed whatever string it
    was handed. It does now: anything that is not a single SELECT/WITH statement raises
    before touching the connection. This is a guard against an accident in a notebook
    that holds a live production connection, not a security boundary.
    """
    _q = re.sub(r'--[^\n]*', ' ', query)            # strip line comments
    _q = re.sub(r'/\*.*?\*/', ' ', _q, flags=re.S)  # strip block comments
    _q_clean = _q.strip().lstrip('(').lstrip()
    if not _q_clean.lower().startswith(_READ_ONLY_STARTS):
        raise ValueError(f'run_query is read-only: a query must begin with SELECT or WITH '
                         f'(label={label!r}, got {_q_clean[:60]!r}).')
    _forbidden = re.findall(r'(?<![\w.])(insert|update|delete|merge|drop|truncate|alter|create|'
                            r'grant|revoke|exec|execute|sp_\w+|xp_\w+)(?![\w.])', _q, flags=re.I)
    if _forbidden:
        raise ValueError(f'run_query is read-only: refusing statement containing '
                         f'{sorted(set(w.lower() for w in _forbidden))} (label={label!r}).')
    if ';' in _q_clean.rstrip().rstrip(';'):
        raise ValueError(f'run_query is read-only: one statement per call (label={label!r}).')
    if verbose: print(f'Query: {label}\n{query}')
    t0 = time.time()
    cur = sql_conn.cursor(); cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0].lower() for c in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    if label: print(f'  {label}: {len(df):,} rows  ({time.time()-t0:.1f}s)')
    return df

def clean_numbers(val):
    # v1.33.0 (L2): preserve the negative sign — the old regex stripped '-', silently
    # flipping negative adjustments positive. Also avoid lstrip('0') mangling.
    if val is None: return np.nan
    s = str(val).strip()
    neg = s.startswith('-') or (s.startswith('(') and s.endswith(')'))  # (123) = accounting negative
    c = re.sub(r'[^0-9.]', '', s)
    if not c or c == '.': return np.nan
    try:
        v = float(c)
    except ValueError:
        return np.nan
    return -v if neg else v

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f'{name}_{RUN_DATE}.png'), bbox_inches='tight', dpi=CHART_DPI)

def _strip_wh_prefix(name):
    """Strip leading region token (R##, RNW) or 'Distribution Center -' to expose
    the underlying city/location. Used by resolve_state_series() to find sibling
    warehouses for the same physical location.
    Examples:
        'R02 Lake Charles'         -> 'Lake Charles'
        'RNW Del Rio'              -> 'Del Rio'
        'Distribution Center - LA' -> 'LA'
    """
    import re as _re
    if not isinstance(name, str): return ''
    s = name.strip()
    s = _re.sub(r'^R\d{2}\s+', '', s)
    s = _re.sub(r'^RNW\s+', '', s, flags=_re.IGNORECASE)
    s = _re.sub(r'^Distribution Center\s*-\s*', '', s, flags=_re.IGNORECASE)
    return s.strip()

def assign_metro(wh_name, metro_groups=METRO_GROUPS):
    """Assign a warehouse to a metro group by substring match (case-insensitive).
    Returns the metro name, or None if no match. First match wins.
    """
    if not wh_name or not isinstance(wh_name, str): return None
    wh_lc = wh_name.lower().strip()
    for metro, patterns in metro_groups.items():
        if any(p in wh_lc for p in patterns):
            return metro
    return None

## Cell 4.3 — PERIOD SPINE & TRAILING AVERAGE *(new in v1.5.0)*

One calendar week definition for the whole notebook, and the trailing-average helper every metric family uses. Built from the calendar rather than from observed rows, so a week with no data is a visible gap instead of a row the rolling window steps over.

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# PERIOD SPINE & TRAILING AVERAGE — one week definition for the whole notebook
#                                                                     new v1.5.0
# Before this cell existed, each metric family invented its own time bucket: Cell 13
# derived weeks from the ticket rows it happened to see, Cells 13.5/14/15/15.5/16 used
# calendar months, and overtime used payroll weeks. Two consequences, both bad:
#   * a reader comparing two panels on one page was comparing two time bases;
#   * a week with no rows for an entity simply did not exist, so a rolling mean over
#     "the last four rows" quietly reached back six calendar weeks and reported it as a
#     4-week average. That is the defect this cell exists to make impossible.
#
# WEEK_SPINE is built from the CALENDAR, bounded by the run window — not from observed
# data — so every entity is measured against the same weeks and a gap is visible as a
# gap. Overtime is the one deliberate exception: the payroll week is Thu-Wed (Cell 3.1)
# and an operational reporting week is a different object, so OT carries its own spine
# and its panels say so.
#
# TEACHING NOTE — why a trailing average at all. A weekly series on ~250 technicians is
# noisy: one holiday week, one feed reload, one heavy Monday and the line jumps. A
# trailing 4-week mean is the shortest window that damps weekday-mix and single-week
# artifacts while still turning inside a month, which is what you want when the question
# is "has this changed?" rather than "what did last week look like?". It is TRAILING, not
# centred — every point uses only weeks up to and including itself, so no published point
# moves when next week's data lands. A centred mean would restate last month's chart
# every week, which is how a report loses its audience.
# ─────────────────────────────────────────────────────────────────────────────
def week_start_of(s, start_dow=None):
    """Map a datetime-like series to the first day of its week.

    start_dow follows pandas' Monday=0..Sunday=6. Defaults to PROD_WEEK_START_DOW
    (Sunday), so this reproduces Cell 13's Sun-Sat bucketing exactly."""
    if start_dow is None:
        start_dow = PROD_WEEK_START_DOW
    d = pd.to_datetime(s, errors='coerce')
    return (d - pd.to_timedelta((d.dt.dayofweek - start_dow) % 7, unit='D')).dt.normalize()


def build_week_spine(start, end, start_dow=None):
    """Every week touching [start, end], with the day counts each week actually has
    INSIDE the window. A first or last week clipped by the window boundary is flagged
    partial — otherwise a truncated week reads as an operational collapse."""
    if start_dow is None:
        start_dow = PROD_WEEK_START_DOW
    lo, hi = pd.Timestamp(start), pd.Timestamp(end)
    first = week_start_of(pd.Series([lo]), start_dow).iloc[0]
    rows = []
    for ws in pd.date_range(first, hi, freq='7D'):
        w_lo, w_hi = max(ws, lo), min(ws + pd.Timedelta(days=6), hi)
        days = pd.date_range(w_lo, w_hi, freq='D') if w_lo <= w_hi else pd.DatetimeIndex([])
        rows.append({'week_start': ws,
                     'week_end': ws + pd.Timedelta(days=6),
                     'calendar_days_in_week': int(len(days)),
                     'weekday_days_in_week': int((days.dayofweek < 5).sum()),
                     'week_label': ws.strftime('%Y-%m-%d'),
                     # A week belongs wholly to the month containing its LAST in-window
                     # day — the same rule overtime uses, so the two frames agree.
                     'period': (w_hi if len(days) else ws).strftime('%Y-%m')})
    sp = pd.DataFrame(rows)
    sp['is_partial_week'] = sp['calendar_days_in_week'] < 7
    return sp


SPINE_META_COLS = ['week_end', 'weekday_days_in_week', 'calendar_days_in_week',
                   'is_partial_week', 'week_label', 'period']
WEEK_SPINE = build_week_spine(FILTER_START, AS_OF_DATE)
print(f'Week spine: {len(WEEK_SPINE)} weeks, {WEEK_SPINE["week_label"].iloc[0]} -> '
      f'{WEEK_SPINE["week_label"].iloc[-1]} ({PROD_WEEK_LABEL}); '
      f'{int(WEEK_SPINE["is_partial_week"].sum())} partial (window-clipped).')


def attach_spine(df, week_col='week_start', spine=None):
    """Put the spine's week metadata on a weekly frame, replacing any stale copy.
    Idempotent, so a frame can pass through it more than once without growing _x/_y."""
    if df is None or df.empty:
        return df
    sp = WEEK_SPINE if spine is None else spine
    d = df.drop(columns=[c for c in SPINE_META_COLS if c in df.columns])
    return d.merge(sp, left_on=week_col, right_on='week_start',
                   how='left', suffixes=('', '_spine'))


def add_trailing(df, gcols, count_cols=(), rate_cols=(), weeks=None, min_weeks=None,
                 week_col='week_start', spine=None, suffix=None,
                 exclude_partial=None):
    """Add a trailing `weeks`-week mean for each named column, computed on a CALENDAR
    spine rather than on the rows that happen to be present.

    count_cols  volumes (tickets, assets, orders, hours). A week with no rows is a
                genuine ZERO, so each entity is reindexed onto the spine and filled with
                0 before the mean. Without this, a quiet week vanishes and lifts the
                average — the trailing figure would read high exactly when volume fell.
                BUT ONLY INSIDE THE ENTITY'S OWN SPAN. Weeks before an entity first
                appears are left missing, not zeroed: a warehouse that opened in March did
                not record zero assets in January, it did not exist, and zero-filling those
                weeks would quarter its first trailing figure. A gap between two
                observations is a zero; the run-up to the first observation is not data.
    rate_cols   ratios, stocks and durations (per-100, %, ADC, median days). A week with
                no rows has NO rate, so it stays NaN and min_weeks decides whether the
                window still qualifies. Filling a rate with 0 would be a lie.

    Partial weeks (clipped by the window edge) are dropped from EVERY trailing window,
    counts and rates alike — see TRAILING_EXCLUDE_PARTIAL_WEEKS in Cell 3.5 for why the
    rate case matters too. They are still returned and still plotted, just marked.

    Emits `t4w_weeks_observed`, the number of real weeks behind each trailing point, so a
    thin window is visible in the sheet instead of implied.
    """
    weeks = ROLL_WEEKS if weeks is None else weeks
    min_weeks = TRAILING_MIN_WEEKS if min_weeks is None else min_weeks
    suffix = TRAILING_SUFFIX if suffix is None else suffix
    if exclude_partial is None:
        exclude_partial = TRAILING_EXCLUDE_PARTIAL_WEEKS
    sp = (WEEK_SPINE if spine is None else spine)
    if df is None or df.empty:
        return df
    count_cols = [c for c in count_cols if c in df.columns]
    rate_cols = [c for c in rate_cols if c in df.columns]
    if not (count_cols or rate_cols):
        return df
    gcols = list(gcols)

    # Spine metadata is dropped off the incoming frame and re-supplied by the spine, so a
    # frame that already carries week_end/is_partial_week does not collide on the merge.
    src = df.drop(columns=[c for c in SPINE_META_COLS if c in df.columns]).copy()
    src['_present'] = 1
    if gcols:
        grid = df[gcols].drop_duplicates().merge(sp, how='cross')
    else:
        grid = sp.copy()
    full = (grid.merge(src, on=gcols + [week_col], how='left')
                .sort_values(gcols + [week_col]))
    full['_present'] = full['_present'].fillna(0)
    _partial = full['is_partial_week'].fillna(False).astype(bool)

    def _roll_mean(s):
        return s.rolling(weeks, min_periods=min_weeks).mean()

    def _by_group(col, fn):
        return (full.groupby(gcols, dropna=False)[col].transform(fn) if gcols
                else fn(full[col]))

    # An entity's span starts at its first observed week. Spine weeks before that are not
    # zeros, they are non-existence — see the count_cols note above.
    full['_seen'] = _by_group('_present', lambda s: s.cummax())

    for c in count_cols:
        full['_v'] = pd.to_numeric(full[c], errors='coerce').fillna(0.0)
        full['_v'] = full['_v'].mask(full['_seen'] < 1)
        if exclude_partial:
            full['_v'] = full['_v'].mask(_partial)
        full[c + suffix] = _by_group('_v', _roll_mean)
    for c in rate_cols:
        full['_v'] = pd.to_numeric(full[c], errors='coerce')
        if exclude_partial:
            full['_v'] = full['_v'].mask(_partial)
        full[c + suffix] = _by_group('_v', _roll_mean)

    full['_v'] = full['_present'].mask(_partial) if exclude_partial else full['_present']
    full['t4w_weeks_observed'] = _by_group(
        '_v', lambda s: s.rolling(weeks, min_periods=1).sum())
    full = full.drop(columns=['_v', '_present', '_seen'])

    # Return only the weeks the caller actually had. The zero-filled spine rows were
    # scaffolding for the window, not new data to publish — a caller that wants the gaps
    # made explicit can merge against WEEK_SPINE itself.
    keep = df[gcols + [week_col]].drop_duplicates()
    return (keep.merge(full, on=gcols + [week_col], how='left')
                .sort_values(gcols + [week_col]).reset_index(drop=True))


def weekly_denominator(df, gcols, num_col, den_col, out_col, weeks=None, min_weeks=None,
                       suffix=None):
    """Trailing ratio computed as trailing-numerator / trailing-denominator, NOT as the
    mean of four weekly ratios.

    TEACHING NOTE, and it matters for every rate on this dashboard. The mean of weekly
    ratios weights a 40-ticket week the same as a 400-ticket week; the ratio of trailing
    sums weights by volume, which is what "the last four weeks ran at X" means. They can
    differ by a lot when volume swings — a holiday week with three tickets and one
    redelivery reads 33 per 100 and drags a mean-of-ratios badly. Pooled is the honest
    one, so rate columns that have both parts available get this treatment; ones where the
    denominator never made it onto the frame fall back to add_trailing's mean-of-ratios
    and are named so you can tell which you are reading.
    """
    weeks = ROLL_WEEKS if weeks is None else weeks
    min_weeks = TRAILING_MIN_WEEKS if min_weeks is None else min_weeks
    suffix = TRAILING_SUFFIX if suffix is None else suffix
    if df is None or df.empty or num_col not in df.columns or den_col not in df.columns:
        return df
    t = add_trailing(df[list(gcols) + ['week_start', num_col, den_col]],
                     gcols, count_cols=[num_col, den_col], weeks=weeks, min_weeks=min_weeks,
                     suffix='_ttmp')
    t[out_col + suffix] = (t[num_col + '_ttmp']
                           / t[den_col + '_ttmp'].replace(0, np.nan)).round(3)
    cols = list(gcols) + ['week_start', out_col + suffix]
    return df.drop(columns=[out_col + suffix], errors='ignore').merge(
        t[cols], on=list(gcols) + ['week_start'], how='left')

Week spine: 85 weeks, 2024-12-29 -> 2026-08-09 (Sun-Sat); 2 partial (window-clipped).


## Cell 5 — LEVENSHTEIN DISTANCE

In [11]:
# Damerau–Levenshtein edit distance for fuzzy first-name matching (Pass 4).
# rapidfuzz (C++) when installed; otherwise a pure-Python fallback with the same
# semantics, including adjacent-transposition ('jhon'→'john' = distance 1).
if _RAPIDFUZZ_OK:
    def levenshtein(a, b): return _DL.distance(a, b)
else:
    def levenshtein(a, b):
        if a==b: return 0
        if not a: return len(b)
        if not b: return len(a)
        la,lb=len(a),len(b)
        d=[[0]*(lb+1) for _ in range(la+1)]
        for i in range(la+1): d[i][0]=i
        for j in range(lb+1): d[0][j]=j
        for i in range(1,la+1):
            for j in range(1,lb+1):
                cost=0 if a[i-1]==b[j-1] else 1
                d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+cost)
                if i>1 and j>1 and a[i-1]==b[j-2] and a[i-2]==b[j-1]: d[i][j]=min(d[i][j],d[i-2][j-2]+cost)
        return d[la][lb]
print('levenshtein() ready.')

levenshtein() ready.


## Cell 6 — NICKNAME DICTIONARY

In [12]:
# Bidirectional nickname dictionary (Pass 3): 'mike'→Michael AND Michael→'mike'.
# Includes Spanish hypocorisms (pepe/chuy/beto...) for the field workforce.
# Retained as-is by explicit decision — do not prune without a match-rate test.
NICKNAME_TO_CANONICAL_RAW = [
    ('chris',['Christopher','Christian','Christina','Christine']),('kris',['Christopher','Kristopher']),
    ('mike',['Michael']),('mikey',['Michael']),('matt',['Matthew']),('dan',['Daniel']),('danny',['Daniel']),
    ('dave',['David']),('rob',['Robert']),('bob',['Robert']),('bobby',['Robert']),('robbie',['Robert']),
    ('robby',['Robert']),('jim',['James']),('jimmy',['James']),('jamie',['James']),('joe',['Joseph']),
    ('joey',['Joseph']),('tom',['Thomas']),('tommy',['Thomas']),('bill',['William']),('billy',['William']),
    ('will',['William']),('liam',['William']),('rick',['Richard','Ricardo','Frederick']),
    ('ricky',['Richard','Ricardo']),('rich',['Richard']),('steve',['Steven','Stephen']),
    ('ed',['Edward','Eduardo','Edwin']),('eddie',['Edward','Eduardo']),('ted',['Edward','Theodore']),
    ('tony',['Anthony','Antonio']),('nick',['Nicholas','Nicolas']),('pat',['Patrick','Patricia']),
    ('tim',['Timothy']),('timmy',['Timothy']),('jon',['Jonathan','Jonathon']),
    ('johnny',['John','Jonathan']),('jack',['John','Jackson']),('jeff',['Jeffrey','Geoffrey']),
    ('andy',['Andrew','Andres']),('drew',['Andrew']),('ron',['Ronald','Ronaldo']),
    ('ronnie',['Ronald']),('ken',['Kenneth']),('kenny',['Kenneth']),('ben',['Benjamin']),
    ('benny',['Benjamin','Benito']),('greg',['Gregory']),('sam',['Samuel','Samantha']),
    ('josh',['Joshua']),('alex',['Alexander','Alejandro','Alexandra']),
    ('nate',['Nathaniel','Nathan']),('zach',['Zachary','Zachariah']),('zack',['Zachary']),
    ('mitch',['Mitchell']),('ray',['Raymond','Raymundo']),('larry',['Lawrence','Lorenzo']),
    ('terry',['Terrence','Terrell']),('jerry',['Gerald','Jeremiah','Jerome']),
    ('chuck',['Charles']),('charlie',['Charles']),('fred',['Frederick','Fredrick','Alfredo']),
    ('hank',['Henry']),('harry',['Henry','Harold','Harrison']),('phil',['Philip','Phillip']),
    ('wes',['Wesley']),('vince',['Vincent']),('vinny',['Vincent','Vincenzo']),
    ('abe',['Abraham']),('gabe',['Gabriel']),('len',['Leonard']),('walt',['Walter']),
    ('doug',['Douglas']),('al',['Albert','Alan','Alfonso']),('bert',['Albert','Robert','Herbert']),
    ('don',['Donald']),('gene',['Eugene']),('manny',['Manuel','Emmanuel']),('marty',['Martin']),
    ('art',['Arthur']),('curt',['Curtis']),('ernie',['Ernest','Ernesto']),
    ('frank',['Franklin','Francisco','Francis']),('frankie',['Franklin','Francisco','Frank']),
    ('jr',['Junior']),('lupe',['Guadalupe']),('max',['Maximilian','Maxwell','Maximo']),
    ('reggie',['Reginald']),('rudy',['Rudolph','Rodolfo']),
    ('ty',['Tyler','Tyrone','Tyson']),('vic',['Victor']),
    ('liz',['Elizabeth']),('beth',['Elizabeth']),('lisa',['Elizabeth']),
    ('kate',['Katherine','Kathryn','Kaitlyn']),('kathy',['Katherine','Kathryn']),
    ('katie',['Katherine']),('sue',['Susan','Suzanne']),('susie',['Susan']),
    ('jen',['Jennifer']),('jenny',['Jennifer']),('jenn',['Jennifer']),('amy',['Amelia','Amy']),
    ('meg',['Megan','Margaret']),('maggie',['Margaret']),('pam',['Pamela']),('barb',['Barbara']),
    ('deb',['Deborah','Debra']),('debbie',['Deborah','Debra']),('carol',['Caroline','Carolyn']),
    ('tina',['Christina']),('sandy',['Sandra','Alexandra']),('cindy',['Cynthia']),
    ('angie',['Angela','Angelica']),('ang',['Angela']),('steph',['Stephanie']),
    ('stacy',['Stacey','Stacy']),('nikki',['Nicole','Nichole']),('mia',['Maria']),
    ('maria',['Maria','Marie']),('anna',['Annette','Annalisa']),
    ('nicky',['Nichole','Nicole']),('joanie',['Joan']),('diana',['Diane']),('dani',['Danielle']),
    ('trish',['Patricia']),('patty',['Patricia']),('bri',['Brianna','Brittany']),
    ('brit',['Brittany','Britney']),('chrissy',['Christina','Christine']),('vero',['Veronica']),
    ('cass',['Cassandra']),('dot',['Dorothy']),('bev',['Beverly']),
    ('mel',['Melanie','Melissa','Melinda']),('missy',['Melissa']),('mindy',['Melinda']),
    ('pepe',['Jose']),('chuy',['Jesus']),('nacho',['Ignacio']),('chava',['Salvador']),
    ('memo',['Guillermo']),('beto',['Roberto','Alberto']),('pancho',['Francisco']),
    ('paco',['Francisco']),('chela',['Graciela']),('chucho',['Jesus']),('vale',['Valeria']),
    ('lalo',['Eduardo']),('pipe',['Felipe']),('nico',['Nicolas']),
]
NICKNAME_TO_CANONICAL = {}
for nick,canonicals in NICKNAME_TO_CANONICAL_RAW:
    if nick not in NICKNAME_TO_CANONICAL: NICKNAME_TO_CANONICAL[nick]=[]
    for c in canonicals:
        if c not in NICKNAME_TO_CANONICAL[nick]: NICKNAME_TO_CANONICAL[nick].append(c)
CANONICAL_TO_NICKNAMES = defaultdict(list)
for nick,canonicals in NICKNAME_TO_CANONICAL.items():
    for canon in canonicals: CANONICAL_TO_NICKNAMES[canon.lower()].append(nick)

def standardize_first_name(raw):
    if not raw or not isinstance(raw,str): return []
    raw_lc=raw.lower().strip(); seen,results=set(),[raw]; seen.add(raw_lc)
    for cand in NICKNAME_TO_CANONICAL.get(raw_lc,[]):
        if cand.lower() not in seen: results.append(cand); seen.add(cand.lower())
    for nick in CANONICAL_TO_NICKNAMES.get(raw_lc,[]):
        if nick.lower() not in seen: results.append(nick.title()); seen.add(nick.lower())
    return results
print(f'Nickname map: {len(NICKNAME_TO_CANONICAL)} entries')

Nickname map: 152 entries


## Cell 7.1 — TRANSACTIONS EXTRACT + H3 PROBE

In [13]:
print(f'Extracting data ({FILTER_START} to {FILTER_END})...')
t0 = time.time()
_reasons_sql = ',\n      '.join(f"'{r}'" for r in INCLUDED_REASONS)

df_tx = run_query(f"""
SELECT TRIM(TX.Order_Num) AS order_num, TRIM(TX.Record_ID) AS record_id,
    TRIM(ISNULL(TX.Tech_Warehouse,'')) AS tech_warehouse,
    TRIM(ISNULL(TX.TechFirstName,'')) AS techfirstname,
    TRIM(ISNULL(TX.TechLastName,''))  AS techlastname,
    TX.Reason AS reason,
    TRY_CONVERT(DATE, TX.Completed_Date) AS completed_date
FROM dbo.[SERP TRANSACTIONS] AS TX WITH (NOLOCK)
-- v1.2.0: the blanket "NOT LIKE 'Z%'" exclusion is GONE. From 2026-06 the virtual
  -- warehouse 'Z CS' carries real completed field tickets; excluding them here deleted
  -- up to 24% of the numerator while the technician's active day stayed in the
  -- denominator, which is the whole of the apparent July productivity cliff. Virtual
  -- rows now arrive tagged and are re-attributed to a physical warehouse in Cell 8.1b.
WHERE TX.[Status] <> 'Canceled'
  AND TX.Order_Num <> ''
  -- v1.33.0 (H3): filter on the CONVERTED date, matching the SELECT. The raw-string
  -- compare could (a) drop the entire last day if values carry a time component
  -- ('2026-07-30 14:22' > '2026-07-30' as strings) and (b) admit malformed strings
  -- that TRY_CONVERT NULLs in the SELECT, creating NaT rows downstream.
  AND TRY_CONVERT(DATE, TX.Completed_Date) >= '{FILTER_START}'
  AND TRY_CONVERT(DATE, TX.Completed_Date) <= '{FILTER_END}'
  AND TX.Reason IN ({_reasons_sql})
OPTION (RECOMPILE, MAXDOP 4)
""", 'Transactions')
# v1.3.0 (audit C1): the previous display(df_tx.head(3)) rendered record_id and
# order_num — record/account-level patient references — into committed notebook output,
# against CLAUDE.md rule 2. A shape/null summary answers the same question ("did the
# extract work?") without putting a single patient reference on screen.
print(f'  df_tx: {len(df_tx):,} rows x {df_tx.shape[1]} cols')
print('  nulls by column: ' + ', '.join(f'{c}={int(n)}' for c, n in df_tx.isna().sum().items() if n))
print(f"  completed_date range: {df_tx['completed_date'].min()} -> {df_tx['completed_date'].max()}")

# v1.3.0 (audit D6): windowed. Unparseable rows have no usable date, so they cannot be
# filtered by Completed_Date — the raw string is bounded instead, which lets the engine
# seek rather than scan the whole table for one number on every run.
_h3 = run_query(f"""
SELECT COUNT(*) AS bad_rows FROM dbo.[SERP TRANSACTIONS] WITH (NOLOCK)
WHERE TRY_CONVERT(DATE, Completed_Date) IS NULL
  AND Completed_Date IS NOT NULL AND LTRIM(RTRIM(Completed_Date)) <> ''
  AND [Status] <> 'Canceled' AND Order_Num <> ''
  AND LEFT(LTRIM(Completed_Date), 4) >= '{FILTER_START[:4]}'
""", 'H3 unparseable Completed_Date probe (windowed by raw year)')
_h3_n = int(_h3.iloc[0, 0]) if len(_h3) else 0
if _h3_n > 0:
    print(f'  *** H3: {_h3_n:,} transaction rows have UNPARSEABLE Completed_Date and are')
    print(f'      EXCLUDED from all analytics. Pull samples and fix the upstream feed.')
else:
    print('  H3 probe: 0 unparseable Completed_Date rows — verification item closed for this run.')


Extracting data (2025-01-01 to 2026-08-14)...
  Transactions: 467,718 rows  (44.9s)
  df_tx: 467,718 rows x 7 cols
  nulls by column: 
  completed_date range: 2025-01-01 -> 2026-08-14
  H3 unparseable Completed_Date probe (windowed by raw year): 1 rows  (11.9s)
  H3 probe: 0 unparseable Completed_Date rows — verification item closed for this run.


## Cell 7.2 — EMPLOYEES & WAREHOUSE HIERARCHY

In [14]:
df_emp = run_query("""
SELECT count(id) AS emp_duplicate_count,
    TRIM([FirstName]) AS empfirstname, TRIM([LastName]) AS emplastname,
    TRIM([Warehouse Name]) AS [location],
    MAX([Department Name]) AS dept, MAX([Job Title]) AS title,
    MAX([Employee Number]) AS eid, MAX([Username]) AS username
FROM SERP_DME_EMPLOYEES
GROUP BY TRIM([FirstName]),TRIM([LastName]),TRIM([Warehouse Name])
""", 'Employees')

df_hier = run_query("""
SELECT DISTINCT TRIM([Warehouse Name]) AS warehouse, [State Province] as [State],
    LEFT(TRIM([Warehouse Name]),3) AS region, TRIM([Group]) AS vp
FROM SERP_WAREHOUSES WITH (NOLOCK)
""", 'Warehouse hierarchy')


  Employees: 2,003 rows  (0.1s)
  Warehouse hierarchy: 138 rows  (0.0s)


## Cell 7.3 — REDELIVERIES EXTRACT

In [15]:
df_redel = run_query(f"""
SELECT TRIM(RD.Orig_Order) AS orig_order_num,
    TRY_CONVERT(DATE,RD.Completion_DateTime) AS rd_date,
    RD.Completion_DateTime AS rd_datetime_raw,
    TRIM(RD.Products) AS rd_products,
    TRIM(ISNULL(SE.[FirstName],'')) AS techfirstname,
    TRIM(ISNULL(SE.[LastName],''))  AS techlastname,
    TRIM(ISNULL(RD.Tech_Warehouse,'')) AS tech_warehouse
FROM dbo.[Re-Delivery Report] RD
LEFT JOIN dbo.SERP_DME_EMPLOYEES SE ON SE.Username=RD.Tech
WHERE (TRY_CONVERT(DATE,RD.Completion_DateTime) BETWEEN '{FILTER_START}' AND '{FILTER_END}')
   OR (TRY_CONVERT(DATE,RD.Completion_DateTime) IS NULL
       AND RD.Completion_DateTime LIKE '202_-%')  -- keep parse-fail rows that look in-range
""", 'Redeliveries')
if 'tech_warehouse' not in df_redel.columns: df_redel['tech_warehouse']=''


  Redeliveries: 188,437 rows  (5.5s)


## Cell 7.4 — LOST EQUIPMENT & INVENTORY EXTRACT *(source replaced in v1.3.0)*

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# LOST EQUIPMENT & INVENTORY EXTRACT                            rewritten v1.3.0
#
# v1.2.0 and earlier queried SERP_ACTIVE_TAGGED_INV WHERE ATI.Lost IS NOT NULL.
# That column is NULL on all 583,530 rows, so the panel was structurally empty and
# had been since at least 2025-01. The live register is SERP_LOST_EQUIPMENT (see the
# source audit in Cell 3.5). The old ATI probe is gone with the old query.
#
# PII DISCIPLINE (CLAUDE.md rule 2): SERP_LOST_EQUIPMENT carries Patient Firstname /
# Lastname / Patient ID, Bill-To ID, Caregiver names, WorkPhone, MobilePhone, Email,
# Ship-To Address1/2, City and Postal. NONE of them are selected. Only [Ship-To State]
# is taken, as a geographic fallback for state attribution. v1.2.0's ATI query pulled
# Bill_to_ID to count distinct patients — that was an account number under rule 2, the
# count was never used on a page, and it is not carried forward.
# ─────────────────────────────────────────────────────────────────────────────
df_lost_raw = run_query(f"""
SELECT LE.[Lost Date]         AS lost_date_raw,
       LE.[Lost Reason]       AS lost_reason,
       LE.[Warehouse]         AS lost_warehouse,
       LE.[Product Name]      AS product_name,
       LE.[Product ID]        AS product_id,
       LE.[Asset Tag]         AS asset_tag,
       LE.[Unit Cost]         AS unit_cost_raw,
       LE.[Resolution]        AS resolution,
       LE.[Resolved Date]     AS resolved_date_raw,
       LE.[Discharged Days]   AS discharged_days_raw,
       LE.[Delivery Order ID] AS delivery_order_id,
       LE.[Pickup Order ID]   AS pickup_order_id,
       LE.[Reason for Pickup] AS reason_for_pickup,
       LE.[Ship-To State]     AS ship_to_state
FROM {LOST_TABLE} AS LE WITH (NOLOCK)
""", 'Lost equipment (SERP_LOST_EQUIPMENT)')

# Inventory denominator — v1.2.0 extracted, cast and then never used this frame.
# v1.3.0 consumes it: lost assets as a share of tagged inventory on hand, which is the
# only way a 40-asset month at a small site and a 40-asset month at a large one can be
# compared. Counts are a CURRENT snapshot, so the ratio is "lost that month against
# inventory today" — fine for relative site comparison, not a point-in-time rate.
df_inventory_total = run_query("""
SELECT TRIM(WH.[Warehouse Name]) AS tech_warehouse,
    COUNT_BIG(ATI.Asset_Tag) AS total_inventory_count,
    SUM(CONVERT(float, COALESCE(ATI.Unit_Cost, MP.Unit_Cost_Last_Price))) AS total_inventory_amount
FROM SERP_ACTIVE_TAGGED_INV ATI WITH (NOLOCK)
JOIN SERP_WAREHOUSES WH WITH (NOLOCK) ON WH.ID=ATI.Warehouse_ID
LEFT OUTER JOIN SERP_MASTER_PRODUCTS MP ON MP.ID=ATI.Master_ID
WHERE WH.[Warehouse Name] NOT LIKE 'Z%' AND MP.Active='Yes' AND MP.Asset_Tag_Not_Required = 'No'
GROUP BY TRIM(WH.[Warehouse Name])
""", 'Inventory totals')
df_inventory_total['total_inventory_count'] = pd.to_numeric(df_inventory_total['total_inventory_count'], errors='coerce').fillna(0).astype(int)
df_inventory_total['total_inventory_amount'] = pd.to_numeric(df_inventory_total['total_inventory_amount'], errors='coerce').fillna(0).astype(float)

# v1.3.0: df_master (product-name lookup) dropped — SERP_LOST_EQUIPMENT carries
# Product Name directly, and the frame was extracted and never read.
print(f'All queries complete in {time.time()-t0:.1f}s')


  Lost equipment (SERP_LOST_EQUIPMENT): 91,023 rows  (1.3s)
  Inventory totals: 85 rows  (0.3s)
All queries complete in 64.1s


## Cell 7.5 — PATIENT CENSUS (APC/ADC), DAILY GRAIN *(rewritten v1.5.0)*
Pulled per warehouse per DAY so every grain can pool patient-days over the dates it actually observed, instead of summing per-warehouse monthly averages taken over different day sets. One customer-exclusion filter, shared with the APC snapshot. Virtual (`Z%`) census is kept and tagged, not deleted.

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# PATIENT CENSUS (APC/ADC) — DAILY GRAIN                          rewritten v1.5.0
#
# WHY THIS WAS REWRITTEN. v1.4.0 pulled census pre-averaged in SQL, per warehouse-month,
# and three defects came with that shape:
#
#   1. TWO DEFINITIONS OF CENSUS IN ONE NOTEBOOK. The APC snapshot excluded facility
#      '(F)', inpatient-unit '(IPU)' and 'Contract Test' customers. The ADC series — the
#      one every census metric and the lost-equipment rate actually consume — excluded
#      none of them. So ADC carried census the company's own APC definition does not.
#      One filter list now (CENSUS_CUSTOMER_EXCLUDE_LIKE), applied to both, and the size
#      of the difference is printed every run instead of being invisible.
#
#   2. COMPANY ADC WAS A SUM OF PER-WAREHOUSE AVERAGES OVER DIFFERENT DAY SETS. AVG(APC)
#      was computed per warehouse over the dates THAT warehouse appears in the feed, and
#      the panels then summed those averages across warehouses. When coverage differs by
#      site — one onboarded mid-month, one that stops reporting, any feed gap — the sum
#      corresponds to no actual day. That is live right now in two places: the partial
#      current month, and the five sites the insights log records going silent after
#      Jun-2026. Pulling daily rows lets every grain compute ADC as pooled patient-days
#      over distinct dates observed, which is the ratio-of-pooled-sums rule Cell 13
#      already applies everywhere else.
#
#   3. VIRTUAL WAREHOUSES WERE DROPPED FROM THE NUMERATOR ONLY. 'Z%' census was excluded
#      while the denominator kept the technicians who serve it (Cell 8.1b re-attributes
#      their tickets) — the same asymmetry that produced the false July "productivity
#      drop", re-introduced in a new panel. Virtual census is now KEPT and tagged. It
#      cannot be re-attributed the way a ticket can (a census row carries no technician),
#      so it stays in the COMPANY total, is labelled, and is reported separately.
#
# COST: one daily aggregate instead of one monthly one. ~590 days x ~72 warehouses is
# tens of thousands of rows — trivial, and it is the only shape that supports a weekly
# grain at all.
# ─────────────────────────────────────────────────────────────────────────────
_apc_snap_df = run_query(f"""
SELECT TOP 1 [date] FROM SERP_APC_DAILY
WHERE [date] <= '{FILTER_END}'
ORDER BY [date] DESC
""", 'APC snapshot-date probe')
if len(_apc_snap_df) == 0 or pd.isna(_apc_snap_df.iloc[0, 0]):
    raise RuntimeError(f'APC: no rows on or before {FILTER_END}. Cannot compute snapshot.')
_apc_snap_date = pd.to_datetime(_apc_snap_df.iloc[0, 0]).date()
_apc_lag_days = (AS_OF_DATE - _apc_snap_date).days
print(f'  APC snapshot date: {_apc_snap_date.isoformat()} '
      f'(lag: {_apc_lag_days} day(s) behind AS_OF_DATE)')
if _apc_lag_days > 7:
    print('  *** WARNING: APC feed is >7 days stale. Census-per-technician and '
          'lost-cost-per-patient-day are both unreliable at the tail.')

# The exclusion predicate is built ONCE from config and reused, so the snapshot and the
# series can no longer drift apart the way they did in v1.4.0.
_cx = ' AND '.join(f"APC.customer NOT LIKE '{p}'" for p in CENSUS_CUSTOMER_EXCLUDE_LIKE)
_cx_case = ' AND '.join(f"APC.customer NOT LIKE '{p}'" for p in CENSUS_CUSTOMER_EXCLUDE_LIKE)

df_apc = run_query(f"""
SELECT TRIM(APC.warehourse) AS warehouse, SUM(CAST(APC.total AS FLOAT)) AS apc
FROM SERP_APC_DAILY AS APC WITH (NOLOCK)
WHERE APC.[date] = '{_apc_snap_date.isoformat()}'
  AND APC.warehourse NOT LIKE '{VIRTUAL_WH_PREFIX}%'
  AND {_cx}
GROUP BY TRIM(APC.warehourse)
""", 'APC snapshot')
df_apc['apc'] = df_apc['apc'].apply(clean_numbers)

# ── the daily census frame every census metric is built from ─────────────────
# Both sums come back in one pass so the cost of the excluded population is measurable
# rather than asserted. NULL customer is counted separately: 'NOT LIKE' is NULL-unsafe,
# so a NULL customer falls out of the filtered sum silently — v1.4.0's snapshot had the
# same behaviour, and it should be visible, not inherited.
df_census_daily = run_query(f"""
SELECT CAST(APC.[date] AS DATE)     AS census_date,
       TRIM(APC.warehourse)         AS warehouse,
       SUM(CAST(APC.total AS FLOAT)) AS pt_count_all,
       SUM(CASE WHEN {_cx_case}
                THEN CAST(APC.total AS FLOAT) ELSE 0 END) AS pt_count,
       SUM(CASE WHEN APC.customer IS NULL
                THEN CAST(APC.total AS FLOAT) ELSE 0 END) AS pt_count_null_customer
FROM SERP_APC_DAILY AS APC WITH (NOLOCK)
WHERE APC.[date] >= '{FILTER_START}' AND APC.[date] <= '{FILTER_END}'
GROUP BY CAST(APC.[date] AS DATE), TRIM(APC.warehourse)
""", 'Census daily (warehouse x date)')

df_census_daily['census_date'] = pd.to_datetime(df_census_daily['census_date'], errors='coerce')
_n_nat = int(df_census_daily['census_date'].isna().sum())
if _n_nat:
    print(f'  *** WARNING: {_n_nat:,} census rows have an unparseable date — DROPPED.')
    df_census_daily = df_census_daily[df_census_daily['census_date'].notna()].copy()
for _c in ('pt_count', 'pt_count_all', 'pt_count_null_customer'):
    df_census_daily[_c] = pd.to_numeric(df_census_daily[_c], errors='coerce').fillna(0.0)
df_census_daily['warehouse'] = df_census_daily['warehouse'].fillna('').astype(str).str.strip()
df_census_daily['_is_virtual_census'] = (df_census_daily['warehouse'].str.upper()
                                         .str.startswith(VIRTUAL_WH_PREFIX.upper()))
df_census_daily['week_start'] = week_start_of(df_census_daily['census_date'])
df_census_daily['period'] = df_census_daily['census_date'].dt.strftime('%Y-%m')

# ── DEFECT 1 MADE VISIBLE: what the customer filter is worth ─────────────────
_all_pd = float(df_census_daily['pt_count_all'].sum())
_inc_pd = float(df_census_daily['pt_count'].sum())
_null_pd = float(df_census_daily['pt_count_null_customer'].sum())
_excl_pct = (1 - _inc_pd / max(_all_pd, 1e-9)) * 100
print(f'Census filter (v1.5.0 correction) — patient-days in window: '
      f'{_all_pd:,.0f} unfiltered vs {_inc_pd:,.0f} after '
      f'{CENSUS_CUSTOMER_EXCLUDE_LIKE}: {_excl_pct:.1f}% excluded.')
print(f'  v1.4.0 ADC used the UNFILTERED figure while the APC snapshot used the filtered '
      f'one. Every census-per-technician number from v1.4.0 is high by this much, and '
      f'lost-cost-per-patient-day is low by it.')
if _null_pd > 0:
    print(f'  NOTE: {_null_pd:,.0f} patient-days ({_null_pd / max(_all_pd, 1e-9) * 100:.2f}%) '
          f'carry a NULL customer and fall OUT of the filtered figure, because NOT LIKE is '
          f'NULL-unsafe. Inherited from the v1.4.0 snapshot definition, not introduced here. '
          f'If those are real patients the filter needs an explicit IS NULL branch — raised '
          f'in questions-for-cfo.md.')

# ── DEFECT 3 MADE VISIBLE: virtual-warehouse census ─────────────────────────
_virt_pd = float(df_census_daily.loc[df_census_daily['_is_virtual_census'], 'pt_count'].sum())
_virt_pct = _virt_pd / max(_inc_pd, 1e-9) * 100
print(f'Virtual-warehouse census: {_virt_pd:,.0f} patient-days ({_virt_pct:.2f}% of the '
      f'window) on {df_census_daily.loc[df_census_daily["_is_virtual_census"], "warehouse"].nunique()} '
      f'{VIRTUAL_WH_PREFIX}-prefixed codes.')
if not CENSUS_KEEP_VIRTUAL_WH:
    df_census_daily = df_census_daily[~df_census_daily['_is_virtual_census']].copy()
    print('  CENSUS_KEEP_VIRTUAL_WH=False -> dropped (reproduces v1.4.0).')
else:
    print('  Kept and tagged. It counts at COMPANY level and cannot reach a site page — a '
          'census row carries no technician, so Cell 8.1b\'s ticket re-attribution does not '
          'apply to it. v1.4.0 dropped it from the numerator while keeping the technicians '
          'who serve it in the denominator.')
_virt_by_month = (df_census_daily[df_census_daily['_is_virtual_census']]
                  .groupby('period', as_index=False).agg(pt_days=('pt_count', 'sum')))
if len(_virt_by_month):
    print(f'  Virtual census by month: '
          f'{dict(zip(_virt_by_month["period"], _virt_by_month["pt_days"].round(0)))}')

# ── canonical ADC at warehouse x month, as a RATIO OF POOLED SUMS ───────────
# adc = patient-days / distinct dates OBSERVED for that warehouse-month. Kept per
# warehouse-month for continuity (df_adc is consumed by Cell 14); the entity-grain
# aggregation in Cell 13.5 re-pools from df_census_daily rather than summing these
# averages, which is the fix for defect 2.
df_adc = (df_census_daily.groupby(['warehouse', 'period'], as_index=False)
          .agg(pt_days=('pt_count', 'sum'), census_days=('census_date', 'nunique')))
df_adc['adc'] = (df_adc['pt_days'] / df_adc['census_days'].replace(0, np.nan)).round(2)
df_adc[['yr', 'mo']] = df_adc['period'].str.split('-', expand=True).astype(int)

# Coverage diagnostic: a warehouse-month observed on far fewer dates than the month has
# is exactly the case where v1.4.0's summed averages went wrong. Report it, do not fix it
# silently — a site that genuinely closed and one whose feed broke look identical here.
_days_in_month = (df_census_daily.groupby('period', as_index=False)
                  .agg(month_days=('census_date', 'nunique')))
_cov = df_adc.merge(_days_in_month, on='period', how='left')
_cov['coverage_pct'] = (_cov['census_days'] / _cov['month_days'].replace(0, np.nan) * 100).round(1)
tbl_census_coverage = _cov[_cov['coverage_pct'] < 90][
    ['warehouse', 'period', 'census_days', 'month_days', 'coverage_pct', 'pt_days']
].sort_values(['period', 'coverage_pct']).reset_index(drop=True)
print(f'Census coverage: {len(tbl_census_coverage):,} of {len(df_adc):,} warehouse-months are '
      f'observed on <90% of the dates that month has data for '
      f'(sheet Census_Coverage — these are the rows v1.4.0\'s summed averages distorted).')
print(f'Census extract: {len(df_census_daily):,} warehouse-days | '
      f'{df_census_daily["warehouse"].nunique():,} warehouses | '
      f'{df_census_daily["census_date"].nunique():,} distinct dates | '
      f'company ADC latest month '
      f'{df_census_daily[df_census_daily["period"] == df_census_daily["period"].max()]["pt_count"].sum() / max(df_census_daily[df_census_daily["period"] == df_census_daily["period"].max()]["census_date"].nunique(), 1):,.0f}')

  APC snapshot-date probe: 1 rows  (0.2s)
  APC snapshot date: 2026-08-14 (lag: 0 day(s) behind AS_OF_DATE)
  APC snapshot: 63 rows  (0.1s)
  Census daily (warehouse x date): 36,387 rows  (1.3s)
Census filter (v1.5.0 correction) — patient-days in window: 12,329,304 unfiltered vs 12,125,323 after ['(F)%', '(IPU)%', '%Contract Test%']: 1.7% excluded.
  v1.4.0 ADC used the UNFILTERED figure while the APC snapshot used the filtered one. Every census-per-technician number from v1.4.0 is high by this much, and lost-cost-per-patient-day is low by it.
Virtual-warehouse census: 45,877 patient-days (0.38% of the window) on 12 Z-prefixed codes.
  Kept and tagged. It counts at COMPANY level and cannot reach a site page — a census row carries no technician, so Cell 8.1b's ticket re-attribution does not apply to it. v1.4.0 dropped it from the numerator while keeping the technicians who serve it in the denominator.
  Virtual census by month: {'2025-01': 2421.0, '2025-02': 1768.0, '2025-03': 1961.0, '

## Cell 7.6 — PAYROLL HOURS (PLC) EXTRACT + DEDUPLICATION *(new in v1.3.0)*

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# PAYROLL HOURS (PLC) EXTRACT + DEDUPLICATION                        new v1.3.0
#
# v1.0.0 skipped payroll on purpose ("no metric here needs hours"). The overtime
# section needs it. Three guards are load-bearing — see Cell 3.5 for the evidence:
#   1. Department_Name IS NOT NULL      — 709 aggregate rows carry 2,600-3,100 hours
#                                        each (1.05M hours, more than all real rows).
#   2. WorkDate < AS_OF_DATE + 1 day    — the feed holds FUTURE-dated rows (to
#                                        2026-08-19 against an AS_OF of 2026-08-13).
#   3. de-duplication, below            — the feed re-loads overlapping windows.
#
# Only [Hours] is trusted. Reg_Hrs / OT1_Hrs / OT2_Hrs / Paid_Hrs / Est_* are pulled
# ONLY so the diagnostic can show why they are unusable; no metric reads them.
#
# TEACHING NOTE — why the half-open upper bound: WorkDate is typed DATE here, so
# 'WorkDate <= FILTER_END' would be safe, but the half-open form survives the column
# becoming DATETIME upstream (a plain <= would then silently drop the final day's
# intra-day rows). Same reasoning as the H3 fix on Completed_Date.
# ─────────────────────────────────────────────────────────────────────────────
_ot_depts_sql = ','.join(f"'{d}'" for d in OT_DEPTS_ALL)
df_hours_raw = run_query(f"""
SELECT TRIM(ISNULL(P.ID,''))            AS plc_id,
       P.Employee_Name                  AS employee_name,
       CAST(P.WorkDate AS DATE)         AS workdate,
       CAST(P.[Hours] AS FLOAT)         AS hours,
       ISNULL(P.Pay_Type,'(blank)')     AS pay_type,
       P.Department_Name                AS dept,
       ISNULL(P.Location_Name,'')       AS location_name,
       CAST(P.SystemUpdatedDate AS DATE) AS sys_updated,
       -- diagnostic only: proven unusable, never summed into a metric
       CAST(ISNULL(P.Reg_Hrs,0) AS FLOAT) AS reg_hrs_unusable,
       CAST(ISNULL(P.OT1_Hrs,0) AS FLOAT) AS ot1_hrs_unusable,
       -- deterministic day-of-week, immune to @@DATEFIRST. 1900-01-01 was a Monday,
       -- so this is Monday=0 .. Saturday=5, Sunday=6 (lifted from tech_workload L3).
       (DATEDIFF(day, '1900-01-01', CAST(P.WorkDate AS DATE)) % 7) AS dow_mon0
FROM {PLC_TABLE} AS P WITH (NOLOCK)
WHERE P.Department_Name IN ({_ot_depts_sql})
  AND P.Employee_Name IS NOT NULL
  AND P.WorkDate >= '{FILTER_START}'
  AND P.WorkDate < DATEADD(day, 1, '{FILTER_END}')
""", 'Payroll hours (PLC)')

_plc_rows_raw = len(df_hours_raw)
df_hours_raw['hours'] = pd.to_numeric(df_hours_raw['hours'], errors='coerce').fillna(0.0)
df_hours_raw['workdate'] = pd.to_datetime(df_hours_raw['workdate'], errors='coerce')
df_hours_raw = df_hours_raw[df_hours_raw['workdate'].notna()].copy()

# ── DEDUPLICATION ────────────────────────────────────────────────────────────
# The feed appends re-loaded windows rather than replacing them, so an employee-day
# can appear many times (worst observed: 30 copies of 2026-06-22). Dropping exact
# duplicates on the full business tuple is the conservative fix: it cannot merge two
# genuinely different entries, because any difference in id, pay type, location or
# hours keeps both rows. The only thing it can lose is a genuine duplicate — two
# byte-identical rows that were both real — which payroll does not produce.
_dupe_mask = df_hours_raw.duplicated(subset=PLC_DEDUP_KEYS, keep='first')
_n_dupes = int(_dupe_mask.sum())
df_hours_raw = df_hours_raw[~_dupe_mask].copy()
print(f'  PLC de-duplication: {_n_dupes:,} duplicate rows dropped '
      f'({_n_dupes / max(_plc_rows_raw, 1) * 100:.1f}% of {_plc_rows_raw:,}) -> {len(df_hours_raw):,} kept')

# Cross-check against the alternative rule "keep only the newest load per employee-day"
# — if the two disagree materially, the feed is doing something we do not understand yet.
_latest = (df_hours_raw.groupby(['employee_name', 'workdate'])['sys_updated'].transform('max'))
_alt_hours = float(df_hours_raw.loc[df_hours_raw['sys_updated'].eq(_latest), 'hours'].sum())
_kept_hours = float(df_hours_raw['hours'].sum())
print(f'  Hours after dedup: {_kept_hours:,.0f}  |  newest-load-only alternative: '
      f'{_alt_hours:,.0f}  (delta {(_alt_hours / max(_kept_hours, 1) - 1) * 100:+.1f}%)')
if abs(_alt_hours / max(_kept_hours, 1) - 1) > 0.05:
    print('  *** NOTE: the two de-duplication rules differ by >5%. Rows within a single '
          'load may already be duplicated, or a load may revise hours rather than repeat '
          'them. Confirm with payroll before publishing OT dollars.')

_dup_by_month = (df_hours_raw.assign(_p=df_hours_raw['workdate'].dt.strftime('%Y-%m'))
                 .groupby('_p').agg(rows=('hours', 'size'), hours=('hours', 'sum'),
                                    employees=('employee_name', 'nunique')))
print('\nPLC hours by month after de-duplication (all departments in scope):')
print(_dup_by_month.round(0).to_string())
print(f"\nDepartments pulled: {sorted(df_hours_raw['dept'].dropna().unique().tolist())}")
print(f"Pay types present:  {sorted(df_hours_raw['pay_type'].dropna().unique().tolist())}")


  Payroll hours (PLC): 123,305 rows  (1.0s)
  PLC de-duplication: 15,335 duplicate rows dropped (12.4% of 123,305) -> 107,970 kept
  Hours after dedup: 915,263  |  newest-load-only alternative: 914,616  (delta -0.1%)

PLC hours by month after de-duplication (all departments in scope):
         rows    hours  employees
_p                               
2025-01  4480  37611.0        185
2025-02  4034  33857.0        187
2025-03  4426  36532.0        188
2025-04  4503  37182.0        191
2025-05  4818  39856.0        201
2025-06  4741  39390.0        208
2025-07  5301  44333.0        225
2025-08  5313  44436.0        230
2025-09  5767  48192.0        251
2025-10  6256  52619.0        265
2025-11  6213  51495.0        283
2025-12  7077  59920.0        294
2026-01  7129  60296.0        307
2026-02  6409  55278.0        305
2026-03  5406  45997.0        319
2026-04  5493  47530.0        261
2026-05  6017  51963.0        276
2026-06  5962  53161.0        299
2026-07  6907  61024.0        312


## Cell 8.1 — DATES, NaT GUARD, SCHEDULE PERIOD & METRO

In [19]:
df_tx['completed_date'] = pd.to_datetime(df_tx['completed_date'],errors='coerce')
# v1.33.0 (H3): NaT guard. With the WHERE now on TRY_CONVERT this should be zero;
# if it isn't, rows were admitted that can't be dated and would silently form
# NaN month groups (or crash later int casts). Fail loudly, drop, and report.
_n_nat = int(df_tx['completed_date'].isna().sum())
if _n_nat > 0:
    print(f'*** WARNING: {_n_nat:,} transaction rows have unparseable Completed_Date — DROPPED.')
    df_tx = df_tx[df_tx['completed_date'].notna()].copy()
df_tx['delivery_year']  = df_tx['completed_date'].dt.year
df_tx['delivery_month'] = df_tx['completed_date'].dt.month
df_tx['month_date']     = df_tx['completed_date'].values.astype('datetime64[M]')
_dow = df_tx['completed_date'].dt.dayofweek
df_tx['schedule_period'] = np.select([_dow==5,_dow==6],['Saturday','Sunday'],default='Weekday')
df_tx['metro'] = df_tx['tech_warehouse'].apply(assign_metro)


## Cell 8.1b — VIRTUAL ("Z") WAREHOUSE RESOLUTION *(new in v1.2.0)*
Re-attributes virtual-warehouse tickets to the physical warehouse the technician
actually worked, so real field work stops vanishing out of the numerator.

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# VIRTUAL ("Z") WAREHOUSE RESOLUTION                                    v1.2.0
#
# WHY THIS CELL EXISTS. 'Z%' warehouse codes are virtual holding buckets, not
# buildings. v1.1.0 dropped them in SQL, which was harmless while they carried
# only housekeeping rows (1-2% of tickets through 2026-05). From 2026-06 'Z CS'
# began carrying ordinary completed field work — Additional Equipment (D), New
# Admit (D), Hospital Discharge (D), Respiratory Distress (D) — reaching 24.3% of
# eligible weekday tickets in Jul-2026. The tickets were deleted but the
# technician's active DAY was not, so tickets-per-active-day fell 13.2% company
# wide in July while real volume rose. This cell puts the work back.
#
# RESOLUTION LADDER (most-specific evidence first — a technician's other tickets
# that day are the strongest evidence of where they physically worked):
#   1. same_day   — modal non-virtual warehouse for that technician, that date
#   2. same_week  — modal non-virtual warehouse for that technician, that week
#   3. same_month — modal non-virtual warehouse for that technician, that month
#   4. VIRTUAL_UNRESOLVED_LABEL — kept and labelled, never silently dropped
# Ties break alphabetically, so assignment is deterministic across runs.
#
# TEACHING NOTE — why modal-from-tickets rather than the employee roster's home
# warehouse: SERP_DME_EMPLOYEES is a CURRENT snapshot, so it would back-date
# today's assignment onto last year's tickets. The technician's own same-day route
# is contemporaneous evidence and needs no join. Running here — before the matcher
# in Cell 9.3 — also means the warehouse-dependent match passes (P4-P7) see a real
# warehouse instead of 'Z CS', which no employee record will ever match.
#
# CAVEAT to repeat in any output that uses this: a resolved warehouse is INFERRED,
# not stamped on the ticket. It is sound for site-level trend and workload; it is
# not evidence about an individual without ticket-level review.
# ─────────────────────────────────────────────────────────────────────────────
_wh_raw = df_tx['tech_warehouse'].fillna('').astype(str).str.strip()
df_tx['_wh_ticket_stamped'] = _wh_raw
df_tx['_is_virtual_wh'] = (_wh_raw.str.upper().str.startswith(VIRTUAL_WH_PREFIX.upper())
                           | (_wh_raw == ''))
df_tx['_tech_key'] = (df_tx['techfirstname'].fillna('').str.strip() + '|'
                      + df_tx['techlastname'].fillna('').str.strip())
df_tx['_wh_resolution'] = 'ticket_stamped'

_n_virt = int(df_tx['_is_virtual_wh'].sum())
print(f'Virtual-warehouse ticket rows in window: {_n_virt:,} of {len(df_tx):,} '
      f'({_n_virt / max(len(df_tx), 1) * 100:.1f}%)')
if _n_virt:
    print('  by ticket-stamped code:')
    print(df_tx.loc[df_tx['_is_virtual_wh'], '_wh_ticket_stamped']
          .replace('', '(blank)').value_counts().head(10).to_string())

if not INCLUDE_VIRTUAL_WH_IN_PRODUCTIVITY:
    df_tx = df_tx[~df_tx['_is_virtual_wh']].copy()   # v1.1.0 parity, reconciliation only
    print('  INCLUDE_VIRTUAL_WH_IN_PRODUCTIVITY=False -> virtual rows DROPPED (v1.1.0 parity).')
elif _n_virt:
    # Modal maps are built ONLY from non-virtual rows carrying a real technician
    # name: a blank name would pool every anonymous row into one group and hand it
    # a meaningless "modal" warehouse.
    _blank_tech = df_tx['_tech_key'].isin(['|', ''])
    _real = df_tx.loc[~df_tx['_is_virtual_wh'] & ~_blank_tech,
                      ['_tech_key', 'completed_date', 'tech_warehouse']].copy()
    _real['_week']  = _real['completed_date'].dt.to_period('W')
    _real['_month'] = _real['completed_date'].dt.to_period('M')

    def _modal_wh_map(keys):
        """Most-frequent non-virtual warehouse per key set; alphabetical tiebreak."""
        g = (_real.groupby(keys + ['tech_warehouse'], dropna=False)
             .size().reset_index(name='_n'))
        g = g.sort_values(keys + ['_n', 'tech_warehouse'],
                          ascending=[True] * len(keys) + [False, True])
        return g.drop_duplicates(subset=keys).set_index(keys)['tech_warehouse']

    _m_day   = _modal_wh_map(['_tech_key', 'completed_date'])
    _m_week  = _modal_wh_map(['_tech_key', '_week'])
    _m_month = _modal_wh_map(['_tech_key', '_month'])

    _v  = df_tx['_is_virtual_wh']
    _vd = df_tx.loc[_v]
    _res = pd.Series(pd.NA, index=_vd.index, dtype=object)
    _src = pd.Series(pd.NA, index=_vd.index, dtype=object)
    for _lvl, _map, _ix in [
        ('same_day',   _m_day,
         pd.MultiIndex.from_arrays([_vd['_tech_key'], _vd['completed_date']])),
        ('same_week',  _m_week,
         pd.MultiIndex.from_arrays([_vd['_tech_key'], _vd['completed_date'].dt.to_period('W')])),
        ('same_month', _m_month,
         pd.MultiIndex.from_arrays([_vd['_tech_key'], _vd['completed_date'].dt.to_period('M')])),
    ]:
        _need = _res.isna()
        if not _need.any():
            break
        _cand = pd.Series(_map.reindex(_ix).values, index=_vd.index)
        _fill = _need & _cand.notna()
        _res[_fill] = _cand[_fill]
        _src[_fill] = _lvl

    _res = _res.fillna(VIRTUAL_UNRESOLVED_LABEL)
    _src = _src.fillna('unresolved')
    df_tx.loc[_v, 'tech_warehouse'] = _res
    df_tx.loc[_v, '_wh_resolution'] = _src

    # metro was assigned off the ticket-stamped name in Cell 8.1 — redo it now the
    # warehouse is physical, or the DFW/Houston/San Antonio pages miss the recovery.
    df_tx['metro'] = df_tx['tech_warehouse'].apply(assign_metro)

    print('\nVirtual-warehouse resolution by month (ticket rows):')
    print(df_tx.loc[_v].groupby([df_tx.loc[_v, 'month_date'].dt.strftime('%Y-%m'),
                                 '_wh_resolution']).size().unstack(fill_value=0).to_string())
    _n_unres = int((df_tx['_wh_resolution'] == 'unresolved').sum())
    print(f'\nResolved: {_n_virt - _n_unres:,} / {_n_virt:,} '
          f'({(_n_virt - _n_unres) / max(_n_virt, 1) * 100:.2f}%).  '
          f'Unresolved kept as "{VIRTUAL_UNRESOLVED_LABEL}": {_n_unres:,}')
    if _n_unres:
        print('  Unresolved = the technician has no non-virtual ticket that month (mostly '
              'the older "Z Equipment Collections" workflow). Reported, not deleted; they '
              'carry state="Unknown" and get their own warehouse page.')


Virtual-warehouse ticket rows in window: 17,444 of 467,718 (3.7%)
  by ticket-stamped code:
_wh_ticket_stamped
Z CS                        11790
Z Equipment Collections      5399
Z Equipment Needed            225
Z Fort Smith (Shut Down)       24
(blank)                         3
Z Tuscaloosa                    2
Z Orlando                       1

Virtual-warehouse resolution by month (ticket rows):
_wh_resolution  same_day  same_month  same_week  unresolved
month_date                                                 
2025-01                0           0          0         342
2025-02                0           0          0         407
2025-03                3          21          8         356
2025-04                0           0          0         433
2025-05                5          56         14         304
2025-06               12          52         55         355
2025-07                9          95        117         211
2025-08               27         233        113         1

## Cell 8.2 — REASON CLASSIFICATION & TICKET TYPE

In [21]:
_REASON_MAP = [
    (r'priority 1','Urgent',1),(r'priority 2 - exchange','Exchange/Service',2),
    (r'priority 2 - new admit','New Admission',2),(r'priority 2 - respiratory','Respiratory',2),
    (r'priority 2','Exchange/Service',2),(r'priority 3 - additional','Additional Equip',3),
    (r'priority 3 - change','Admin/Change',3),(r'priority 3 - customer','Patient Request',3),
    (r'priority 3 - live disc','Live Discharge',3),(r'priority 3 - o2','O2 Refill',3),
    (r'priority 3 - patient exp','Patient Expired',3),(r'priority 3 - respite','Respite',3),
    (r'priority 3','Routine P3',3),(r'split order','Split Order',4),
]
def _classify(reason):
    if not reason or not isinstance(reason,str): return ('Other',5)
    r=reason.lower()
    for pat,cat,pri in _REASON_MAP:
        if re.search(pat,r): return (cat,pri)
    return ('Other',5)
def _ticket_type(reason):
    if not reason or not isinstance(reason,str): return 'Other'
    if reason.strip().upper()=='SPLIT ORDER': return 'Split'
    m=re.search(r'\(([DPS])\)\s*$',reason.strip())
    return {'D':'Delivery','P':'Pickup','S':'Service'}[m.group(1)] if m else 'Other'

_unique_reasons    = df_tx['reason'].dropna().unique()
_reason_cat_lookup = {r:_classify(r)[0] for r in _unique_reasons}
_reason_pri_lookup = {r:_classify(r)[1] for r in _unique_reasons}
_type_lookup       = {r:_ticket_type(r) for r in _unique_reasons}
df_tx['reason_category'] = df_tx['reason'].map(_reason_cat_lookup).fillna('Other')
df_tx['priority_level']  = df_tx['reason'].map(_reason_pri_lookup).fillna(5).astype(int)
df_tx['ticket_type']     = df_tx['reason'].map(_type_lookup).fillna('Other')

print(f'Rows: {len(df_tx):,}')
print(df_tx['schedule_period'].value_counts().to_string())
# State diagnostic moved to Cell 9 (v1.26.1) — state isn't merged in yet here.
print('\nMetro distribution:')
print(df_tx['metro'].value_counts(dropna=False).to_string())


Rows: 467,718
schedule_period
Weekday     436834
Saturday     20139
Sunday       10745

Metro distribution:
metro
None           369805
San Antonio     35446
Houston         34566
DFW             27901


## Cell 9.1 — EMPLOYEE INDEXES & ROLE CLASSIFIERS

In [22]:
# Name normalizer used by every matching pass: strip apostrophes, periods,
# spaces, hyphens; lowercase. "O'Brien" == "OBrien" == "o brien".
# MANUAL_NAME_CORRECTIONS (Pass 0) fixes known-bad spellings before any lookup.
def _clean(s):
    if not s or not isinstance(s,str): return ''
    return re.sub(r"['\.\s\-]",'',s).lower()

MANUAL_NAME_CORRECTIONS = {
    ('damein','combs'):('Damien','Combs'),
    ('damien','combs'):('Damien','Combs'),
}

emp_idx={}; _emp_by_last=defaultdict(list)
for row in df_emp.itertuples(index=False):
    fn,ln=_clean(row.empfirstname),_clean(row.emplastname)
    if fn and ln: emp_idx[(fn,ln)]=row; _emp_by_last[ln].append((fn,row))
print(f'Employee index: {len(emp_idx):,}')

def _classify_role(dept_raw,title_raw):
    d=(dept_raw or '').lower().strip(); t=(title_raw or '').lower().strip()
    if any(x in t for x in FIELD_TECH_TITLES) or any(x in d for x in FIELD_TECH_DEPTS): return 'field_tech'
    if any(x in t for x in INTERNAL_OPS_TITLES) or any(x in d for x in INTERNAL_OPS_DEPTS): return 'internal_ops'
    return 'other'

_dispatcher_last_names=set()
for row in df_emp.itertuples(index=False):
    if any(d in (row.dept or '').lower() for d in INTERNAL_OPS_DEPTS):
        ln_c=_clean(row.emplastname)
        if ln_c: _dispatcher_last_names.add(ln_c)

_tech_by_wh_first={}; _wh_first_collisions=[]
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,fn_c=_clean(row.location),_clean(row.empfirstname)
    if not (wh_c and fn_c): continue
    key=(wh_c,fn_c)
    if key in _tech_by_wh_first: _wh_first_collisions.append({'wh':row.location,'first':row.empfirstname})
    else: _tech_by_wh_first[key]=row

_field_tech_by_wh_last={}; _wh_last_collisions={}
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)!='field_tech': continue
    wh_c,ln_c=_clean(row.location),_clean(row.emplastname)
    if not (wh_c and ln_c): continue
    key=(wh_c,ln_c)
    if key in _field_tech_by_wh_last: _wh_last_collisions.setdefault(key,[_field_tech_by_wh_last[key]]).append(row)
    else: _field_tech_by_wh_last[key]=row
for key in list(_wh_last_collisions): _field_tech_by_wh_last.pop(key,None)

_internal_ops_first_names=set(); _internal_ops_last_names=set()
for row in df_emp.itertuples(index=False):
    if _classify_role(row.dept,row.title)=='internal_ops':
        fn_c=_clean(row.empfirstname); ln_c=_clean(row.emplastname)
        if fn_c: _internal_ops_first_names.add(fn_c)
        if ln_c: _internal_ops_last_names.add(ln_c)


Employee index: 1,995


## Cell 9.2 — MATCHER FUNCTION (PASSES 0–5)

In [23]:
def _match_name(fn_raw,ln_raw,wh_raw=''):
    """Passes 0-5 name matching. Passes 6-7 run in the caller loop."""
    fn_lc,ln_lc=fn_raw.lower(),ln_raw.lower()
    corrected=MANUAL_NAME_CORRECTIONS.get((fn_lc,ln_lc))
    if corrected: fn_raw,ln_raw=corrected
    fn_c,ln_c=_clean(fn_raw),_clean(ln_raw); wh_c=_clean(wh_raw) if wh_raw else ''
    if (fn_c,ln_c) in emp_idx: return emp_idx[(fn_c,ln_c)],('P0' if corrected else 'P1')
    for cand in standardize_first_name(fn_raw)[1:]:
        key=(_clean(cand),ln_c)
        if key in emp_idx: return emp_idx[key],'P2'
    best_row,best_dist,best_same_wh=None,FUZZY_EDIT_DIST+1,False
    for (emp_fn,emp_row) in _emp_by_last.get(ln_c,[]):
        d=levenshtein(fn_c,emp_fn); same_wh=(wh_c!='' and _clean(getattr(emp_row,'location','') or '')==wh_c)
        if d<best_dist or (d==best_dist and same_wh and not best_same_wh):
            best_row,best_dist,best_same_wh=emp_row,d,same_wh
    if best_row is not None and best_dist<=FUZZY_EDIT_DIST: return best_row,'P3'
    if ln_c in _dispatcher_last_names and wh_c:
        match=_tech_by_wh_first.get((wh_c,fn_c))
        if match is not None: return match,'P4'
    if wh_c and ln_c:
        match=_field_tech_by_wh_last.get((wh_c,ln_c))
        if match is not None: return match,'P5_fallback'
    return None,None


## Cell 9.3 — MATCHING LOOP (PASSES 6–7 OVERRIDES)

In [24]:
# THE MATCHING LOOP — one pass over each distinct (first, last, warehouse) name
# combination seen on tickets (not per ticket row: cheaper and idempotent).
# Order of operations per name: blank check → passes 0–5 via _match_name() →
# passes 5b/6/7 internal-ops overrides (a dispatcher name on a ticket usually
# means the dispatcher ENTERED it; if a same-warehouse field tech shares the
# last/first name, re-attribute to the tech) → collision bookkeeping.
_unique_names=df_tx[['techfirstname','techlastname','tech_warehouse']].drop_duplicates()
_match_results=[]
_pass_counts={'P0':0,'P1':0,'P2':0,'P3':0,'P4':0,'P5_fallback':0,'P5_intops_override':0,
              'P6_intops_first_override':0,'P7_intops_last_override':0,
              'ambiguous_collision':0,'unmatched_blank':0,'unmatched':0}

for row in _unique_names.itertuples(index=False):  # each distinct name-warehouse combo, once
    fn,ln,wh=row.techfirstname,row.techlastname,row.tech_warehouse
    is_blank=not fn.strip() and not ln.strip()
    pass_used,matched,collision_candidates=None,None,[]
    if is_blank:
        _pass_counts['unmatched_blank']+=1
    else:
        matched,pass_used=_match_name(fn,ln,wh)
        if matched is not None and _classify_role(matched.dept,matched.title)=='internal_ops' and wh and ln:
            _p5=_field_tech_by_wh_last.get((_clean(wh),_clean(ln)))
            if _p5 is not None: matched,pass_used=_p5,'P5_intops_override'
        if wh and ln and pass_used!='P5_intops_override':
            fn_c,ln_c,wh_c=_clean(fn),_clean(ln),_clean(wh)
            if fn_c in _internal_ops_first_names:
                _p6=_field_tech_by_wh_last.get((wh_c,ln_c))
                _cur_ok=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c)
                if _p6 is not None and not _cur_ok: matched,pass_used=_p6,'P6_intops_first_override'
        if wh and fn and pass_used not in ('P5_intops_override','P6_intops_first_override'):
            fn_c7,ln_c7,wh_c7=_clean(fn),_clean(ln),_clean(wh)
            if ln_c7 in _internal_ops_last_names:
                _p7=_tech_by_wh_first.get((wh_c7,fn_c7))
                _cur_ok7=(matched is not None and _classify_role(matched.dept,matched.title)=='field_tech' and _clean(getattr(matched,'location','') or '')==wh_c7)
                if _p7 is not None and not _cur_ok7: matched,pass_used=_p7,'P7_intops_last_override'
        # If still unmatched, record whether it's an AMBIGUOUS collision (two field
        # techs share this last name at this warehouse) — exported for HR review.
        if matched is None and wh and ln:
            key=(_clean(wh),_clean(ln))
            if key in _wh_last_collisions: collision_candidates=_wh_last_collisions[key]; _pass_counts['ambiguous_collision']+=1
        _pass_counts['unmatched' if matched is None else pass_used]=_pass_counts.get('unmatched' if matched is None else pass_used,0)+1
    _rb=_classify_role(matched.dept,matched.title) if matched else None
    _match_results.append({'techfirstname':fn,'techlastname':ln,'tech_warehouse':wh,
        '_is_blank_tech':is_blank,'_is_unmatched':(not is_blank)and(matched is None),
        '_is_ambiguous':(not is_blank)and(matched is None)and len(collision_candidates)>0,
        '_collision_candidates':'; '.join(f'{r.empfirstname} {r.emplastname} ({r.eid})' for r in collision_candidates) if collision_candidates else '',
        '_matched_dept':matched.dept if matched else None,'_matched_title':matched.title if matched else None,
        '_matched_eid':matched.eid if matched else None,'_matched_first':matched.empfirstname if matched else None,
        '_matched_last':matched.emplastname if matched else None,'_matched_location':matched.location if matched else None,
        '_matched_role_bucket':_rb,'_is_internal_ops':_rb=='internal_ops','_pass_used':pass_used})

## Cell 9.4 — MERGE RESULTS, HIERARCHY & STATE RESOLUTION

In [25]:
_df_match=pd.DataFrame(_match_results)
df_tx=df_tx.merge(_df_match,on=['techfirstname','techlastname','tech_warehouse'],how='left')
for col in ['_is_blank_tech','_is_unmatched','_is_ambiguous','_is_internal_ops']: df_tx[col]=df_tx[col].fillna(False)
df_tx['_matched_role_bucket']=df_tx['_matched_role_bucket'].fillna('unknown')
df_tx['_unattributed']=df_tx['_is_blank_tech']|df_tx['_is_unmatched']|df_tx['_is_internal_ops']

_intops_override_passes={'P5_intops_override','P6_intops_first_override','P7_intops_last_override'}
_correction_mask=df_tx['_pass_used'].isin(_intops_override_passes)
df_tx['_raw_techfirstname']=df_tx['techfirstname']; df_tx['_raw_techlastname']=df_tx['techlastname']
df_tx['_name_was_corrected']=_correction_mask
df_tx.loc[_correction_mask,'techfirstname']=df_tx.loc[_correction_mask,'_matched_first'].fillna(df_tx.loc[_correction_mask,'techfirstname'])
df_tx.loc[_correction_mask,'techlastname'] =df_tx.loc[_correction_mask,'_matched_last'].fillna(df_tx.loc[_correction_mask,'techlastname'])
df_tx=df_tx.merge(df_hier.rename(columns={'warehouse':'tech_warehouse'}),on='tech_warehouse',how='left')
# Layer 1 — value already present from SERP_WAREHOUSES.[State Province] merge.
# Layer 2 — sibling-city lookup: another warehouse with the same trimmed city
#           name (prefix stripped) that has a populated state. Self-healing
#           because as ops backfills the master, this layer picks it up.
# Layer 3 — WAREHOUSE_STATE_OVERRIDES (curated in Cell 3) for orphans.
# Layer 4 — 'Unknown' with a LOUD warning listing every warehouse so the data
#           team can fix the source. NO prefix fallback (it fabricated R0/R1/DI/RN).

def resolve_state_series(wh_series, current_state_series, hier_df):
    """3-layer state resolution. Returns (new_state_series, audit_dict).

    v1.3.0 (audit B5): the docstring used to claim this was vectorized. It is not — the
    body is a per-row Python loop with .loc assignment, ~21,000 iterations on the live
    data. Correct, but do not assume it is cheap when adding a caller.
    hier_df: a DataFrame with columns ['warehouse','state'] from df_hier."""
    out = current_state_series.copy()
    missing_mask = out.isna() | (out.astype(str).str.strip()=='')
    audit = {'sibling':[], 'override':[], 'unknown':[]}
    if not missing_mask.any():
        return out, audit

    # Build city->state map from hier_df where state IS populated
    _hier_clean = hier_df[hier_df['state'].notna() & (hier_df['state'].astype(str).str.strip()!='')].copy()
    _hier_clean['_city'] = _hier_clean['warehouse'].apply(_strip_wh_prefix)
    # If a city maps to >1 distinct state in the master, that's ambiguous — drop it
    _city_states = _hier_clean.groupby('_city')['state'].nunique()
    _unambig_cities = set(_city_states[_city_states==1].index)
    _city_to_state = (_hier_clean[_hier_clean['_city'].isin(_unambig_cities)]
                      .drop_duplicates('_city').set_index('_city')['state'].to_dict())

    for idx in out[missing_mask].index:
        wh = wh_series.loc[idx]
        if not isinstance(wh, str) or not wh.strip():
            audit['unknown'].append(wh); out.loc[idx] = 'Unknown'; continue
        # Layer 2: sibling city
        city = _strip_wh_prefix(wh)
        if city in _city_to_state:
            out.loc[idx] = _city_to_state[city]
            audit['sibling'].append((wh, city, _city_to_state[city])); continue
        # Layer 3: override
        if wh in WAREHOUSE_STATE_OVERRIDES:
            out.loc[idx] = WAREHOUSE_STATE_OVERRIDES[wh]
            audit['override'].append((wh, WAREHOUSE_STATE_OVERRIDES[wh])); continue
        # Layer 4: unknown
        out.loc[idx] = 'Unknown'
        audit['unknown'].append(wh)
    return out, audit

def _print_state_audit(audit, label):
    print(f'\n[{label}] State resolution audit:')
    if audit['sibling']:
        print(f'  Sibling-city resolved ({len(audit["sibling"])}):')
        # dedupe: show distinct (wh, state) pairs only once
        _seen = set()
        for wh, city, st in audit['sibling']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} (city='{city}') -> {st}")
                _seen.add((wh, st))
    if audit['override']:
        print(f'  Override-resolved ({len(audit["override"])}):')
        _seen = set()
        for wh, st in audit['override']:
            if (wh, st) not in _seen:
                print(f"    {wh:<40} -> {st} [override]")
                _seen.add((wh, st))
    if audit['unknown']:
        _unique_unknown = sorted(set(audit['unknown']))
        print(f'  *** UNRESOLVED ({len(_unique_unknown)} distinct warehouses) — set to state="Unknown" ***')
        for wh in _unique_unknown:
            print(f'    {wh}')
        print('  ACTION: Add to WAREHOUSE_STATE_OVERRIDES (Cell 3) or fix SERP_WAREHOUSES.[State Province].')

df_tx['state'], _state_audit = resolve_state_series(df_tx['tech_warehouse'], df_tx['state'], df_hier)
_print_state_audit(_state_audit, 'df_tx')

# Remove R0/R1/DI/RN/etc region codes for states
_bogus = df_tx['state'].astype(str).str.match(r'^R\d|^DI$|^RN$', na=False)
if _bogus.any():
    _bogus_whs = sorted(df_tx.loc[_bogus,'tech_warehouse'].dropna().unique())
    print(f'\n*** WARNING: {_bogus.sum():,} rows still have region-code states. Warehouses: {_bogus_whs}')

_tot=df_tx['order_num'].nunique(); _n_field=(~df_tx['_unattributed']).sum()
_n_blank=df_tx['_is_blank_tech'].sum(); _n_unmatch=df_tx['_is_unmatched'].sum()
_n_ambig=df_tx['_is_ambiguous'].sum(); _n_intops=df_tx['_is_internal_ops'].sum(); _n_unattr=df_tx['_unattributed'].sum()
print(f'Matching complete. {_tot:,} tickets — field: {_n_field:,} ({_n_field/_tot*100:.1f}%) | dark: {_n_unattr:,} ({_n_unattr/_tot*100:.1f}%)')
print('\nState distribution (top 10):')
print(df_tx['state'].value_counts(dropna=False).head(10).to_string())


[df_tx] State resolution audit:
  Sibling-city resolved (8798):
    R05 Birmingham (Hoover)                  (city='Birmingham (Hoover)') -> AL
    RNW San Antonio WH 2                     (city='San Antonio WH 2') -> TX
    RNW San Antonio                          (city='San Antonio') -> TX
    R08 Gadsden (Rainbow City)               (city='Gadsden (Rainbow City)') -> AL
    R10 Winchester                           (city='Winchester') -> VA
    R10 Lorton                               (city='Lorton') -> VA
    RNW McAllen                              (city='McAllen') -> TX
    RNW Corpus Christi                       (city='Corpus Christi') -> TX
    RNW Laredo                               (city='Laredo') -> TX
    RNW Irving                               (city='Irving') -> TX
    R08 Enterprise (Dothan)                  (city='Enterprise (Dothan)') -> AL
    R08 Birmingham (Hoover)                  (city='Birmingham (Hoover)') -> AL
    RNW Del Rio                              (ci

## Cell 10 — VISIT DEDUPLICATION (EXCHANGE CONSOLIDATION)

In [26]:
_KEY=['record_id','techfirstname','techlastname','completed_date']
# 'exchange pair' = same patient + tech + date with BOTH a Pickup and a Delivery row. Only the Pickup is the redundant half; 
# the Delivery represents the visit. Service tickets (and anything else) in the same group are independent work and must be kept.
_tt_set=df_tx.groupby(_KEY)['ticket_type'].transform(lambda s:('Pickup' in s.values)and('Delivery' in s.values))
df_tx['_is_exchange_pair']=_tt_set.fillna(False).astype(bool)
# Drop ONLY the Pickup row(s) inside an exchange pair; keep every other row.
df_tx['_keep'] = ~(df_tx['_is_exchange_pair'] & (df_tx['ticket_type']=='Pickup'))
df_visits=df_tx[df_tx['_keep']].copy()
# only the Delivery half of an exchange pair becomes 'Exchange'.
# Service (or other) tickets sharing the (patient, tech, date) group are independent work and keep their own type.
df_visits['visit_type']=np.where(df_visits['_is_exchange_pair']&(df_visits['ticket_type']=='Delivery'),'Exchange',df_visits['ticket_type'])
n_raw,n_visits=len(df_tx),len(df_visits); n_ex=int(df_visits['_is_exchange_pair'].sum())
print(f'Raw: {n_raw:,}  -> Visits: {n_visits:,}  ({(1-n_visits/n_raw)*100:.1f}% reduction)  Exchange pairs: {n_ex:,}')
print(df_visits['visit_type'].value_counts().to_string())
_ex_kept=df_visits[df_visits['_is_exchange_pair']].groupby(_KEY)['order_num'].count()
print('PASS: Exchange dedup validated.' if (_ex_kept>1).sum()==0 else f'WARNING: {(_ex_kept>1).sum():,} groups >1 kept row.')

Raw: 467,718  -> Visits: 437,180  (6.5% reduction)  Exchange pairs: 32,853
visit_type
Delivery    233037
Pickup      118986
Service      43621
Exchange     30839
Split         5807
Other         4890


## Cell 11.1 — REDELIVERY DATE RESCUE & WINDOW FILTER

In [27]:
# Re-Delivery Report dates are VARCHAR (locked lesson: filter in Python, not SQL).
# Rescue rows TRY_CONVERT missed via a second pandas parse of the raw string,
# count what stays unparseable (dropped LOUDLY), refilter to the window, and
# build event_key = (orig_order, redelivery date) — the dedup unit everywhere.
print(f'df_redel raw rows (post-SQL filter): {len(df_redel):,}')
_df_redel_orig=df_redel.copy()
_rd_from_convert=pd.to_datetime(_df_redel_orig['rd_date'],errors='coerce')
_rd_from_raw    =pd.to_datetime(_df_redel_orig['rd_datetime_raw'],errors='coerce')
_best_date=_rd_from_convert.where(_rd_from_convert.notna(),_rd_from_raw)
_df_redel_orig['_best_rd_date']=_best_date
_n_rescued=(_rd_from_convert.isna()&_rd_from_raw.notna()).sum()
if _n_rescued>0: print(f'  Rescued via raw parse: {_n_rescued:,}')
_n_unparseable = (_rd_from_convert.isna() & _rd_from_raw.isna()).sum()
if _n_unparseable>0:
    print(f'  *** WARNING: {_n_unparseable:,} redelivery rows have UNPARSEABLE Completion_DateTime')
    print(f'      These rows are DROPPED from all redelivery analytics.')
    print(f'      Sample unparseable raw values: {_df_redel_orig.loc[(_rd_from_convert.isna() & _rd_from_raw.isna()),"rd_datetime_raw"].dropna().astype(str).head(5).tolist()}')
_mask=(_df_redel_orig['_best_rd_date']>=pd.Timestamp(FILTER_START))&(_df_redel_orig['_best_rd_date']<=pd.Timestamp(FILTER_END))
df_redel=_df_redel_orig[_mask].copy()
print(f'  Window filter: {len(df_redel):,} kept / {(~_mask).sum():,} dropped (incl. {_n_unparseable:,} unparseable)')
df_redel['rd_date']=df_redel['_best_rd_date'].dt.date; df_redel.drop(columns=['_best_rd_date'],inplace=True)
df_redel['event_key'] = (df_redel['orig_order_num'].astype(str).str.strip()
                          + '|' + df_redel['rd_date'].astype(str))
print(f'After filter: {len(df_redel):,} rows  |  {df_redel["event_key"].nunique():,} unique (orig_order, rd_date) events')

def _parse_products(raw):
    if not raw or not isinstance(raw,str) or raw.strip()=='': return ['Unknown / Blank']
    items=[p.strip() for p in raw.replace('\r\n','\n').replace('\r','\n').split('\n') if p.strip()]
    return items or ['Unknown / Blank']

df_redel_exploded=(df_redel.copy().assign(product_list=lambda d:d['rd_products'].apply(_parse_products)).explode('product_list').rename(columns={'product_list':'product'}).reset_index(drop=True))
df_redel_exploded['product']=df_redel_exploded['product'].str.strip()
df_redel_exploded.loc[df_redel_exploded['product'].isna()|(df_redel_exploded['product']==''),'product']='Unknown / Blank'


df_redel raw rows (post-SQL filter): 188,437
  Window filter: 188,437 kept / 0 dropped (incl. 0 unparseable)
After filter: 188,437 rows  |  118,464 unique (orig_order, rd_date) events


## Cell 11.2 — LINK REDELIVERIES TO ORIGINATING TICKETS

In [28]:
df_tx['order_num']=df_tx['order_num'].astype(str).str.strip()
df_redel_exploded['orig_order_num']=df_redel_exploded['orig_order_num'].astype(str).str.strip()
_order_wh_counts=df_tx.groupby(['order_num','tech_warehouse'],dropna=False).size().reset_index(name='_n')
_modal_wh=(_order_wh_counts.sort_values(['order_num','_n','tech_warehouse'],ascending=[True,False,True]).drop_duplicates(subset=['order_num'],keep='first')[['order_num','tech_warehouse']])
# v1.5.0: completed_date is carried through so redeliveries can be bucketed into the
# originating WEEK. v1.4.0 carried only delivery_year/delivery_month, and a month cannot
# be resolved back into a week.
_tx_keys=(df_tx[['order_num','tech_warehouse','region','vp','state','metro','techfirstname','techlastname','delivery_year','delivery_month','schedule_period','completed_date']].merge(_modal_wh,on=['order_num','tech_warehouse'],how='inner').drop_duplicates(subset=['order_num']).rename(columns={'tech_warehouse':'tx_warehouse','region':'tx_region','vp':'tx_vp','state':'tx_state','metro':'tx_metro','techfirstname':'tx_techfirstname','techlastname':'tx_techlastname','delivery_year':'tx_delivery_year','delivery_month':'tx_delivery_month','schedule_period':'tx_schedule_period','completed_date':'tx_completed_date'}))
redel_linked=df_redel_exploded.merge(_tx_keys,left_on='orig_order_num',right_on='order_num',how='inner')
print(f'Unlinked: {len(df_redel_exploded)-len(redel_linked):,}  Linked: {len(redel_linked):,}')
# reconcile UNLINKED redeliveries by month (of the redelivery date). These are events whose originating order is outside the 
# ticket window (e.g. a Jan-2025 redelivery of a Dec-2024 original) or under a non-included Reason.
# They are EXCLUDED from every linked redelivery metric — the early months of the window undercount by design.
# This table makes the exclusion auditable in Excel (sheet 'Redel_Unlinked_Monthly'); footnote any board chart that spans the edge.
_unlinked = df_redel_exploded[~df_redel_exploded['orig_order_num'].isin(set(_tx_keys['order_num']))].copy()
_unlinked['_rd_dt'] = pd.to_datetime(_unlinked['rd_date'], errors='coerce')
tbl_redel_unlinked_monthly = (_unlinked
    .assign(delivery_year=_unlinked['_rd_dt'].dt.year, delivery_month=_unlinked['_rd_dt'].dt.month)
    .groupby(['delivery_year','delivery_month'], dropna=False, as_index=False)
    .agg(unlinked_events=('event_key','nunique'), unlinked_items=('product','count')))
tbl_redel_unlinked_monthly['period'] = (tbl_redel_unlinked_monthly['delivery_year'].astype('Int64').astype(str)
    + '-' + tbl_redel_unlinked_monthly['delivery_month'].astype('Int64').astype(str).str.zfill(2))
print(f'  Unlinked reconciliation: {tbl_redel_unlinked_monthly["unlinked_events"].sum():,} events across '
      f'{len(tbl_redel_unlinked_monthly)} month(s) — exported to the Redel_Unlinked_Monthly '
      f'sheet (v1.3.0: that sheet is now actually written; the claim predated it).')
for _src,_dst in [('tx_completed_date','orig_completed_date'),('tx_warehouse','tech_warehouse'),('tx_region','region'),('tx_vp','vp'),('tx_state','state'),('tx_metro','metro'),('tx_techfirstname','techfirstname'),('tx_techlastname','techlastname'),('tx_delivery_year','delivery_year'),('tx_delivery_month','delivery_month'),('tx_schedule_period','schedule_period')]:
    redel_linked[_dst]=redel_linked[_src]
redel_linked.drop(columns=[c for c in redel_linked.columns if c.startswith('tx_')],inplace=True,errors='ignore')

redel_linked['period']=(redel_linked['delivery_year'].astype(int).astype(str)+'-'+redel_linked['delivery_month'].astype(int).astype(str).str.zfill(2))
tbl_redel_product=(redel_linked.groupby('product').agg(
    redelivery_count=('event_key','nunique'),
    warehouses=('tech_warehouse',lambda x:x.nunique())
).reset_index().sort_values('redelivery_count',ascending=False))
tbl_redel_monthly=(redel_linked.groupby(['delivery_year','delivery_month']).agg(
    redelivery_count=('event_key','nunique'),
    redelivery_items=('product','count')   # count of product rows (items returned)
).reset_index().sort_values(['delivery_year','delivery_month']))
tbl_redel_monthly['period']=tbl_redel_monthly['delivery_year'].astype(str)+'-'+tbl_redel_monthly['delivery_month'].astype(str).str.zfill(2)
# v1.3.0 (audit B2): this used ALL tickets from df_tx while Cell 15's version of the
# same rate used ATTRIBUTED tickets only, so the two never reconciled. Both are now on
# attributed tickets, and the column name says which basis it is.
_tx_monthly_total = (df_tx[~df_tx['_unattributed']]
    .groupby(['delivery_year','delivery_month'],dropna=False,as_index=False)
    .agg(total_attributed_tickets=('order_num','nunique')))
tbl_redel_monthly = tbl_redel_monthly.merge(_tx_monthly_total,on=['delivery_year','delivery_month'],how='left')
tbl_redel_monthly['redel_pct_of_attributed_tickets'] = (tbl_redel_monthly['redelivery_count']/tbl_redel_monthly['total_attributed_tickets'].replace(0,np.nan)*100).round(2)
display(tbl_redel_product.head(10))
print(f'Redelivery exploded rows: {len(redel_linked):,}')

Unlinked: 51,456  Linked: 456,334
  Unlinked reconciliation: 9,875 events across 19 month(s) — exported to the Redel_Unlinked_Monthly sheet (v1.3.0: that sheet is now actually written; the claim predated it).


,product,redelivery_count,warehouses
242,"Oxygen Cylinder, E Tank(100)",17881,91
167,Full Electric Hospital Bed(294),10753,90
228,Over Bed Table(303),10157,90
239,"Oxygen Concentrator, 5 Liter(305)",9932,88
322,Pressure Prevention Foam Mattress(314),9241,89
37,7' Oxygen Tubing(3),8176,88
233,"Oxygen Cannula, Adult(16)",7882,88
475,Water Trap(40),7863,88
187,Humidifier Bottle Adapter(11),7501,87
248,"Oxygen Gauge, Portable(309)",7253,86


Redelivery exploded rows: 456,334


## Cell 11.3 — ORDER → PHYSICAL WAREHOUSE MAP *(new in v1.3.0, shared by stock outs + lost equipment)*

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# ORDER -> PHYSICAL WAREHOUSE MAP (shared attribution key)            new v1.3.0
#
# Two v1.3.0 sections need to know where a record with no usable warehouse of its own
# physically belongs:
#   * stock outs — 'Z Equipment Needed' on almost every row, but the [Order] number
#     matches the ticket feed for 20,230 of 21,438 EnRoute orders;
#   * lost equipment — some rows sit in 'Z Equipment Collections' / 'Z CS', but they
#     carry a Delivery Order ID and a Pickup Order ID.
# The ticket feed already knows the physical warehouse for an order, and after Cell
# 8.1b that warehouse is resolved rather than virtual. So build the map once here and
# let both sections use it, instead of each inventing its own attribution.
#
# TEACHING NOTE — modal, not first or last: an order can appear on several tickets
# (delivery, then a service call) and occasionally at different sites. The modal
# warehouse is the site that handled most of the order's tickets, with an alphabetical
# tiebreak so the map is stable across runs. Same rule as Cell 8.1b, deliberately —
# one attribution model for the whole notebook is worth more than a locally optimal one.
# ─────────────────────────────────────────────────────────────────────────────
_omap_src = df_tx[df_tx['tech_warehouse'].notna()
                  & (df_tx['tech_warehouse'] != VIRTUAL_UNRESOLVED_LABEL)].copy()
_omap_counts = (_omap_src.groupby(['order_num', 'tech_warehouse'], dropna=False)
                .size().reset_index(name='_n'))
_omap_counts = _omap_counts.sort_values(['order_num', '_n', 'tech_warehouse'],
                                        ascending=[True, False, True])
order_wh_map = (_omap_counts.drop_duplicates(subset=['order_num'], keep='first')
                [['order_num', 'tech_warehouse']])
# carry the hierarchy so a consumer gets warehouse + vp + state + metro in one join
_wh_attrs = (df_tx[['tech_warehouse', 'vp', 'state', 'metro']].drop_duplicates('tech_warehouse'))
order_wh_map = order_wh_map.merge(_wh_attrs, on='tech_warehouse', how='left')
order_wh_map = order_wh_map.rename(columns={'tech_warehouse': 'ord_warehouse', 'vp': 'ord_vp',
                                            'state': 'ord_state', 'metro': 'ord_metro'})

# Completion date per order — lets the stock-out section measure time-to-fulfil and
# tell a genuinely unfulfilled order from one whose Status was simply never updated.
order_completion = (df_tx.groupby('order_num', as_index=False)
                    .agg(ord_completed_date=('completed_date', 'max'),
                         ord_ticket_count=('record_id', 'nunique')))
order_wh_map = order_wh_map.merge(order_completion, on='order_num', how='left')
print(f'order_wh_map: {len(order_wh_map):,} orders -> physical warehouse '
      f'({order_wh_map["ord_warehouse"].nunique():,} distinct warehouses)')


order_wh_map: 463,174 orders -> physical warehouse (101 distinct warehouses)


## Cell 12 — LOST EQUIPMENT PREP: PARSE, BULK-EVENT EXCLUSION, SITE RESOLUTION *(rewritten v1.3.0)*

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# LOST EQUIPMENT PREP — PARSE, EXCLUDE BULK EVENTS, RESOLVE SITE   rewritten v1.3.0
#
# Four things have to happen before a single number is trustworthy:
#   1. PARSE the date. [Lost Date] is '01/02/2025<br/>(147)' — an HTML fragment. A
#      naive port fails on 100% of 91,023 rows and yields an empty panel exactly like
#      the one this rewrite replaces, so the parse failure count is asserted, not printed.
#   2. EXCLUDE known bulk events (Cell 3.5), reported separately for completeness. Any
#      NEW spike is warned about loudly but never auto-excluded — surfacing beats
#      silent suppression, and an analyst has to add the date deliberately.
#   3. RESOLVE the site. Some rows sit in virtual buckets ('Z Equipment Collections',
#      'Z CS'); their Delivery/Pickup Order IDs go through order_wh_map.
#   4. DECLARE the staleness. The feed ends 2026-05 against an AS_OF of 2026-08-13, so
#      every page that shows lost equipment must say so.
#
# TEACHING NOTE — why the bulk event is excluded rather than winsorised or smoothed:
# 2025-08-01 is not a large observation of the same process, it is a different process
# (a reconciliation write-off, 99.9% of it into virtual buckets). Smoothing would
# average two unlike things; excluding it and reporting it separately keeps both the
# trend readable and the total reconcilable.
# ─────────────────────────────────────────────────────────────────────────────
_n_lost_pulled = len(df_lost_raw)

# 1 — parse: strip the '<br/>(147)' tail, then a strict MM/DD/YYYY parse.
def _parse_lost_date(s):
    """SERP_LOST_EQUIPMENT dates carry an HTML tail. Strip it, then parse strictly."""
    if not isinstance(s, str):
        return pd.NaT
    return pd.to_datetime(re.sub(LOST_DATE_TAIL_RE, '', s).strip(),
                          format=LOST_DATE_FMT, errors='coerce')

df_lost_raw['lost_date'] = df_lost_raw['lost_date_raw'].apply(_parse_lost_date)
df_lost_raw['resolved_date'] = df_lost_raw['resolved_date_raw'].apply(_parse_lost_date)
_n_unparsed = int(df_lost_raw['lost_date'].isna().sum())
print(f'Lost equipment: {_n_lost_pulled:,} rows pulled  |  unparseable Lost Date: {_n_unparsed:,}')
if _n_unparsed:
    print(f'  Sample unparseable raw values: '
          f'{df_lost_raw.loc[df_lost_raw["lost_date"].isna(), "lost_date_raw"].dropna().astype(str).head(5).tolist()}')
if _n_unparsed > 0.02 * max(_n_lost_pulled, 1):
    raise AssertionError(
        f'{_n_unparsed:,} of {_n_lost_pulled:,} Lost Date values failed to parse (>2%). '
        f'The upstream format has changed — fix LOST_DATE_TAIL_RE / LOST_DATE_FMT in Cell 3.5 '
        f'before trusting any lost-equipment number. This assert exists because a silent '
        f'parse failure is what kept this panel empty for over a year.')

df_lost_raw['lost_cost'] = df_lost_raw['unit_cost_raw'].apply(clean_numbers).fillna(0.0)
df_lost_raw['discharged_days'] = df_lost_raw['discharged_days_raw'].apply(clean_numbers)
df_lost_raw['asset_tag'] = df_lost_raw['asset_tag'].astype(str).str.strip().str.lstrip('\t')
df_lost_raw['lost_warehouse'] = df_lost_raw['lost_warehouse'].fillna('').astype(str).str.strip()

# ── GRAIN: one row per (asset, lost date) ────────────────────────────────────
# The source repeats an asset across rows (2,783 duplicate asset-days in the window,
# concentrated on batch dates such as 2026-05-15). Left alone, three things break:
# lost_asset_count (a distinct count) disagrees with the recovered/unresolved counts
# (row sums), and lost_asset_cost double-counts the unit cost of every repeated asset.
# An asset recorded lost on a date is ONE event, so that is the grain — after this,
# distinct counts and row sums are the same number by construction.
_pre_dedup = len(df_lost_raw)
df_lost_raw = df_lost_raw.sort_values(
    ['asset_tag', 'lost_date', 'resolved_date'], na_position='last'
).drop_duplicates(subset=['asset_tag', 'lost_date'], keep='first').copy()
_n_asset_dupes = _pre_dedup - len(df_lost_raw)
if _n_asset_dupes:
    print(f'  Grain: {_n_asset_dupes:,} duplicate (asset, lost date) rows dropped '
          f'({_n_asset_dupes / max(_pre_dedup, 1) * 100:.1f}%) -> {len(df_lost_raw):,} events. '
          f'Kept the row with the earliest resolution, so a recovery is never lost to dedup.')

# 2 — bulk events + automatic spike surveillance
_dated = df_lost_raw[df_lost_raw['lost_date'].notna()]
_by_day = _dated.groupby(_dated['lost_date'].dt.date).size()
_median_day = float(_by_day.median()) if len(_by_day) else 0.0
_bulk_dates = {pd.Timestamp(d).date() for d in LOST_BULK_EVENT_DATES}
_spikes = _by_day[_by_day > LOST_SPIKE_FACTOR * max(_median_day, 1)]
_unlisted = [d for d in _spikes.index if d not in _bulk_dates]
if len(_spikes):
    print(f'\nSpike surveillance (>{LOST_SPIKE_FACTOR:g}x the {_median_day:,.0f}/day median):')
    for _d in sorted(_spikes.index):
        _tag = 'EXCLUDED (listed in LOST_BULK_EVENT_DATES)' if _d in _bulk_dates else '*** NOT EXCLUDED ***'
        print(f'  {_d}: {_by_day[_d]:,} rows  {_tag}')
if _unlisted:
    print(f'  *** ACTION: {len(_unlisted)} spike date(s) are NOT in LOST_BULK_EVENT_DATES and '
          f'remain in every metric and trend: {_unlisted}. Confirm whether each is a bulk '
          f'administrative event; if so add it to Cell 3.5. Nothing is excluded automatically.')

_bulk_mask = df_lost_raw['lost_date'].dt.date.isin(_bulk_dates)
tbl_lost_bulk_events = (df_lost_raw[_bulk_mask]
    .assign(lost_date=lambda d: d['lost_date'].dt.strftime('%Y-%m-%d'))
    .groupby(['lost_date', 'lost_warehouse'], dropna=False, as_index=False)
    .agg(excluded_assets=('asset_tag', 'nunique'), excluded_rows=('asset_tag', 'size'),
         excluded_cost=('lost_cost', 'sum'), products=('product_name', 'nunique'))
    .sort_values('excluded_cost', ascending=False))
if _bulk_mask.any():
    print(f'\nBULK EVENTS EXCLUDED from all metrics and trends (reported for completeness, '
          f'and exported to the Lost_Bulk_Events sheet):')
    print(f'  {int(_bulk_mask.sum()):,} rows / '
          f'{df_lost_raw.loc[_bulk_mask, "asset_tag"].nunique():,} assets / '
          f'${df_lost_raw.loc[_bulk_mask, "lost_cost"].sum():,.0f} unit cost on '
          f'{sorted({str(d) for d in df_lost_raw.loc[_bulk_mask, "lost_date"].dt.date})}')
    print(f'  Top holding buckets: '
          f'{df_lost_raw.loc[_bulk_mask, "lost_warehouse"].value_counts().head(3).to_dict()}')

# 3 — site resolution: stamped warehouse -> order join -> virtual label
_lost = df_lost_raw[df_lost_raw['lost_date'].notna() & ~_bulk_mask].copy()
_lost['_is_virtual_wh'] = (_lost['lost_warehouse'].str.upper().str.startswith(VIRTUAL_WH_PREFIX.upper())
                           | (_lost['lost_warehouse'] == ''))
_lost['tech_warehouse'] = _lost['lost_warehouse']
_lost['_wh_resolution'] = np.where(_lost['_is_virtual_wh'], 'unresolved', 'stamped')
for _col in ('delivery_order_id', 'pickup_order_id'):
    _need = _lost['_wh_resolution'].eq('unresolved')
    if not _need.any():
        break
    _key = _lost.loc[_need, _col].astype(str).str.strip()
    _hit = _key.map(order_wh_map.set_index('order_num')['ord_warehouse'])
    _fill = _hit.notna()
    _idx = _hit[_fill].index
    _lost.loc[_idx, 'tech_warehouse'] = _hit[_fill].values
    _lost.loc[_idx, '_wh_resolution'] = f'order_join:{_col}'
_lost.loc[_lost['_wh_resolution'].eq('unresolved'), 'tech_warehouse'] = VIRTUAL_UNRESOLVED_LABEL

# hierarchy + metro off the resolved warehouse, then the shared 4-layer state resolution
_lost = _lost.merge(df_hier.rename(columns={'warehouse': 'tech_warehouse'})[
    ['tech_warehouse', 'state', 'region', 'vp']].drop_duplicates('tech_warehouse'),
    on='tech_warehouse', how='left')  # dedup guards against a fan-out on the merge
_lost['state'] = _lost['state'].where(_lost['state'].notna(), _lost['ship_to_state'])
_lost['state'], _lost_state_audit = resolve_state_series(_lost['tech_warehouse'], _lost['state'], df_hier)
_print_state_audit(_lost_state_audit, 'lost equipment')
_lost['metro'] = _lost['tech_warehouse'].apply(assign_metro)
print(f'\nSite resolution: {_lost["_wh_resolution"].value_counts().to_dict()}')

# window filter (the source may pre-date FILTER_START)
_lost_filt = _lost[(_lost['lost_date'] >= pd.Timestamp(FILTER_START))
                   & (_lost['lost_date'] <= pd.Timestamp(FILTER_END))].copy()
_lost_filt['lost_year'] = _lost_filt['lost_date'].dt.year.astype(int)
_lost_filt['lost_month'] = _lost_filt['lost_date'].dt.month.astype(int)
_lost_filt['period'] = (_lost_filt['lost_year'].astype(str) + '-'
                        + _lost_filt['lost_month'].astype(str).str.zfill(2))

# 4 — staleness: declare it once here, then banner it on every page that uses it
LOST_MAX_DATE = _lost_filt['lost_date'].max() if len(_lost_filt) else pd.NaT
LOST_LAG_DAYS = int((pd.Timestamp(AS_OF_DATE) - LOST_MAX_DATE).days) if pd.notna(LOST_MAX_DATE) else None
LOST_IS_STALE = LOST_LAG_DAYS is not None and LOST_LAG_DAYS > LOST_STALENESS_WARN_DAYS
LOST_STALE_NOTE = ''
if pd.notna(LOST_MAX_DATE):
    LOST_STALE_NOTE = (f'Lost equipment: source current only through '
                       f'{LOST_MAX_DATE.strftime("%Y-%m-%d")} ({LOST_LAG_DAYS}d behind as-of)')
print(f'\nLost equipment in window: {len(_lost_filt):,} rows / '
      f'{_lost_filt["asset_tag"].nunique():,} assets / ${_lost_filt["lost_cost"].sum():,.0f}')
print(f'  Latest lost date in feed: {LOST_MAX_DATE.date() if pd.notna(LOST_MAX_DATE) else "n/a"} '
      f'(AS_OF_DATE {AS_OF_DATE})')
if LOST_IS_STALE:
    print(f'  *** STALE FEED: lost equipment lags AS_OF_DATE by {LOST_LAG_DAYS} days '
          f'(threshold {LOST_STALENESS_WARN_DAYS}). Every panel and sheet carries a banner; '
          f'do NOT read the final month as a decline — it is missing data, not recovery.')

# resolution classes + censoring-aware recovery. An asset lost last week has had no
# chance to be found, so a raw recovery rate would fall every month by construction.
_lost_filt['is_recovered'] = _lost_filt['resolution'].isin(LOST_RESOLUTION_RECOVERED)
_lost_filt['is_discarded'] = _lost_filt['resolution'].isin(LOST_RESOLUTION_DISCARDED)
_lost_filt['is_unresolved'] = ~(_lost_filt['is_recovered'] | _lost_filt['is_discarded'])
_lost_filt['days_to_resolve'] = (_lost_filt['resolved_date'] - _lost_filt['lost_date']).dt.days
_lost_filt['cohort_mature'] = ((pd.Timestamp(AS_OF_DATE) - _lost_filt['lost_date']).dt.days
                               >= LOST_RECOVERY_MATURITY_DAYS)
print(f"  Resolution mix: recovered {_lost_filt['is_recovered'].sum():,} | "
      f"discarded {_lost_filt['is_discarded'].sum():,} | unresolved {_lost_filt['is_unresolved'].sum():,}")
_dtr = _lost_filt.loc[_lost_filt['days_to_resolve'].notna(), 'days_to_resolve']
if len(_dtr):
    print(f'  Days to resolve (resolved only) — P25 {_dtr.quantile(.25):.0f} | '
          f'median {_dtr.median():.0f} | P75 {_dtr.quantile(.75):.0f}')
print(f'  Recovery rate is reported only for cohorts >= {LOST_RECOVERY_MATURITY_DAYS} days old '
      f'({_lost_filt["cohort_mature"].sum():,} of {len(_lost_filt):,} rows); newer cohorts are '
      f'censored and would understate recovery by construction.')


Lost equipment: 91,023 rows pulled  |  unparseable Lost Date: 0
  Grain: 2,114 duplicate (asset, lost date) rows dropped (2.3%) -> 88,909 events. Kept the row with the earliest resolution, so a recovery is never lost to dedup.

Spike surveillance (>8x the 99/day median):
  2025-01-31: 1,588 rows  *** NOT EXCLUDED ***
  2025-04-30: 1,504 rows  *** NOT EXCLUDED ***
  2025-05-31: 920 rows  *** NOT EXCLUDED ***
  2025-08-01: 30,899 rows  EXCLUDED (listed in LOST_BULK_EVENT_DATES)
  *** ACTION: 3 spike date(s) are NOT in LOST_BULK_EVENT_DATES and remain in every metric and trend: [datetime.date(2025, 1, 31), datetime.date(2025, 4, 30), datetime.date(2025, 5, 31)]. Confirm whether each is a bulk administrative event; if so add it to Cell 3.5. Nothing is excluded automatically.

BULK EVENTS EXCLUDED from all metrics and trends (reported for completeness, and exported to the Lost_Bulk_Events sheet):
  30,899 rows / 30,899 assets / $3,054,981 unit cost on ['2025-08-01']
  Top holding buckets: {

## Cell 13.0 — ROUTING-BREAK DIAGNOSTIC (D1–D5) *(new in v1.2.0)*
Run this first whenever a series steps up or down. It separates "the feed moved"
from "operations moved" — the July-2026 drop was entirely the former.

In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# ROUTING-BREAK DIAGNOSTIC (D1–D5)                                      v1.2.0
#
# Built to answer "why did tickets per technician fall in July across nearly every
# location?" — and kept because the same five checks will diagnose the next such
# step. Every panel is a MONTHLY series so a break shows as a level shift instead
# of being smoothed away by the 4-week rolling average.
#
#   D1  filter funnel      — how many raw rows each WHERE clause removes, by month.
#                            A filter whose retention rate MOVES is a feed change,
#                            not an operations change.
#   D2  virtual warehouses — eligible tickets by 'Z%' code and month, with reasons.
#   D3  the mechanism      — tech-days mixing real and virtual warehouses, plus the
#                            as-reported and corrected company ratios side by side.
#   D4  denominator health — per-warehouse active-day sum vs distinct (tech, date).
#   D5  entity churn       — warehouses that appear or go silent.
#
# TEACHING NOTE — the general lesson. A ratio breaks in exactly three ways:
# numerator loss, denominator inflation, or a real change. July 2026 was the first
# two at once, and both were invisible because the page plotted only the ratio.
# Instrument the funnel, not just the result.
# ─────────────────────────────────────────────────────────────────────────────
diag_funnel = pd.DataFrame(); diag_virtual_wh = pd.DataFrame()
diag_resolution = pd.DataFrame(); diag_wh_denominator = pd.DataFrame()

if not RUN_ROUTING_DIAGNOSTIC:
    print('RUN_ROUTING_DIAGNOSTIC=False — skipped.')
else:
    # ── D1: what each extraction filter removes, by month ────────────────────
    # Re-queried unfiltered: df_tx is already past every WHERE clause, so the rows
    # we most need to count are exactly the ones it cannot see.
    print('=' * 96); print('D1) FILTER FUNNEL BY MONTH'); print('=' * 96)
    diag_funnel = run_query(f"""
WITH base AS (
  SELECT TRY_CONVERT(DATE, Completed_Date) AS cd,
         TRIM(ISNULL(Tech_Warehouse,'')) AS wh, Reason AS reason,
         [Status] AS status, TRIM(ISNULL(Order_Num,'')) AS ord
  FROM dbo.[SERP TRANSACTIONS] WITH (NOLOCK)
  WHERE TRY_CONVERT(DATE, Completed_Date) >= '{FILTER_START}'
    AND TRY_CONVERT(DATE, Completed_Date) <= '{FILTER_END}')
SELECT YEAR(cd) AS yr, MONTH(cd) AS mo,
  COUNT(*) AS s0_all_rows,
  SUM(CASE WHEN status <> 'Canceled' THEN 1 ELSE 0 END) AS s1_not_canceled,
  SUM(CASE WHEN status <> 'Canceled' AND ord <> '' THEN 1 ELSE 0 END) AS s2_has_order,
  SUM(CASE WHEN status <> 'Canceled' AND ord <> ''
            AND reason IN ({_reasons_sql}) THEN 1 ELSE 0 END) AS s3_reason_ok,
  SUM(CASE WHEN status <> 'Canceled' AND ord <> '' AND reason IN ({_reasons_sql})
            AND wh LIKE '{VIRTUAL_WH_PREFIX}%' THEN 1 ELSE 0 END) AS virtual_wh_rows
FROM base GROUP BY YEAR(cd), MONTH(cd) ORDER BY yr, mo
""", 'D1 filter funnel')
    for _c in diag_funnel.columns:
        diag_funnel[_c] = pd.to_numeric(diag_funnel[_c], errors='coerce')
    diag_funnel['period'] = (diag_funnel['yr'].astype(int).astype(str) + '-'
                             + diag_funnel['mo'].astype(int).astype(str).str.zfill(2))
    diag_funnel['reason_excluded'] = diag_funnel['s2_has_order'] - diag_funnel['s3_reason_ok']
    diag_funnel['pct_reason_excluded'] = (diag_funnel['reason_excluded']
                                          / diag_funnel['s2_has_order'] * 100).round(1)
    diag_funnel['pct_virtual_wh'] = (diag_funnel['virtual_wh_rows']
                                     / diag_funnel['s3_reason_ok'] * 100).round(1)
    print(diag_funnel[['period', 's0_all_rows', 's1_not_canceled', 's2_has_order',
                       's3_reason_ok', 'reason_excluded', 'pct_reason_excluded',
                       'virtual_wh_rows', 'pct_virtual_wh']].to_string(index=False))
    print('\nREAD: a jump in pct_virtual_wh or pct_reason_excluded is a FEED change. Rows '
          'dropped by the reason whitelist never reach df_tx at all, so if a reason code '
          'is renamed upstream its tickets vanish in silence — watch pct_reason_excluded '
          'as closely as the headline metric.')

    # ── D2: virtual-warehouse volume by code ─────────────────────────────────
    print('\n' + '=' * 96); print('D2) VIRTUAL-WAREHOUSE TICKETS BY CODE AND MONTH'); print('=' * 96)
    if df_tx['_is_virtual_wh'].any():
        _v2 = df_tx[df_tx['_is_virtual_wh']].copy()
        _v2['period'] = _v2['month_date'].dt.strftime('%Y-%m')
        _v2['code'] = _v2['_wh_ticket_stamped'].replace('', '(blank)')
        diag_virtual_wh = (_v2.groupby(['code', 'period'], as_index=False)
            .agg(tickets=('order_num', 'nunique'), techs=('_tech_key', 'nunique')))
        print(diag_virtual_wh.pivot_table(index='code', columns='period', values='tickets',
                                          aggfunc='sum', fill_value=0).to_string())
        _recent = sorted(_v2['period'].unique())[-3:]
        print(f'\nTop reasons on virtual-warehouse tickets ({", ".join(_recent)}):')
        print(_v2[_v2['period'].isin(_recent)]['reason'].value_counts().head(8).to_string())
        print('\nREAD: administrative codes carry administrative reasons. Delivery and '
              'discharge reasons on a virtual code mean real field work is parked there '
              'and must be re-attributed, not dropped.')
    else:
        print('No virtual-warehouse rows in window.')

    # ── D3: the mechanism, and the corrected ratio ────────────────────────────
    print('\n' + '=' * 96)
    print('D3) MECHANISM — numerator loss on days the denominator still counts in full')
    print('=' * 96)
    _d3 = df_tx[(df_tx['schedule_period'] == 'Weekday') & (df_tx['_tech_key'] != '|')].copy()
    _d3['period'] = _d3['month_date'].dt.strftime('%Y-%m')
    _d3['_real_ord'] = np.where(~_d3['_is_virtual_wh'], _d3['order_num'], np.nan)
    _d3['_virt_ord'] = np.where(_d3['_is_virtual_wh'], _d3['order_num'], np.nan)
    _td = (_d3.groupby(['_tech_key', 'completed_date', 'period'], as_index=False)
           .agg(real_tickets=('_real_ord', 'nunique'), virt_tickets=('_virt_ord', 'nunique')))
    _td['_mixed'] = (_td['real_tickets'] > 0) & (_td['virt_tickets'] > 0)
    _td['_vonly'] = (_td['real_tickets'] == 0) & (_td['virt_tickets'] > 0)
    diag_resolution = (_td.groupby('period', as_index=False)
        .agg(tech_days=('completed_date', 'count'),
             real_tickets=('real_tickets', 'sum'), virt_tickets=('virt_tickets', 'sum'),
             mixed_days=('_mixed', 'sum'), virt_only_days=('_vonly', 'sum')))
    diag_resolution['virt_share_of_tickets_pct'] = (diag_resolution['virt_tickets']
        / (diag_resolution['real_tickets'] + diag_resolution['virt_tickets']) * 100).round(1)
    diag_resolution['mixed_pct_of_days'] = (diag_resolution['mixed_days']
        / diag_resolution['tech_days'] * 100).round(1)
    # v1.1.0 as reported: virtual tickets dropped, but the day still counted unless
    # the technician did nothing else that day.
    diag_resolution['as_reported_v1_1_0'] = (diag_resolution['real_tickets']
        / (diag_resolution['tech_days'] - diag_resolution['virt_only_days'])).round(3)
    diag_resolution['corrected_v1_2_0'] = ((diag_resolution['real_tickets']
        + diag_resolution['virt_tickets']) / diag_resolution['tech_days']).round(3)
    diag_resolution['artifact_tickets_per_day'] = (diag_resolution['corrected_v1_2_0']
        - diag_resolution['as_reported_v1_1_0']).round(3)
    print(diag_resolution.to_string(index=False))
    print('\nREAD: where as_reported and corrected diverge, the "decline" is the filter, '
          'not the field. artifact_tickets_per_day is productivity handed back per '
          'technician-day. Company level only — see D4 for the site-level effect.')

    # ── D4: is the warehouse-grain denominator honest? ────────────────────────
    print('\n' + '=' * 96)
    print('D4) DENOMINATOR HEALTH — per-warehouse active-day sum vs distinct (tech, date)')
    print('=' * 96)
    _pairs = _d3[['_tech_key', 'completed_date', 'tech_warehouse', 'period']].drop_duplicates()
    diag_wh_denominator = (_pairs.groupby('period', as_index=False)
        .agg(wh_active_day_sum=('tech_warehouse', 'count')))
    _honest = (_d3[['_tech_key', 'completed_date', 'period']].drop_duplicates()
               .groupby('period', as_index=False).size()
               .rename(columns={'size': 'distinct_tech_days'}))
    _spread = (_pairs.groupby(['period', '_tech_key'])['tech_warehouse'].nunique()
               .groupby('period').mean().round(2).reset_index(name='avg_wh_per_tech_day'))
    _spm = (_d3[['period', '_tech_key', 'tech_warehouse']].drop_duplicates()
            .groupby(['period', '_tech_key']).size().groupby('period').mean()
            .round(2).reset_index(name='avg_wh_per_tech_month'))
    diag_wh_denominator = (diag_wh_denominator.merge(_honest, on='period')
                           .merge(_spread, on='period').merge(_spm, on='period'))
    diag_wh_denominator['inflation_pct'] = ((diag_wh_denominator['wh_active_day_sum']
        / diag_wh_denominator['distinct_tech_days'] - 1) * 100).round(1)
    print(diag_wh_denominator.to_string(index=False))
    print('\nREAD: inflation_pct is how much every warehouse/metro/VP page understates '
          'tickets-per-active-day when a whole day is credited to each warehouse a '
          'technician touched. APPORTION_TECH_DAYS=True removes it, and the apportioned '
          'days then sum exactly to distinct_tech_days (Cell 13 asserts this).')

    # ── D5: entity churn ─────────────────────────────────────────────────────
    print('\n' + '=' * 96); print('D5) WAREHOUSE CHURN — appearing / going silent'); print('=' * 96)
    _wm = (_d3[~_d3['_unattributed']]
           .pivot_table(index='tech_warehouse', columns='period', values='order_num',
                        aggfunc='nunique', fill_value=0))
    if _wm.shape[1] >= 2:
        _last, _prev = _wm.columns[-1], _wm.columns[-2]
        for _lbl, _mask in [('WENT SILENT (volume last month, none this month)',
                             (_wm[_prev] > 0) & (_wm[_last] == 0)),
                            ('NEW (none last month, volume this month)',
                             (_wm[_prev] == 0) & (_wm[_last] > 0))]:
            _hits = _wm[_mask]
            print(f'\n{_lbl} — {_prev} -> {_last}: {len(_hits)}')
            if len(_hits):
                print(_hits.iloc[:, -6:].to_string())
        print('\nREAD: a closed or renamed site both ends its own series and pushes work '
              'onto its neighbours, so check churn before reading any site-level move. '
              'AS_OF_DATE truncates the final month — expect it to look light.')


D1) FILTER FUNNEL BY MONTH


  D1 filter funnel: 20 rows  (9.0s)
 period  s0_all_rows  s1_not_canceled  s2_has_order  s3_reason_ok  reason_excluded  pct_reason_excluded  virtual_wh_rows  pct_virtual_wh
2025-01        39696            33048         33048         23000            10048                 30.4              342             1.5
2025-02        40764            33212         33212         21427            11785                 35.5              407             1.9
2025-03        37787            30184         30184         23617             6567                 21.8              388             1.6
2025-04        39161            32077         32077         23442             8635                 26.9              433             1.8
2025-05        35504            27845         27845         23499             4346                 15.6              379             1.6
2025-06        40631            31594         31594         23174             8420                 26.7              474             2.0
2025-

## Cell 13 — WEEKLY TECHNICIAN PRODUCTIVITY BY ENTITY

In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# WEEKLY TECHNICIAN PRODUCTIVITY BY ENTITY
# Sun–Sat weeks (FLSA-aligned). Weekday tickets only. Ratio-of-pooled-sums.
# Grains produced: warehouse, metro, VP, company
#
# v1.2.0 — THE DENOMINATOR IS NOW APPORTIONED. v1.1.0 credited a whole active day
# to every warehouse a technician touched that day, so the per-warehouse day sum
# ran 34.9% above the honest company total in Jul-2026 (~3% historically) and
# every site page read low. Each technician-day is now split across warehouses by
# that day's ticket share (`active_day_equiv`), which sums back exactly to
# distinct (tech, date) — asserted at the bottom of this cell. Company figures are
# unchanged by apportionment; warehouse, metro and VP figures rise.
# `active_weekdays` (whole days) is kept alongside for the ranking eligibility
# floor and for reconciliation against v1.1.0.
#
# TEACHING NOTE — why ticket share rather than 1/n_warehouses: a technician who
# runs nine stops out of one site and one out of another spent about 90% of the day
# at the first; ticket share tracks that, while 1/n would hand each site half a
# day. Neither is exact (a stop is not an hour), so read single-warehouse days as
# exact and split days as apportioned — `denominator_inflation_pct` reports how
# much of the entity's week rests on that assumption.
# ─────────────────────────────────────────────────────────────────────────────
_win_start = pd.Timestamp(FILTER_START)
_win_end   = pd.Timestamp(AS_OF_DATE)

_wk_src = df_tx[(~df_tx['_unattributed']) & (df_tx['schedule_period'] == 'Weekday')].copy()
# v1.5.0: one shared week definition (Cell 4.3). Arithmetically identical to the local
# expression it replaces; shared so census, lost equipment, redeliveries and stock outs
# cannot drift into a different bucketing.
_wk_src['week_start'] = week_start_of(_wk_src['completed_date'])
_wk_src['_tech_key'] = _wk_src['techfirstname'].str.strip() + '|' + _wk_src['techlastname'].str.strip()

# v1.5.0 — the week index IS the shared WEEK_SPINE (Cell 4.3), so every metric family is
# bucketed identically and a week with no rows still exists as a row. The local derivation
# it replaces flagged a week partial on its WEEKDAY count (< 5), which missed a week clipped
# only on its weekend; the spine flags on calendar days.
_week_index = WEEK_SPINE.copy()

_GRAIN = ['techfirstname', 'techlastname', '_tech_key', 'tech_warehouse', 'vp', 'state', 'metro']

# Day level first — apportionment is a property of a DAY, not of a week.
_techday = (_wk_src.groupby(_GRAIN + ['completed_date', 'week_start'], dropna=False, as_index=False)
    .agg(day_tickets=('order_num', 'nunique'),
         day_virtual_tickets=('_is_virtual_wh', 'sum')))
_techday['_tech_day_tickets'] = (_techday.groupby(['_tech_key', 'completed_date'])['day_tickets']
                                 .transform('sum'))
if APPORTION_TECH_DAYS:
    _techday['active_day_share'] = (_techday['day_tickets']
                                    / _techday['_tech_day_tickets'].replace(0, np.nan))
else:
    _techday['active_day_share'] = 1.0

# Tech-week base at (tech, warehouse, metro, vp, week) — one row per tech per warehouse.
_techday_wk = (_techday.groupby(_GRAIN + ['week_start'], dropna=False, as_index=False)
    .agg(total_tickets=('day_tickets', 'sum'),
         virtual_tickets=('day_virtual_tickets', 'sum'),
         active_weekdays=('completed_date', 'nunique'),
         active_day_equiv=('active_day_share', 'sum')))

def _dash_weekly(gcols):
    """Pooled weekly rollup for any entity grain ([]=company)."""
    keys = gcols + ['week_start']
    agg = (_techday_wk.groupby(keys, dropna=False, as_index=False)
        .agg(total_tickets=('total_tickets', 'sum'),
             virtual_tickets=('virtual_tickets', 'sum'),
             total_active_tech_days=('active_day_equiv', 'sum'),
             total_active_tech_days_whole=('active_weekdays', 'sum'),
             tech_count=('_tech_key', 'nunique'))
        .merge(_week_index, on='week_start', how='left'))
    agg['tickets_per_active_day_per_tech'] = (agg['total_tickets']
        / agg['total_active_tech_days'].replace(0, np.nan)).round(3)
    # v1.1.0's whole-day denominator, kept so any reader can reconcile the change
    agg['tickets_per_active_day_wholeday'] = (agg['total_tickets']
        / agg['total_active_tech_days_whole'].replace(0, np.nan)).round(3)
    agg['denominator_inflation_pct'] = ((agg['total_active_tech_days_whole']
        / agg['total_active_tech_days'].replace(0, np.nan) - 1) * 100).round(1)
    # share of the numerator that had to be re-attributed from a virtual warehouse
    agg['virtual_ticket_share_pct'] = (agg['virtual_tickets']
        / agg['total_tickets'].replace(0, np.nan) * 100).round(1)
    _cal_denom = (agg['tech_count'] * agg['weekday_days_in_week']).replace(0, np.nan)
    agg['weekday_daily_tickets_per_tech'] = (agg['total_tickets'] / _cal_denom).round(3)
    agg = agg.sort_values((gcols if gcols else []) + ['week_start']).reset_index(drop=True)
    # v1.5.0 — TRAILING 4-WEEK AVERAGES ON THE SHARED SPINE, AND POOLED FOR THE RATIO.
    # Two changes from v1.4.0's rolling mean, both of which move numbers:
    #   * it rolled over the last four ROWS PRESENT. An entity with a quiet week had no row,
    #     so the window silently reached back five or six calendar weeks and was published as
    #     a 4-week average. add_trailing reindexes onto WEEK_SPINE first, so an absent week
    #     is a zero for counts and a gap for rates.
    #   * it averaged four weekly RATIOS, which weights a 40-ticket holiday week the same as
    #     a 700-ticket one. The trailing ratio is now trailing tickets / trailing tech-days.
    agg = add_trailing(agg, gcols,
                       count_cols=['total_tickets', 'virtual_tickets',
                                   'total_active_tech_days', 'total_active_tech_days_whole'],
                       rate_cols=['tickets_per_active_day_per_tech', 'virtual_ticket_share_pct',
                                  'tech_count', 'denominator_inflation_pct'])
    _s = TRAILING_SUFFIX
    agg['tickets_per_active_day_per_tech' + _s] = (agg['total_tickets' + _s]
        / agg['total_active_tech_days' + _s].replace(0, np.nan)).round(3)
    agg['virtual_ticket_share_pct' + _s] = (agg['virtual_tickets' + _s]
        / agg['total_tickets' + _s].replace(0, np.nan) * 100).round(1)
    # Back-compat alias: v1.4.0 sheets and any saved pivot referred to rolling_4wk_avg.
    agg['rolling_4wk_avg'] = agg['tickets_per_active_day_per_tech' + _s]
    return agg

dash_wk_prod_wh    = _dash_weekly(['tech_warehouse','vp','state','metro'])
dash_wk_prod_metro = _dash_weekly(['metro'])
dash_wk_prod_vp    = _dash_weekly(['vp'])
dash_wk_prod_co    = _dash_weekly([])
print(f'Weekly productivity: wh={len(dash_wk_prod_wh):,} metro={len(dash_wk_prod_metro):,} '
      f'vp={len(dash_wk_prod_vp):,} co={len(dash_wk_prod_co):,} ({len(_week_index)} weeks)')

# Reconciliation guard: apportioned days must equal distinct (tech, date) company-wide.
# This is the grain rule made executable — if it ever fails, a later merge has
# duplicated rows and every entity ratio is wrong.
_appor = float(_techday_wk['active_day_equiv'].sum())
_honest_days = int(_wk_src[_wk_src['_tech_key'] != '|']
                   .drop_duplicates(['_tech_key', 'completed_date']).shape[0])
_whole = int(_techday_wk['active_weekdays'].sum())
print(f'Denominator check — apportioned {_appor:,.1f} | distinct (tech,date) {_honest_days:,} '
      f'| whole-day sum {_whole:,} (v1.1.0 used the last, inflating it '
      f'{(_whole / max(_honest_days, 1) - 1) * 100:.1f}%).')
if APPORTION_TECH_DAYS and abs(_appor - _honest_days) > max(1.0, 0.001 * _honest_days):
    print('*** WARNING: apportioned days do not reconcile to distinct (tech, date). '
          'Investigate before circulating — the grain rule is broken somewhere.')
_vshare = (dash_wk_prod_co['virtual_tickets'].sum()
           / max(dash_wk_prod_co['total_tickets'].sum(), 1) * 100)
print(f'Numerator recovered from virtual warehouses: {_vshare:.1f}% of company tickets in window.')


Weekly productivity: wh=5,418 metro=340 vp=500 co=85 (85 weeks)
Denominator check — apportioned 79,504.0 | distinct (tech,date) 79,504 | whole-day sum 84,109 (v1.1.0 used the last, inflating it 5.8%).
Numerator recovered from virtual warehouses: 2.6% of company tickets in window.


## Cell 13.5 — CENSUS PER TECHNICIAN *(rewritten v1.5.0, fixed v1.6.0)*

Weekly, with a trailing 4-week average, at every grain. **Three denominators are published side by side** — ADC over distinct technicians (headline), ADC over the average number of technicians on the road per weekday (the v1.4.0 basis, retained for reconciliation), and the attendance rate that is the whole difference between them.

**v1.6.0 fixes two defects here:** the unmapped-census diagnostic conflated virtual warehouses (unresolvable by construction) with a real warehouse-vocabulary gap and warned about the wrong one; and the trailing ratios divided a four-week census by a two-week technician count, reading high for exactly the sites that had a quiet week.

In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# CENSUS PER TECHNICIAN — WEEKLY, THREE DENOMINATORS         rewritten v1.5.0
#
# THE CORRECTION. v1.4.0 published ADC / (weekday tech-days / weekdays in month) under the
# title "Census per Active Technician". The arithmetic was right; the number is ~40% above
# ADC per technician, and the whole gap is the average weekday ATTENDANCE rate. From this
# notebook's own Jul-2026 figures: 4,112 apportioned tech-days / 23 weekdays = 178.8
# "average technicians on the road", against 251 distinct technicians on tickets and ~300
# on payroll. ADC 23,356 therefore reads 130.6 / 93.1 / 77.9 depending on which
# denominator you pick, and the pack printed the first while readers heard the second.
#
# WHY THAT IS A DEFINITIONAL ERROR AND NOT A LABEL QUIBBLE. ADC is a STOCK — patients on
# service on an average day. Tech-days are a FLOW — attendance events. A patient does not
# leave service because their technician took PTO, so the caseload is carried by the whole
# roster every day. A stock divided by an attendance average answers no question anyone
# asked. Read the other way it IS a real quantity — coverage intensity, "how thin are we in
# the field?" — but that is not caseload and must never sit beside a per-technician
# staffing target.
#
# The productivity metric is untouched and stays as it is: tickets / tech-days is
# flow / flow, which is the case this shared denominator was correct for. Inheriting it for
# a stock numerator was the mistake.
#
# SO ALL THREE ARE PUBLISHED, and the wedge between them is a column rather than a
# footnote:
#   census_per_tech_headcount       ADC / distinct technicians in the week   [HEADLINE]
#   census_per_active_tech_weekday  the v1.4.0 basis, retained so the restatement is
#                                   measurable and the last pack reconciles line by line
#   attendance_rate_pct             tech-days / (technicians x weekdays) — the wedge
#
# RETIRED CAVEAT. v1.4.0 warned on every panel that "ADC is a 7-day average, tech-days are
# weekdays only". That is nearly harmless — census is a stock, so a 7-day mean and a 5-day
# mean differ by the weekend admit/discharge rhythm, low single digits — and it pointed
# readers at a few-percent effect while a ~40% one was documented as a deliberate design
# choice. The panels carry the attendance wedge instead.
#
# WEEKLY GRAIN (v1.5.0). Weekly distinct-technician counts are noisier than monthly: a
# technician on a week's PTO leaves the denominator entirely, which lifts the ratio for
# reasons that are not caseload. That is what the trailing 4-week average is for, and it is
# why the suppression floor moved onto the DENOMINATOR. The monthly frame is retained as
# the reconciliation bridge to the v1.4.0 pack, not for plotting.
#
# SUPPRESSION FLOOR, CORRECTED. v1.4.0 tested tech_days < 20 — but the ratio's denominator
# is tech_days / weekdays, so at the floor it suppressed at ~0.95 technicians, i.e. barely
# at all, and on a weekly grain that floor would be ~4.3x too loose. The test is now on the
# denominators themselves: avg_active_techs and techs_distinct.
# ─────────────────────────────────────────────────────────────────────────────
_td_wk = _techday.copy()          # from Cell 13: apportioned, weekday-only, Sun-Sat weeks
_td_wk['week_start'] = week_start_of(_td_wk['completed_date'])
_td_wk['period'] = _td_wk['completed_date'].dt.strftime('%Y-%m')

# Warehouse -> (vp, state, metro), ONE row per warehouse. If a warehouse carried two
# hierarchy rows the census merge would fan out and silently double the patient-days.
_wh_ent_all = _techday[['tech_warehouse', 'vp', 'state', 'metro']].drop_duplicates()
_wh_multi = _wh_ent_all.groupby('tech_warehouse').size()
_wh_multi = _wh_multi[_wh_multi > 1]
if len(_wh_multi):
    print(f'*** WARNING: {len(_wh_multi)} warehouse(s) map to more than one (vp, state, metro): '
          f'{list(_wh_multi.index)[:5]}. Taking the first; census at VP/metro grain may be '
          f'attributed inconsistently. Fix the hierarchy in Cell 9.4.')
_wh_ent = _wh_ent_all.drop_duplicates(subset=['tech_warehouse'], keep='first')

# Daily census -> entity. Kept at DAILY grain so every aggregation below re-pools
# patient-days over distinct dates instead of averaging averages (Cell 7.5, defect 2).
census_daily_ent = df_census_daily.merge(_wh_ent, left_on='warehouse',
                                         right_on='tech_warehouse', how='left')
# ── UNMAPPED CENSUS — TWO DIFFERENT THINGS, REPORTED SEPARATELY  (fixed v1.6.0) ──
# v1.5.0 counted every unmapped patient-day together and warned "check the warehouse
# vocabulary in SERP_APC_DAILY against SERP_WAREHOUSES". That warning fired on its own
# change: v1.5.0 started KEEPING virtual ('Z%') census (Cell 7.5, defect 3), and a virtual
# code can NEVER map here — Cell 8.1b re-attributes virtual TICKETS to physical warehouses,
# so no 'Z%' value survives as a tech_warehouse for census to join against. 100% of virtual
# census therefore landed in the "vocabulary gap" bucket, inflated it, and pointed the
# analyst at SERP_WAREHOUSES for something that is a design decision two cells earlier.
# The two are now separate numbers with separate meanings:
#   virtual        expected, and unresolvable without a new key on the census feed
#   vocabulary gap a real, actionable mismatch between two warehouse name lists
# Only the second can trip the warning.
_cd_tot = float(census_daily_ent['pt_count'].sum())
_cd_is_unmapped = census_daily_ent['tech_warehouse'].isna()
_cd_is_virtual = census_daily_ent['_is_virtual_census'].fillna(False).astype(bool)

_cd_virtual = float(census_daily_ent.loc[_cd_is_unmapped & _cd_is_virtual, 'pt_count'].sum())
_cd_gap = float(census_daily_ent.loc[_cd_is_unmapped & ~_cd_is_virtual, 'pt_count'].sum())
_cd_virtual_pct = _cd_virtual / max(_cd_tot, 1e-9) * 100
_cd_gap_pct = _cd_gap / max(_cd_tot, 1e-9) * 100
_cd_unmapped_pct = _cd_virtual_pct + _cd_gap_pct          # kept for the sheets
_gap_wh = sorted(census_daily_ent.loc[_cd_is_unmapped & ~_cd_is_virtual, 'warehouse'].unique())
_virt_wh = sorted(census_daily_ent.loc[_cd_is_unmapped & _cd_is_virtual, 'warehouse'].unique())
print(f'Census mapping: {_cd_unmapped_pct:.1f}% of patient-days cannot reach a site page, '
      f'from two unrelated causes:')
print(f'  {_cd_virtual_pct:>5.1f}%  VIRTUAL warehouses ({len(_virt_wh)} codes: {_virt_wh[:5]}) '
      f'— EXPECTED. A census row carries no technician, so the Cell 8.1b ticket '
      f're-attribution cannot be applied to it. Not a data-quality problem.')
print(f'  {_cd_gap_pct:>5.1f}%  VOCABULARY GAP ({len(_gap_wh)} codes: {_gap_wh[:5]}) — physical '
      f'sites in SERP_APC_DAILY the ticket feed never resolved. This one is ACTIONABLE.')
print('  Both stay in the COMPANY total, so site rows sum to less than the company row by '
      'design; the company row remains the reconciliation truth.')
if _cd_gap_pct > CENSUS_WARN_UNMAPPED_PCT:
    print(f'  *** WARNING: the VOCABULARY GAP alone exceeds {CENSUS_WARN_UNMAPPED_PCT}%. '
          f'Site-level census-per-technician UNDERSTATES the true load. Reconcile the '
          f'warehouse names in SERP_APC_DAILY against SERP_WAREHOUSES before publishing.')
tbl_census_unmapped = pd.DataFrame(
    [{'cause': 'virtual warehouse (expected)', 'codes': len(_virt_wh),
      'patient_days': round(_cd_virtual, 1), 'pct_of_census': round(_cd_virtual_pct, 2),
      'actionable': False, 'warehouses': ', '.join(_virt_wh[:30])},
     {'cause': 'warehouse vocabulary gap (actionable)', 'codes': len(_gap_wh),
      'patient_days': round(_cd_gap, 1), 'pct_of_census': round(_cd_gap_pct, 2),
      'actionable': True, 'warehouses': ', '.join(_gap_wh[:30])}])

# Month basis, so the monthly bridge uses exactly the weekday denominator v1.4.0 used.
_cal_days = pd.DataFrame({'d': pd.date_range(pd.Timestamp(FILTER_START),
                                             pd.Timestamp(AS_OF_DATE), freq='D')})
_cal_days['period'] = _cal_days['d'].dt.strftime('%Y-%m')
_MONTH_BASIS = (_cal_days.groupby('period', as_index=False)
                .agg(weekdays_in_period=('d', lambda s: int((s.dt.dayofweek < 5).sum())),
                     calendar_days_in_period=('d', 'nunique')))
_MONTH_BASIS['is_partial_week'] = False    # keeps add_trailing's partial-week logic inert


def _census_frame(gcols, tcol, basis_frame, day_basis_col):
    """Census per technician at any entity grain and either time grain.

    One function for weekly and monthly, so the two cannot drift: the v1.4.0 monthly
    figure has to stay exactly reproducible for the restatement to be credible."""
    gcols = list(gcols)
    keys = gcols + [tcol]
    techs = (_td_wk.groupby(keys, dropna=False, as_index=False)
             .agg(tech_days=('active_day_share', 'sum'),
                  techs_distinct=('_tech_key', 'nunique'),
                  active_weekdays_whole=('completed_date', 'nunique'),
                  tickets=('day_tickets', 'sum')))

    # ── APPORTIONED HEADCOUNT (techs_equiv)  fixed v1.6.0 ────────────────────
    # techs_distinct counts a technician who works two VPs ONCE IN EACH, while tech_days is
    # apportioned between them. Dividing an apportioned numerator by a whole-count
    # denominator is exactly the defect v1.2.0 fixed for the productivity metric, and it was
    # sitting in the census denominator: at VP grain attendance read far below the company
    # figure, and patients-per-technician far below it too, purely because the same person
    # was counted twice.
    #
    # techs_equiv gives each technician to an entity in proportion to the share of their own
    # days spent there. A technician entirely at one site counts as 1 there; one splitting
    # 50/50 counts as 0.5 at each. It therefore SUMS TO THE TRUE DISTINCT COUNT company-wide,
    # so the company figures — and the restatement table this release rests on — are
    # unchanged, while every entity grain becomes internally consistent and additive.
    _tt = (_td_wk.groupby(['_tech_key', tcol], as_index=False)
           .agg(_tech_total=('active_day_share', 'sum')))
    _te = (_td_wk.groupby(gcols + ['_tech_key', tcol], dropna=False, as_index=False)
           .agg(_here=('active_day_share', 'sum'))
           .merge(_tt, on=['_tech_key', tcol], how='left'))
    _te['_equiv'] = _te['_here'] / _te['_tech_total'].replace(0, np.nan)
    techs = techs.merge(_te.groupby(keys, dropna=False, as_index=False)
                        .agg(techs_equiv=('_equiv', 'sum')), on=keys, how='left')
    techs['techs_equiv'] = techs['techs_equiv'].round(3)
    # Company uses the whole feed (mapped or not, virtual or not) so the company row stays
    # the reconciliation truth; entity grains can only use the mapped subset.
    _cs = (census_daily_ent[census_daily_ent['tech_warehouse'].notna()]
           if gcols else census_daily_ent)
    cen = (_cs.groupby(keys, dropna=False, as_index=False)
           .agg(pt_days=('pt_count', 'sum'),
                pt_days_unfiltered=('pt_count_all', 'sum'),
                census_days=('census_date', 'nunique'),
                census_warehouses=('warehouse', 'nunique')))
    # ADC = pooled patient-days / distinct dates OBSERVED. Never a sum of per-warehouse
    # averages (Cell 7.5, defect 2) — that number corresponds to no real day.
    cen['adc'] = (cen['pt_days'] / cen['census_days'].replace(0, np.nan)).round(2)
    cen['adc_unfiltered'] = (cen['pt_days_unfiltered']
                             / cen['census_days'].replace(0, np.nan)).round(2)

    out = techs.merge(cen, on=keys, how='outer').merge(basis_frame, on=tcol, how='left')
    # One name for the day basis whatever the grain, so the pooled trailing arithmetic below
    # does not need to know whether it is looking at a week or a month.
    out['weekday_days'] = pd.to_numeric(out[day_basis_col], errors='coerce')
    out['avg_active_techs'] = (out['tech_days']
                               / out['weekday_days'].replace(0, np.nan)).round(3)
    # THE HEADLINE: a stock over a headcount — the APPORTIONED headcount, so the figure is
    # additive across entities and comparable to the company row.
    out['census_per_tech_headcount'] = (out['adc']
                                        / out['techs_equiv'].replace(0, np.nan)).round(2)
    # The v1.4.0 basis, retained so the restatement is measurable.
    out['census_per_active_tech_weekday'] = (out['adc']
        / out['avg_active_techs'].replace(0, np.nan)).round(2)
    # The wedge, as a column. This is the whole 85-vs-120 story in one number.
    out['attendance_rate_pct'] = (out['tech_days']
        / (out['techs_equiv'] * out['weekday_days']).replace(0, np.nan) * 100).round(1)
    out['restatement_vs_v140_pct'] = ((out['census_per_tech_headcount']
        / out['census_per_active_tech_weekday'].replace(0, np.nan) - 1) * 100).round(1)
    # Denominator-honest alternative that sidesteps the choice entirely: patient-days
    # carried per technician-day actually worked.
    out['pt_days_per_tech_day'] = (out['pt_days'] / out['tech_days'].replace(0, np.nan)).round(2)
    out['tickets_per_active_day'] = (out['tickets']
                                     / out['tech_days'].replace(0, np.nan)).round(3)

    # Suppression on the DENOMINATORS, not on a period total. Inputs stay in the sheet so
    # every suppression is auditable; only the derived ratios are withheld.
    out['ratio_suppressed'] = ((out['avg_active_techs'].fillna(0) < CENSUS_MIN_ACTIVE_TECHS)
                               | (out['techs_equiv'].fillna(0) < CENSUS_MIN_TECHS))
    for _rc in ('census_per_tech_headcount', 'census_per_active_tech_weekday',
                'attendance_rate_pct', 'pt_days_per_tech_day', 'restatement_vs_v140_pct'):
        out[_rc] = out[_rc].mask(out['ratio_suppressed'])
    out['denominator_note'] = ('headline = ADC / distinct technicians; v1.4.0 basis = '
                               'ADC / avg technicians on the road per weekday')
    for _c in ('tech_days', 'pt_days', 'pt_days_unfiltered'):
        out[_c] = pd.to_numeric(out[_c], errors='coerce').round(1)
    return out.sort_values(gcols + [tcol]).reset_index(drop=True)


# v1.6.0 — techs_distinct and avg_active_techs moved from RATES to COUNTS. A week in which
# a site had no technicians on the road genuinely had ZERO of them, so they zero-fill inside
# the entity's span exactly as tickets and patient-days do. Treated as rates they were
# averaged over present weeks only, which left every census trailing ratio mixing a
# numerator pooled over four weeks against a denominator averaged over two — so the ratio
# read HIGH for precisely the sites that had a quiet week. weekday_days is carried as a
# count too, so the attendance wedge can be pooled instead of averaged.
_CENSUS_COUNTS = ['pt_days', 'census_days', 'tech_days', 'tickets',
                  'techs_distinct', 'techs_equiv', 'avg_active_techs', 'weekday_days']
_CENSUS_RATES = ['adc', 'attendance_rate_pct', 'census_per_tech_headcount',
                 'census_per_active_tech_weekday', 'pt_days_per_tech_day',
                 'tickets_per_active_day']


def _dash_census_weekly(gcols):
    out = _census_frame(gcols, 'week_start', WEEK_SPINE, 'weekday_days_in_week')
    out = add_trailing(out, gcols, count_cols=_CENSUS_COUNTS, rate_cols=_CENSUS_RATES)
    # POOLED trailing ratios override the mean-of-ratios ones add_trailing produced. For a
    # ratio, the mean of four weekly values weights a quiet week the same as a busy one;
    # pooling weights by volume, which is what "the last four weeks ran at X" means. Kept
    # explicit here rather than hidden in the helper, because ADC's numerator and
    # denominator are both sums and techs_distinct genuinely can only be averaged.
    _s = TRAILING_SUFFIX
    # EVERY census trailing ratio is rebuilt from the pooled count columns, so numerator and
    # denominator always rest on the same four weeks. add_trailing's mean-of-ratios values
    # are overwritten — including attendance, which v1.5.0 left averaged.
    out['adc' + _s] = (out['pt_days' + _s] / out['census_days' + _s].replace(0, np.nan)).round(2)
    out['census_per_tech_headcount' + _s] = (out['adc' + _s]
        / out['techs_equiv' + _s].replace(0, np.nan)).round(2)
    out['census_per_active_tech_weekday' + _s] = (out['adc' + _s]
        / out['avg_active_techs' + _s].replace(0, np.nan)).round(2)
    out['attendance_rate_pct' + _s] = (out['tech_days' + _s]
        / (out['techs_equiv' + _s] * out['weekday_days' + _s]).replace(0, np.nan)
        * 100).round(1)
    out['tickets_per_active_day' + _s] = (out['tickets' + _s]
        / out['tech_days' + _s].replace(0, np.nan)).round(3)
    out['pt_days_per_tech_day' + _s] = (out['pt_days' + _s]
        / out['tech_days' + _s].replace(0, np.nan)).round(2)
    # A suppressed week must not reappear through its own trailing average.
    for _rc in ('census_per_tech_headcount', 'census_per_active_tech_weekday',
                'attendance_rate_pct', 'pt_days_per_tech_day'):
        out[_rc + _s] = out[_rc + _s].mask(out['ratio_suppressed'])
    return out


def _dash_census_monthly(gcols):
    return _census_frame(gcols, 'period', _MONTH_BASIS, 'weekdays_in_period')


dash_wk_census_wh    = _dash_census_weekly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_wk_census_metro = _dash_census_weekly(['metro'])
dash_wk_census_vp    = _dash_census_weekly(['vp'])
dash_wk_census_co    = _dash_census_weekly([])
dash_mo_census_wh    = _dash_census_monthly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_mo_census_metro = _dash_census_monthly(['metro'])
dash_mo_census_vp    = _dash_census_monthly(['vp'])
dash_mo_census_co    = _dash_census_monthly([])
print(f'Weekly census/tech: wh={len(dash_wk_census_wh):,} metro={len(dash_wk_census_metro):,} '
      f'vp={len(dash_wk_census_vp):,} co={len(dash_wk_census_co):,} '
      f'(monthly bridge: co={len(dash_mo_census_co):,})')

# ── Reconciliation 1: census and productivity must share one technician population ──
_cens_techdays = float(dash_wk_census_co['tech_days'].sum())
_prod_techdays = float(_techday_wk['active_day_equiv'].sum())
print(f'Denominator tie-out — census {_cens_techdays:,.1f} tech-days vs productivity '
      f'{_prod_techdays:,.1f} (delta {_cens_techdays - _prod_techdays:+,.1f}).')
if abs(_cens_techdays - _prod_techdays) > max(1.0, 0.001 * max(_prod_techdays, 1)):
    print('*** WARNING: the census and productivity panels are NOT on the same technician '
          'population. Do not read one against the other until this reconciles.')

# ── Reconciliation 2: THE 85-vs-120 TABLE ────────────────────────────────────
# The answer to the question that triggered the v1.5.0 audit, printed every run so nobody
# re-derives it. Monthly, because that is the grain the disputed pack was published on.
tbl_census_denominator_recon = dash_mo_census_co[
    ['period', 'adc', 'adc_unfiltered', 'techs_distinct', 'techs_equiv', 'tech_days',
     'weekdays_in_period', 'avg_active_techs', 'census_per_tech_headcount',
     'census_per_active_tech_weekday', 'attendance_rate_pct',
     'restatement_vs_v140_pct']].copy()
print('\n' + '=' * 100)
print('CENSUS PER TECHNICIAN — WHICH DENOMINATOR (company, monthly bridge to the v1.4.0 pack)')
print('=' * 100)
print(tbl_census_denominator_recon.to_string(index=False))
if len(tbl_census_denominator_recon):
    _l = tbl_census_denominator_recon.iloc[-1]
    print(f'\n  Latest month {_l["period"]}: ADC {_l["adc"]:,.0f} carried by '
          f'{_l["techs_distinct"]:,.0f} distinct technicians '
          f'({_l["avg_active_techs"]:,.0f} on the road on an average weekday, '
          f'{_l["attendance_rate_pct"]:.0f}% attendance).')
    print(f'    HEADLINE  ADC / technicians       = {_l["census_per_tech_headcount"]:>7,.1f}')
    print(f'    v1.4.0    ADC / avg techs on road = '
          f'{_l["census_per_active_tech_weekday"]:>7,.1f}   '
          f'({_l["restatement_vs_v140_pct"]:+.0f}% restatement)')
    print('    The difference is ATTENDANCE — not census, not productivity. Quote the '
          'headline for caseload; quote the v1.4.0 basis only for field-coverage questions, '
          'and say which one you used.')

# ── Reconciliation 3: the apportioned headcount must be additive ────────────
# If it is not, the entity rows are not comparable to the company row and the headline
# census figure cannot be read across grains.
_co_equiv = float(dash_wk_census_co['techs_equiv'].sum())
_wh_equiv = float(dash_wk_census_wh['techs_equiv'].sum())
print(f'Apportioned-headcount tie-out — warehouse sum {_wh_equiv:,.1f} vs company '
      f'{_co_equiv:,.1f} (delta {_wh_equiv - _co_equiv:+,.1f}); raw distinct-count sum would '
      f'be {float(dash_wk_census_wh["techs_distinct"].sum()):,.0f}, which double-counts every '
      f'technician who works more than one site.')
if abs(_wh_equiv - _co_equiv) > max(1.0, 0.001 * max(_co_equiv, 1)):
    print('*** WARNING: the apportioned headcount does not sum across warehouses. '
          'Census per technician is NOT comparable between grains until this reconciles.')

_sup_n = int(dash_wk_census_wh['ratio_suppressed'].sum())
print(f'\nSuppressed (thin denominator: <{CENSUS_MIN_ACTIVE_TECHS} avg techs or '
      f'<{CENSUS_MIN_TECHS} distinct techs): {_sup_n:,} of {len(dash_wk_census_wh):,} '
      f'warehouse-weeks.')
print('Company census per technician, last 8 weeks:')
print(dash_wk_census_co[['week_label', 'adc', 'techs_distinct', 'avg_active_techs',
                         'census_per_tech_headcount',
                         'census_per_tech_headcount' + TRAILING_SUFFIX,
                         'census_per_active_tech_weekday', 'attendance_rate_pct',
                         'is_partial_week']].tail(8).to_string(index=False))

Census mapping: 1.0% of patient-days cannot reach a site page, from two unrelated causes:
    0.4%  VIRTUAL warehouses (12 codes: ['Z Akron (Shut Down)', 'Z CS', 'Z Columbia - SC (Shut Down)', 'Z Equipment Collections', 'Z Fort Smith (Shut Down)']) — EXPECTED. A census row carries no technician, so the Cell 8.1b ticket re-attribution cannot be applied to it. Not a data-quality problem.
    0.6%  VOCABULARY GAP (6 codes: ['R03 Hot Springs', 'R11 Columbia - SC', 'R16 Stafford', 'RNW College Station', 'RNW Lufkin']) — physical sites in SERP_APC_DAILY the ticket feed never resolved. This one is ACTIONABLE.
  Both stay in the COMPANY total, so site rows sum to less than the company row by design; the company row remains the reconciliation truth.


ValueError: Boolean array expected for the condition, not int64

## Cell 14 — LOST EQUIPMENT BY ENTITY *(rewritten v1.5.0)*

**Bug fix:** VP and metro rows divided their lost cost by the WHOLE COMPANY's patient-days, understating `lost_cost_per_1k_pt_days` several-fold at those grains. Every grain now joins census at its own grain, and the cell proves it. Also moved to weekly + trailing 4-week; monthly retained as the reconciliation bridge.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOST EQUIPMENT BY ENTITY — WEEKLY + TRAILING 4-WEEK          rewritten v1.5.0
#
# WHAT CHANGED IN v1.5.0, and the first item is a bug fix, not a presentation change.
#
# 1. THE CENSUS DENOMINATOR AT VP AND METRO GRAIN WAS THE WHOLE COMPANY'S. v1.3.0/v1.4.0
#    branched on `if 'tech_warehouse' in gcols`, merging per-warehouse ADC in that branch
#    and the COMPANY total in the else branch. Both the metro and the VP grain took the
#    else branch, so every VP row and every metro row divided its lost cost by the entire
#    company's patient-days. lost_cost_per_1k_pt_days on a VP page was understated by
#    roughly (company census / that VP's census) — order of 5-10x for a VP, far more for a
#    metro — the metric was not comparable across grains, and warehouse rows did not
#    aggregate to their VP. Company and warehouse grains were correct, which is exactly
#    why nobody caught it: the error is invisible unless you compare two grains.
#    Every grain now joins census AT ITS OWN GRAIN, from the daily census frame.
#
# 2. WEEKLY, with a trailing 4-week average, like every other metric. The v1.3.0 comment
#    said monthly was chosen because "per-site weekly counts are mostly 0-2 and would
#    chart as noise". That reasoning was sound for a raw weekly count and is the reason the
#    trailing average exists: the panel plots weekly bars for honesty and the trailing line
#    for signal, and the ratio line is suppressed where the denominator is too thin to
#    carry one. Monthly frames are retained as the reconciliation bridge.
#
# 3. Census figures now exclude facility '(F)' / inpatient-unit '(IPU)' / 'Contract Test'
#    customers, matching the APC definition (Cell 7.5, defect 1). That RAISES
#    lost_cost_per_1k_pt_days versus v1.4.0, because the denominator got smaller.
#
# Two denominators are still offered, because a raw asset count cannot compare a large site
# to a small one:
#   * lost_pct_of_inventory   — assets lost / tagged inventory on hand. Inventory is a
#     CURRENT snapshot, so this is "this week's losses against today's stock": sound for
#     ranking sites against each other, not a point-in-time rate.
#   * lost_cost_per_1k_pt_days — cost / (patient-days x 1000). The census-normalised view;
#     it moves with how much equipment is actually out in the field.
#
# CENSORING, unchanged: recovery_rate_pct is computed ONLY over cohorts at least
# LOST_RECOVERY_MATURITY_DAYS old. Recent weeks show NaN rather than a falsely low number —
# the alternative is a chart that always declines at the right-hand edge.
#
# STALENESS, unchanged and material: the feed ends before the rest of the dashboard, so the
# final weeks are MISSING DATA, not recovery. LOST_STALE_NOTE banners every page.
# ─────────────────────────────────────────────────────────────────────────────
_lost_filt['week_start'] = week_start_of(_lost_filt['lost_date'])

# Census at any entity grain and week, pooled from the daily frame. This function is the
# fix for defect 1 above: there is no longer a branch that can fall back to the company.
def _census_for(gcols, tcol='week_start'):
    gcols = list(gcols)
    _cs = (census_daily_ent[census_daily_ent['tech_warehouse'].notna()]
           if gcols else census_daily_ent)
    c = (_cs.groupby(gcols + [tcol], dropna=False, as_index=False)
         .agg(pt_days=('pt_count', 'sum'), census_days=('census_date', 'nunique')))
    c['adc'] = (c['pt_days'] / c['census_days'].replace(0, np.nan)).round(2)
    return c


def _dash_lost_weekly(gcols):
    gcols = list(gcols)
    keys = gcols + ['week_start']
    agg = (_lost_filt.groupby(keys, dropna=False, as_index=False)
        .agg(lost_asset_count=('asset_tag', 'nunique'),
             lost_asset_cost=('lost_cost', 'sum'),
             lost_product_types=('product_name', 'nunique'),
             recovered_assets=('is_recovered', 'sum'),
             discarded_assets=('is_discarded', 'sum'),
             unresolved_assets=('is_unresolved', 'sum'),
             mature_assets=('cohort_mature', 'sum'),
             median_days_to_resolve=('days_to_resolve', 'median')))
    # recovery rate on mature cohorts only
    _mat = (_lost_filt[_lost_filt['cohort_mature']].groupby(keys, dropna=False, as_index=False)
            .agg(_mature_n=('asset_tag', 'size'), _mature_recovered=('is_recovered', 'sum')))
    agg = agg.merge(_mat, on=keys, how='left')
    agg['recovery_rate_pct'] = np.where(
        agg['_mature_n'].fillna(0) > 0,
        (agg['_mature_recovered'] / agg['_mature_n'].replace(0, np.nan) * 100).round(1), np.nan)
    agg = agg.drop(columns=['_mature_n', '_mature_recovered'])
    agg['lost_asset_cost'] = agg['lost_asset_cost'].round(2)

    # inventory denominator exists at warehouse grain only — it is a per-site snapshot
    if 'tech_warehouse' in gcols:
        agg = agg.merge(df_inventory_total, on='tech_warehouse', how='left')
        agg['lost_pct_of_inventory'] = (agg['lost_asset_count']
            / agg['total_inventory_count'].replace(0, np.nan) * 100).round(3)

    # CENSUS AT THIS GRAIN — the v1.5.0 fix. Every grain, including metro and VP.
    agg = agg.merge(_census_for(gcols), on=keys, how='left')
    agg['lost_cost_per_1k_pt_days'] = (agg['lost_asset_cost']
        / agg['pt_days'].replace(0, np.nan) * 1000).round(2)
    agg['lost_assets_per_1k_pt_days'] = (agg['lost_asset_count']
        / agg['pt_days'].replace(0, np.nan) * 1000).round(3)
    agg['census_grain'] = ('company' if not gcols else '+'.join(gcols))

    agg = attach_spine(agg)
    agg = add_trailing(agg, gcols,
                       count_cols=['lost_asset_count', 'lost_asset_cost', 'pt_days',
                                   'recovered_assets', 'unresolved_assets'],
                       rate_cols=['lost_cost_per_1k_pt_days', 'lost_assets_per_1k_pt_days',
                                  'lost_pct_of_inventory', 'recovery_rate_pct',
                                  'median_days_to_resolve', 'adc'])
    # Pooled trailing rate rather than a mean of four weekly rates — a week with two
    # assets lost and a normal census would otherwise carry the same weight as a bad week.
    _s = TRAILING_SUFFIX
    agg['lost_cost_per_1k_pt_days' + _s] = (agg['lost_asset_cost' + _s]
        / agg['pt_days' + _s].replace(0, np.nan) * 1000).round(2)
    agg['lost_assets_per_1k_pt_days' + _s] = (agg['lost_asset_count' + _s]
        / agg['pt_days' + _s].replace(0, np.nan) * 1000).round(3)
    # staleness travels with the data, so no sheet can be read without it
    agg['source_through'] = (LOST_MAX_DATE.strftime('%Y-%m-%d') if pd.notna(LOST_MAX_DATE) else '')
    return agg.sort_values(gcols + ['week_start']).reset_index(drop=True)


def _dash_lost_monthly(gcols):
    """Retained as the reconciliation bridge to the v1.3.0/v1.4.0 packs. NOT plotted.
    Note that lost_cost_per_1k_pt_days here is the CORRECTED per-grain figure, so it will
    not match a v1.4.0 VP or metro sheet — that is the bug being fixed, not a regression."""
    gcols = list(gcols)
    keys = gcols + ['period']
    agg = (_lost_filt.groupby(keys, dropna=False, as_index=False)
        .agg(lost_asset_count=('asset_tag', 'nunique'),
             lost_asset_cost=('lost_cost', 'sum'),
             recovered_assets=('is_recovered', 'sum'),
             unresolved_assets=('is_unresolved', 'sum'),
             mature_assets=('cohort_mature', 'sum'),
             median_days_to_resolve=('days_to_resolve', 'median')))
    _mat = (_lost_filt[_lost_filt['cohort_mature']].groupby(keys, dropna=False, as_index=False)
            .agg(_mature_n=('asset_tag', 'size'), _mature_recovered=('is_recovered', 'sum')))
    agg = agg.merge(_mat, on=keys, how='left')
    agg['recovery_rate_pct'] = np.where(
        agg['_mature_n'].fillna(0) > 0,
        (agg['_mature_recovered'] / agg['_mature_n'].replace(0, np.nan) * 100).round(1), np.nan)
    agg = agg.drop(columns=['_mature_n', '_mature_recovered'])
    agg = agg.merge(_census_for(gcols, 'period'), on=keys, how='left')
    agg['lost_cost_per_1k_pt_days'] = (agg['lost_asset_cost']
        / agg['pt_days'].replace(0, np.nan) * 1000).round(2)
    agg['lost_asset_cost'] = agg['lost_asset_cost'].round(2)
    agg[['lost_year', 'lost_month']] = agg['period'].str.split('-', expand=True).astype(int)
    agg['source_through'] = (LOST_MAX_DATE.strftime('%Y-%m-%d') if pd.notna(LOST_MAX_DATE) else '')
    return agg.sort_values(gcols + ['period']).reset_index(drop=True)


dash_wk_lost_wh    = _dash_lost_weekly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_wk_lost_metro = _dash_lost_weekly(['metro'])
dash_wk_lost_vp    = _dash_lost_weekly(['vp'])
dash_wk_lost_co    = _dash_lost_weekly([])
dash_mo_lost_wh    = _dash_lost_monthly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_mo_lost_metro = _dash_lost_monthly(['metro'])
dash_mo_lost_vp    = _dash_lost_monthly(['vp'])
dash_mo_lost_co    = _dash_lost_monthly([])
print(f'Weekly lost: wh={len(dash_wk_lost_wh):,} metro={len(dash_wk_lost_metro):,} '
      f'vp={len(dash_wk_lost_vp):,} co={len(dash_wk_lost_co):,}')

# ── Prove the census-denominator fix, rather than asserting it ────────────────
# VP patient-days must sum to at most the company's (less the unmapped share), and each
# VP's census must be a strict subset. Under the v1.4.0 code every VP carried the company
# figure, so this check would have failed by a factor of the VP count.
_vp_pt = float(dash_mo_lost_vp.drop_duplicates(['vp', 'period'])['pt_days'].sum())
_co_pt = float(dash_mo_lost_co.drop_duplicates(['period'])['pt_days'].sum())
_n_vp = dash_mo_lost_vp['vp'].nunique()
print(f'Census denominator check — VP patient-days sum {_vp_pt:,.0f} vs company {_co_pt:,.0f} '
      f'across {_n_vp} VPs ({_vp_pt / max(_co_pt, 1e-9) * 100:.1f}% of company; the shortfall '
      f'is census on warehouses the ticket feed never resolved).')
if _vp_pt > _co_pt * 1.001:
    print('*** WARNING: VP census exceeds company census. The per-grain join is fanning out — '
          'check for a warehouse mapping to two VPs before publishing any normalised rate.')
elif _n_vp > 1 and _vp_pt / max(_co_pt, 1e-9) > 0.999 and _n_vp > 1:
    print('  (Sanity note: VP census equals company census almost exactly. Verify this is '
          'complete mapping and not the v1.4.0 defect reappearing.)')

# Product and reason views — 99.2% of rows are reason 'Not found', so the catastrophic
# reasons (Fire, Flood, Hurricane, Insect Infestation) are the informative tail and are
# listed separately rather than buried under a dominant category.
tbl_lost_products = (_lost_filt.groupby('product_name', dropna=False, as_index=False)
    .agg(lost_assets=('asset_tag', 'nunique'), lost_cost=('lost_cost', 'sum'),
         warehouses=('tech_warehouse', 'nunique'),
         recovered=('is_recovered', 'sum'), unresolved=('is_unresolved', 'sum'))
    .sort_values('lost_cost', ascending=False).reset_index(drop=True))
tbl_lost_products['lost_cost'] = tbl_lost_products['lost_cost'].round(2)

tbl_lost_reasons = (_lost_filt.groupby(['lost_reason', 'week_start'], dropna=False, as_index=False)
    .agg(lost_assets=('asset_tag', 'nunique'), lost_cost=('lost_cost', 'sum'))
    .sort_values(['week_start', 'lost_cost'], ascending=[True, False]).reset_index(drop=True))

tbl_lost_resolution = (_lost_filt.groupby(['week_start', 'resolution'], dropna=False, as_index=False)
    .agg(assets=('asset_tag', 'nunique'), cost=('lost_cost', 'sum'),
         median_days_to_resolve=('days_to_resolve', 'median')))
tbl_lost_resolution['cost'] = tbl_lost_resolution['cost'].round(2)

print(f'Lost product rows: {len(tbl_lost_products):,} | reason rows: {len(tbl_lost_reasons):,} '
      f'| resolution rows: {len(tbl_lost_resolution):,}')
if len(dash_wk_lost_co):
    print('\nCompany weekly, last 8 weeks (bulk events excluded):')
    print(dash_wk_lost_co[['week_label', 'lost_asset_count',
                           'lost_asset_count' + TRAILING_SUFFIX, 'lost_asset_cost',
                           'lost_asset_cost' + TRAILING_SUFFIX, 'recovery_rate_pct',
                           'lost_cost_per_1k_pt_days',
                           'lost_cost_per_1k_pt_days' + TRAILING_SUFFIX,
                           'is_partial_week']].tail(8).to_string(index=False))
    print('\nCompany monthly bridge:')
    print(dash_mo_lost_co[['period', 'lost_asset_count', 'lost_asset_cost',
                           'recovery_rate_pct', 'pt_days',
                           'lost_cost_per_1k_pt_days']].to_string(index=False))
_cat = _lost_filt[~_lost_filt['lost_reason'].isin(['Not found'])]
if len(_cat):
    print(f'\nNon-"Not found" reasons (the informative tail): '
          f'{_cat.groupby("lost_reason")["lost_cost"].sum().round(0).sort_values(ascending=False).to_dict()}')

## Cell 15 — REDELIVERIES BY ENTITY *(rewritten v1.5.0)*

Weekly + trailing 4-week, keyed to the **originating** ticket's week — so the most recent weeks are structurally censored, not improving. The trailing rate is pooled (trailing redeliveries / trailing tickets), not a mean of weekly rates.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# REDELIVERIES BY ENTITY — WEEKLY + TRAILING 4-WEEK              rewritten v1.5.0
#
# Week = the week of the ORIGINATING ticket, not of the redelivery visit. That is the same
# choice v1.1.0 made at monthly grain and it is the right one: the question is "how often
# does work done in week W come back?", so the event has to be attributed to the week that
# caused it. Consequence, which has to be stated wherever this is read: THE MOST RECENT
# WEEKS ARE STRUCTURALLY LOW. A ticket completed last Friday has had almost no time to
# generate a redelivery, so the right-hand end of the series is censored, not improving.
# The trailing 4-week average inherits that censoring — it is not a fix for it.
#
# Rate denominator = attributed tickets for the SAME entity and week (grain rule). df_tx
# shares the column names, so numerator and denominator key identically.
#
# The trailing rate is POOLED (trailing redeliveries / trailing tickets), not the mean of
# four weekly rates: a holiday week with 30 tickets and one redelivery reads 3.3 per 100
# and would otherwise carry the same weight as a 700-ticket week.
# ─────────────────────────────────────────────────────────────────────────────
_txa = df_tx[~df_tx['_unattributed']].copy()
_txa['week_start'] = week_start_of(_txa['completed_date'])
# orig_completed_date is the originating ticket's completion date, carried through the link
# in Cell 11.2 (added in v1.5.0 — v1.4.0 carried only the month, which cannot make a week).
redel_linked['week_start'] = week_start_of(redel_linked['orig_completed_date'])
_n_noweek = int(redel_linked['week_start'].isna().sum())
if _n_noweek:
    print(f'*** WARNING: {_n_noweek:,} linked redeliveries have no originating completion date '
          f'and cannot be placed in a week — excluded from the weekly panels, still present in '
          f'the monthly bridge.')


def _dash_redel_weekly(gcols):
    gcols = list(gcols)
    keys = gcols + ['week_start']
    agg = (redel_linked[redel_linked['week_start'].notna()]
           .groupby(keys, dropna=False, as_index=False)
           .agg(redelivery_count=('event_key', 'nunique'),
                redelivery_items=('product', 'count')))
    tx_w = (_txa.groupby(keys, dropna=False, as_index=False)
            .agg(total_tickets=('order_num', 'nunique')))
    # Outer join, not left: a week with tickets and zero redeliveries is a real 0.0 rate and
    # belongs on the chart. A left join off the redelivery frame would delete exactly the
    # good weeks.
    agg = agg.merge(tx_w, on=keys, how='outer')
    for _c in ('redelivery_count', 'redelivery_items'):
        agg[_c] = agg[_c].fillna(0).astype(int)
    agg['redel_per_100_tickets'] = (agg['redelivery_count']
        / agg['total_tickets'].replace(0, np.nan) * 100).round(2)
    agg = attach_spine(agg)
    agg = add_trailing(agg, gcols,
                       count_cols=['redelivery_count', 'redelivery_items', 'total_tickets'],
                       rate_cols=['redel_per_100_tickets'])
    _s = TRAILING_SUFFIX
    agg['redel_per_100_tickets' + _s] = (agg['redelivery_count' + _s]
        / agg['total_tickets' + _s].replace(0, np.nan) * 100).round(2)
    return agg.sort_values(gcols + ['week_start']).reset_index(drop=True)


def _dash_redel_monthly(gcols):
    """Reconciliation bridge to the v1.1.0-v1.4.0 packs. Not plotted."""
    gcols = list(gcols)
    keys = gcols + ['delivery_year', 'delivery_month']
    agg = (redel_linked.groupby(keys, dropna=False, as_index=False)
        .agg(redelivery_count=('event_key', 'nunique'),
             redelivery_items=('product', 'count')))
    tx_m = (_txa.groupby(keys, dropna=False, as_index=False)
        .agg(total_tickets=('order_num', 'nunique')))
    agg = agg.merge(tx_m, on=keys, how='outer')
    for _c in ('redelivery_count', 'redelivery_items'):
        agg[_c] = agg[_c].fillna(0).astype(int)
    agg['redel_per_100_tickets'] = (agg['redelivery_count']
        / agg['total_tickets'].replace(0, np.nan) * 100).round(2)
    agg['period'] = (agg['delivery_year'].astype('Int64').astype(str) + '-'
                     + agg['delivery_month'].astype('Int64').astype(str).str.zfill(2))
    return agg.sort_values(gcols + ['delivery_year', 'delivery_month']).reset_index(drop=True)


dash_wk_redel_wh    = _dash_redel_weekly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_wk_redel_metro = _dash_redel_weekly(['metro'])
dash_wk_redel_vp    = _dash_redel_weekly(['vp'])
dash_wk_redel_co    = _dash_redel_weekly([])
dash_mo_redel_wh    = _dash_redel_monthly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_mo_redel_metro = _dash_redel_monthly(['metro'])
dash_mo_redel_vp    = _dash_redel_monthly(['vp'])
dash_mo_redel_co    = _dash_redel_monthly([])
print(f'Weekly redeliveries: wh={len(dash_wk_redel_wh):,} metro={len(dash_wk_redel_metro):,} '
      f'vp={len(dash_wk_redel_vp):,} co={len(dash_wk_redel_co):,}')
if len(dash_wk_redel_co):
    print('Company weekly, last 8 weeks (rightmost weeks are censored — see the cell header):')
    print(dash_wk_redel_co[['week_label', 'redelivery_count', 'total_tickets',
                            'redel_per_100_tickets',
                            'redel_per_100_tickets' + TRAILING_SUFFIX,
                            'is_partial_week']].tail(8).to_string(index=False))

## Cell 15.5 — OVERTIME BY ENTITY *(weekly in v1.5.0)*

Weekly + trailing 4-week on its **own Thu–Wed payroll spine** — deliberately not the Sun–Sat spine the other panels use, because re-bucketing the result would re-create the defect v1.4.0 fixed. Site attribution now apportions by ticket share over that same payroll week rather than the month.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OVERTIME BY ENTITY                                                 new v1.3.0
#
# WHY INFERRED AND NOT READ FROM THE FEED. PLC_EMPLOYEE_HOURS has Reg_Hrs, OT1_Hrs,
# OT2_Hrs, Paid_Hrs and Est_*. All are unusable: they are pay-period values repeated on
# every daily row (Pay_Type='Work' shows Hours 958,685 against Reg_Hrs 13,626,530 —
# ~14x), OT1_Hrs is zero for every month from 2026-02 onward, and OT2_Hrs is zero
# throughout. The diagnostic at the bottom of this cell proves it on the live data
# every run, so nobody re-litigates it. OT is therefore inferred FLSA-style from daily
# [Hours], the approach tech_workload v1.35.0 uses.
#
# WHAT IS NEW versus v1.35.0: Pay_Type separates PTO and Holiday from worked time, so
# paid leave no longer inflates the weekly total. This retires the standing caveat "if
# PLC Hours includes PTO, OT is overstated" — worth 18% in Jul-2026 (10,264 hours
# including leave vs 8,384 excluding). Both figures are emitted; ot_hours is the
# leave-excluded one, ot_hours_all_paytypes reconciles to the old basis.
#
# WEEK AND MONTH (CHANGED IN v1.4.0): the workweek start is OT_WEEK_START_DOW from
# Cell 3.1 and is THURSDAY, so weeks run Thu -> Wed to match the Paylocity payroll week.
# v1.3.0 used Sun-Sat, an FLSA textbook default that was never checked against the
# payroll system; it split a Thu-Wed payroll week across two reporting weeks, so a
# technician could show overtime in a week payroll never paid it, and none in the week
# payroll did. The threshold applies to the EMPLOYEE, not the warehouse — the statute is
# about the person. A week is attributed wholly to the month containing its LAST day
# (Wednesday, under the current setting), so a week spanning a month boundary lands in
# the later month. Documented simplification: do not expect a day-by-day tie to payroll.
#
# THE OLD BASIS IS RETAINED, not discarded: ot_hours_sunsat travels on the weekly frame
# and the company-level size of the restatement is printed below on every run, so the
# change can be quantified by whoever has to explain a moved number.
#
# ATTRIBUTION: PLC has no usable warehouse key (2 of 139 Location_Name values match
# SERP_WAREHOUSES), so hours reach a site through the technician name match and are
# then apportioned across that technician's warehouses by ticket share — the same rule
# Cell 13 uses for active days. One attribution model for the whole notebook.
#
# SCOPE: warehouse/metro/VP panels use Patient Care Technician only, so OT and
# productivity share one population. Company pages additionally carry an
# all-departments line for total labour cost.
# ─────────────────────────────────────────────────────────────────────────────
def parse_plc(raw):
    """PLC Employee_Name is 'Last, First Middle' -> ('First','Last').
    LIFTED VERBATIM from tech_workload v1.35.0 Cell 13 — re-sync if it changes."""
    if not raw or not isinstance(raw, str):
        return ('', '')
    p = raw.split(',', 1)
    last = p[0].strip().title()
    first = p[1].strip().split()[0].title() if len(p) > 1 and p[1].strip() else ''
    return (first, last)

_hrs = df_hours_raw.copy()
_hrs[['h_first', 'h_last']] = pd.DataFrame(
    _hrs['employee_name'].apply(parse_plc).tolist(), index=_hrs.index)
_hrs['_plc_key'] = _hrs['h_first'].str.strip() + '|' + _hrs['h_last'].str.strip()
_hrs['is_worked'] = _hrs['pay_type'].isin(OT_PAY_TYPES_WORKED)
_hrs['is_leave'] = _hrs['pay_type'].isin(OT_PAY_TYPES_LEAVE)
_unknown_pt = sorted(set(_hrs.loc[~(_hrs['is_worked'] | _hrs['is_leave']), 'pay_type'].unique()))
if _unknown_pt:
    print(f'*** WARNING: pay type(s) {_unknown_pt} are neither in OT_PAY_TYPES_WORKED nor '
          f'OT_PAY_TYPES_LEAVE. Their hours are EXCLUDED from the 40-hour threshold and from '
          f'leave. Classify them in Cell 3.5 — an unclassified pay type silently understates OT.')

# dow_mon0 is Monday=0..Sunday=6, computed deterministically in SQL (immune to
# @@DATEFIRST, which is session-scoped and has bitten this codebase before). Days since
# the configured week start = (dow_mon0 - OT_WEEK_START_DOW) % 7.
_hrs['week_start'] = _hrs['workdate'] - pd.to_timedelta(
    (_hrs['dow_mon0'] - OT_WEEK_START_DOW) % 7, unit='D')
_hrs['week_end'] = _hrs['week_start'] + pd.Timedelta(days=6)
# v1.3.0's Sun-Sat bucketing, kept purely so the restatement can be measured.
_hrs['week_start_sunsat'] = _hrs['workdate'] - pd.to_timedelta(
    (_hrs['dow_mon0'] + 1) % 7, unit='D')
# The boundary is load-bearing for every OT number below, so assert it rather than trust
# the arithmetic: every week must open on the configured payroll weekday.
assert (_hrs['week_start'].dt.dayofweek == OT_WEEK_START_DOW).all(), (
    f'week_start does not land on {OT_WEEK_LABEL[:3]} — check OT_WEEK_START_DOW and dow_mon0')
print(f'Payroll workweek: {OT_WEEK_LABEL} (OT_WEEK_START_DOW={OT_WEEK_START_DOW}); '
      f'productivity week stays {PROD_WEEK_LABEL}.')

def _ot_weekly(df):
    """Employee-week hours -> inferred OT. Threshold applies to the person."""
    w = (df.groupby(['_plc_key', 'h_first', 'h_last', 'dept', 'week_start', 'week_end'],
                    dropna=False, as_index=False)
         .agg(week_hours_all=('hours', 'sum'),
              week_hours_worked=('hours', lambda s: float(s[df.loc[s.index, 'is_worked']].sum())),
              week_hours_leave=('hours', lambda s: float(s[df.loc[s.index, 'is_leave']].sum())),
              days_worked=('workdate', 'nunique')))
    w['ot_hours'] = (w['week_hours_worked'] - OT_WEEKLY_THRESHOLD).clip(lower=0).round(2)
    w['reg_hours'] = w['week_hours_worked'].clip(upper=OT_WEEKLY_THRESHOLD).round(2)
    w['ot_hours_all_paytypes'] = (w['week_hours_all'] - OT_WEEKLY_THRESHOLD).clip(lower=0).round(2)
    w['week_basis'] = OT_WEEK_LABEL
    w['delivery_year'] = w['week_end'].dt.year
    w['delivery_month'] = w['week_end'].dt.month
    w['period'] = (w['delivery_year'].astype(str) + '-'
                   + w['delivery_month'].astype(str).str.zfill(2))
    return w

df_ot_weekly = _ot_weekly(_hrs[_hrs['dept'].isin(OT_DEPTS_TECH)])
df_ot_weekly_all = _ot_weekly(_hrs)

# ── RESTATEMENT: what moving the week boundary did to overtime ───────────────
# Printed, not buried. Someone will be asked why last month's OT number changed.
_tech_worked = _hrs[_hrs['dept'].isin(OT_DEPTS_TECH) & _hrs['is_worked']]

def _ot_total_on(basis_col):
    """Total inferred OT for the technician population under one week-bucketing."""
    _w = _tech_worked.groupby(['_plc_key', basis_col], as_index=False)['hours'].sum()
    return float((_w['hours'] - OT_WEEKLY_THRESHOLD).clip(lower=0).sum())

_ot_new, _ot_old = _ot_total_on('week_start'), _ot_total_on('week_start_sunsat')
_ot_delta_pct = (_ot_new / _ot_old - 1) * 100 if _ot_old else float('nan')
print(f'Week-boundary restatement — {PCT_DEPT} inferred OT over the window:')
print(f'  {OT_WEEK_LABEL} (v1.4.0, payroll-aligned): {_ot_new:>10,.0f} h')
print(f'  Sun-Sat (v1.3.0 basis)                  : {_ot_old:>10,.0f} h'
      f'   -> {_ot_delta_pct:+.1f}% restatement')
print('  Same hours, same threshold — only the week boundary moved. The Thu-Wed figure '
      'is the one that ties to payroll.')

# Edge guard, lifted from v1.35.0: a partial first week can only UNDERstate OT.
_first_wk = df_ot_weekly['week_start'].min()
if pd.notna(_first_wk) and _first_wk < pd.Timestamp(FILTER_START):
    print(f'  OT edge note: the first workweek ({_first_wk.date()}) starts before FILTER_START, '
          f'so its OT is understated (pre-window days were not pulled). Same at the tail: the '
          f'week containing AS_OF_DATE is incomplete.')

# ── v1.5.0: OVERTIME MOVES TO WEEKLY, ON ITS OWN PAYROLL SPINE ───────────────
# Every other metric is bucketed Sun-Sat (WEEK_SPINE). Overtime is NOT, and this is
# deliberate, not an oversight: the 40-hour threshold is applied over the Thu-Wed Paylocity
# week, and re-bucketing the RESULT into Sun-Sat weeks would split a single payroll week's
# overtime across two reporting weeks — the exact defect v1.4.0 fixed when it moved the
# threshold off Sun-Sat. So OT carries OT_WEEK_SPINE, its panels are labelled with the
# payroll week, and the offset (OT weeks open on Thursday, productivity weeks on Sunday) is
# stated wherever the two appear on one page. They are comparable as trends; they are not
# alignable week-for-week, and nobody should try.
OT_WEEK_SPINE = build_week_spine(FILTER_START, AS_OF_DATE, OT_WEEK_START_DOW)
print(f'Overtime spine: {len(OT_WEEK_SPINE)} {OT_WEEK_LABEL} payroll weeks '
      f'(productivity uses {len(WEEK_SPINE)} {PROD_WEEK_LABEL} weeks — offset by '
      f'{(PROD_WEEK_START_DOW - OT_WEEK_START_DOW) % 7} days, not alignable week-for-week).')

_OT_COUNTS = ['total_hours', 'worked_hours', 'leave_hours', 'ot_hours',
              'ot_hours_all_paytypes', 'reg_hours']
_OT_RATES = ['ot_pct_of_worked', 'ot_hours_per_employee', 'ot_hours_per_tech',
             'leave_pct_of_total', 'pct_employees_with_ot', 'employees', 'technicians']


def _ot_weekly_company(w, label):
    """Company / all-department weekly overtime. No site attribution — this is the labour
    cost lens, and unattributable hours belong in it."""
    k = ['week_start', 'week_end']
    m = (w.groupby(k, as_index=False)
         .agg(total_hours=('week_hours_all', 'sum'), worked_hours=('week_hours_worked', 'sum'),
              leave_hours=('week_hours_leave', 'sum'), ot_hours=('ot_hours', 'sum'),
              reg_hours=('reg_hours', 'sum'),
              ot_hours_all_paytypes=('ot_hours_all_paytypes', 'sum'),
              employees=('_plc_key', 'nunique')))
    # employees_with_ot is counted on the EMPLOYEE. At weekly grain each employee appears
    # once per week anyway, but the pattern is kept so the monthly bridge below cannot
    # reintroduce the v1.3.0 bug where week-rows were counted and the figure exceeded
    # the headcount sitting next to it.
    _with_ot = (w[w['ot_hours'] > 0].groupby(k, as_index=False)
                .agg(employees_with_ot=('_plc_key', 'nunique')))
    m = m.merge(_with_ot, on=k, how='left')
    m['employees_with_ot'] = m['employees_with_ot'].fillna(0).astype(int)
    m['pct_employees_with_ot'] = (m['employees_with_ot']
                                  / m['employees'].replace(0, np.nan) * 100).round(1)
    m['ot_pct_of_worked'] = (m['ot_hours'] / m['worked_hours'].replace(0, np.nan) * 100).round(2)
    m['ot_hours_per_employee'] = (m['ot_hours'] / m['employees'].replace(0, np.nan)).round(2)
    m['leave_pct_of_total'] = (m['leave_hours'] / m['total_hours'].replace(0, np.nan) * 100).round(2)
    m['ot_overstatement_pct'] = ((m['ot_hours_all_paytypes'] / m['ot_hours'].replace(0, np.nan) - 1)
                                 * 100).round(1)
    m['scope'] = label
    m['week_basis'] = OT_WEEK_LABEL
    for c in _OT_COUNTS:
        m[c] = m[c].round(1)
    m = attach_spine(m, spine=OT_WEEK_SPINE)
    m = add_trailing(m, [], count_cols=_OT_COUNTS, rate_cols=_OT_RATES, spine=OT_WEEK_SPINE)
    _s = TRAILING_SUFFIX
    # Pooled, not a mean of weekly percentages: a light week must not weigh as much as a
    # heavy one when the question is "what share of hours were overtime lately?".
    m['ot_pct_of_worked' + _s] = (m['ot_hours' + _s]
        / m['worked_hours' + _s].replace(0, np.nan) * 100).round(2)
    m['ot_hours_per_employee' + _s] = (m['ot_hours' + _s]
        / m['employees' + _s].replace(0, np.nan)).round(2)
    return m.sort_values('week_start').reset_index(drop=True)


def _ot_monthly_company(w, label):
    """Monthly bridge to the v1.3.0/v1.4.0 packs. A week is attributed wholly to the month
    of its LAST day, so a week spanning a month boundary lands in the later month —
    documented simplification, do not expect a day-by-day tie to payroll."""
    _mk = ['delivery_year', 'delivery_month', 'period']
    m = (w.groupby(_mk, as_index=False)
         .agg(total_hours=('week_hours_all', 'sum'), worked_hours=('week_hours_worked', 'sum'),
              leave_hours=('week_hours_leave', 'sum'), ot_hours=('ot_hours', 'sum'),
              ot_hours_all_paytypes=('ot_hours_all_paytypes', 'sum'),
              employees=('_plc_key', 'nunique')))
    _with_ot = (w[w['ot_hours'] > 0].groupby(_mk, as_index=False)
                .agg(employees_with_ot=('_plc_key', 'nunique')))
    m = m.merge(_with_ot, on=_mk, how='left')
    m['employees_with_ot'] = m['employees_with_ot'].fillna(0).astype(int)
    m['pct_employees_with_ot'] = (m['employees_with_ot']
                                  / m['employees'].replace(0, np.nan) * 100).round(1)
    m['ot_pct_of_worked'] = (m['ot_hours'] / m['worked_hours'].replace(0, np.nan) * 100).round(2)
    m['ot_hours_per_employee'] = (m['ot_hours'] / m['employees'].replace(0, np.nan)).round(2)
    m['leave_pct_of_total'] = (m['leave_hours'] / m['total_hours'].replace(0, np.nan) * 100).round(2)
    m['ot_overstatement_pct'] = ((m['ot_hours_all_paytypes'] / m['ot_hours'].replace(0, np.nan) - 1)
                                 * 100).round(1)
    m['scope'] = label
    for c in ('total_hours', 'worked_hours', 'leave_hours', 'ot_hours', 'ot_hours_all_paytypes'):
        m[c] = m[c].round(1)
    return m.sort_values(['delivery_year', 'delivery_month']).reset_index(drop=True)


dash_wk_ot_co      = _ot_weekly_company(df_ot_weekly, PCT_DEPT)
dash_wk_ot_alldept = _ot_weekly_company(df_ot_weekly_all, 'All payroll departments')
dash_mo_ot_co      = _ot_monthly_company(df_ot_weekly, PCT_DEPT)
dash_mo_ot_alldept = _ot_monthly_company(df_ot_weekly_all, 'All payroll departments')

tbl_ot_by_dept = (df_ot_weekly_all.groupby(['dept', 'week_start'], as_index=False)
    .agg(total_hours=('week_hours_all', 'sum'), worked_hours=('week_hours_worked', 'sum'),
         ot_hours=('ot_hours', 'sum'), employees=('_plc_key', 'nunique')))
tbl_ot_by_dept['ot_pct_of_worked'] = (tbl_ot_by_dept['ot_hours']
    / tbl_ot_by_dept['worked_hours'].replace(0, np.nan) * 100).round(2)
for _c in ('total_hours', 'worked_hours', 'ot_hours'):
    tbl_ot_by_dept[_c] = tbl_ot_by_dept[_c].round(1)
tbl_ot_by_dept = add_trailing(tbl_ot_by_dept, ['dept'],
                              count_cols=['ot_hours', 'worked_hours'],
                              rate_cols=['ot_pct_of_worked'], spine=OT_WEEK_SPINE)

# ── site attribution: PLC person -> ticket technician -> warehouse ───────────
# Reuses the reviewed matcher (exact -> nickname -> fuzzy on shared last name) rather
# than inventing a second one. Techs are keyed on the resolved ticket name so v1.2.0's
# internal-ops corrections carry through.
_plc_idx = {}
_plc_by_last = defaultdict(list)
# NOTE: itertuples() mangles column names that start with an underscore into positional
# aliases (_1, _2, ...), so '_plc_key' is renamed before iterating rather than accessed
# as an attribute — a silent AttributeError trap.
_plc_people = (_hrs[_hrs['dept'].isin(OT_DEPTS_TECH)][['h_first', 'h_last', '_plc_key']]
               .drop_duplicates().rename(columns={'_plc_key': 'plckey'}))
for r in _plc_people.itertuples(index=False):
    fn, ln = _clean(r.h_first), _clean(r.h_last)
    if fn and ln:
        _plc_idx[(fn, ln)] = r.plckey
        _plc_by_last[ln].append((fn, r.plckey))

_tech_names = (df_tx[~df_tx['_unattributed']][['techfirstname', 'techlastname']]
               .drop_duplicates())
_ot_map_rows = []
for row in _tech_names.itertuples(index=False):
    fn, ln = row.techfirstname, row.techlastname
    ln_c = _clean(ln)
    matched = _plc_idx.get((_clean(fn), ln_c))
    if matched is None:
        for cand in standardize_first_name(fn):
            matched = _plc_idx.get((_clean(cand), ln_c))
            if matched is not None:
                break
    if matched is None:
        best, best_d = None, FUZZY_EDIT_DIST + 1
        for (i_fn, key) in _plc_by_last.get(ln_c, []):
            d = levenshtein(_clean(fn), i_fn)
            if d < best_d:
                best, best_d = key, d
        if best is not None and best_d <= FUZZY_EDIT_DIST:
            matched = best
    if matched is not None:
        _ot_map_rows.append({'techfirstname': fn, 'techlastname': ln, '_plc_key': matched})
df_ot_map = pd.DataFrame(_ot_map_rows, columns=['techfirstname', 'techlastname', '_plc_key'])
_n_plc_people = _hrs[_hrs['dept'].isin(OT_DEPTS_TECH)]['_plc_key'].nunique()
_n_mapped = df_ot_map['_plc_key'].nunique()
print(f'Payroll->ticket match: {_n_mapped:,} of {_n_plc_people:,} {PCT_DEPT} payroll people '
      f'({_n_mapped / max(_n_plc_people, 1) * 100:.1f}%) tie to an attributed technician.')
_unmapped_hours = float(df_ot_weekly.loc[~df_ot_weekly['_plc_key'].isin(set(df_ot_map['_plc_key'])),
                                         'ot_hours'].sum())
_all_ot = float(df_ot_weekly['ot_hours'].sum())
print(f'  OT hours that CANNOT be attributed to a site: {_unmapped_hours:,.0f} of {_all_ot:,.0f} '
      f'({_unmapped_hours / max(_all_ot, 1) * 100:.1f}%). They stay in the company total — the '
      f'company row is the reconciliation truth; site rows sum to less by this amount.')

# ── v1.5.0: ticket share is computed per PAYROLL WEEK, not per month ─────────
# v1.4.0 apportioned a technician's monthly OT across sites by their MONTHLY ticket share.
# At weekly grain that is wrong twice over: it smears a week's overtime across whichever
# sites the technician touched all month, and it cannot produce a weekly site figure at all.
# Ticket share is now measured over the same Thu-Wed span the overtime was computed on, so
# the hours land where the technician actually was that week.
_txa_ot = df_tx[~df_tx['_unattributed']].copy()
_txa_ot['week_start'] = week_start_of(_txa_ot['completed_date'], OT_WEEK_START_DOW)
_tick_wk = (_txa_ot.groupby(['techfirstname', 'techlastname', 'tech_warehouse', 'vp', 'state',
                             'metro', 'week_start'], dropna=False, as_index=False)
            .agg(_tickets=('order_num', 'nunique')))
_tick_wk['_tech_week_tickets'] = (_tick_wk.groupby(['techfirstname', 'techlastname', 'week_start'])
                                  ['_tickets'].transform('sum'))
_tick_wk['_share'] = np.where(OT_APPORTION_BY_TICKET_SHARE,
                              _tick_wk['_tickets'] / _tick_wk['_tech_week_tickets'].replace(0, np.nan),
                              1.0)

_OT_SITE_VALS = ['ot_hours', 'worked_hours', 'total_hours', 'leave_hours',
                 'ot_hours_all_paytypes']
_ot_tech_wk = (df_ot_weekly.groupby(['_plc_key', 'week_start'], as_index=False)
               .agg(ot_hours=('ot_hours', 'sum'), worked_hours=('week_hours_worked', 'sum'),
                    total_hours=('week_hours_all', 'sum'), leave_hours=('week_hours_leave', 'sum'),
                    ot_hours_all_paytypes=('ot_hours_all_paytypes', 'sum')))
_ot_site = (_ot_tech_wk.merge(df_ot_map, on='_plc_key', how='inner')
            .merge(_tick_wk, on=['techfirstname', 'techlastname', 'week_start'], how='inner'))
for _c in _OT_SITE_VALS:
    _ot_site[_c] = _ot_site[_c] * _ot_site['_share']


def _dash_ot_weekly(gcols):
    gcols = list(gcols)
    keys = gcols + ['week_start']
    agg = (_ot_site.groupby(keys, dropna=False, as_index=False)
        .agg(ot_hours=('ot_hours', 'sum'), worked_hours=('worked_hours', 'sum'),
             total_hours=('total_hours', 'sum'), leave_hours=('leave_hours', 'sum'),
             ot_hours_all_paytypes=('ot_hours_all_paytypes', 'sum'),
             technicians=('_plc_key', 'nunique')))
    agg['ot_pct_of_worked'] = (agg['ot_hours'] / agg['worked_hours'].replace(0, np.nan) * 100).round(2)
    agg['ot_hours_per_tech'] = (agg['ot_hours'] / agg['technicians'].replace(0, np.nan)).round(2)
    agg['leave_pct_of_total'] = (agg['leave_hours'] / agg['total_hours'].replace(0, np.nan) * 100).round(2)
    for c in _OT_SITE_VALS:
        agg[c] = agg[c].round(1)
    agg['week_basis'] = OT_WEEK_LABEL
    agg = attach_spine(agg, spine=OT_WEEK_SPINE)
    agg = add_trailing(agg, gcols, count_cols=_OT_SITE_VALS,
                       rate_cols=['ot_pct_of_worked', 'ot_hours_per_tech', 'leave_pct_of_total',
                                  'technicians'],
                       spine=OT_WEEK_SPINE)
    _s = TRAILING_SUFFIX
    agg['ot_pct_of_worked' + _s] = (agg['ot_hours' + _s]
        / agg['worked_hours' + _s].replace(0, np.nan) * 100).round(2)
    agg['ot_hours_per_tech' + _s] = (agg['ot_hours' + _s]
        / agg['technicians' + _s].replace(0, np.nan)).round(2)
    return agg.sort_values(gcols + ['week_start']).reset_index(drop=True)


def _dash_ot_monthly(gcols):
    """Monthly bridge. Uses the same weekly site frame, rolled up by the month of the
    payroll week's end date, so it reconciles to the weekly sheets exactly."""
    gcols = list(gcols)
    _s = _ot_site.copy()
    _s['period'] = (_s['week_start'] + pd.Timedelta(days=6)).dt.strftime('%Y-%m')
    keys = gcols + ['period']
    agg = (_s.groupby(keys, dropna=False, as_index=False)
        .agg(ot_hours=('ot_hours', 'sum'), worked_hours=('worked_hours', 'sum'),
             total_hours=('total_hours', 'sum'), leave_hours=('leave_hours', 'sum'),
             ot_hours_all_paytypes=('ot_hours_all_paytypes', 'sum'),
             technicians=('_plc_key', 'nunique')))
    agg['ot_pct_of_worked'] = (agg['ot_hours'] / agg['worked_hours'].replace(0, np.nan) * 100).round(2)
    agg['ot_hours_per_tech'] = (agg['ot_hours'] / agg['technicians'].replace(0, np.nan)).round(2)
    agg['leave_pct_of_total'] = (agg['leave_hours'] / agg['total_hours'].replace(0, np.nan) * 100).round(2)
    for c in _OT_SITE_VALS:
        agg[c] = agg[c].round(1)
    agg[['delivery_year', 'delivery_month']] = agg['period'].str.split('-', expand=True).astype(int)
    return agg.sort_values(gcols + ['period']).reset_index(drop=True)


dash_wk_ot_wh    = _dash_ot_weekly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_wk_ot_metro = _dash_ot_weekly(['metro'])
dash_wk_ot_vp    = _dash_ot_weekly(['vp'])
dash_mo_ot_wh    = _dash_ot_monthly(['tech_warehouse', 'vp', 'state', 'metro'])
dash_mo_ot_metro = _dash_ot_monthly(['metro'])
dash_mo_ot_vp    = _dash_ot_monthly(['vp'])
print(f'Weekly OT: wh={len(dash_wk_ot_wh):,} metro={len(dash_wk_ot_metro):,} '
      f'vp={len(dash_wk_ot_vp):,} co={len(dash_wk_ot_co):,} alldept={len(dash_wk_ot_alldept):,}')

# ── Reconciliation: apportionment must not create or destroy hours ────────────
# One legitimate leak: a technician on payroll in a week who ran no attributed tickets that
# week has no site for their hours to land on. Rather than warn and leave the reader to work
# out whether that explains the gap, quantify it and only escalate if it does NOT.
_matched_keys = set(df_ot_map['_plc_key'])
_site_ot = float(dash_wk_ot_wh['ot_hours'].sum())
_attr_ot = float(_ot_tech_wk[_ot_tech_wk['_plc_key'].isin(_matched_keys)]['ot_hours'].sum())
_delta = _site_ot - _attr_ot
_ticket_tech_weeks = set(map(tuple, _tick_wk.merge(
    df_ot_map, on=['techfirstname', 'techlastname'], how='inner')[['_plc_key', 'week_start']].values))
_no_ticket_week = _ot_tech_wk[_ot_tech_wk['_plc_key'].isin(_matched_keys)
    & ~_ot_tech_wk[['_plc_key', 'week_start']].apply(tuple, axis=1).isin(_ticket_tech_weeks)]
_explained = float(_no_ticket_week['ot_hours'].sum())
print(f'Apportionment check — site-level OT {_site_ot:,.0f} vs attributable OT {_attr_ot:,.0f} '
      f'(delta {_delta:+,.0f}).')
print(f'  Explained by {len(_no_ticket_week):,} (technician, payroll week) pairs on payroll with '
      f'no attributed tickets that week: {_explained:,.0f} OT hours. Those have no site to land '
      f'on and stay in the company total only.')
print('  NOTE: this gap is LARGER at weekly grain than the monthly figure v1.4.0 printed, and '
      'that is expected — a technician with no tickets in one week may still have had tickets '
      'that month. It is a coarser attribution being replaced by a truer one, not a regression.')
_unexplained = abs(_delta) - _explained
if _unexplained > max(1.0, 0.005 * max(_attr_ot, 1)):
    print(f'  *** WARNING: {_unexplained:,.0f} OT hours of the gap are NOT explained by that '
          f'population. Apportionment is losing or duplicating hours somewhere — investigate '
          f'before publishing OT dollars.')
else:
    print('  Fully explained — apportionment neither creates nor destroys hours.')

# ── the diagnostic that keeps the recorded-OT columns retired ─────────────────
_diag = (_hrs.assign(_wk=week_start_of(_hrs['workdate'], OT_WEEK_START_DOW))
         .groupby('_wk')
         .agg(hours_daily_trusted=('hours', 'sum'),
              reg_hrs_column=('reg_hrs_unusable', 'sum'),
              ot1_hrs_column=('ot1_hrs_unusable', 'sum')))
_diag['reg_col_vs_hours_x'] = (_diag['reg_hrs_column']
                               / _diag['hours_daily_trusted'].replace(0, np.nan)).round(1)
tbl_ot_feed_diagnostic = _diag.reset_index().rename(columns={'_wk': 'week_start'})
print('\nWhy the recorded OT columns are not used (Reg_Hrs vs the daily Hours we trust, '
      'last 8 payroll weeks):')
print(tbl_ot_feed_diagnostic.tail(8).round(0).to_string(index=False))
print('  Reg_Hrs runs an order of magnitude above summed daily Hours (pay-period values '
      'repeated per row) and OT1_Hrs collapses to 0 from 2026-02 on. Inference from '
      '[Hours] is the only defensible route until the feed is fixed.')
print(f'\nCompany OT by {OT_WEEK_LABEL} payroll week, last 8 weeks:')
print(dash_wk_ot_co[['week_label', 'worked_hours', 'leave_hours', 'ot_hours',
                     'ot_hours' + TRAILING_SUFFIX, 'ot_pct_of_worked',
                     'ot_pct_of_worked' + TRAILING_SUFFIX, 'employees',
                     'is_partial_week']].tail(8).to_string(index=False))
print('\nCompany OT monthly bridge (leave-excluded vs the old all-pay-types basis):')
print(dash_mo_ot_co[['period', 'worked_hours', 'leave_hours', 'ot_hours',
                     'ot_hours_all_paytypes', 'ot_overstatement_pct', 'ot_pct_of_worked',
                     'employees']].to_string(index=False))

## Cell 16 — STOCK OUTS: INCIDENCE, TIME-TO-FULFIL & REAL BACKLOG *(weekly in v1.5.0)*

Weekly + trailing 4-week by **creation** week. Outcome columns at the right-hand edge are censored — an order created last week has had no time to be fulfilled — so the trailing average inherits that censoring rather than correcting it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STOCK OUTS — INCIDENCE, TIME-TO-FULFIL, AND THE REAL BACKLOG   rewritten v1.3.0
#
# WHAT CHANGED AND WHY. v1.1.0 read Status='EnRoute' as the open backlog and trended it
# as an aging view, with the caveat that fulfilled stock-outs "disappear from this query
# entirely". Both the metric and the caveat were wrong:
#   * 19,891 of the 21,438 EnRoute orders carry a completion date, so rows ARE retained
#     after fulfilment — nothing disappears. BUT 6,035 of those completions sit only on a
#     CANCELED ticket, so genuine fulfilment is 13,856 (64.7% of in-window orders), not
#     93%. Treating a canceled ticket as a delivery overstates fulfilment by ~28 points,
#     which is exactly the error the three-way outcome below exists to prevent.
#   * Every other status value is frozen at a single 2026-01-20 load (Completed,
#     Approval Needed, Received, Next Stop, Pending Review all created that day;
#     Canceled and Reconciled stop there too). Status is simply not maintained.
#   * So "oldest open 1,021 days" was a stale row, not an aging crisis.
# Therefore EnRoute BY CREATION MONTH is true stock-out incidence, time-to-fulfil comes
# from the ticket join (median 6 days, P75 9, P90 17), and orders resolve three ways:
# fulfilled 64.7% / canceled-abandoned 28.2% / genuinely open 7.1%. That last group is the
# real backlog — 1,518 orders, median 115 days old.
#
# ATTRIBUTION. v1.1.0 stated no physical warehouse exists on stock-out rows, and showed
# an n/a notice on every warehouse and metro page. It does exist: 5,726 EnRoute rows
# carry one of 65 real warehouses, and the order join resolves 76.0% of orders to a real
# site. Resolution ladder: order join on the analysis ticket set -> order join widened to
# tickets under any reason -> the row's own non-virtual Warehouse -> Territory when it
# exactly matches a known warehouse -> unresolved. Unresolved orders are labelled and
# still counted at company level, so the company row is the reconciliation truth.
#
# PII (CLAUDE.md rule 2): the table carries Patient Firstname/Lastname, Caregiver names,
# WorkPhone, MobilePhone, Email and free-text Order Notes. None are selected. Ship To
# Address is pulled only to extract a two-letter state and is dropped immediately.
# ─────────────────────────────────────────────────────────────────────────────
_so_raw = run_query(f"""
SELECT [Order]           AS so_order,
       [Order Type]      AS order_type,
       [Warehouse]       AS holding_warehouse,
       [Territory]       AS territory,
       [Creation Date]   AS creation_date_raw,
       [Status]          AS status,
       [Priority]        AS priority,
       ProductName       AS product,
       Quantity          AS qty,
       [Ship To Address] AS _ship_addr,
       [Reason For]      AS reason_for
FROM {STOCKOUT_TABLE} WITH (NOLOCK)
""", 'Stock outs (all statuses)')

dash_mo_stockout_wh = pd.DataFrame(); dash_mo_stockout_metro = pd.DataFrame()
dash_mo_stockout_vp = pd.DataFrame(); dash_mo_stockout_co = pd.DataFrame()
dash_mo_stockout_state = pd.DataFrame(); dash_stockout_products = pd.DataFrame()
dash_wk_stockout_wh = pd.DataFrame(); dash_wk_stockout_metro = pd.DataFrame()
dash_wk_stockout_vp = pd.DataFrame(); dash_wk_stockout_co = pd.DataFrame()
dash_wk_stockout_state = pd.DataFrame()
dash_stockout_backlog = pd.DataFrame(); dash_stockout_canceled = pd.DataFrame()
tbl_stockout_status_recon = pd.DataFrame()
tbl_stockout_fulfil = pd.DataFrame()

if len(_so_raw) == 0:
    print('Stock outs: query returned 0 rows — table empty or renamed.')
else:
    _so = _so_raw.copy()
    # Creation Date is MM/DD/YYYY VARCHAR — parsed Python-side (locked lesson).
    _so['creation_date'] = pd.to_datetime(_so['creation_date_raw'], format='%m/%d/%Y', errors='coerce')
    _retry = _so['creation_date'].isna()
    if _retry.any():
        _so.loc[_retry, 'creation_date'] = pd.to_datetime(_so.loc[_retry, 'creation_date_raw'],
                                                          errors='coerce')
    _n_bad = int(_so['creation_date'].isna().sum())
    if _n_bad:
        print(f'  *** {_n_bad:,} rows with unparseable Creation Date — excluded from trend, '
              f'listed in the backlog sheet with a blank date.')
    _so['state'] = _so['_ship_addr'].astype(str).str.extract(r',\s*([A-Za-z]{2})\s*,\s*\d{5}')[0].str.upper()
    _so = _so.drop(columns=['_ship_addr'])          # dropped immediately, before any grouping
    _so['state'] = _so['state'].fillna('Unknown')
    _so['qty'] = pd.to_numeric(_so['qty'], errors='coerce').fillna(1).astype(int)
    _so['so_order'] = _so['so_order'].astype(str).str.strip()
    _so['holding_warehouse'] = _so['holding_warehouse'].fillna('').astype(str).str.strip()
    _so['territory'] = _so['territory'].fillna('').astype(str).str.strip()

    # Status reconciliation: show the reader that only EnRoute is maintained.
    tbl_stockout_status_recon = (_so.groupby('status', dropna=False, as_index=False)
        .agg(rows=('so_order', 'size'), orders=('so_order', 'nunique'),
             min_created=('creation_date', 'min'), max_created=('creation_date', 'max')))
    tbl_stockout_status_recon['is_event_log'] = tbl_stockout_status_recon['status'].eq(STOCKOUT_EVENT_STATUS)
    _frozen = tbl_stockout_status_recon[~tbl_stockout_status_recon['is_event_log']]
    print(f'\nStatus universe — only {STOCKOUT_EVENT_STATUS} is maintained:')
    print(tbl_stockout_status_recon.to_string(index=False))
    if len(_frozen) and _frozen['max_created'].max() is not pd.NaT:
        print(f'  Non-{STOCKOUT_EVENT_STATUS} statuses stop at '
              f'{pd.to_datetime(_frozen["max_created"]).max()} — a frozen load, not a live '
              f'workflow. They are reported here for reconciliation and excluded from trends.')

    _ev_all = _so[_so['status'].eq(STOCKOUT_EVENT_STATUS) & _so['creation_date'].notna()].copy()
    # Respect the dashboard window. Every other panel is bounded by FILTER_START/END; the
    # stock-out feed reaches back to 2023-10 with 1-6 orders a month, and leaving those in
    # both breaks comparability and squeezes the informative range off the chart. The
    # pre-window volume is reported rather than silently dropped.
    _pre = _ev_all[_ev_all['creation_date'] < pd.Timestamp(FILTER_START)]
    _ev = _ev_all[(_ev_all['creation_date'] >= pd.Timestamp(FILTER_START))
                  & (_ev_all['creation_date'] <= pd.Timestamp(FILTER_END))].copy()
    if len(_pre):
        print(f'  {_pre["so_order"].nunique():,} stock-out orders created BEFORE {FILTER_START} '
              f'({_pre["creation_date"].min().date()} onward) are outside the dashboard window '
              f'and excluded from every panel, matching the other metrics.')

    # ── fulfilment lookup, UNFILTERED BY REASON ──────────────────────────────
    # order_wh_map is built from df_tx, which is filtered to INCLUDED_REASONS. Measuring
    # fulfilment against it understates badly: 1,165 stock-out orders sit under
    # 'Priority 4 - System Update/Correction (D)' alone, a reason the analysis whitelist
    # deliberately excludes, and their delivery would look like an open backlog item
    # forever. Fulfilment is a question about the ORDER, not about the analysis
    # population, so it is asked of the ticket feed with no reason filter at all.
    # The EXISTS clause restricts this server-side to stock-out orders, so it returns
    # ~21k rows instead of the whole ticket table.
    _so_tx = run_query(f"""
SELECT TRIM(TX.Order_Num) AS so_order,
       -- fulfilment must come from a LIVE ticket: a canceled ticket carries a
       -- Completed_Date too, and counting it inflates fulfilment by ~28 points.
       MAX(CASE WHEN TX.[Status] <> 'Canceled'
                THEN TRY_CONVERT(DATE, TX.Completed_Date) END) AS tx_completed_date,
       MAX(CASE WHEN TX.[Status]  = 'Canceled'
                THEN TRY_CONVERT(DATE, TX.Completed_Date) END) AS tx_canceled_date,
       SUM(CASE WHEN TX.[Status]  = 'Canceled' THEN 1 ELSE 0 END) AS tx_canceled_tickets,
       COUNT(*) AS tx_tickets,
       -- attribution takes any status: a canceled ticket still tells us which site held it
       MAX(CASE WHEN TRIM(ISNULL(TX.Tech_Warehouse,'')) NOT LIKE '{VIRTUAL_WH_PREFIX}%'
                 AND TRIM(ISNULL(TX.Tech_Warehouse,'')) <> ''
                THEN TRIM(TX.Tech_Warehouse) END) AS tx_any_real_wh
FROM dbo.[SERP TRANSACTIONS] AS TX WITH (NOLOCK)
WHERE TX.Order_Num <> ''
  AND EXISTS (SELECT 1 FROM {STOCKOUT_TABLE} AS SO WITH (NOLOCK)
              WHERE TRIM(SO.[Order]) = TRIM(TX.Order_Num))
GROUP BY TRIM(TX.Order_Num)
""", 'Stock-out fulfilment lookup (all reasons, status-aware)')
    for _c in ('tx_completed_date', 'tx_canceled_date'):
        _so_tx[_c] = pd.to_datetime(_so_tx[_c], errors='coerce')
    for _c in ('tx_canceled_tickets', 'tx_tickets'):
        _so_tx[_c] = pd.to_numeric(_so_tx[_c], errors='coerce').fillna(0).astype(int)
    _so_tx = _so_tx.drop_duplicates('so_order').set_index('so_order')

    # ── site resolution ──────────────────────────────────────────────────────
    _known_wh = set(df_hier['warehouse'].dropna().astype(str).str.strip())
    _omap = order_wh_map.set_index('order_num')
    _ev['ord_warehouse'] = _ev['so_order'].map(_omap['ord_warehouse'])
    _ev['tx_any_real_wh'] = _ev['so_order'].map(_so_tx['tx_any_real_wh'])
    _ev['ord_completed_date'] = _ev['so_order'].map(_so_tx['tx_completed_date'])
    _ev['tech_warehouse'] = _ev['ord_warehouse']
    _ev['_so_resolution'] = np.where(_ev['tech_warehouse'].notna(), 'order_join', 'unresolved')
    # layer 2: any real warehouse seen on the order, including on tickets whose reason is
    # outside the analysis whitelist — same evidence, wider net.
    _need = _ev['_so_resolution'].eq('unresolved') & _ev['tx_any_real_wh'].notna()
    _ev.loc[_need, 'tech_warehouse'] = _ev.loc[_need, 'tx_any_real_wh']
    _ev.loc[_need, '_so_resolution'] = 'order_join_all_reasons'
    _need = _ev['_so_resolution'].eq('unresolved')
    _hw = _ev['holding_warehouse']
    _hw_is_real = ~_hw.str.upper().str.startswith(VIRTUAL_WH_PREFIX.upper()) & _hw.isin(_known_wh)
    _own_ok = _need & _hw_is_real
    _ev.loc[_own_ok, 'tech_warehouse'] = _ev.loc[_own_ok, 'holding_warehouse']
    _ev.loc[_own_ok, '_so_resolution'] = 'own_warehouse'
    _need = _ev['_so_resolution'].eq('unresolved')
    _terr_ok = _need & _ev['territory'].isin(_known_wh)
    _ev.loc[_terr_ok, 'tech_warehouse'] = _ev.loc[_terr_ok, 'territory']
    _ev.loc[_terr_ok, '_so_resolution'] = 'territory'
    _ev['tech_warehouse'] = _ev['tech_warehouse'].fillna(VIRTUAL_UNRESOLVED_LABEL)

    # drop_duplicates guards the merge: df_hier is SELECT DISTINCT across four columns,
    # so one warehouse carrying two state spellings would fan every stock-out row out.
    _ev = _ev.merge(df_hier.rename(columns={'warehouse': 'tech_warehouse'})[
        ['tech_warehouse', 'vp', 'state']].rename(columns={'state': '_wh_state'})
        .drop_duplicates('tech_warehouse'), on='tech_warehouse', how='left')
    _ev['state'] = np.where(_ev['_wh_state'].notna() & (_ev['_wh_state'].astype(str).str.strip() != ''),
                            _ev['_wh_state'], _ev['state'])
    _ev = _ev.drop(columns=['_wh_state'])
    _ev['metro'] = _ev['tech_warehouse'].apply(assign_metro)

    _order_res = _ev.drop_duplicates('so_order')['_so_resolution'].value_counts()
    _cov = 1 - (_order_res.get('unresolved', 0) / max(_ev['so_order'].nunique(), 1))
    print(f'\nSite attribution: {_order_res.to_dict()}  -> {_cov * 100:.1f}% of orders resolved '
          f'to a physical warehouse.')
    if _cov < STOCKOUT_ATTRIBUTION_MIN_COVERAGE:
        print(f'  *** WARNING: coverage below {STOCKOUT_ATTRIBUTION_MIN_COVERAGE:.0%}. Warehouse and '
              f'metro stock-out pages are built on a minority of orders — read company/state instead.')

    # ── fulfilment ───────────────────────────────────────────────────────────
    _ev['days_to_fulfil'] = (_ev['ord_completed_date'] - _ev['creation_date']).dt.days
    # A negative value means the ticket completed before the stock-out row was created —
    # a data-entry artifact, not a negative lead time. Excluded from the distribution.
    _neg = int((_ev['days_to_fulfil'] < 0).sum())
    if _neg:
        print(f'  {_neg:,} rows have a completion date BEFORE the stock-out creation date; '
              f'excluded from time-to-fulfil (kept in incidence counts).')
    _ev.loc[_ev['days_to_fulfil'] < 0, 'days_to_fulfil'] = np.nan

    # ── THREE-WAY OUTCOME, not a binary ──────────────────────────────────────
    # A stock-out order ends one of three ways, and collapsing them to
    # "fulfilled / still open" hides the most actionable group:
    #   fulfilled — a live (Reconciled/Completed) ticket carries a completion date
    #   canceled  — the only completion evidence sits on a CANCELED ticket, i.e. the
    #               order was abandoned rather than delivered. 28% of orders land here,
    #               and counting them as fulfilled is what produced a false 93% earlier.
    #   open      — no completion evidence at all: a pending ticket, or no ticket. This
    #               is the genuine backlog.
    _ev['is_fulfilled'] = _ev['ord_completed_date'].notna()
    _ev['tx_canceled_tickets'] = _ev['so_order'].map(_so_tx['tx_canceled_tickets']).fillna(0)
    _ev['is_canceled'] = ~_ev['is_fulfilled'] & (_ev['tx_canceled_tickets'] > 0)
    _ev['is_open'] = ~_ev['is_fulfilled'] & ~_ev['is_canceled']
    _ev['outcome'] = np.select([_ev['is_fulfilled'], _ev['is_canceled']],
                               ['fulfilled', 'canceled'], default='open')
    _ev['age_days'] = (pd.Timestamp(AS_OF_DATE) - _ev['creation_date']).dt.days
    _ev['so_year'] = _ev['creation_date'].dt.year.astype(int)
    _ev['so_month'] = _ev['creation_date'].dt.month.astype(int)
    _ev['period'] = _ev['so_year'].astype(str) + '-' + _ev['so_month'].astype(str).str.zfill(2)

    _ord = _ev.drop_duplicates('so_order')
    print(f'\nStock-out EVENTS (Status={STOCKOUT_EVENT_STATUS}, by creation date): '
          f'{_ord["so_order"].nunique():,} orders / {int(_ev["qty"].sum()):,} items, '
          f'{_ev["creation_date"].min().date()} -> {_ev["creation_date"].max().date()}')
    print(f'  Outcome split — fulfilled {int(_ord["is_fulfilled"].sum()):,} '
          f'({_ord["is_fulfilled"].mean() * 100:.1f}%) | '
          f'canceled/abandoned {int(_ord["is_canceled"].sum()):,} '
          f'({_ord["is_canceled"].mean() * 100:.1f}%) | '
          f'genuinely open {int(_ord["is_open"].sum()):,} ({_ord["is_open"].mean() * 100:.1f}%)')
    print('  NOTE: "canceled" means the order\'s only completion evidence is on a CANCELED '
          'ticket — abandoned, not delivered. Counting those as fulfilled overstates '
          'fulfilment by ~28 points, so they are reported as their own outcome.')
    _d = _ord.loc[_ord['days_to_fulfil'].notna(), 'days_to_fulfil']
    if len(_d):
        print(f'  Days to fulfil (fulfilled only) — P25 {_d.quantile(.25):.0f} | '
              f'median {_d.median():.0f} | P75 {_d.quantile(.75):.0f} | P90 {_d.quantile(.90):.0f}')
    _bl = _ord[_ord['is_open']]
    if len(_bl):
        print(f'  Real backlog age (days open) — P25 {_bl["age_days"].quantile(.25):.0f} | '
              f'median {_bl["age_days"].median():.0f} | P75 {_bl["age_days"].quantile(.75):.0f} | '
              f'max {_bl["age_days"].max():.0f}')

    # ── WEEKLY incidence by entity (v1.5.0; monthly retained as the bridge) ──
    # Week = stock-out CREATION week, so the series measures when supply failed, not when
    # it was eventually fixed. Two consequences to repeat wherever this is read:
    #   * the most recent weeks are CENSORED on every outcome column — an order created
    #     last Tuesday has had ~6 days against a median 6-day fulfilment, so fulfilled_pct
    #     is structurally low and open_pct structurally high at the right-hand edge. The
    #     trailing average inherits that; it does not correct it.
    #   * median_days_to_fulfil on a week with four orders is not a median. The line is
    #     suppressed below the denominator floor on the panel, and the trailing version is
    #     pooled over four weeks precisely so there is something legible to plot.
    _ev['week_start'] = week_start_of(_ev['creation_date'])
    _txa_so = df_tx[~df_tx['_unattributed']].copy()
    _txa_so['week_start'] = week_start_of(_txa_so['completed_date'])
    # The ticket denominator needs both time keys, because _so_agg serves the weekly panel
    # and the monthly bridge from one body of code.
    _txa_so['period'] = _txa_so['completed_date'].dt.strftime('%Y-%m')

    _SO_COUNTS = ['stockout_orders', 'stockout_items', 'fulfilled_orders', 'canceled_orders',
                  'open_orders', 'total_tickets']
    _SO_RATES = ['fulfilled_pct', 'canceled_pct', 'open_pct', 'median_days_to_fulfil',
                 'p75_days_to_fulfil', 'median_open_age', 'stockouts_per_100_tickets']

    def _so_agg(gcols, tcol):
        gcols = list(gcols)
        keys = gcols + [tcol]
        # Collapse to one row per ORDER first: an order carries several product rows and
        # counting rows would inflate incidence by the basket size.
        orders = (_ev.groupby(keys + ['so_order'], dropna=False, as_index=False)
                  .agg(_items=('qty', 'sum'), _fulfilled=('is_fulfilled', 'max'),
                       _canceled=('is_canceled', 'max'), _open=('is_open', 'max'),
                       _days=('days_to_fulfil', 'max'), _age=('age_days', 'max')))
        agg = (orders.groupby(keys, dropna=False, as_index=False)
            .agg(stockout_orders=('so_order', 'nunique'), stockout_items=('_items', 'sum'),
                 fulfilled_orders=('_fulfilled', 'sum'),
                 canceled_orders=('_canceled', 'sum'),
                 open_orders=('_open', 'sum'),
                 median_days_to_fulfil=('_days', 'median'),
                 p75_days_to_fulfil=('_days', lambda s: s.quantile(.75)),
                 median_open_age=('_age', 'median')))
        agg['stockout_events'] = agg['stockout_orders']   # renderer/back-compat alias
        for _oc, _pc in (('fulfilled_orders', 'fulfilled_pct'), ('canceled_orders', 'canceled_pct'),
                         ('open_orders', 'open_pct')):
            agg[_pc] = (agg[_oc] / agg['stockout_orders'].replace(0, np.nan) * 100).round(1)
        for c in ('median_days_to_fulfil', 'p75_days_to_fulfil', 'median_open_age'):
            agg[c] = agg[c].round(1)
        # incidence per 100 tickets: is the rise volume-driven or a real supply problem?
        _tx_keys = [c for c in gcols if c in df_tx.columns]
        tx_p = (_txa_so.groupby(_tx_keys + [tcol], dropna=False, as_index=False)
                .agg(total_tickets=('order_num', 'nunique')))
        agg = agg.merge(tx_p, on=_tx_keys + [tcol], how='left')
        agg['stockouts_per_100_tickets'] = (agg['stockout_orders']
            / agg['total_tickets'].replace(0, np.nan) * 100).round(2)
        return agg.sort_values(gcols + [tcol]).reset_index(drop=True)

    def _so_weekly(gcols):
        gcols = list(gcols)
        agg = attach_spine(_so_agg(gcols, 'week_start'))
        agg = add_trailing(agg, gcols, count_cols=_SO_COUNTS, rate_cols=_SO_RATES)
        _s = TRAILING_SUFFIX
        # Pooled trailing rates. Outcome shares especially: a week with three orders, two
        # of them fulfilled, is 67% and must not weigh as much as a 300-order week.
        for _oc, _pc in (('fulfilled_orders', 'fulfilled_pct'),
                         ('canceled_orders', 'canceled_pct'), ('open_orders', 'open_pct')):
            agg[_pc + _s] = (agg[_oc + _s]
                             / agg['stockout_orders' + _s].replace(0, np.nan) * 100).round(1)
        agg['stockouts_per_100_tickets' + _s] = (agg['stockout_orders' + _s]
            / agg['total_tickets' + _s].replace(0, np.nan) * 100).round(2)
        return agg

    def _so_monthly(gcols):
        """Reconciliation bridge to the v1.3.0/v1.4.0 packs. Not plotted."""
        agg = _so_agg(gcols, 'period')
        agg[['so_year', 'so_month']] = agg['period'].str.split('-', expand=True).astype(int)
        return agg

    dash_wk_stockout_co    = _so_weekly([])
    dash_wk_stockout_state = _so_weekly(['state'])
    dash_wk_stockout_vp    = _so_weekly(['vp'])
    dash_wk_stockout_metro = _so_weekly(['metro'])
    dash_wk_stockout_wh    = _so_weekly(['tech_warehouse', 'vp', 'state', 'metro'])
    dash_mo_stockout_co    = _so_monthly([])
    dash_mo_stockout_state = _so_monthly(['state'])
    dash_mo_stockout_vp    = _so_monthly(['vp'])
    dash_mo_stockout_metro = _so_monthly(['metro'])
    dash_mo_stockout_wh    = _so_monthly(['tech_warehouse', 'vp', 'state', 'metro'])

    dash_stockout_products = (_ev.groupby('product', dropna=False, as_index=False)
        .agg(orders=('so_order', 'nunique'), items=('qty', 'sum'),
             open_orders=('is_open', 'sum'), canceled_orders=('is_canceled', 'sum'),
             median_days_to_fulfil=('days_to_fulfil', 'median'),
             states=('state', lambda s: ', '.join(sorted(s.dropna().unique()[:12]))))
        .sort_values('items', ascending=False).reset_index(drop=True))

    # The backlog sheet is the GENUINELY open set only — canceled/abandoned orders get
    # their own sheet, because "chase this" and "explain why this was dropped" are
    # different actions for different people.
    _bl_cols = ['so_order', 'order_type', 'priority', 'tech_warehouse', 'vp', 'state', 'product',
                'qty', 'creation_date', 'age_days', 'reason_for', '_so_resolution']
    dash_stockout_backlog = (_ord[_ord['is_open']][_bl_cols]
        .assign(creation_date=lambda d: d['creation_date'].dt.strftime('%Y-%m-%d'))
        .sort_values('age_days', ascending=False, na_position='last').reset_index(drop=True))
    dash_stockout_canceled = (_ord[_ord['is_canceled']][_bl_cols]
        .assign(creation_date=lambda d: d['creation_date'].dt.strftime('%Y-%m-%d'))
        .sort_values('age_days', ascending=False, na_position='last').reset_index(drop=True))

    # v1.5.0: weekly cohorts. The right-hand weeks are censored on every outcome
    # column — see the section header — so read the trailing columns, not the last row.
    _ord = _ord.assign(week_start=week_start_of(_ord['creation_date']))
    tbl_stockout_fulfil = (_ord.groupby('week_start', as_index=False)
        .agg(orders=('so_order', 'nunique'), fulfilled=('is_fulfilled', 'sum'),
             canceled=('is_canceled', 'sum'), still_open=('is_open', 'sum'),
             p25_days=('days_to_fulfil', lambda s: s.quantile(.25)),
             median_days=('days_to_fulfil', 'median'),
             p75_days=('days_to_fulfil', lambda s: s.quantile(.75)),
             p90_days=('days_to_fulfil', lambda s: s.quantile(.90))))
    for _oc, _pc in (('fulfilled', 'fulfilled_pct'), ('canceled', 'canceled_pct'),
                     ('still_open', 'open_pct')):
        tbl_stockout_fulfil[_pc] = (tbl_stockout_fulfil[_oc]
            / tbl_stockout_fulfil['orders'].replace(0, np.nan) * 100).round(1)
    for c in ('p25_days', 'median_days', 'p75_days', 'p90_days'):
        tbl_stockout_fulfil[c] = tbl_stockout_fulfil[c].round(1)
    tbl_stockout_fulfil = attach_spine(tbl_stockout_fulfil)
    tbl_stockout_fulfil = add_trailing(
        tbl_stockout_fulfil, [],
        count_cols=['orders', 'fulfilled', 'canceled', 'still_open'],
        rate_cols=['fulfilled_pct', 'canceled_pct', 'open_pct', 'median_days'])
    for _oc, _pc in (('fulfilled', 'fulfilled_pct'), ('canceled', 'canceled_pct'),
                     ('still_open', 'open_pct')):
        tbl_stockout_fulfil[_pc + TRAILING_SUFFIX] = (
            tbl_stockout_fulfil[_oc + TRAILING_SUFFIX]
            / tbl_stockout_fulfil['orders' + TRAILING_SUFFIX].replace(0, np.nan)
            * 100).round(1)
    print('\nWeekly incidence and fulfilment (company), last 10 weeks:')
    print(tbl_stockout_fulfil[['week_label', 'orders', 'fulfilled', 'canceled',
                               'still_open', 'fulfilled_pct',
                               'fulfilled_pct' + TRAILING_SUFFIX, 'median_days',
                               'is_partial_week']].tail(10).to_string(index=False))
    print(f'\nWeekly stock outs: wh={len(dash_wk_stockout_wh):,} metro={len(dash_wk_stockout_metro):,} '
          f'vp={len(dash_wk_stockout_vp):,} state={len(dash_wk_stockout_state):,} '
          f'co={len(dash_wk_stockout_co):,} (monthly bridge co={len(dash_mo_stockout_co):,})')
    print('CAVEAT to repeat on any stock-out output: incidence is measured from the only '
          'maintained status value, so a stock-out that was created and resolved entirely '
          'outside that workflow would not appear. Fulfilment is inferred from the ticket '
          'feed, not from the stock-out record itself.')


## Cell 17 — TOP & BOTTOM TECHNICIANS PER GROUPING

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOP & BOTTOM TECHNICIANS PER GROUPING
# Ranking metric: pooled tickets-per-active-day, on the v1.2.0 APPORTIONED
# denominator (active_day_equiv) so a technician who splits days across warehouses
# is not penalised at every one of them. Eligibility stays on WHOLE active weekdays
# — that floor is a presence test ("did this person work enough to rank?"), not a
# productivity measure, and fractional days would quietly raise the bar.
# Eligibility: >= DASH_MIN_ACTIVE_DAYS active weekdays
# inside the grouping. Redelivery stats joined at the SAME grain as the pool (grain-discipline rule). 
# Top and bottom lists are kept disjoint: a tech can never appear on both sides.
# ─────────────────────────────────────────────────────────────────────────────
_redel_by_tech = (redel_linked[redel_linked['techfirstname'].fillna('').str.strip().ne('')]
    .groupby(['techfirstname','techlastname','tech_warehouse'], dropna=False, as_index=False)
    .agg(redelivery_count=('event_key','nunique')))

def _dash_topbottom(gcols, level_name):
    """Pooled per-tech stats within each entity of `gcols`; returns top/bottom N."""
    keys = ['techfirstname','techlastname','tech_warehouse'] + [c for c in gcols
            if c not in ('techfirstname','techlastname','tech_warehouse')]
    pool = (_techday_wk.groupby(keys, dropna=False, as_index=False)
        .agg(total_tickets=('total_tickets','sum'),
             active_weekdays=('active_weekdays','sum'),
             active_day_equiv=('active_day_equiv','sum'),
             virtual_tickets=('virtual_tickets','sum'),
             weeks_active=('week_start','nunique')))
    pool = pool.merge(_redel_by_tech, on=['techfirstname','techlastname','tech_warehouse'], how='left')
    pool['redelivery_count'] = pool['redelivery_count'].fillna(0).astype(int)
    pool['tickets_per_active_day'] = (pool['total_tickets']
        / pool['active_day_equiv'].replace(0, np.nan)).round(3)
    # v1.1.0's whole-day figure, retained so a mover can be reconciled to last week's list
    pool['tickets_per_active_day_wholeday'] = (pool['total_tickets']
        / pool['active_weekdays'].replace(0, np.nan)).round(3)
    pool['virtual_ticket_share_pct'] = (pool['virtual_tickets']
        / pool['total_tickets'].replace(0, np.nan) * 100).round(1)
    pool['redel_per_100_tickets'] = (pool['redelivery_count']
        / pool['total_tickets'].replace(0, np.nan) * 100).round(2)
    pool = pool[pool['active_weekdays'] >= DASH_MIN_ACTIVE_DAYS].copy()
    out = []
    ents = pool[gcols].drop_duplicates().itertuples(index=False) if gcols else [None]
    for ent in ents:
        sub = pool
        if ent is not None:
            for c, v in zip(gcols, ent):
                sub = sub[(sub[c] == v) | (sub[c].isna() & pd.isna(v))]
        sub = sub.sort_values(['tickets_per_active_day','total_tickets'],
                              ascending=[False, False]).reset_index(drop=True)
        n = min(DASH_TOP_N, len(sub))
        top = sub.head(n).copy(); top['rank_group'] = 'Top'
        # disjoint: bottom drawn only from rows not already in top
        bot = sub.iloc[n:].tail(min(DASH_TOP_N, max(0, len(sub) - n))).copy()
        bot['rank_group'] = 'Bottom'
        blk = pd.concat([top, bot], ignore_index=True)
        blk['grouping_level'] = level_name
        out.append(blk)
    if not out: return pd.DataFrame()
    res = pd.concat(out, ignore_index=True)
    _stat_cols = ['grouping_level','rank_group','techfirstname','techlastname','tech_warehouse',
                  'total_tickets','active_weekdays','active_day_equiv','weeks_active',
                  'tickets_per_active_day','tickets_per_active_day_wholeday',
                  'virtual_ticket_share_pct','redelivery_count','redel_per_100_tickets']
    return res[[c for c in gcols if c not in _stat_cols] + _stat_cols]

dash_tb_wh    = _dash_topbottom(['tech_warehouse'], 'Warehouse')  # tech's own wh = the entity
dash_tb_metro = _dash_topbottom(['metro'], 'Metro')
dash_tb_vp    = _dash_topbottom(['vp'], 'VP')
dash_tb_co    = _dash_topbottom([], 'Company')
print(f'Top/Bottom lists: wh={len(dash_tb_wh):,} metro={len(dash_tb_metro):,} '
      f'vp={len(dash_tb_vp):,} co={len(dash_tb_co):,} rows '
      f'(N={DASH_TOP_N}/side, floor={DASH_MIN_ACTIVE_DAYS} active days)')

## Cell 17.5 — RECOMMENDATION EVIDENCE *(new in v1.6.0)*

The VP scorecard and per-technician detail that Cells 22 and 23 rank and write up. Everything the recommendation documents assert is traceable to a row here, and both tables ship in the Excel workbook. Each technician is compared to their **own VP's median**, not the company's, so route density is not mistaken for effort.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RECOMMENDATION EVIDENCE — VP SCORECARD & PER-TECHNICIAN DETAIL     new v1.6.0
#
# Everything the VP recommendation documents say has to be traceable to a number in this
# cell. Nothing downstream invents a finding: Cell 21 ranks what is built here, writes it
# up, and attaches the caveats. That split exists so the wording can be edited without
# touching the arithmetic, and so a VP who disputes a recommendation can be shown the row.
#
# TWO OBJECTS:
#   vp_scorecard   one row per (VP, metric): the VP's trailing figure, the company's, the
#                  gap, whether the gap is adverse, and the VP's rank among VPs.
#   tech_perf_vp   one row per (VP, technician) over the recommendation window: efficiency
#                  on the denominator-honest basis, overtime, redeliveries, and how far
#                  each sits from its OWN VP's median.
#
# WHO CAN BE NAMED, AND WHY THIS IS DELIBERATELY NARROW. CLAUDE.md is explicit that
# lost-equipment attribution is proximity, not fault, and never grounds for discipline
# without ticket-level review. So:
#   NAMED     tickets per active day (denominator-honest, eligibility floor applied) and
#             overtime hours / OT share. These are the two the metric can actually attach
#             to a person, and they are what the request asked for.
#   NOT NAMED lost equipment (proximity only), attendance/PTO (employment-sensitive and
#             the feed cannot tell PTO from a scheduling gap), stock outs (supply-side,
#             nothing to do with the technician who did or did not get the delivery).
#             Those recommendations name SITES instead.
# Every named list travels with COACHING_CAVEAT attached to the row, not just to the
# document, so the caveat cannot be separated from the name by a copy-paste.
#
# COMPARISON BASE: each technician is measured against the median of their OWN VP, not the
# company. A VP running dense urban routes and one running 200-mile rural routes have
# different achievable ticket rates, and ranking both against one company median would
# hand the rural VP a list of "underperformers" that is really a map of drive time.
# ─────────────────────────────────────────────────────────────────────────────
_recs_cut = pd.Timestamp(AS_OF_DATE) - pd.Timedelta(weeks=RECS_LOOKBACK_WEEKS)
print(f'Recommendation window: {_recs_cut.date()} -> {AS_OF_DATE} '
      f'({RECS_LOOKBACK_WEEKS} weeks). Eligibility floor {RECS_MIN_ACTIVE_DAYS} active '
      f'weekdays in that window.')

# ── metric registry: what a "bad" direction means for each metric ────────────
# direction  +1 = higher is better, -1 = lower is better, 0 = context only (never ranked
#                 as a problem; reported for interpretation)
_SCORE_METRICS = [
    ('tickets_per_active_day_per_tech', 'Tickets per active technician-day', 'productivity',
     +1, 'dash_wk_prod_vp', 'dash_wk_prod_co', 'tickets/day'),
    ('attendance_rate_pct', 'Weekday attendance rate', 'census', +1,
     'dash_wk_census_vp', 'dash_wk_census_co', '%'),
    ('census_per_tech_headcount', 'Patients on service per technician', 'census', 0,
     'dash_wk_census_vp', 'dash_wk_census_co', 'patients'),
    ('ot_pct_of_worked', 'Overtime as % of worked hours', 'overtime', -1,
     'dash_wk_ot_vp', 'dash_wk_ot_co', '%'),
    ('ot_hours_per_tech', 'Overtime hours per technician per week', 'overtime', -1,
     'dash_wk_ot_vp', 'dash_wk_ot_co', 'hours'),
    ('lost_cost_per_1k_pt_days', 'Lost equipment $ per 1,000 patient-days', 'lost equipment',
     -1, 'dash_wk_lost_vp', 'dash_wk_lost_co', '$'),
    ('recovery_rate_pct', 'Lost-asset recovery rate (mature cohorts)', 'lost equipment', +1,
     'dash_wk_lost_vp', 'dash_wk_lost_co', '%'),
    ('redel_per_100_tickets', 'Redeliveries per 100 tickets', 'redeliveries', -1,
     'dash_wk_redel_vp', 'dash_wk_redel_co', 'per 100'),
    ('stockouts_per_100_tickets', 'Stock outs per 100 tickets', 'stock outs', -1,
     'dash_wk_stockout_vp', 'dash_wk_stockout_co', 'per 100'),
    ('open_pct', 'Stock-out orders still open', 'stock outs', -1,
     'dash_wk_stockout_vp', 'dash_wk_stockout_co', '%'),
    ('canceled_pct', 'Stock-out orders abandoned rather than filled', 'stock outs', -1,
     'dash_wk_stockout_vp', 'dash_wk_stockout_co', '%'),
]


def _window_mean(df, col, gcols):
    """Mean of the TRAILING series over the recommendation window.

    Why the trailing series and not the raw weekly one: a single week decides nothing, and
    the trailing column already excludes partial weeks and pools its ratio. Averaging the
    trailing series over 13 weeks gives a figure that is stable enough to put in front of a
    VP and still recent enough to be actionable. `weeks` is emitted so a thin base is
    visible rather than implied."""
    tcol = col + TRAILING_SUFFIX
    use = tcol if (df is not None and not df.empty and tcol in df.columns) else col
    if df is None or df.empty or use not in df.columns or 'week_start' not in df.columns:
        return pd.DataFrame(columns=list(gcols) + ['value', 'weeks', 'latest'])
    d = df[df['week_start'] >= _recs_cut].copy()
    d['_v'] = pd.to_numeric(d[use], errors='coerce')
    d = d[d['_v'].notna()].sort_values('week_start')
    if d.empty:
        return pd.DataFrame(columns=list(gcols) + ['value', 'weeks', 'latest'])
    if not list(gcols):
        # The company frame has no grouping column, and groupby([]) raises rather than
        # treating the whole frame as one group.
        return pd.DataFrame([{'value': float(d['_v'].mean()),
                              'weeks': int(d['week_start'].nunique()),
                              'latest': float(d['_v'].iloc[-1])}])
    return (d.groupby(list(gcols), dropna=False, as_index=False)
            .agg(value=('_v', 'mean'), weeks=('week_start', 'nunique'),
                 latest=('_v', 'last')))


_score_rows = []
_missing_metrics = []
for _col, _label, _family, _dir, _vp_frame, _co_frame, _unit in _SCORE_METRICS:
    _vpd = globals().get(_vp_frame)
    _cod = globals().get(_co_frame)
    if _vpd is None or _cod is None or _vpd.empty or _col not in _vpd.columns:
        _missing_metrics.append(_col)
        continue
    _v = _window_mean(_vpd, _col, ['vp'])
    _c = _window_mean(_cod, _col, [])
    _co_val = float(_c['value'].iloc[0]) if len(_c) else np.nan
    for r in _v.itertuples(index=False):
        if pd.isna(r.vp):
            continue      # an unassigned-hierarchy bucket is not a VP and gets no report
        _gap = r.value - _co_val
        _score_rows.append({
            'vp': r.vp, 'metric': _col, 'metric_label': _label, 'family': _family,
            'unit': _unit, 'direction': _dir,
            'vp_value': round(float(r.value), 3), 'company_value': round(_co_val, 3)
            if pd.notna(_co_val) else np.nan,
            'gap_vs_company': round(float(_gap), 3) if pd.notna(_gap) else np.nan,
            'gap_pct_of_company': (round(float(_gap) / abs(_co_val) * 100, 1)
                                   if pd.notna(_co_val) and _co_val else np.nan),
            'weeks_observed': int(r.weeks), 'latest_week_value': round(float(r.latest), 3),
        })
vp_scorecard = pd.DataFrame(_score_rows)
if _missing_metrics:
    print(f'  NOTE: {len(_missing_metrics)} metric(s) absent from the VP frames and skipped: '
          f'{_missing_metrics}. Recommendations will not cover them.')

if len(vp_scorecard):
    # adverse = the gap points the wrong way for this metric's direction
    vp_scorecard['is_adverse'] = np.where(
        vp_scorecard['direction'] == 0, False,
        vp_scorecard['gap_vs_company'] * vp_scorecard['direction'] < 0)
    # severity is normalised so metrics in different units can be ranked against each other:
    # how far off the company figure this VP sits, as a share of the company figure.
    vp_scorecard['severity_pct'] = np.where(
        vp_scorecard['is_adverse'], vp_scorecard['gap_pct_of_company'].abs(), 0.0)
    # ...and ranked within each metric so "worst of N VPs" can be stated plainly.
    vp_scorecard['rank_in_metric'] = (vp_scorecard.groupby('metric')['vp_value']
        .rank(ascending=False, method='min').astype('Int64'))
    _flip = vp_scorecard['direction'] == -1
    vp_scorecard.loc[_flip, 'rank_in_metric'] = (
        vp_scorecard[_flip].groupby('metric')['vp_value']
        .rank(ascending=True, method='min').astype('Int64'))
    vp_scorecard['vps_in_metric'] = vp_scorecard.groupby('metric')['vp'].transform('nunique')
    # A thin base cannot carry a recommendation to a VP...
    vp_scorecard['too_thin'] = vp_scorecard['weeks_observed'] < RECS_MIN_WEEKS
    # ...and neither can a gap of rounding. Without a materiality floor a VP sitting 0.03
    # percentage points off the company figure gets a formal recommendation, which teaches
    # the reader to ignore the list. Immaterial gaps stay in the scorecard, where the number
    # speaks for itself, and simply do not become recommendations.
    vp_scorecard['immaterial'] = (vp_scorecard['severity_pct'].fillna(0)
                                  < RECS_MIN_SEVERITY_PCT)
    vp_scorecard = vp_scorecard.sort_values(['vp', 'severity_pct'],
                                            ascending=[True, False]).reset_index(drop=True)
    _n_actionable = int((vp_scorecard['is_adverse'] & ~vp_scorecard['immaterial']
                         & ~vp_scorecard['too_thin']).sum())
    print(f'VP scorecard: {len(vp_scorecard):,} (VP, metric) rows across '
          f'{vp_scorecard["vp"].nunique()} VPs and {vp_scorecard["metric"].nunique()} metrics; '
          f'{int(vp_scorecard["is_adverse"].sum())} adverse, {_n_actionable} of those material '
          f'enough to recommend on, {int(vp_scorecard["too_thin"].sum())} on too thin a base.')
else:
    print('*** WARNING: VP scorecard is EMPTY — no VP recommendations can be produced. '
          'Check that the weekly VP frames exist and carry the metric columns.')

# ── per-technician detail, within the VP ─────────────────────────────────────
# Efficiency comes from the same apportioned tech-week frame the rankings use, so a
# technician who splits days across warehouses is not penalised at each of them.
_tw = _techday_wk[_techday_wk['week_start'] >= _recs_cut].copy()
_tech_eff = (_tw.groupby(['vp', 'techfirstname', 'techlastname', '_tech_key'],
                         dropna=False, as_index=False)
             .agg(total_tickets=('total_tickets', 'sum'),
                  active_weekdays=('active_weekdays', 'sum'),
                  active_day_equiv=('active_day_equiv', 'sum'),
                  weeks_active=('week_start', 'nunique')))
_tech_eff['tickets_per_active_day'] = (_tech_eff['total_tickets']
    / _tech_eff['active_day_equiv'].replace(0, np.nan)).round(3)
# The technician's main site, for the report line. Modal warehouse by ticket volume.
_tech_wh = (_tw.groupby(['_tech_key', 'tech_warehouse'], dropna=False, as_index=False)
            .agg(_t=('total_tickets', 'sum'))
            .sort_values(['_tech_key', '_t'], ascending=[True, False])
            .drop_duplicates('_tech_key')[['_tech_key', 'tech_warehouse']]
            .rename(columns={'tech_warehouse': 'main_warehouse'}))
_tech_eff = _tech_eff.merge(_tech_wh, on='_tech_key', how='left')

# Overtime, per technician, over the same window. Uses the RAW per-person weekly frame,
# not the site-apportioned one: the 40-hour threshold applies to the PERSON, and a
# technician's overtime is not divisible across the sites they happened to visit.
_ot_tech = pd.DataFrame(columns=['_tech_key', 'ot_hours', 'worked_hours', 'ot_pct_of_worked',
                                 'weeks_with_ot', 'ot_weeks_observed'])
if 'df_ot_weekly' in globals() and len(df_ot_weekly) and 'df_ot_map' in globals():
    _otw = df_ot_weekly[df_ot_weekly['week_start'] >= _recs_cut].copy()
    _map = df_ot_map.copy()
    _map['_tech_key'] = (_map['techfirstname'].str.strip() + '|' + _map['techlastname'].str.strip())
    _otw = _otw.merge(_map[['_plc_key', '_tech_key']].drop_duplicates('_plc_key'),
                      on='_plc_key', how='inner')
    _ot_tech = (_otw.groupby('_tech_key', as_index=False)
                .agg(ot_hours=('ot_hours', 'sum'), worked_hours=('week_hours_worked', 'sum'),
                     weeks_with_ot=('ot_hours', lambda s: int((s > 0).sum())),
                     ot_weeks_observed=('week_start', 'nunique')))
    _ot_tech['ot_pct_of_worked'] = (_ot_tech['ot_hours']
        / _ot_tech['worked_hours'].replace(0, np.nan) * 100).round(2)
    _ot_tech['ot_hours_per_week'] = (_ot_tech['ot_hours']
        / _ot_tech['ot_weeks_observed'].replace(0, np.nan)).round(2)
else:
    print('  NOTE: no payroll match available, so overtime recommendations cannot name '
          'technicians. Efficiency recommendations are unaffected.')

# Redeliveries per technician, at the same grain
_redel_tech = pd.DataFrame(columns=['_tech_key', 'redelivery_count'])
if ('redel_linked' in globals() and len(redel_linked)
        and {'techfirstname', 'techlastname'} <= set(redel_linked.columns)):
    _rl = redel_linked.copy()
    if 'week_start' in _rl.columns:
        _rl = _rl[_rl['week_start'] >= _recs_cut]
    _rl = _rl[_rl['techfirstname'].fillna('').str.strip().ne('')]
    _rl['_tech_key'] = (_rl['techfirstname'].str.strip() + '|' + _rl['techlastname'].str.strip())
    _redel_tech = (_rl.groupby('_tech_key', as_index=False)
                   .agg(redelivery_count=('event_key', 'nunique')))

else:
    print('  NOTE: the redelivery link carries no technician name, so per-technician '
          'redelivery counts are unavailable. Efficiency and overtime are unaffected.')

tech_perf_vp = (_tech_eff.merge(_ot_tech, on='_tech_key', how='left')
                         .merge(_redel_tech, on='_tech_key', how='left'))
tech_perf_vp['redelivery_count'] = tech_perf_vp['redelivery_count'].fillna(0).astype(int)
tech_perf_vp['redel_per_100_tickets'] = (tech_perf_vp['redelivery_count']
    / tech_perf_vp['total_tickets'].replace(0, np.nan) * 100).round(2)
# Eligibility: the SAME presence floor the published rankings use. Below it a technician's
# rate rests on too few days to put their name in a document.
tech_perf_vp['rank_eligible'] = tech_perf_vp['active_weekdays'] >= RECS_MIN_ACTIVE_DAYS

# Compare each technician to the median of their OWN VP — see the cell header.
_elig = tech_perf_vp[tech_perf_vp['rank_eligible'] & tech_perf_vp['vp'].notna()]
_vp_med = (_elig.groupby('vp', as_index=False)
           .agg(vp_median_tickets_per_active_day=('tickets_per_active_day', 'median'),
                vp_median_ot_pct=('ot_pct_of_worked', 'median'),
                vp_eligible_techs=('_tech_key', 'nunique')))
tech_perf_vp = tech_perf_vp.merge(_vp_med, on='vp', how='left')
tech_perf_vp['eff_vs_vp_median_pct'] = ((tech_perf_vp['tickets_per_active_day']
    / tech_perf_vp['vp_median_tickets_per_active_day'].replace(0, np.nan) - 1) * 100).round(1)
tech_perf_vp['ot_pct_vs_vp_median_pp'] = (tech_perf_vp['ot_pct_of_worked']
    - tech_perf_vp['vp_median_ot_pct']).round(2)
tech_perf_vp['eff_percentile_in_vp'] = (tech_perf_vp[tech_perf_vp['rank_eligible']]
    .groupby('vp')['tickets_per_active_day'].rank(pct=True).mul(100).round(1))
# The caveat rides on the ROW, so a name cannot be copied out of the table without it.
tech_perf_vp['interpretation_caveat'] = COACHING_CAVEAT
tech_perf_vp = tech_perf_vp.sort_values(['vp', 'tickets_per_active_day'],
                                        na_position='last').reset_index(drop=True)
print(f'Technician detail: {len(tech_perf_vp):,} (VP, technician) rows | '
      f'{int(tech_perf_vp["rank_eligible"].sum()):,} clear the {RECS_MIN_ACTIVE_DAYS}-day floor | '
      f'{int(tech_perf_vp["ot_hours"].notna().sum()):,} carry payroll-matched overtime.')
if len(tech_perf_vp) and tech_perf_vp['rank_eligible'].sum() == 0:
    print(f'*** WARNING: no technician clears the {RECS_MIN_ACTIVE_DAYS}-day floor inside a '
          f'{RECS_LOOKBACK_WEEKS}-week window. Lower RECS_MIN_ACTIVE_DAYS or lengthen '
          f'RECS_LOOKBACK_WEEKS — as configured, no recommendation can name anyone.')

## Cell 18 — DASHBOARD PAGES → PDF *(3×4 layout, single-axis panels, v1.5.0)*

No panel has two y-axes any more: volumes on rows 0–1, the four normalised rates on row 2, each weekly with its trailing 4-week average. Family colours were machine-checked for chroma, contrast and colour-vision separation.

**v1.6.0:** every page is also captured as a single-page PDF in memory, so Cell 23 can assemble the publishable document in reading order.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DASHBOARD PAGE RENDERER                                       rewritten v1.5.0
# One landscape page per entity — company, each VP, each metro, each warehouse.
#
# TWO STRUCTURAL CHANGES, and the first is a correctness fix, not decoration.
#
# 1. NO PANEL HAS TWO Y-AXES ANY MORE. v1.3.0/v1.4.0 drew overtime, lost equipment,
#    redeliveries and stock outs as "bars on the left axis, rate line on the right". A
#    twin-axis chart has no defensible crossing point: the point where the line appears
#    to overtake the bars is an artifact of two independently chosen scales, so the same
#    data supports opposite readings depending on how the axes were auto-ranged. v1.4.0's
#    own census panel already refused to do this and said why in its docstring — the rule
#    was written down and then applied to one panel out of five. It now applies to all of
#    them: a VOLUME panel carries the weekly count and its trailing average (same unit,
#    one axis), and each normalised RATE gets its own panel on row 2.
#
# 2. LAYOUT 2x4 -> 3x4. Row 0 volumes, row 1 volumes + the top/bottom table, row 2 the
#    four normalised rates. Splitting the rates out is what paid for dropping the second
#    axis, and it means a rate is now readable rather than squeezed against a bar chart.
#
#   (0,0) technician productivity — weekly tickets/active day + trailing 4wk
#   (0,1) census per technician   — weekly, headline + the retained v1.4.0 basis
#   (0,2) overtime hours          — weekly (Thu-Wed payroll weeks) + trailing 4wk
#   (0,3) lost equipment assets   — weekly + trailing 4wk, with staleness banner
#   (1,0) redeliveries            — weekly events + trailing 4wk
#   (1,1) stock outs              — weekly orders + trailing 4wk
#   (1,2:4) top/bottom technician table
#   (2,0) overtime % of worked hours          (2,1) lost $ per 1,000 patient-days
#   (2,2) redeliveries per 100 tickets        (2,3) stock outs per 100 tickets
#
# COLOR. One hue per metric family, held constant across every page, so a reader learns
# "purple is equipment" once. Each panel is a SINGLE-series chart plus a black trailing
# reference line, which is why the family hues only need to separate pairwise as adjacent
# panels rather than as co-plotted series — the panel title names the measure. The set was
# machine-checked, not eyeballed: lightness band, chroma floor, CVD separation and
# normal-vision separation all pass on the adjacent pairlist. Two v1.4.0 colors had to go —
# overtime's #8c564b and stock outs' #7f7f7f both failed the chroma floor (they read as
# gray, and gray is reserved on this page for the recessive re-attribution rug), and
# redeliveries' #e377c2 sat under 3:1 against the surface. The census teal is kept and its
# sub-3:1 contrast is relieved the way the rules allow: a direct value label on every
# panel, plus the Excel sheet as the table view.
#
# THE TRAILING LINE IS THE SAME INK IN EVERY PANEL (black, dashed). It is not a series —
# it is a reference, and giving it a family hue would make it compete with the data it
# summarises.
#
# v1.3.0's inline-output fix is retained: only company and VP pages render inline, so the
# saved notebook does not carry ~110 figures against the rule that outputs are never
# committed.
# ─────────────────────────────────────────────────────────────────────────────
from matplotlib.backends.backend_pdf import PdfPages

# Machine-validated family hues (see the header). PALETTE keys, so a future change is
# made in one place and every panel follows.
PALETTE['prod']     = PALETTE['attributed']   # '#1f77b4'
PALETTE['census']   = '#17becf'
PALETTE['census_alt'] = '#0b6f7a'             # the retained v1.4.0 basis: same hue, darker
PALETTE['overtime'] = '#e6550d'
PALETTE['lost']     = '#9467bd'
PALETTE['redel']    = '#d6609e'
PALETTE['stockout'] = '#7f9c1e'
PALETTE['trailing'] = '#000000'
_RUG_INK = '#7f7f7f'          # recessive, non-data: data-integrity rug only

_S = TRAILING_SUFFIX


def _wk_axis(ax, df, label_every=4):
    """Weekly x axis: real dates, thinned tick labels. A weekly series over 19 months is
    ~85 points, so every label would be unreadable — one in four, always including the
    most recent week, which is the one anybody looks for first."""
    ax.tick_params(labelsize=7)
    ax.grid(axis='y', ls='--', alpha=0.3)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    for lb in ax.get_xticklabels():
        lb.set_fontsize(6.5)


def _mark_routing_change(ax, df):
    try:
        _brk = pd.Timestamp(ROUTING_CHANGE_DATE)
        if df['week_start'].min() <= _brk <= df['week_start'].max():
            ax.axvline(_brk, color='#d62728', ls=':', lw=1.4, zorder=1)
            ax.annotate(f'routing change {ROUTING_CHANGE_DATE}', xy=(_brk, 0.02),
                        xycoords=('data', 'axes fraction'), rotation=90, fontsize=5.5,
                        color='#d62728', ha='right', va='bottom')
    except Exception:
        pass


def _label_last(ax, x, y, color, fmt='{:,.1f}'):
    """Selective direct label — the latest value only. A number on every point is noise,
    and this is also the contrast relief for the low-contrast series."""
    s = pd.to_numeric(y, errors='coerce')
    i = s.last_valid_index()
    if i is None:
        return
    ax.annotate(fmt.format(s.loc[i]), xy=(x.loc[i], s.loc[i]), xytext=(0, 8),
                textcoords='offset points', ha='center', fontsize=7,
                fontweight='bold', color='#333333')


def _panel_weekly_volume(ax, df, col, title, ylab, color, note=None, min_denom_col=None,
                         min_denom=None):
    """Weekly bars + trailing 4-week mean. ONE axis: both are in the same unit, which is
    the whole reason they may share it.

    Partial weeks (clipped by the window edge) are drawn hatched, because a truncated week
    is low for calendar reasons and gets read as a collapse otherwise. They are also out of
    the trailing window — see add_trailing."""
    if df is None or df.empty or col not in df.columns:
        ax.text(0.5, 0.5, 'No data in window', ha='center', va='center', fontsize=8)
        ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off'); return
    d = df.sort_values('week_start').reset_index(drop=True)
    y = pd.to_numeric(d[col], errors='coerce').fillna(0)
    _pf = d['is_partial_week'].fillna(False).values.astype(bool)
    # 2px surface gap between adjacent bars: width 5 of a 7-day slot.
    ax.bar(d['week_start'][~_pf], y[~_pf], width=5, color=color, alpha=0.85,
           label='Weekly', linewidth=0)
    if _pf.any():
        ax.bar(d['week_start'][_pf], y[_pf], width=5, color=color, alpha=0.35,
               hatch='///', edgecolor='white', linewidth=0.4, label='Partial week')
    if col + _S in d.columns:
        t = pd.to_numeric(d[col + _S], errors='coerce')
        ax.plot(d['week_start'], t, color=PALETTE['trailing'], lw=2.0, ls='--',
                label=f'Trailing {ROLL_WEEKS}wk avg', zorder=3)
        _label_last(ax, d['week_start'], t, PALETTE['trailing'], '{:,.0f}')
    _mark_routing_change(ax, d)
    ax.set_ylabel(ylab, fontsize=6.5)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=14 if note else 6)
    _wk_axis(ax, d)
    ax.legend(fontsize=6, loc='best')
    if note:
        ax.annotate(note, xy=(0.5, 1.005), xycoords='axes fraction', ha='center', va='bottom',
                    fontsize=5.5, style='italic', color='#b35400')


def _panel_weekly_rate(ax, df, col, title, ylab, color, note=None, min_denom_col=None,
                       min_denom=10, fmt='{:,.2f}'):
    """A normalised rate, weekly, on its own axis — never sharing one with the volume it
    was divided by.

    min_denom_col/min_denom suppress the weekly points whose denominator is too thin to
    carry a rate: a median over three orders is not a median, and left in it rescales the
    axis and hides the informative range. The TRAILING line is not suppressed the same way
    — pooling four weeks is exactly how a thin week becomes reportable — so the trailing
    series is often the only thing visible on a small site, which is intended. Values stay
    in the Excel sheet; only plotted points are withheld."""
    if df is None or df.empty or col not in df.columns:
        ax.text(0.5, 0.5, 'No data in window', ha='center', va='center', fontsize=8)
        ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off'); return
    d = df.sort_values('week_start').reset_index(drop=True)
    y = pd.to_numeric(d[col], errors='coerce')
    _thin = 0
    if min_denom_col and min_denom_col in d.columns:
        _dn = pd.to_numeric(d[min_denom_col], errors='coerce').fillna(0)
        _mask = _dn < min_denom
        _thin = int(_mask.sum())
        y = y.mask(_mask)
    if not y.notna().any() and not (col + _S in d.columns
                                    and pd.to_numeric(d[col + _S], errors='coerce').notna().any()):
        ax.text(0.5, 0.5, 'Denominator too thin at this grain\nto carry a rate',
                ha='center', va='center', fontsize=7.5)
        ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off'); return
    _pf = d['is_partial_week'].fillna(False).values.astype(bool)
    # A partial week's rate is not comparable to a full one — a 2-day payroll week shows 0%
    # overtime because nobody reached 40 hours, and a 1-day ticket week put stock-outs at 32
    # per 100 against a 27 baseline. Drawn hollow and detached, so it is visible and clearly
    # not part of the trend. It is out of the trailing window too (Cell 4.3).
    ax.plot(d['week_start'].mask(pd.Series(_pf, index=d.index)), y, color=color, lw=1.2,
            marker='o', ms=4, alpha=0.7, mec='white', mew=0.8,
            label='Weekly' + (f' (n>={min_denom})' if _thin else ''))
    if _pf.any():
        ax.plot(d['week_start'][_pf], y[_pf], ls='none', marker='o', ms=5, mfc='white',
                mec=color, mew=1.2, label='Partial week (excluded from trailing)')
    if col + _S in d.columns:
        t = pd.to_numeric(d[col + _S], errors='coerce')
        ax.plot(d['week_start'], t, color=PALETTE['trailing'], lw=2.0, ls='--',
                label=f'Trailing {ROLL_WEEKS}wk (pooled)', zorder=3)
        _label_last(ax, d['week_start'], t, PALETTE['trailing'], fmt)
    else:
        _label_last(ax, d['week_start'], y, color, fmt)
    _mark_routing_change(ax, d)
    ax.set_ylabel(ylab, fontsize=6.5)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=14 if note else 6)
    _wk_axis(ax, d)
    ax.legend(fontsize=6, loc='best')
    if note:
        ax.annotate(note, xy=(0.5, 1.005), xycoords='axes fraction', ha='center', va='bottom',
                    fontsize=5.5, style='italic', color='#b35400')


def _panel_productivity(ax, wk, title):
    """Tickets per active technician-day, weekly + trailing.

    v1.5.0 — THE DATA-INTEGRITY OVERLAY LOST ITS SECOND AXIS. v1.2.0 added a grey band on
    a twin axis showing the share of tickets re-attributed from a virtual warehouse, so the
    July artifact could never hide again. Keeping that signal but dropping the second scale:
    it is now a RUG along the top of the axes — one tick per week whose re-attributed share
    clears the threshold — plus the peak share stated in the corner. Same warning, no
    invented crossing point."""
    if wk is None or wk.empty:
        ax.text(0.5, 0.5, 'No weekly data', ha='center', va='center')
        ax.set_title(title, fontsize=9, fontweight='bold'); return
    d = wk.sort_values('week_start').reset_index(drop=True)
    m = 'tickets_per_active_day_per_tech'
    y = pd.to_numeric(d[m], errors='coerce')
    _pf = d['is_partial_week'].fillna(False).values.astype(bool)
    ax.plot(d['week_start'], y, color=PALETTE['prod'], lw=1.3, marker='o', ms=3.5,
            alpha=0.75, label='Weekly actual')
    if _pf.any():
        ax.plot(d['week_start'][_pf], y[_pf], ls='none', marker='o', ms=5,
                mfc='white', mec=PALETTE['prod'], label='Partial week')
    t = pd.to_numeric(d.get(m + _S, d.get('rolling_4wk_avg')), errors='coerce')
    ax.plot(d['week_start'], t, color=PALETTE['trailing'], lw=2.2, ls='--',
            label=f'Trailing {ROLL_WEEKS}wk (pooled)', zorder=3)
    _label_last(ax, d['week_start'], t, PALETTE['trailing'], '{:,.2f}')
    # re-attribution rug, on THIS axis, no second scale
    if 'virtual_ticket_share_pct' in d.columns:
        vs = pd.to_numeric(d['virtual_ticket_share_pct'], errors='coerce').fillna(0)
        _hit = vs > VIRTUAL_RUG_THRESHOLD_PCT
        if _hit.any():
            ax.plot(d['week_start'][_hit], np.full(int(_hit.sum()), 1.0),
                    transform=ax.get_xaxis_transform(), ls='none', marker='|', ms=7,
                    mew=1.4, color=_RUG_INK, clip_on=False,
                    label=f'>{VIRTUAL_RUG_THRESHOLD_PCT:.0f}% re-attributed from virtual WH')
            ax.annotate(f'peak re-attributed share {vs.max():.0f}% of tickets',
                        xy=(0.99, 0.02), xycoords='axes fraction', ha='right', va='bottom',
                        fontsize=5.5, style='italic', color='#777')
    _mark_routing_change(ax, d)
    ax.set_ylabel('tickets per active tech-day', fontsize=6.5)
    ax.set_title(title, fontsize=9, fontweight='bold')
    _wk_axis(ax, d)
    ax.legend(fontsize=6, loc='best')
    if 'denominator_inflation_pct' in d.columns:
        _inf = pd.to_numeric(d['denominator_inflation_pct'], errors='coerce').mean()
        if pd.notna(_inf) and _inf > 1:
            ax.annotate(f'active days apportioned across warehouses '
                        f'(whole-day denominator would be {_inf:.0f}% higher)',
                        xy=(0.99, 0.08), xycoords='axes fraction', ha='right', va='bottom',
                        fontsize=5.5, style='italic', color='#777')


def _panel_census(ax, ce, title):
    """Census per technician, weekly, TWO denominators on one axis.

    Both series are the same measure in the same unit — patients on service per technician —
    so they legitimately share an axis; that is the point of drawing them together. The
    headline (ADC / distinct technicians) is solid and forward; the retained v1.4.0 basis
    (ADC / average technicians on the road per weekday) is the same hue, darker and dotted,
    so it reads as a subordinate variant of the same thing rather than a rival series. The
    gap between the lines IS the attendance wedge — the ~40% that made the pack read 120
    where ADC / technicians reads ~85-93 — and having it visible on the page is the whole
    reason both are plotted.

    Line style and a direct label carry the distinction as well as color, so the pair
    survives a red-green reader and a monochrome printout, and the label doubles as the
    contrast relief the teal needs against the surface."""
    if ce is None or ce.empty or CENSUS_HEADLINE_METRIC not in ce.columns:
        ax.text(0.5, 0.5, 'No census data at this grain', ha='center', va='center', fontsize=8)
        ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off'); return
    d = ce.sort_values('week_start').reset_index(drop=True)
    y = pd.to_numeric(d[CENSUS_HEADLINE_METRIC], errors='coerce')
    if not y.notna().any():
        ax.text(0.5, 0.5, f'Census ratio suppressed at this grain\n'
                          f'(every week under {CENSUS_MIN_ACTIVE_TECHS:g} average technicians)',
                ha='center', va='center', fontsize=7.5)
        ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off'); return
    _pf = d['is_partial_week'].fillna(False).values.astype(bool)
    # A clipped week loses whole days of attendance, so techs_distinct drops and the ratio
    # jumps. Hollow and out of the trailing window, like every other rate panel.
    ax.plot(d['week_start'].mask(pd.Series(_pf, index=d.index)), y, color=PALETTE['census'],
            lw=1.2, marker='o', ms=4, mec='white', mew=0.8, alpha=0.75,
            label='Weekly: ADC / technicians')
    if _pf.any():
        ax.plot(d['week_start'][_pf], y[_pf], ls='none', marker='o', ms=5, mfc='white',
                mec=PALETTE['census'], mew=1.2, label='Partial week (excluded from trailing)')
    t = pd.to_numeric(d.get(CENSUS_HEADLINE_METRIC + _S), errors='coerce')
    if t is not None and t.notna().any():
        ax.plot(d['week_start'], t, color=PALETTE['trailing'], lw=2.0, ls='--',
                label=f'Trailing {ROLL_WEEKS}wk (pooled)', zorder=3)
        _label_last(ax, d['week_start'], t, PALETTE['trailing'], '{:,.1f}')
    y2 = pd.to_numeric(d.get('census_per_active_tech_weekday'), errors='coerce')
    if y2 is not None and y2.notna().any():
        # The WEDGE is the finding, so it is drawn as an object rather than left for the
        # reader to measure between two lines. Two same-unit series 40% apart also compress
        # each other's variation; shading the space between them makes the separation read as
        # the point of the chart instead of as a scaling accident.
        ax.fill_between(d['week_start'], y, y2, color=PALETTE['census'], alpha=0.13, lw=0,
                        zorder=0, label='attendance wedge (v1.4.0 overstatement)')
        ax.plot(d['week_start'], y2, color=PALETTE['census_alt'], lw=1.4, ls=':',
                label='v1.4.0 basis: ADC / techs on the road')
        _label_last(ax, d['week_start'], y2, PALETTE['census_alt'], '{:,.1f}')
    _mark_routing_change(ax, d)
    ax.set_ylabel('patients on service per technician', fontsize=6.5)
    _att = pd.to_numeric(d.get('attendance_rate_pct'), errors='coerce')
    _note = 'gap between the lines = weekday attendance'
    if _att is not None and _att.notna().any():
        _note += f' ({_att.mean():.0f}% avg)'
    _sup = int(d['ratio_suppressed'].fillna(False).sum()) if 'ratio_suppressed' in d else 0
    if _sup:
        _note += f' | {_sup} thin week(s) hidden'
    ax.set_title(title, fontsize=9, fontweight='bold', pad=14)
    _wk_axis(ax, d)
    ax.legend(fontsize=6, loc='best')
    ax.annotate(_note, xy=(0.5, 1.005), xycoords='axes fraction', ha='center', va='bottom',
                fontsize=5.5, style='italic', color='#b35400')


# v1.4.0 — TOP/BOTTOM HIGHLIGHTING.
# Top and Bottom are a STATUS (a state a row is in), not a data series, so they take
# reserved status colours rather than family hues. Three rules are load-bearing:
#
#  1. COLOUR IS NEVER THE ONLY CHANNEL. Green-vs-red is precisely the pair a red-green
#     colour-blind reader (~8% of men) cannot separate, and the two inks are nearly
#     identical in greyscale (relative luminance 0.089 vs 0.081) so a mono printout
#     loses it too. Every row therefore keeps its 'Top'/'Bottom' word and gains a
#     triangle glyph. Remove either and the panel stops being readable for those readers.
#  2. CONTRAST WAS COMPUTED, NOT EYEBALLED. Body text on the row tints measures 17.2:1
#     (top) and 16.3:1 (bottom); the status-coloured rank label 6.2:1 and 6.3:1 on its
#     own fill. All clear the 4.5:1 bar for body text.
#  3. THE SATURATED STATUS HUES ARE NOT USED BEHIND TEXT. White on #0ca30c measures
#     3.35:1 and fails, which is why the fills are tints and the saturation lives in
#     the ink instead.
_TB_STYLE = {'Top':    {'fill': '#d6efd6', 'ink': '#006300', 'glyph': '▲'},
             'Bottom': {'fill': '#fadcdc', 'ink': '#a11212', 'glyph': '▼'}}


def _panel_topbottom(ax, tb, title):
    ax.axis('off'); ax.set_title(title, fontsize=9, fontweight='bold')
    if tb is None or tb.empty:
        ax.text(0.5, 0.5, f'No technicians clear the {DASH_MIN_ACTIVE_DAYS}-active-day floor',
                ha='center', va='center', fontsize=8); return
    cols = ['rank_group', 'techfirstname', 'techlastname', 'tech_warehouse',
            'total_tickets', 'active_weekdays', 'tickets_per_active_day', 'redel_per_100_tickets']
    d = tb[cols].copy()
    _groups = d['rank_group'].tolist()          # positional, before the column is relabelled
    d['rank_group'] = [f"{_TB_STYLE.get(g, {}).get('glyph', '')} {g}" for g in _groups]
    d.columns = ['Rank', 'First', 'Last', 'Warehouse', 'Tickets', 'ActDays', 'Tkts/Day', 'Redel/100']
    t = ax.table(cellText=d.values, colLabels=d.columns, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(6); t.scale(1, 1.25)
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor('white'); cell.set_linewidth(0.8)   # 'spacer' between fills
        if r == 0:
            cell.set_text_props(fontweight='bold'); cell.set_facecolor('#dbe5f1'); continue
        _st = _TB_STYLE.get(_groups[r - 1])
        if _st is None: continue
        cell.set_facecolor(_st['fill'])
        if c == 0:
            cell.set_text_props(fontweight='bold', color=_st['ink'])


# v1.6.0 — every page is also captured as a single-page PDF in memory, keyed by
# (level, name), so Cell 22 can assemble the publishable document in reading order:
# company, then each VP followed immediately by that VP's recommendations, then metros and
# warehouses. Capturing per page beats slicing the combined PDF by index afterwards, which
# breaks silently the first time an entity is added or removed.
dash_page_pdfs = {}


def _entity_page(pdf, name, level, wk, ce, ot, lost, redel, so, tb, save_png=False):
    fig = plt.figure(figsize=DASH_PAGE_INCHES)
    gs = fig.add_gridspec(3, 4)
    ax = {(r, c): fig.add_subplot(gs[r, c]) for r in (0, 1, 2) for c in range(4)
          if not (r == 1 and c in (2, 3))}
    ax[(1, 2)] = fig.add_subplot(gs[1, 2:4])
    fig.suptitle(f'{level}: {name}   |   {FILTER_START} → {FILTER_END}   |   weekly '
                 f'({PROD_WEEK_LABEL}) with trailing {ROLL_WEEKS}-week average   |   '
                 f'generated {RUN_DATE}', fontsize=13, fontweight='bold')

    # ── row 0 + row 1: volumes, one axis each ────────────────────────────────
    _panel_productivity(ax[(0, 0)], wk,
                        f'Technician Productivity — tickets per active tech-day')
    _panel_census(ax[(0, 1)], ce, 'Census per Technician — ADC / technicians')
    _panel_weekly_volume(ax[(0, 2)], ot, 'ot_hours',
                         f'Overtime Hours — {OT_WEEK_LABEL} payroll weeks', 'OT hours',
                         PALETTE['overtime'],
                         note=f'inferred >{OT_WEEKLY_THRESHOLD:.0f}h per {OT_WEEK_LABEL} week on '
                              f'worked hours; PTO & holiday excluded. Payroll weeks open '
                              f'Thursday — offset from the {PROD_WEEK_LABEL} panels, not '
                              f'alignable week-for-week')
    _panel_weekly_volume(ax[(0, 3)], lost, 'lost_asset_count',
                         'Lost Equipment — assets recorded lost', 'assets',
                         PALETTE['lost'],
                         note=(LOST_STALE_NOTE if LOST_IS_STALE else None) or
                              ('bulk events excluded' if LOST_BULK_EVENT_DATES else None))
    _panel_weekly_volume(ax[(1, 0)], redel, 'redelivery_count',
                         'Redeliveries — events, by originating week', 'redeliveries',
                         PALETTE['redel'],
                         note='attributed to the week of the ORIGINATING ticket, so the most '
                              'recent weeks are censored, not improving')
    _panel_weekly_volume(ax[(1, 1)], so, 'stockout_orders',
                         'Stock Outs — orders, by creation week', 'stock-out orders',
                         PALETTE['stockout'],
                         note='incidence by creation week; fulfilment inferred from the ticket '
                              'feed, and recent weeks are censored on every outcome column')
    _panel_topbottom(ax[(1, 2)], tb, f'Top {DASH_TOP_N} / Bottom {DASH_TOP_N} Technicians '
                                     f'(≥{DASH_MIN_ACTIVE_DAYS} active days; tickets/active day)')

    # ── row 2: the normalised rates that used to ride a second y-axis ────────
    _panel_weekly_rate(ax[(2, 0)], ot, 'ot_pct_of_worked',
                       'Overtime as % of worked hours', '% of worked hours',
                       PALETTE['overtime'], min_denom_col='worked_hours', min_denom=200,
                       fmt='{:,.1f}%',
                       note='a clipped payroll week shows ~0% because nobody reached the '
                            '40-hour line, not because overtime stopped — hollow points')
    _panel_weekly_rate(ax[(2, 1)], lost, 'lost_cost_per_1k_pt_days',
                       'Lost equipment $ per 1,000 patient-days', '$ / 1k patient-days',
                       PALETTE['lost'], min_denom_col='pt_days', min_denom=1000,
                       note='census denominator is now this entity\'s own — v1.4.0 divided VP '
                            'and metro rows by the WHOLE COMPANY, understating them several-fold')
    _panel_weekly_rate(ax[(2, 2)], redel, 'redel_per_100_tickets',
                       'Redeliveries per 100 tickets', 'per 100 tickets',
                       PALETTE['redel'], min_denom_col='total_tickets', min_denom=50)
    _panel_weekly_rate(ax[(2, 3)], so, 'stockouts_per_100_tickets',
                       'Stock outs per 100 tickets', 'per 100 tickets',
                       PALETTE['stockout'], min_denom_col='total_tickets', min_denom=50)

    fig.tight_layout(rect=[0, 0, 1, 0.955])
    pdf.savefig(fig)
    _one = io.BytesIO()
    fig.savefig(_one, format='pdf')
    _one.seek(0)
    dash_page_pdfs[(level, name)] = _one
    if save_png:
        save_fig(fig, f'Dash_{level}_{re.sub(r"[^A-Za-z0-9]+", "_", str(name))}')
        plt.show()          # inline only for the pages leadership circulates
    plt.close(fig)


def _sub(df, col, val):
    return (df[df[col] == val] if (df is not None and not df.empty and col in df.columns)
            else pd.DataFrame())


_pdf_path = os.path.join(OUT_DIR, DASH_PDF_NAME)
with PdfPages(_pdf_path) as _pdf:
    # Company page first, then VPs, metros, warehouses (board reading order)
    _entity_page(_pdf, 'DME Express — All Operations', 'Company',
                 dash_wk_prod_co, dash_wk_census_co, dash_wk_ot_co, dash_wk_lost_co,
                 dash_wk_redel_co, dash_wk_stockout_co, dash_tb_co, save_png=True)
    for _v in sorted(dash_wk_prod_vp['vp'].dropna().unique()):
        _entity_page(_pdf, _v, 'VP', _sub(dash_wk_prod_vp, 'vp', _v),
                     _sub(dash_wk_census_vp, 'vp', _v), _sub(dash_wk_ot_vp, 'vp', _v),
                     _sub(dash_wk_lost_vp, 'vp', _v), _sub(dash_wk_redel_vp, 'vp', _v),
                     _sub(dash_wk_stockout_vp, 'vp', _v), _sub(dash_tb_vp, 'vp', _v),
                     save_png=True)
    for _m in sorted(dash_wk_prod_metro['metro'].dropna().unique()):
        _entity_page(_pdf, _m, 'Metro', _sub(dash_wk_prod_metro, 'metro', _m),
                     _sub(dash_wk_census_metro, 'metro', _m), _sub(dash_wk_ot_metro, 'metro', _m),
                     _sub(dash_wk_lost_metro, 'metro', _m), _sub(dash_wk_redel_metro, 'metro', _m),
                     _sub(dash_wk_stockout_metro, 'metro', _m), _sub(dash_tb_metro, 'metro', _m))
    for _w in sorted(dash_wk_prod_wh['tech_warehouse'].dropna().unique()):
        _entity_page(_pdf, _w, 'Warehouse', _sub(dash_wk_prod_wh, 'tech_warehouse', _w),
                     _sub(dash_wk_census_wh, 'tech_warehouse', _w),
                     _sub(dash_wk_ot_wh, 'tech_warehouse', _w),
                     _sub(dash_wk_lost_wh, 'tech_warehouse', _w),
                     _sub(dash_wk_redel_wh, 'tech_warehouse', _w),
                     _sub(dash_wk_stockout_wh, 'tech_warehouse', _w),
                     _sub(dash_tb_wh, 'tech_warehouse', _w))
print(f'Dashboard PDF written: {_pdf_path}')
print(f'  {len(dash_page_pdfs)} entity page(s) captured for the consolidated document.')
print(f'  Layout 3x4, single-axis panels only. Volumes rows 0-1, normalised rates row 2. '
      f'Every panel: weekly series + trailing {ROLL_WEEKS}-week average.')

## Cell 19 — EXCEL EXPORT *(weekly grain, v1.5.0; recommendation evidence added v1.6.0)*

`*_Weekly` sheets are the published grain and carry the trailing columns (suffix `_t4w`, lighter header). `*_Monthly` sheets are retained only as reconciliation bridges. `VP_Scorecard` and `Tech_Detail_By_VP` are the evidence behind the recommendation documents, published so a disputed recommendation can be traced to its row.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EXCEL EXPORT                                                  extended v1.5.0
# One sheet per grouping x metric family. WEEKLY sheets are the published grain and carry
# the trailing 4-week columns (suffix '_t4w'); MONTHLY sheets are retained purely as the
# reconciliation bridge to the v1.3.0/v1.4.0 packs and are not plotted anywhere.
# No patient identifiers exported.
# ─────────────────────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

_D_PCT = {'vp_value', 'company_value', 'gap_vs_company', 'gap_pct_of_company',
          'severity_pct', 'latest_week_value', 'eff_vs_vp_median_pct',
          'ot_pct_vs_vp_median_pp', 'eff_percentile_in_vp', 'ot_hours_per_week',
          'vp_median_tickets_per_active_day', 'vp_median_ot_pct', 'active_day_equiv',
          'pct_of_census', 'tickets_per_active_day_per_tech', 'weekday_daily_tickets_per_tech', 'rolling_4wk_avg',
          'tickets_per_active_day', 'redel_per_100_tickets', 'lost_asset_cost',
          'tickets_per_active_day_wholeday', 'active_day_equiv', 'denominator_inflation_pct',
          'virtual_ticket_share_pct',
          'ot_hours', 'ot_pct_of_worked', 'ot_hours_per_tech', 'ot_hours_per_employee',
          'worked_hours', 'leave_hours', 'total_hours', 'ot_hours_all_paytypes', 'reg_hours',
          'leave_pct_of_total', 'ot_overstatement_pct',
          'lost_pct_of_inventory', 'lost_cost_per_1k_pt_days', 'lost_assets_per_1k_pt_days',
          'recovery_rate_pct', 'median_days_to_resolve', 'lost_cost', 'cost',
          'median_days_to_fulfil', 'p75_days_to_fulfil', 'median_open_age', 'fulfilled_pct',
          'stockouts_per_100_tickets', 'p25_days', 'median_days', 'p75_days', 'p90_days',
          'canceled_pct', 'open_pct', 'pct_employees_with_ot',
          'adc', 'adc_unfiltered', 'pt_days', 'tech_days', 'avg_active_techs',
          'census_per_active_tech_weekday', 'census_per_tech_headcount',
          'attendance_rate_pct', 'restatement_vs_v140_pct', 'pt_days_per_tech_day',
          'coverage_pct', 't4w_weeks_observed'}
# Every trailing column formats like the column it summarises.
_D_PCT |= {c + TRAILING_SUFFIX for c in _D_PCT}
_HDR_FILL = PatternFill('solid', fgColor='1F4E79'); _HDR_FONT = Font(color='FFFFFF', bold=True)
# Trailing columns get their own header tint so a reader can see at a glance which columns
# are derived from a 4-week window rather than measured in the week on the row.
_HDR_FILL_T4W = PatternFill('solid', fgColor='2E75B6')

_TB_XL = {
    'Top':    {'fill': PatternFill('solid', fgColor='D6EFD6'),
               'font': Font(bold=True, color='006300'), 'glyph': '▲'},
    'Bottom': {'fill': PatternFill('solid', fgColor='FADCDC'),
               'font': Font(bold=True, color='A11212'), 'glyph': '▼'},
}


def _dws(wb, name, df, fcol=1, rank_col=None):
    ws = wb.create_sheet(name[:31])
    if df is None or df.empty:
        ws.cell(1, 1, 'No data in window'); return
    d = df.copy()
    for _dc in ('week_start', 'week_end', 'creation_date', 'lost_date', 'census_date',
                'min_created', 'max_created'):
        if _dc in d.columns:
            d[_dc] = pd.to_datetime(d[_dc], errors='coerce').dt.strftime('%Y-%m-%d')
    for j, col in enumerate(d.columns, 1):
        c = ws.cell(1, j, col)
        c.fill = _HDR_FILL_T4W if str(col).endswith(TRAILING_SUFFIX) else _HDR_FILL
        c.font = _HDR_FONT
        c.alignment = Alignment(horizontal='center')
    for i, row in enumerate(d.itertuples(index=False), 2):
        for j, (col, val) in enumerate(zip(d.columns, row), 1):
            if isinstance(val, (pd.Timestamp,)): val = val.strftime('%Y-%m-%d')
            elif pd.isna(val): val = None
            elif isinstance(val, (np.integer,)): val = int(val)
            elif isinstance(val, (np.floating,)): val = float(val)
            elif isinstance(val, (np.bool_,)): val = bool(val)
            cell = ws.cell(i, j, val)
            if col in _D_PCT: cell.number_format = '0.00'
            elif isinstance(val, int) and not isinstance(val, bool): cell.number_format = '#,##0'
    # Paint Top/Bottom rows after the values are written, so the fill cannot be clobbered.
    if rank_col and rank_col in list(d.columns):
        _rc = list(d.columns).index(rank_col) + 1
        for _i, _val in enumerate(d[rank_col].astype(str).values, 2):
            _st = _TB_XL.get(_val)
            if _st is None:
                continue
            for _j in range(1, len(d.columns) + 1):
                ws.cell(_i, _j).fill = _st['fill']
            _lab = ws.cell(_i, _rc)
            _lab.font = _st['font']
            _lab.value = f"{_st['glyph']} {_val}"     # redundant encoding, not colour alone
    ws.freeze_panes = ws.cell(2, fcol + 1)
    for j, col in enumerate(d.columns, 1):
        _w = max(len(str(col)), int(d[col].astype(str).str.len().quantile(0.9)) if len(d) else 8)
        ws.column_dimensions[get_column_letter(j)].width = min(38, max(10, _w + 2))


wb = Workbook(); wb.remove(wb.active)
_readme = pd.DataFrame({'OpsDashboard README': [
    f'Generated {RUN_DATE} | window {FILTER_START} to {FILTER_END} | source notebook v{NB_VERSION}',
    '',
    '=== v1.6.0 ===',
    'FIXED: the census mapping diagnostic reported virtual warehouses and a real warehouse-',
    '  vocabulary gap as one number and warned about the wrong one. Virtual census can never',
    '  map to a site (a census row carries no technician), so it was inflating the actionable',
    '  figure. The two are now split — see Census_Unmapped_Split.',
    'FIXED: census trailing ratios divided a four-week census by a two-week technician count.',
    '  techs_distinct and avg_active_techs are COUNTS (a week with no technicians had zero),',
    '  not rates, so they zero-fill like every other count. The old treatment read HIGH for',
    '  exactly the sites that had a quiet week. attendance_rate_pct is pooled now too.',
    'NEW: an editable Word metric dictionary, an editable Word recommendations document per',
    '  VP, and one consolidated publishable PDF. VP_Scorecard and Tech_Detail_By_VP hold the',
    '  evidence behind every recommendation, so a disputed item can be traced to its row.',
    'Technicians are named ONLY for tickets-per-active-day and overtime. Lost equipment,',
    '  attendance and stock outs name SITES — attribution for those does not support naming a',
    '  person. Every named list carries the coaching caveat, which also rides on the data rows.',
    '',
    '=== v1.5.0 — READ THIS BEFORE COMPARING ANY NUMBER TO A v1.4.0 PACK ===',
    '',
    'A) CENSUS PER TECHNICIAN IS RESTATED, and this is the change that prompted the release.',
    '   v1.4.0 published ADC / (weekday tech-days / weekdays in month) as "Census per Active',
    '   Technician" — ~120 on the company page. That is ~40% above ADC / technician count, and',
    '   the whole gap is average weekday ATTENDANCE (71-74%: PTO, sick, training, part-time,',
    '   and weekend work, which the weekday filter discards). Jul-2026: 4,112 tech-days / 23',
    '   weekdays = 178.8 "technicians on the road" against 251 distinct technicians on tickets',
    '   and ~300 on payroll, so ADC 23,356 reads 130.6 / 93.1 / 77.9 by denominator.',
    '   WHY IT IS AN ERROR AND NOT A LABEL QUIBBLE: ADC is a STOCK (patients on service on an',
    '   average day); tech-days are a FLOW (attendance). A patient does not leave service',
    '   because their technician took PTO, so caseload is carried by the whole roster and the',
    '   honest denominator is HEADCOUNT. Read the other way the v1.4.0 number is real but it',
    '   measures FIELD COVERAGE, not caseload, and must not sit beside a per-technician target.',
    '   Three measures now travel together on every Census sheet:',
    '     census_per_tech_headcount       ADC / distinct technicians    <- HEADLINE',
    '     census_per_active_tech_weekday  the v1.4.0 basis, retained for reconciliation',
    '     attendance_rate_pct             the wedge between them',
    '   Co_Census_Denominator_Recon is the month-by-month bridge from one to the other.',
    '   RETIRED: v1.4.0\'s caveat "ADC is a 7-day average, tech-days are weekdays only". Census',
    '   is a stock, so that is worth low single digits — it pointed readers at the wrong effect.',
    '   PRODUCTIVITY IS UNAFFECTED: tickets / tech-days is flow / flow, which is the case the',
    '   shared denominator was correct for. Overtime, redeliveries and stock outs never used',
    '   census at all.',
    '',
    'B) BUG FIX — VP AND METRO LOST-EQUIPMENT RATES WERE DIVIDED BY THE WHOLE COMPANY.',
    '   v1.3.0/v1.4.0 branched on whether the grain contained a warehouse; metro and VP both',
    '   fell to the else branch and took the COMPANY total as their patient-day denominator.',
    '   lost_cost_per_1k_pt_days on a VP page was understated by roughly (company census / that',
    '   VP\'s census) — order of 5-10x for a VP, more for a metro — and warehouse rows did not',
    '   aggregate to their VP. Company and warehouse grains were correct, which is why it went',
    '   unnoticed: the error is invisible unless you compare two grains. Every grain now joins',
    '   census at its own grain, and Cell 14 proves it (VP patient-days must not exceed the',
    '   company\'s). VP AND METRO EQUIPMENT RATES WILL RISE SHARPLY AGAINST v1.4.0. That is the',
    '   fix, not a regression.',
    '',
    'C) CENSUS SOURCE CORRECTED. Cell 7.5 had TWO definitions of census: the APC snapshot',
    '   excluded facility (F), inpatient-unit (IPU) and Contract Test customers, and the ADC',
    '   series that every metric actually consumed excluded none of them. One filter now, and',
    '   the notebook prints what it is worth. ADC also used to be a SUM OF PER-WAREHOUSE',
    '   MONTHLY AVERAGES taken over different day sets, which corresponds to no actual day when',
    '   site coverage differs — live right now for the partial current month and the five sites',
    '   that went silent after Jun-2026. ADC is now pooled patient-days / distinct dates at',
    '   every grain. Census_Coverage lists the warehouse-months the old shape distorted.',
    '   Virtual (Z%) census is no longer DELETED from the numerator while its technicians stay',
    '   in the denominator — the same asymmetry that caused the false July productivity drop.',
    '   It counts at company level and is reported separately; it cannot reach a site page',
    '   because a census row carries no technician.',
    '',
    'D) EVERY METRIC IS WEEKLY WITH A TRAILING 4-WEEK AVERAGE (columns ending _t4w, shown with',
    '   a lighter header). Previously productivity was weekly and everything else monthly, so',
    '   two panels on one page were two different time bases. Trailing averages are computed on',
    '   a CALENDAR week spine, not on the rows present: a rolling mean over "the last four rows"',
    '   silently reaches back six weeks when an entity has a quiet week. Counts treat a missing',
    '   week as zero; rates leave it missing. Partial (window-clipped) weeks are excluded from',
    '   count windows and drawn hatched. t4w_weeks_observed reports how many real weeks each',
    '   trailing point rests on. Rate trailing columns are POOLED (trailing numerator /',
    '   trailing denominator), not a mean of four weekly rates — a 30-ticket holiday week must',
    '   not weigh as much as a 700-ticket week.',
    '   OVERTIME KEEPS ITS OWN Thu-Wed PAYROLL WEEK and is NOT alignable week-for-week with the',
    f'  {PROD_WEEK_LABEL} panels. OT site attribution now uses each technician\'s ticket share',
    '   over that same payroll week, not their monthly share.',
    '   CENSORING, unchanged by the grain change and worse to read at weekly resolution:',
    '   redeliveries are attributed to the ORIGINATING week and stock-out outcomes to the',
    '   CREATION week, so the rightmost weeks of both are structurally incomplete — not',
    '   improving. Lost equipment ends at source_through; a final-week decline is missing data.',
    '',
    'E) PDF LAYOUT 2x4 -> 3x4, AND NO PANEL HAS TWO Y-AXES. The old overtime, lost-equipment,',
    '   redelivery and stock-out panels put bars on the left scale and a rate line on the right.',
    '   A twin-axis chart has no defensible crossing point — where the line appears to overtake',
    '   the bars is an artifact of two independently auto-ranged scales. Volumes are now rows',
    '   0-1 and the four normalised rates have their own panels on row 2. Two family colours',
    '   were replaced because they failed a machine check for chroma and contrast (overtime',
    '   #8c564b and stock outs #7f7f7f read as gray; redeliveries #e377c2 sat under 3:1).',
    '',
    '=== v1.4.0 (still in force) ===',
    f'PAYROLL WEEK: overtime is inferred over {OT_WEEK_LABEL} weeks (Paylocity), not Sun-Sat.',
    '  Set OT_WEEK_START_DOW=6 in Cell 3.1 to reproduce v1.3.0. The boundary is derived from',
    '  the feed by sql_explorer_2026-08-15 Cell 8 — re-run it whenever payroll changes',
    '  processor or period. OT_Weekly_Detail carries week_basis so any row is traceable.',
    'TOP/BOTTOM HIGHLIGHTING: green rows = top, red = bottom, in the PDF and the TopBottom_*',
    '  sheets. Colour is never the only channel — the Top/Bottom word and a triangle glyph',
    '  carry the same meaning for red-green colour-blind readers and monochrome printouts.',
    '',
    '=== v1.2.0 (still in force) ===',
    'The Jul-2026 "productivity drop" was a MEASUREMENT ARTIFACT, not an operations change:',
    '  (a) virtual warehouse Z CS began carrying real field tickets in Jun-2026 (24.3% of',
    '      eligible weekday tickets by Jul); v1.1.0 deleted them from the numerator but kept',
    '      the technician active day in the denominator. They are re-attributed to the',
    '      warehouse the technician actually worked that day/week/month — see the Diag_* sheets.',
    '  (b) warehouse-grain active days were double-counted (+34.9% vs the honest company total',
    '      in Jul-2026). Days are apportioned by that day\'s ticket share;',
    '      tickets_per_active_day_wholeday reproduces v1.1.0.',
    'Company Jun->Jul tickets/active-day: -13.2% as reported vs -2.1% corrected. Volume ROSE 6.4%.',
    '',
    '=== v1.3.0 (still in force) ===',
    'OVERTIME: INFERRED, not read from the feed. PLC Reg_Hrs/OT1_Hrs/OT2_Hrs/Paid_Hrs/Est_* are',
    '  pay-period values repeated on every daily row (~14x daily Hours) and OT1_Hrs is 0 from',
    '  2026-02 on. PTO and Holiday are EXCLUDED from the 40-hour threshold;',
    '  ot_hours_all_paytypes reconciles to the older, ~18%-overstated basis. The payroll feed',
    '  re-loads overlapping windows, so rows are de-duplicated first — without that, Jun-2026',
    '  reads 66% OT. Warehouse/metro/VP panels cover Patient Care Technicians only (same',
    '  population as productivity); Co_OT_AllDept covers all departments. Hours reach a site',
    '  via the technician name match, then ticket share, so site rows sum to LESS than company.',
    'LOST EQUIPMENT: source is SERP_LOST_EQUIPMENT. The previous source',
    '  (SERP_ACTIVE_TAGGED_INV.Lost) is NULL on all 583,530 rows and the panel had been',
    '  structurally empty. CURRENT ONLY THROUGH source_through. Known bulk events are excluded',
    '  from every metric and trend and reported in Lost_Bulk_Events. recovery_rate_pct is',
    '  computed only over cohorts >= 90 days old; recent periods are censored.',
    'STOCK OUTS: Status=EnRoute is not an open backlog — rows are retained after fulfilment and',
    '  every other status is frozen at a 2026-01-20 load, so EnRoute by creation date is',
    '  INCIDENCE. Orders resolve THREE ways: fulfilled 64.7%, canceled/abandoned 28.2%',
    '  (completion only on a CANCELED ticket — counting those as delivered overstates',
    '  fulfilment by ~28 points), genuinely open 7.1% = StockOut_Backlog, median 115 days old.',
    '  StockOut_Canceled is the abandoned set: chasing an open order and explaining a dropped',
    '  one are different jobs. Attribution comes from joining [Order] to the ticket feed (76.0%',
    '  of orders); unresolved orders still count at company level.',
    '',
    f'Top/Bottom: {DASH_TOP_N} per side, >={DASH_MIN_ACTIVE_DAYS} active weekdays, ranked on '
    f'tickets/active day; lists disjoint.',
    'Matching/attribution logic lifted from tech_workload v1.35.0 — keep in sync (it still',
    '  carries the v1.1.0 Z-warehouse exclusion and whole-day denominator, so its July figures',
    '  differ), and it has NOT had the census or lost-equipment fixes above.']})
_dws(wb, 'README', _readme)

# ── WEEKLY: the published grain ──────────────────────────────────────────────
for _nm, _df in [
    ('Co_Prod_Weekly', dash_wk_prod_co), ('VP_Prod_Weekly', dash_wk_prod_vp),
    ('Metro_Prod_Weekly', dash_wk_prod_metro), ('WH_Prod_Weekly', dash_wk_prod_wh),
    ('Co_Census_Weekly', dash_wk_census_co), ('VP_Census_Weekly', dash_wk_census_vp),
    ('Metro_Census_Weekly', dash_wk_census_metro), ('WH_Census_Weekly', dash_wk_census_wh),
    ('Co_OT_Weekly', dash_wk_ot_co), ('Co_OT_AllDept_Weekly', dash_wk_ot_alldept),
    ('VP_OT_Weekly', dash_wk_ot_vp), ('Metro_OT_Weekly', dash_wk_ot_metro),
    ('WH_OT_Weekly', dash_wk_ot_wh), ('OT_By_Department_Weekly', tbl_ot_by_dept),
    ('Co_Lost_Weekly', dash_wk_lost_co), ('VP_Lost_Weekly', dash_wk_lost_vp),
    ('Metro_Lost_Weekly', dash_wk_lost_metro), ('WH_Lost_Weekly', dash_wk_lost_wh),
    ('Co_Redel_Weekly', dash_wk_redel_co), ('VP_Redel_Weekly', dash_wk_redel_vp),
    ('Metro_Redel_Weekly', dash_wk_redel_metro), ('WH_Redel_Weekly', dash_wk_redel_wh),
    ('Co_Stockout_Weekly', dash_wk_stockout_co), ('State_Stockout_Weekly', dash_wk_stockout_state),
    ('VP_Stockout_Weekly', dash_wk_stockout_vp), ('Metro_Stockout_Weekly', dash_wk_stockout_metro),
    ('WH_Stockout_Weekly', dash_wk_stockout_wh),
]:
    _dws(wb, _nm, _df)

# ── The census restatement, front and centre ─────────────────────────────────
_dws(wb, 'Co_Census_Denom_Recon', tbl_census_denominator_recon)
_dws(wb, 'Census_Coverage', tbl_census_coverage)
_dws(wb, 'Census_Unmapped_Split', tbl_census_unmapped)

# ── v1.6.0: the evidence behind the VP recommendation documents. Published so a VP who
# disputes a recommendation can be shown the row it came from, rather than the document.
_dws(wb, 'VP_Scorecard', vp_scorecard)
_dws(wb, 'Tech_Detail_By_VP', tech_perf_vp)

# ── MONTHLY: reconciliation bridges to the v1.3.0/v1.4.0 packs. Not plotted. ─
for _nm, _df in [
    ('Co_Census_Monthly', dash_mo_census_co), ('VP_Census_Monthly', dash_mo_census_vp),
    ('Metro_Census_Monthly', dash_mo_census_metro), ('WH_Census_Monthly', dash_mo_census_wh),
    ('Co_OT_Monthly', dash_mo_ot_co), ('Co_OT_AllDept_Monthly', dash_mo_ot_alldept),
    ('VP_OT_Monthly', dash_mo_ot_vp), ('Metro_OT_Monthly', dash_mo_ot_metro),
    ('WH_OT_Monthly', dash_mo_ot_wh),
    ('Co_Lost_Monthly', dash_mo_lost_co), ('VP_Lost_Monthly', dash_mo_lost_vp),
    ('Metro_Lost_Monthly', dash_mo_lost_metro), ('WH_Lost_Monthly', dash_mo_lost_wh),
    ('Co_Redel_Monthly', dash_mo_redel_co), ('VP_Redel_Monthly', dash_mo_redel_vp),
    ('Metro_Redel_Monthly', dash_mo_redel_metro), ('WH_Redel_Monthly', dash_mo_redel_wh),
    ('Co_Stockout_Monthly', dash_mo_stockout_co),
    ('State_Stockout_Monthly', dash_mo_stockout_state),
    ('VP_Stockout_Monthly', dash_mo_stockout_vp),
    ('Metro_Stockout_Monthly', dash_mo_stockout_metro),
    ('WH_Stockout_Monthly', dash_mo_stockout_wh),
]:
    _dws(wb, _nm, _df)

# ── top/bottom sheets get the green/red row highlighting ─────────────────────
for _nm, _df in [('TopBottom_Co', dash_tb_co), ('TopBottom_VP', dash_tb_vp),
                 ('TopBottom_Metro', dash_tb_metro), ('TopBottom_WH', dash_tb_wh)]:
    _dws(wb, _nm, _df, 2, rank_col='rank_group')

# ── detail and diagnostics ───────────────────────────────────────────────────
for _nm, _df in [('OT_Weekly_Detail', df_ot_weekly),
                 ('OT_Feed_Diagnostic', tbl_ot_feed_diagnostic),
                 ('Lost_Products', tbl_lost_products),
                 ('Lost_Reasons_Weekly', tbl_lost_reasons),
                 ('Lost_Resolution_Weekly', tbl_lost_resolution),
                 ('Lost_Bulk_Events', tbl_lost_bulk_events),
                 ('StockOut_Fulfilment', tbl_stockout_fulfil),
                 ('StockOut_Products', dash_stockout_products),
                 ('StockOut_Backlog', dash_stockout_backlog),
                 ('StockOut_Canceled', dash_stockout_canceled),
                 ('StockOut_Status_Recon', tbl_stockout_status_recon),
                 # v1.2.0 diagnostics — kept with the report so a reader can audit the
                 # correction instead of taking the headline on trust.
                 ('Diag_Funnel_Monthly', diag_funnel),
                 ('Diag_VirtualWH_Monthly', diag_virtual_wh),
                 ('Diag_Mechanism_Monthly', diag_resolution),
                 ('Diag_WH_Denominator', diag_wh_denominator),
                 ('Redel_Unlinked_Monthly', tbl_redel_unlinked_monthly)]:
    _dws(wb, _nm, _df)

_xlsx_path = os.path.join(OUT_DIR, DASH_XLSX_NAME)
wb.save(_xlsx_path)
print(f'Excel written: {_xlsx_path} ({len(wb.sheetnames)} sheets)')
print(f'  Weekly sheets are the published grain (trailing columns end "{TRAILING_SUFFIX}"); '
      f'Monthly sheets are reconciliation bridges only.')
try:
    sql_conn.close(); print('DB connection closed.')
except Exception:
    pass

## Cell 20 — DOCUMENT MODEL & RENDERERS *(new in v1.6.0)*

One content model rendered two ways: **python-docx** for the editable Word files and **reportlab** for the narrative pages of the publishable PDF. Writing the content twice would guarantee the Word file the analyst edits and the PDF the CFO reads eventually disagree about what a metric means.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DOCUMENT MODEL & RENDERERS — one content source, two outputs        new v1.6.0
#
# The deliverables are an EDITABLE Word document and a publishable PDF containing the same
# words. Writing that content twice guarantees they drift, and a metric definition that
# differs between the Word file the analyst edits and the PDF the CFO reads is worse than
# having only one of them. So content is built once as a list of blocks and rendered twice:
# python-docx for the editable file, reportlab for the PDF pages that get merged into the
# distribution document.
#
# BLOCK TYPES (a block is a plain dict, so the model stays inspectable and diffable):
#   {'t':'h1'|'h2'|'h3', 'text':...}          headings
#   {'t':'p', 'text':..., 'style':None|'note'|'caveat'}   paragraph
#   {'t':'bullets', 'items':[...]}            bulleted list
#   {'t':'numbers', 'items':[...]}            numbered list
#   {'t':'table', 'cols':[...], 'rows':[[...]], 'widths':[...]|None, 'caption':None}
#   {'t':'kv', 'pairs':[(k,v)]}               two-column fact table, no header
#   {'t':'pagebreak'}
#
# The Word file is genuinely editable — real styles, real tables, no images, no text boxes —
# so an analyst can change a definition, add a caveat, or delete a recommendation and
# re-circulate without re-running the notebook.
# ─────────────────────────────────────────────────────────────────────────────
try:
    import docx
    from docx import Document
    from docx.shared import Pt, Inches, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    _DOCX_OK = True
except ImportError:
    _DOCX_OK = False
    print('*** python-docx not available — Word deliverables will be SKIPPED. '
          'Run Cell 1 (%pip install) and re-run this cell.')

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import landscape, LETTER
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import (BaseDocTemplate, Frame, PageTemplate, SimpleDocTemplate,
                                    Paragraph, Spacer, Table, TableStyle, PageBreak,
                                    KeepTogether, ListFlowable, ListItem)
    from reportlab.lib.enums import TA_LEFT
    _RL_OK = True
except ImportError:
    _RL_OK = False
    print('*** reportlab not available — the consolidated PDF will be SKIPPED. '
          'Run Cell 1 (%pip install) and re-run this cell.')

try:
    from pypdf import PdfReader, PdfWriter
    _PYPDF_OK = True
except ImportError:
    try:
        from PyPDF2 import PdfReader, PdfWriter
        _PYPDF_OK = True
    except ImportError:
        _PYPDF_OK = False
        print('*** pypdf not available — dashboard pages cannot be merged into the '
              'consolidated PDF. Run Cell 1 (%pip install) and re-run this cell.')

# Ink for the documents. Deliberately NOT the chart family hues: a document is text, and
# CLAUDE.md-style caveats need to read as caveats, not as a data series. Contrast against
# white was checked, not eyeballed — 'caveat' is 5.9:1 and 'note' 7.3:1, both clear of the
# 4.5:1 body-text bar, so neither depends on colour alone to be readable.
DOC_INK = {'heading': '1F4E79', 'body': '222222', 'note': '555555', 'caveat': 'B03A00',
           'rule': 'BFBFBF', 'header_fill': '1F4E79', 'zebra': 'F2F6FA'}


def _blk(t, **kw):
    d = {'t': t}
    d.update(kw)
    return d


def h1(text):    return _blk('h1', text=text)
def h2(text):    return _blk('h2', text=text)
def h3(text):    return _blk('h3', text=text)
def p(text, style=None):  return _blk('p', text=text, style=style)
def note(text):  return _blk('p', text=text, style='note')
def caveat(text): return _blk('p', text=text, style='caveat')
def bullets(items): return _blk('bullets', items=list(items))
def numbers(items): return _blk('numbers', items=list(items))
def kv(pairs):   return _blk('kv', pairs=list(pairs))
def pagebreak(): return _blk('pagebreak')


def table(cols, rows, widths=None, caption=None):
    """Table block. Values are stringified here, once, so the two renderers cannot format
    the same number differently."""
    def _fmt(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return '—'
        if isinstance(v, (bool, np.bool_)):
            return 'yes' if v else 'no'
        if isinstance(v, (int, np.integer)):
            return f'{int(v):,}'
        if isinstance(v, (float, np.floating)):
            av = abs(float(v))
            return f'{float(v):,.0f}' if av >= 1000 else f'{float(v):,.2f}'
        return str(v)
    return _blk('table', cols=[str(c) for c in cols],
                rows=[[_fmt(v) for v in r] for r in rows],
                widths=widths, caption=caption)


def df_table(df, cols=None, rename=None, widths=None, caption=None, max_rows=None):
    """Table block straight from a DataFrame, so a document cannot show a column the frame
    does not have — a missing column raises here rather than printing a wrong number."""
    if df is None or len(df) == 0:
        return p('No data in window at this grain.', style='note')
    cols = list(cols) if cols else list(df.columns)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f'df_table: columns absent from the frame: {missing}')
    d = df[cols]
    if max_rows:
        d = d.head(max_rows)
    labels = [(rename or {}).get(c, c) for c in cols]
    return table(labels, d.values.tolist(), widths=widths, caption=caption)


# ── Word renderer ────────────────────────────────────────────────────────────
def _docx_setup(doc):
    st = doc.styles['Normal']
    st.font.name = 'Calibri'
    st.font.size = Pt(10)
    st.paragraph_format.space_after = Pt(6)
    for nm, sz in (('Heading 1', 18), ('Heading 2', 14), ('Heading 3', 11.5)):
        s = doc.styles[nm]
        s.font.name = 'Calibri'
        s.font.size = Pt(sz)
        s.font.color.rgb = RGBColor.from_string(DOC_INK['heading'])
        s.font.bold = True


def render_docx(blocks, path, title=None, subtitle=None):
    """Render the block model to an editable .docx. Returns the path, or None if
    python-docx is unavailable."""
    if not _DOCX_OK:
        return None
    doc = Document()
    _docx_setup(doc)
    for sec in doc.sections:
        sec.left_margin = sec.right_margin = Inches(0.8)
        sec.top_margin = sec.bottom_margin = Inches(0.7)
    if title:
        t = doc.add_paragraph(title)
        t.style = doc.styles['Heading 1']
    if subtitle:
        s = doc.add_paragraph(subtitle)
        s.runs[0].font.size = Pt(10)
        s.runs[0].font.color.rgb = RGBColor.from_string(DOC_INK['note'])
    for b in blocks:
        t = b['t']
        if t in ('h1', 'h2', 'h3'):
            doc.add_paragraph(b['text'], style=doc.styles[{'h1': 'Heading 1', 'h2': 'Heading 2',
                                                          'h3': 'Heading 3'}[t]])
        elif t == 'p':
            par = doc.add_paragraph(b['text'])
            if b.get('style') in ('note', 'caveat'):
                r = par.runs[0]
                r.font.size = Pt(9)
                r.font.color.rgb = RGBColor.from_string(DOC_INK[b['style']])
                r.italic = b['style'] == 'note'
                r.bold = b['style'] == 'caveat'
        elif t in ('bullets', 'numbers'):
            style = 'List Bullet' if t == 'bullets' else 'List Number'
            for it in b['items']:
                doc.add_paragraph(str(it), style=style)
        elif t == 'kv':
            tb = doc.add_table(rows=0, cols=2)
            tb.style = 'Light Grid Accent 1'
            tb.alignment = WD_TABLE_ALIGNMENT.LEFT
            for k, v in b['pairs']:
                cells = tb.add_row().cells
                cells[0].text = str(k)
                cells[1].text = '' if v is None else str(v)
                for pr in cells[0].paragraphs:
                    for r in pr.runs:
                        r.bold = True
                for c in cells:
                    for pr in c.paragraphs:
                        for r in pr.runs:
                            r.font.size = Pt(9)
        elif t == 'table':
            tb = doc.add_table(rows=1, cols=len(b['cols']))
            tb.style = 'Light Grid Accent 1'
            for j, c in enumerate(b['cols']):
                cell = tb.rows[0].cells[j]
                cell.text = c
                for pr in cell.paragraphs:
                    for r in pr.runs:
                        r.bold = True
                        r.font.size = Pt(8.5)
            for row in b['rows']:
                cells = tb.add_row().cells
                for j, v in enumerate(row):
                    cells[j].text = str(v)
                    for pr in cells[j].paragraphs:
                        for r in pr.runs:
                            r.font.size = Pt(8.5)
            if b.get('caption'):
                cap = doc.add_paragraph(b['caption'])
                cap.runs[0].font.size = Pt(8)
                cap.runs[0].italic = True
                cap.runs[0].font.color.rgb = RGBColor.from_string(DOC_INK['note'])
        elif t == 'pagebreak':
            doc.add_page_break()
    doc.save(path)
    return path


# ── PDF renderer (reportlab) ─────────────────────────────────────────────────
def _rl_styles(k=1.0):
    """Paragraph styles, scaled by k.

    k exists because the consolidated PDF puts narrative pages and dashboard pages in ONE
    document, and mixing page sizes in a distributed PDF makes every reader fight their
    viewer. The dashboard pages are large by necessity (twelve panels), so the narrative
    pages match that page size and scale their type up to suit it — 9pt body text on a
    23-inch page reads like 6pt on Letter. The text column itself stays a readable measure
    (see DOC_TEXT_WIDTH_IN); the extra page width becomes margin, not line length."""
    ss = getSampleStyleSheet()
    base = dict(fontName='Helvetica', textColor=colors.HexColor('#' + DOC_INK['body']),
                leading=12.5 * k, alignment=TA_LEFT)
    out = {
        'h1': ParagraphStyle('d_h1', parent=ss['Heading1'], fontSize=17 * k, leading=21 * k,
                             spaceAfter=8 * k, spaceBefore=2 * k, fontName='Helvetica-Bold',
                             textColor=colors.HexColor('#' + DOC_INK['heading'])),
        'h2': ParagraphStyle('d_h2', parent=ss['Heading2'], fontSize=13 * k, leading=16 * k,
                             spaceAfter=6 * k, spaceBefore=10 * k, fontName='Helvetica-Bold',
                             textColor=colors.HexColor('#' + DOC_INK['heading'])),
        'h3': ParagraphStyle('d_h3', parent=ss['Heading3'], fontSize=10.5 * k, leading=13 * k,
                             spaceAfter=4 * k, spaceBefore=8 * k, fontName='Helvetica-Bold',
                             textColor=colors.HexColor('#' + DOC_INK['heading'])),
        'p': ParagraphStyle('d_p', fontSize=9.2 * k, spaceAfter=5 * k, **base),
        'note': ParagraphStyle('d_note', fontSize=8.2 * k, spaceAfter=5 * k,
                               fontName='Helvetica-Oblique',
                               textColor=colors.HexColor('#' + DOC_INK['note']),
                               leading=10.5 * k),
        'caveat': ParagraphStyle('d_cav', fontSize=8.4 * k, spaceAfter=6 * k,
                                 fontName='Helvetica-Bold',
                                 textColor=colors.HexColor('#' + DOC_INK['caveat']),
                                 leading=11 * k),
        'cell': ParagraphStyle('d_cell', fontSize=7.4 * k, leading=9 * k,
                               textColor=colors.HexColor('#' + DOC_INK['body'])),
        'cellh': ParagraphStyle('d_cellh', fontSize=7.4 * k, leading=9 * k,
                                fontName='Helvetica-Bold', textColor=colors.white),
        'sub': ParagraphStyle('d_sub', fontSize=9 * k, spaceAfter=10 * k,
                              fontName='Helvetica-Oblique',
                              textColor=colors.HexColor('#' + DOC_INK['note'])),
        'title': ParagraphStyle('d_title', fontSize=30 * k, leading=36 * k, spaceAfter=14 * k,
                                fontName='Helvetica-Bold',
                                textColor=colors.HexColor('#' + DOC_INK['heading'])),
    }
    return out


def _esc(s):
    return (str(s).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;'))


def blocks_to_flowables(blocks, avail_width, S=None):
    S = S or _rl_styles()
    fl = []
    for b in blocks:
        t = b['t']
        if t in ('h1', 'h2', 'h3'):
            fl.append(Paragraph(_esc(b['text']), S[t]))
        elif t == 'p':
            fl.append(Paragraph(_esc(b['text']), S[b.get('style') or 'p']))
        elif t in ('bullets', 'numbers'):
            # The bullet glyph has to scale with the type. At a fixed 8pt beside 14pt body
            # text it renders as a misaligned speck.
            _bs = S['p'].fontSize * 0.9
            items = [ListItem(Paragraph(_esc(i), S['p']), leftIndent=_bs * 1.6)
                     for i in b['items']]
            fl.append(ListFlowable(items, bulletType='bullet' if t == 'bullets' else '1',
                                   start=None if t == 'bullets' else '1',
                                   leftIndent=_bs * 1.8, bulletFontSize=_bs,
                                   bulletOffsetY=-_bs * 0.12))
            fl.append(Spacer(1, 3))
        elif t == 'kv':
            data = [[Paragraph('<b>%s</b>' % _esc(k), S['cell']),
                     Paragraph(_esc('' if v is None else v), S['cell'])]
                    for k, v in b['pairs']]
            tb = Table(data, colWidths=[avail_width * 0.30, avail_width * 0.70], hAlign='LEFT')
            tb.setStyle(TableStyle([
                ('GRID', (0, 0), (-1, -1), 0.4, colors.HexColor('#' + DOC_INK['rule'])),
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ROWBACKGROUNDS', (0, 0), (-1, -1),
                 [colors.white, colors.HexColor('#' + DOC_INK['zebra'])]),
                ('LEFTPADDING', (0, 0), (-1, -1), 4), ('RIGHTPADDING', (0, 0), (-1, -1), 4),
                ('TOPPADDING', (0, 0), (-1, -1), 2.5), ('BOTTOMPADDING', (0, 0), (-1, -1), 2.5),
            ]))
            fl += [tb, Spacer(1, 7)]
        elif t == 'table':
            ncol = len(b['cols'])
            w = b.get('widths')
            widths = ([avail_width * x / sum(w) for x in w] if w
                      else [avail_width / ncol] * ncol)
            data = [[Paragraph(_esc(c), S['cellh']) for c in b['cols']]]
            data += [[Paragraph(_esc(v), S['cell']) for v in r] for r in b['rows']]
            tb = Table(data, colWidths=widths, repeatRows=1, hAlign='LEFT')
            tb.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#' + DOC_INK['header_fill'])),
                ('GRID', (0, 0), (-1, -1), 0.4, colors.HexColor('#' + DOC_INK['rule'])),
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ROWBACKGROUNDS', (0, 1), (-1, -1),
                 [colors.white, colors.HexColor('#' + DOC_INK['zebra'])]),
                ('LEFTPADDING', (0, 0), (-1, -1), 4), ('RIGHTPADDING', (0, 0), (-1, -1), 4),
                ('TOPPADDING', (0, 0), (-1, -1), 2.5), ('BOTTOMPADDING', (0, 0), (-1, -1), 2.5),
            ]))
            fl.append(tb)
            if b.get('caption'):
                fl.append(Paragraph(_esc(b['caption']), S['note']))
            fl.append(Spacer(1, 7))
        elif t == 'pagebreak':
            fl.append(PageBreak())
    return fl


def render_pdf_bytes(blocks, title=None, subtitle=None, page_inches=None,
                     text_width_in=None, font_scale=None, big_title=False, columns=None):
    """Render the block model to PDF bytes at a given page size.

    page_inches defaults to the dashboard page size, so narrative and chart pages are one
    consistent size in the consolidated document — a distributed PDF that alternates page
    size makes every reader fight their viewer.

    COLUMNS EXIST BECAUSE THE PAGE IS WIDE. A 23-inch page needs a ~10-inch text measure to
    be readable, so a single column fills a third of it and leaves the rest blank — over a
    long dictionary that is a lot of paper and it reads as unfinished. Two frames fill the
    page at a proper measure. Content flows column 1 -> column 2 -> next page; a pagebreak
    block still starts a NEW PAGE, which is what a major section wants.
    """
    if not _RL_OK:
        return None
    page_inches = page_inches or DOC_PAGE_INCHES
    k = DOC_FONT_SCALE if font_scale is None else font_scale
    ncol = DOC_COLUMNS if columns is None else columns
    buf = io.BytesIO()
    pagesize = (page_inches[0] * inch, page_inches[1] * inch)
    top = bot = 0.7 * inch
    gutter = DOC_COL_GUTTER_IN * inch
    if ncol <= 1:
        # single column: hold the measure and let the surplus width become margin
        avail = min(text_width_in or DOC_TEXT_WIDTH_IN, page_inches[0] - 1.2) * inch
        side = (pagesize[0] - avail) / 2.0
    else:
        side = 0.9 * inch
        avail = (pagesize[0] - 2 * side - gutter * (ncol - 1)) / ncol
    frame_h = pagesize[1] - top - bot
    doc = BaseDocTemplate(buf, pagesize=pagesize, leftMargin=side, rightMargin=side,
                          topMargin=top, bottomMargin=bot,
                          title=title or 'DME Express Operations Report',
                          author='DME Express Analytics')
    S = _rl_styles(k)
    fl = []
    if title:
        fl.append(Paragraph(_esc(title), S['title' if big_title else 'h1']))
    if subtitle:
        fl.append(Paragraph(_esc(subtitle), S['sub']))
    fl += blocks_to_flowables(blocks, avail, S)

    def _footer(canv, _doc):
        canv.saveState()
        canv.setFont('Helvetica', 7 * k)
        canv.setFillColor(colors.HexColor('#' + DOC_INK['note']))
        canv.drawString(side, 0.4 * inch,
                        f'DME Express — internal operations reporting | window {FILTER_START} '
                        f'to {FILTER_END} | generated {RUN_DATE} | no patient identifiers')
        canv.restoreState()

    frames = [Frame(side + i * (avail + gutter), bot, avail, frame_h, id=f'col{i}',
                    leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)
              for i in range(max(1, ncol))]
    doc.addPageTemplates([PageTemplate(id='body', frames=frames, onPage=_footer)])
    doc.build(fl)
    buf.seek(0)
    return buf


print(f'Document renderers ready — docx: {_DOCX_OK} | reportlab: {_RL_OK} | pypdf: {_PYPDF_OK}')

## Cell 21 — METRIC DICTIONARY → EDITABLE WORD *(new in v1.6.0)*

Every published metric with its data elements, calculation, grain, direction and interpretation assumptions, plus the cross-cutting assumptions and this release's restatements. **Validated against the code on every run** — a documented metric the notebook does not produce fails the build, and undocumented published columns are listed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# METRIC DICTIONARY -> EDITABLE WORD DOCUMENT                        new v1.6.0
#
# Every metric the pack publishes, the exact data elements behind it, the calculation, and
# the assumptions a reader needs in order to interpret it. Written to .docx so an analyst
# can edit a definition and re-circulate without re-running the notebook.
#
# THE DICTIONARY IS CHECKED AGAINST THE CODE, NOT MAINTAINED ALONGSIDE IT. A metric
# dictionary that drifts from the notebook is worse than none, because it is trusted. So
# METRIC_REGISTRY names the frame each metric lives in, and the validation at the bottom
# of this cell does two things on every run:
#   * every documented metric must EXIST as a column in its frame — otherwise the doc
#     describes something that is not produced, and the cell fails loudly;
#   * every published column must be DOCUMENTED — undocumented columns are listed, so a
#     metric added later cannot quietly ship without a definition.
# ─────────────────────────────────────────────────────────────────────────────
# (family, column, label, formula, source data elements, grain/denominator, direction,
#  [assumptions])
METRIC_REGISTRY = [
    # ── PRODUCTIVITY ─────────────────────────────────────────────────────────
    ('Technician productivity', 'dash_wk_prod_co', 'total_tickets',
     'Completed tickets',
     'COUNT DISTINCT Order_Num',
     '[SERP TRANSACTIONS].Order_Num, .Completed_Date, .Tech_Warehouse, .Reason; '
     'technician name matched to SERP_D',
     'Warehouse / metro / VP / company x Sun-Sat week',
     'higher = more volume',
     ['Weekday tickets only — Saturday and Sunday work is excluded from every productivity '
      'figure, so a site that runs weekend routes shows less volume than it delivered.',
      'Only reasons on the INCLUDED_REASONS whitelist count. A reason renamed upstream '
      'silently drops its tickets; diagnostic D1 (Cell 13.0) tracks the funnel for this.',
      'Distinct ORDER, not row: an order with several product lines counts once.',
      'Exchange pairs (a Pickup and a Delivery for the same patient, technician and day) '
      'are consolidated into one visit by Cell 10.']),
    ('Technician productivity', 'dash_wk_prod_co', 'total_active_tech_days',
     'Active technician-days (apportioned)',
     'SUM over (technician, date) of that day\'s ticket share at each warehouse',
     'Derived from [SERP TRANSACTIONS] — one row per technician per calendar date',
     'Same as above; sums exactly to distinct (technician, date) at company level',
     'context (the denominator)',
     ['A technician who works two warehouses in a day contributes a FRACTION of a day to '
      'each, weighted by that day\'s ticket share — not a whole day to both. Without this '
      'the per-warehouse denominator ran ~35% above the honest company total in Jul-2026.',
      'Ticket share is a proxy for time on site. A stop is not an hour, so read '
      'single-warehouse days as exact and split days as apportioned.',
      'Ticket dates carry NO TIME COMPONENT, so an "active day" is a calendar day. A '
      'half-day still counts as one.']),
    ('Technician productivity', 'dash_wk_prod_co', 'tickets_per_active_day_per_tech',
     'Tickets per active technician-day  [HEADLINE]',
     'total_tickets / total_active_tech_days (ratio of pooled sums, not a mean of means)',
     'As above',
     'Per entity per week',
     'higher is better',
     ['DENOMINATOR-HONEST: it divides by days actually worked, so PTO and part-time '
      'schedules do not penalise a technician. This is the metric technicians are ranked '
      'on; calendar-weekday averages are not used.',
      'Pooled, not averaged: summing tickets and days and then dividing weights a busy '
      'week correctly. Averaging weekly ratios would let a 30-ticket week count as much '
      'as a 700-ticket one.']),
    ('Technician productivity', 'dash_wk_prod_co', 'tickets_per_active_day_wholeday',
     'Tickets per active day, whole-day denominator (v1.1.0 basis)',
     'total_tickets / total_active_tech_days_whole',
     'As above',
     'Per entity per week',
     'reconciliation only',
     ['Retained ONLY so a figure published before v1.2.0 can be reconciled. Do not quote '
      'it: it credits a whole day to every warehouse a technician touched.']),
    ('Technician productivity', 'dash_wk_prod_co', 'denominator_inflation_pct',
     'Denominator inflation from apportionment',
     '(whole-day days / apportioned days - 1) x 100',
     'As above',
     'Per entity per week',
     'context',
     ['How much of this entity\'s week rests on the apportionment assumption. A high value '
      'means many split days, so read the site figure as apportioned rather than measured.']),
    ('Technician productivity', 'dash_wk_prod_co', 'virtual_ticket_share_pct',
     'Share of tickets re-attributed from a virtual warehouse',
     'virtual_tickets / total_tickets x 100',
     '[SERP TRANSACTIONS].Tech_Warehouse values beginning "Z"',
     'Per entity per week',
     'context — a data-integrity signal',
     ['From 2026-06 the virtual code "Z CS" began carrying REAL completed field work '
      '(24.3% of eligible weekday tickets by Jul-2026). Those tickets are re-attributed to '
      'the physical warehouse the technician actually worked that day, else that week, else '
      'that month; anything unresolved is labelled, never dropped.',
      'A RESOLVED WAREHOUSE IS INFERRED, NOT STAMPED ON THE TICKET. It comes from the '
      'technician\'s own route. Sound for site-level trend and workload; NOT evidence about '
      'an individual without ticket-level review.',
      'A rising share means read the trend with suspicion — this is the signal that caught '
      'the false July-2026 "productivity drop".']),
    ('Technician productivity', 'dash_wk_prod_co', 'tech_count',
     'Distinct technicians with a ticket',
     'COUNT DISTINCT (first name | last name)',
     'As above',
     'Per entity per week',
     'context',
     ['Identity is a matched NAME, not an employee ID — the ticket feed carries names. '
      'Matching runs exact, then nickname, then fuzzy on a shared last name.',
      'A technician working two warehouses is counted once per warehouse at site grain, so '
      'site counts sum above the company count.']),

    # ── CENSUS PER TECHNICIAN ────────────────────────────────────────────────
    ('Census per technician', 'dash_wk_census_co', 'pt_days',
     'Patient-days',
     'SUM(total)',
     '[SERP_APC_DAILY].total, .date, .warehourse, .customer',
     'Warehouse / metro / VP / company x week',
     'context (the census numerator)',
     ['Excludes facility "(F)", inpatient-unit "(IPU)" and "Contract Test" customers, '
      'matching the company APC definition. Before v1.5.0 the ADC series excluded NONE of '
      'these while the APC snapshot excluded all three — two different definitions of '
      'census in one report.',
      'NOT LIKE is NULL-unsafe, so a row with a NULL customer falls OUT of the filtered '
      'figure. The volume is printed on every run; if those are real patients the filter '
      'needs an explicit IS NULL branch.']),
    ('Census per technician', 'dash_wk_census_co', 'adc',
     'Average daily census (ADC)',
     'pt_days / census_days  (pooled patient-days over distinct dates OBSERVED)',
     'As above',
     'Per entity per week',
     'context',
     ['A RATIO OF POOLED SUMS. Before v1.5.0, company ADC was the SUM of per-warehouse '
      'monthly AVERAGES taken over different day sets — a number corresponding to no actual '
      'day whenever site coverage differed, which it does for the partial current month and '
      'for any site that stops reporting mid-month.',
      'Denominator is dates OBSERVED, not dates in the calendar. A site with a feed gap is '
      'averaged over the days it reported; the Census_Coverage sheet lists every '
      'warehouse-month observed on under 90% of available dates.']),
    ('Census per technician', 'dash_wk_census_co', 'techs_distinct',
     'Distinct technicians',
     'COUNT DISTINCT technician with any weekday ticket in the week',
     'Derived from [SERP TRANSACTIONS]',
     'Per entity per week',
     'context',
     ['The RAW distinct count of technicians who did field work — roughly 250 company-wide, '
      'against roughly 300 on payroll. The ~50 difference is the payroll-match residual '
      '(75.1% of payroll people tie to an attributed technician) plus genuine non-field '
      'roles, and is not yet reconciled.',
      'AT ENTITY GRAIN THIS DOUBLE-COUNTS anyone who worked more than one site, so warehouse '
      'and VP counts sum above the company count. Use techs_equiv as a denominator; this '
      'column is here as a plain body count.']),
    ('Census per technician', 'dash_wk_census_co', 'techs_equiv',
     'Apportioned technician headcount  [the headline denominator]',
     'SUM over technicians of (that technician\'s days at this entity / their days anywhere)',
     'Derived from [SERP TRANSACTIONS]',
     'Per entity per week; SUMS to the distinct company count',
     'context (the headline denominator)',
     ['A technician who splits their time between two sites counts as a FRACTION of a '
      'technician at each, in proportion to the days they spent there. A technician entirely '
      'at one site counts as 1.',
      'WHY NOT A PLAIN DISTINCT COUNT: tech_days is apportioned across entities, so dividing '
      'it by a whole count of technicians — each of whom is counted once at EVERY site they '
      'touch — mixes an apportioned numerator with an unapportioned denominator. That made '
      'attendance and patients-per-technician read far below the company figure at VP and '
      'warehouse grain, purely because the same person was counted twice. This is the same '
      'defect v1.2.0 fixed for the productivity denominator.',
      'Because it sums to the true distinct count company-wide, the company figures are '
      'unchanged by it and the entity figures are additive. The notebook asserts that sum on '
      'every run.']),
    ('Census per technician', 'dash_wk_census_co', 'avg_active_techs',
     'Average technicians on the road per weekday',
     'tech_days / weekday_days',
     'Derived from [SERP TRANSACTIONS]',
     'Per entity per week',
     'context',
     ['Presence, not headcount. Runs ~71-74% of techs_distinct because of PTO, sickness, '
      'training, part-time schedules and weekend work that the weekday filter discards.']),
    ('Census per technician', 'dash_wk_census_co', 'census_per_tech_headcount',
     'Patients on service per technician  [HEADLINE]',
     'adc / techs_equiv (the APPORTIONED headcount)',
     'As above',
     'Per entity per week',
     'context — a load measure, not a performance measure',
     ['THE CASELOAD MEASURE. ADC is a STOCK (patients on service on an average day) and '
      'must be divided by a HEADCOUNT: a patient does not leave service because their '
      'technician took PTO, so the caseload is carried by the whole roster every day.',
      'RESTATEMENT: v1.4.0 published ADC divided by avg_active_techs and called it "census '
      'per active technician" — ~120 company-wide, about 40% above this figure. The gap is '
      'entirely weekday attendance. Any target quoted against the old number is ~40% out.',
      'Weekly headcount is noisier than monthly (a technician on a week\'s PTO leaves the '
      'denominator entirely), which is what the trailing 4-week column is for.',
      'Suppressed where an entity has under 1 average technician or under 2 distinct '
      'technicians in the week — the inputs stay in the sheet so every suppression is '
      'auditable.']),
    ('Census per technician', 'dash_wk_census_co', 'census_per_active_tech_weekday',
     'Patients per technician on the road (v1.4.0 basis)',
     'adc / avg_active_techs',
     'As above',
     'Per entity per week',
     'context — FIELD COVERAGE, not caseload',
     ['A real quantity, but it answers "how thin are we in the field?", NOT "how many '
      'patients does each technician carry?". Never put it beside a per-technician staffing '
      'target.',
      'Retained so every figure in the v1.4.0 pack can be reconciled line by line.']),
    ('Census per technician', 'dash_wk_census_co', 'attendance_rate_pct',
     'Weekday attendance rate',
     'tech_days / (techs_equiv x weekday_days) x 100',
     'Derived from [SERP TRANSACTIONS]',
     'Per entity per week',
     'higher = more days covered',
     ['THE WEDGE between the two census measures above, and the whole explanation of the '
      '85-versus-120 discrepancy.',
      'It is a PRESENCE measure built from tickets, and it cannot distinguish approved PTO '
      'from an unfilled schedule, training, or light duty. Do not read it as absenteeism, '
      'and do not use it for an individual — the feed does not support that.']),
    ('Census per technician', 'dash_wk_census_co', 'tickets_per_active_day',
     'Tickets per active technician-day (on the census frame)',
     'tickets / tech_days',
     'Derived from [SERP TRANSACTIONS]',
     'Per entity per week',
     'higher is better',
     ['The SAME quantity as tickets_per_active_day_per_tech on the productivity frame, '
      'recomputed here so a reader can check that the census and productivity panels rest on '
      'one technician population. If the two disagree, the panels must not be read against '
      'each other — the notebook asserts this on every run.']),
    ('Census per technician', 'dash_wk_census_co', 'pt_days_per_tech_day',
     'Patient-days carried per technician-day worked',
     'pt_days / tech_days',
     'As above',
     'Per entity per week',
     'context',
     ['Sidesteps the headcount-versus-presence choice entirely by putting a flow over a '
      'flow. Useful when the denominator argument is unsettled.']),

    # ── OVERTIME ─────────────────────────────────────────────────────────────
    ('Overtime', 'dash_wk_ot_co', 'worked_hours',
     'Worked hours',
     'SUM([Hours]) where Pay_Type in (Work, On Call Hours), de-duplicated',
     '[PLC_EMPLOYEE_HOURS].Hours, .WorkDate, .Pay_Type, .Department_Name, .Employee_Name, '
     '.SystemUpdatedDate',
     'Employee x Thu-Wed payroll week, rolled to entity',
     'context',
     ['THE FEED RE-LOADS OVERLAPPING WINDOWS, so rows accumulate — 12.4% of pulled rows are '
      'duplicates and one employee-day appeared 30 times. Un-deduplicated, Jun-2026 reads '
      '66% overtime instead of ~16%. Rows are de-duplicated before anything else.',
      'ONLY [Hours] is trusted. Reg_Hrs / OT1_Hrs / OT2_Hrs / Paid_Hrs / Est_* are '
      'PAY-PERIOD values repeated on every daily row (~14x the daily total), OT1_Hrs is 0 '
      'from 2026-02 on, and OT2_Hrs is 0 throughout. A diagnostic re-proves this every run.',
      'PTO and Holiday are excluded — paid leave does not count toward the 40-hour '
      'threshold. 709 aggregate rows with NULL department/employee (1.05M hours, more than '
      'every real row combined) are excluded by an explicit guard.']),
    ('Overtime', 'dash_wk_ot_co', 'ot_hours',
     'Overtime hours (inferred)  [HEADLINE]',
     'MAX(0, worked hours in the Thu-Wed payroll week - 40), per EMPLOYEE, then summed',
     'As above',
     'Employee x payroll week, rolled to entity',
     'lower is better',
     ['INFERRED, NOT READ FROM THE FEED — the recorded OT columns are unusable (above).',
      'The workweek is THURSDAY to WEDNESDAY, matching Paylocity, derived empirically from '
      'the feed by sql_explorer_2026-08-15 Cell 8 rather than assumed. v1.3.0 used the FLSA '
      'textbook Sun-Sat, which split a payroll week across two reporting weeks. Re-run that '
      'probe if payroll changes processor or period.',
      'THE OVERTIME WEEK IS NOT THE PRODUCTIVITY WEEK. Payroll weeks open Thursday, '
      'reporting weeks open Sunday. The two panels are comparable as trends but are NOT '
      'alignable week-for-week — do not read one against the other at a single week.',
      'The threshold applies to the PERSON, so overtime is computed before any site '
      'allocation. Do not expect a day-by-day tie to a payroll register.']),
    ('Overtime', 'dash_wk_ot_co', 'ot_pct_of_worked',
     'Overtime as a share of worked hours',
     'ot_hours / worked_hours x 100',
     'As above',
     'Per entity per payroll week',
     'lower is better',
     ['Runs ~16-19% of worked hours company-wide and is stable month to month.',
      'A window-clipped payroll week shows ~0% because nobody reached the 40-hour line, not '
      'because overtime stopped. Those weeks are drawn hollow and excluded from the trailing '
      'average.']),
    ('Overtime', 'dash_wk_ot_vp', 'ot_hours_per_tech',
     'Overtime hours per technician per week',
     'ot_hours / technicians',
     'As above',
     'Per entity per payroll week',
     'lower is better',
     ['Site-level overtime is APPORTIONED. Payroll has no usable warehouse key (only 2 of '
      '139 Location_Name values match the warehouse master), so hours reach a site through '
      'the technician name match and are then split by that technician\'s ticket share over '
      'the same payroll week.',
      '~9.9% of overtime hours cannot be attributed to any site and remain in the company '
      'total only. SITE ROWS THEREFORE SUM TO LESS THAN THE COMPANY ROW BY DESIGN; the '
      'reconciliation is printed on every run.']),
    ('Overtime', 'dash_wk_ot_co', 'ot_hours_all_paytypes',
     'Overtime including paid leave (old basis)',
     'MAX(0, all-pay-type hours in the payroll week - 40)',
     'As above',
     'Per entity per payroll week',
     'reconciliation only',
     ['Retained to reconcile to the pre-v1.3.0 basis, which overstated overtime by ~18% in '
      'Jul-2026 by counting PTO and Holiday toward the 40-hour threshold.']),
    ('Overtime', 'dash_wk_ot_co', 'leave_pct_of_total',
     'Paid leave as a share of all paid hours',
     'leave_hours / total_hours x 100',
     '[PLC_EMPLOYEE_HOURS].Pay_Type in (PTO, Holiday)',
     'Per entity per payroll week',
     'context',
     ['Useful next to attendance: a low attendance rate with high leave is planned time '
      'off, and a low attendance rate with low leave is an unfilled schedule.']),

    ('Overtime', 'dash_wk_ot_co', 'leave_hours',
     'Paid leave hours',
     'SUM([Hours]) where Pay_Type in (PTO, Holiday), de-duplicated',
     '[PLC_EMPLOYEE_HOURS].Hours, .Pay_Type',
     'Employee x payroll week, rolled to entity',
     'context',
     ['Excluded from the 40-hour overtime threshold, because paid leave does not count '
      'toward it. Reported separately so attendance can be interpreted: low attendance with '
      'high leave is planned time off, low attendance with low leave is an unfilled '
      'schedule.']),
    ('Overtime', 'dash_wk_ot_co', 'ot_hours_per_employee',
     'Overtime hours per employee per week',
     'ot_hours / employees',
     'As above',
     'Per entity per payroll week',
     'lower is better',
     ['Company and all-department scope only. The site-attributed equivalent is '
      'ot_hours_per_tech, which covers Patient Care Technicians alone so that overtime and '
      'productivity share one population.']),
    ('Overtime', 'dash_wk_ot_co', 'pct_employees_with_ot',
     'Share of employees with any overtime',
     'employees with ot_hours > 0 / employees x 100',
     'As above',
     'Per entity per payroll week',
     'context',
     ['Separates breadth from depth: the same overtime total can be a few very long weeks or '
      'most of the workforce slightly over. Counted on the EMPLOYEE — counting week-rows '
      'instead lands about 4x too high and can exceed the headcount beside it.']),

    # ── LOST EQUIPMENT ───────────────────────────────────────────────────────
    ('Lost equipment', 'dash_wk_lost_co', 'lost_asset_count',
     'Assets recorded lost',
     'COUNT DISTINCT asset tag by week of the lost date',
     '[SERP_LOST_EQUIPMENT].[Lost Date], .[Unit Cost], .Warehouse, .[Lost Reason], '
     '.Resolution, .[Resolved Date]',
     'Warehouse / metro / VP / company x week',
     'lower is better',
     ['SOURCE CHANGED in v1.3.0. The previous source (SERP_ACTIVE_TAGGED_INV.Lost) is NULL '
      'on all 583,530 rows, so this panel had been structurally EMPTY, not zero.',
      '[Lost Date] is not a date — values look like "01/02/2025<br/>(147)", an HTML fragment '
      'that SQL date conversion fails on for 100% of rows. It is parsed in Python and the '
      'parse-failure rate is asserted, not printed.',
      'THE FEED IS STALE relative to the rest of the pack (see source_through on every '
      'sheet). A decline in the final weeks is MISSING DATA, not recovery.',
      'Known bulk events are excluded from every metric and trend and reported separately — '
      'the 2025-08-01 event alone was 30,902 rows and $3.06M, 99.9% of it parked in virtual '
      'buckets rather than at physical sites, which is what marks it as a reconciliation '
      'dump rather than operational loss. Nothing is ever excluded that is not listed in '
      'LOST_BULK_EVENT_DATES.',
      'ATTRIBUTION IS PROXIMITY, NOT FAULT. Suitable for pattern detection and coaching '
      'conversations; NEVER for discipline without ticket-level review.']),
    ('Lost equipment', 'dash_wk_lost_co', 'lost_asset_cost',
     'Cost of assets recorded lost',
     'SUM([Unit Cost])',
     'As above',
     'Per entity per week',
     'lower is better',
     ['[Unit Cost] is a currency STRING ("$11.88") and is cleaned before summing.',
      'Unit cost is book value on the asset record, not a replacement quote or a written-off '
      'amount. Treat it as an order of magnitude.']),
    ('Lost equipment', 'dash_wk_lost_co', 'lost_cost_per_1k_pt_days',
     'Lost equipment $ per 1,000 patient-days',
     'lost_asset_cost / pt_days x 1000',
     'Lost-equipment feed + [SERP_APC_DAILY]',
     'Per entity per week — census joined AT THE SAME GRAIN',
     'lower is better',
     ['THIS WAS WRONG BEFORE v1.6.0 AT VP AND METRO GRAIN. Both grains divided by the WHOLE '
      'COMPANY\'s patient-days, understating the rate by roughly (company census / that '
      'entity\'s census) — order of 5-10x for a VP, more for a metro. Company and warehouse '
      'grains were correct. VP and metro figures rise sharply against the v1.4.0 pack.',
      'The census-normalised view is the one to use when comparing a large site to a small '
      'one; a raw asset count cannot.']),
    ('Lost equipment', 'dash_wk_lost_wh', 'lost_pct_of_inventory',
     'Assets lost as a share of tagged inventory',
     'lost_asset_count / total_inventory_count x 100',
     '[SERP_LOST_EQUIPMENT] + tagged inventory snapshot',
     'Warehouse x week (no equivalent exists at other grains)',
     'lower is better',
     ['Inventory is a CURRENT snapshot, so this is "this week\'s losses against today\'s '
      'stock". Sound for ranking sites against each other; NOT a point-in-time rate, and '
      'not comparable across time.']),
    ('Lost equipment', 'dash_wk_lost_co', 'recovery_rate_pct',
     'Recovery rate on mature cohorts',
     'recovered assets / assets in cohorts at least 90 days old x 100',
     '[SERP_LOST_EQUIPMENT].Resolution, .[Resolved Date]',
     'Per entity per week of loss',
     'higher is better',
     ['CENSORED ON PURPOSE. Only cohorts at least 90 days old get a rate — an asset lost '
      'last week has had no time to be found. Recent weeks show blank rather than a falsely '
      'low number; the alternative is a chart that always declines at the right-hand edge.',
      'Runs 13-25% on mature cohorts, with resolution taking a median of ~42 days.']),

    ('Lost equipment', 'dash_wk_lost_co', 'median_days_to_resolve',
     'Median days to resolve a lost asset',
     'MEDIAN([Resolved Date] - [Lost Date])',
     '[SERP_LOST_EQUIPMENT].[Lost Date], .[Resolved Date]',
     'Per entity per week of loss',
     'lower is better',
     ['Company distribution runs P25 21 / median 42 / P75 91 days.',
      'Computed only over assets that HAVE been resolved, so it says nothing about the ones '
      'still open — read it beside the recovery rate, not instead of it.']),

    # ── REDELIVERIES ─────────────────────────────────────────────────────────
    ('Redeliveries', 'dash_wk_redel_co', 'redelivery_count',
     'Redelivery events',
     'COUNT DISTINCT redelivery event, keyed to the ORIGINATING ticket\'s week',
     'Re-delivery report .Orig_Order joined to [SERP TRANSACTIONS].Order_Num',
     'Warehouse / metro / VP / company x week of the ORIGINATING ticket',
     'lower is better',
     ['Attributed to the week of the ticket that CAUSED it, not the week of the return '
      'visit — the question is "how often does work done in week W come back?".',
      'THE MOST RECENT WEEKS ARE STRUCTURALLY LOW. A ticket completed last Friday has had '
      'almost no time to generate a redelivery, so the right-hand end of the series is '
      'CENSORED, not improving. The trailing average inherits that censoring.',
      'Events whose originating order falls outside the ticket window or under a '
      'non-included reason are EXCLUDED and counted separately in Redel_Unlinked_Monthly; '
      'the early weeks of the window undercount by design.',
      'Redelivery dates arrive as text and are parsed in Python, not filtered in SQL.']),
    ('Redeliveries', 'dash_wk_redel_co', 'redel_per_100_tickets',
     'Redeliveries per 100 tickets  [HEADLINE]',
     'redelivery_count / total_tickets x 100',
     'As above',
     'Per entity per week, numerator and denominator on the SAME entity and week',
     'lower is better',
     ['The denominator includes tickets recovered from virtual warehouses from Jun-2026 on, '
      'which lowers the rate versus figures published before v1.2.0. Correct, but it moves '
      'a published number.']),

    # ── STOCK OUTS ───────────────────────────────────────────────────────────
    ('Stock outs', 'dash_wk_stockout_co', 'stockout_orders',
     'Stock-out orders (incidence)',
     'COUNT DISTINCT [Order] with Status = EnRoute, by CREATION week',
     '[SERP_ORDER_MANAGEMENT_STOCK_OUTS].[Order], .[Creation Date], .[Status], .[Warehouse], '
     '.[Territory], .ProductName, .Quantity, .[Reason For]',
     'Warehouse / metro / VP / state / company x creation week',
     'lower is better',
     ['THIS IS INCIDENCE, NOT A BACKLOG. "EnRoute" is the only maintained status value — '
      '19,891 of 21,438 EnRoute orders carry a completion date, so rows are RETAINED after '
      'the stock-out is dealt with. Every other status value is frozen at a single '
      '2026-01-20 load, so an "oldest open" figure from it is a stale row.',
      'A stock-out created and resolved entirely outside that workflow would not appear at '
      'all.',
      'Site attribution comes from joining [Order] to the ticket feed and resolves ~76% of '
      'orders; unresolved orders are labelled and still count at company level, so the '
      'company row is the reconciliation truth.',
      'PII: the source table carries patient and caregiver names, phone numbers, email and '
      'free-text notes. NONE are selected. The ship-to address is read only to extract a '
      'two-letter state and is dropped immediately.']),
    ('Stock outs', 'dash_wk_stockout_co', 'fulfilled_pct',
     'Stock-out orders fulfilled',
     'orders with a completion date on a LIVE (non-canceled) ticket / stockout_orders x 100',
     'As above, joined to [SERP TRANSACTIONS].Completed_Date and .Status',
     'Per entity per creation week',
     'higher is better',
     ['AN ORDER ENDS ONE OF THREE WAYS, not two. A CANCELED ticket also carries a '
      'completion date, so a naive "has a completion date" test returns 93% fulfilment when '
      'the true figure is ~65% — an overstatement of ~28 points.',
      'Fulfilment is measured against the UNFILTERED ticket feed, with no reason whitelist: '
      '1,165 stock-out orders sit under a reason the analysis deliberately excludes, and '
      'measuring against the filtered set left them looking open forever.',
      'CENSORED at the right-hand edge: an order created last week has had days against a '
      'median 6-day fulfilment, so recent weeks read structurally low.']),
    ('Stock outs', 'dash_wk_stockout_co', 'canceled_pct',
     'Stock-out orders abandoned rather than filled',
     'orders whose only completion evidence is a CANCELED ticket / stockout_orders x 100',
     'As above',
     'Per entity per creation week',
     'lower is better',
     ['~28% of stock-outs end with the order DROPPED rather than supplied. That is an '
      'operational question, not a data one, and it is the most actionable group in the '
      'panel — "chase this" and "explain why this was dropped" are different jobs for '
      'different people.']),
    ('Stock outs', 'dash_wk_stockout_co', 'open_pct',
     'Stock-out orders still genuinely open',
     'orders with NO completion evidence at all / stockout_orders x 100',
     'As above',
     'Per entity per creation week',
     'lower is better',
     ['The real backlog: ~7% of orders, with a median age around 115 days.',
      'Censored like fulfilment — recent weeks read structurally HIGH.']),
    ('Stock outs', 'dash_wk_stockout_co', 'median_days_to_fulfil',
     'Median days to fulfil',
     'MEDIAN(completion date - creation date) over fulfilled orders',
     'As above',
     'Per entity per creation week',
     'lower is better',
     ['Company distribution: P25 3 / median 6 / P75 9 / P90 17 days.',
      'Completions dated BEFORE the stock-out was created are a data-entry artifact, not a '
      'negative lead time, and are excluded from the distribution while staying in the '
      'incidence count.',
      'A median over a handful of orders is not a median; the weekly point is suppressed '
      'below the denominator floor and only the pooled trailing figure is plotted.']),
    ('Stock outs', 'dash_wk_stockout_co', 'stockouts_per_100_tickets',
     'Stock outs per 100 tickets',
     'stockout_orders / total_tickets x 100',
     'As above + [SERP TRANSACTIONS]',
     'Per entity per week',
     'lower is better',
     ['Answers whether a rise is volume-driven or a real supply problem.']),
]

# Cross-cutting matter that applies to every metric above and belongs in one place.
_CROSS = [
    ('Reporting grain',
     'Every metric is WEEKLY with a trailing 4-week average (columns ending "_t4w"). Weeks '
     'run Sunday-Saturday, EXCEPT overtime, which runs Thursday-Wednesday to match the '
     'Paylocity payroll week. The two are not alignable week-for-week.'),
    ('Trailing 4-week average',
     'Computed on a CALENDAR week spine, not on the rows present. A rolling mean over "the '
     'last four rows" reaches back five or six calendar weeks whenever an entity has a quiet '
     'week, and reports it as four — it biases UPWARD, because the weeks that vanish are the '
     'slow ones. Counts treat a missing week inside the entity\'s span as ZERO; rates leave '
     'it missing. Weeks before an entity first appears are left missing, not zeroed.'),
    ('Trailing rates are POOLED',
     'A trailing rate is trailing numerator / trailing denominator, never the mean of four '
     'weekly rates. Averaging ratios weights a 30-ticket holiday week the same as a '
     '700-ticket week.'),
    ('Partial weeks',
     'The first and last week of the window are clipped by the window boundary. They are '
     'plotted (hatched bars, hollow points) but EXCLUDED from every trailing average: a '
     'truncated week\'s count is low for calendar reasons, and its rate is unreliable '
     'whenever the numerator carries a weekly threshold or the denominator is small.'),
    ('Distributions, not just means',
     'Where a distribution exists, P25 / median / P75 are reported. A mean hides the tails, '
     'and on these feeds the tails are usually the story.'),
    ('Site rows do not sum to the company row',
     'By design, in three places: overtime that cannot be matched to a site (~9.9%), census '
     'on warehouses the ticket feed never resolved, and stock-out orders whose site could not '
     'be resolved. The COMPANY ROW IS ALWAYS THE RECONCILIATION TRUTH.'),
    ('Attribution caveats — repeat these wherever the numbers are used',
     'Lost-equipment attribution is PROXIMITY, NOT FAULT: suitable for pattern detection and '
     'coaching, never for discipline without ticket-level review. Technicians are ranked on '
     'tickets per ACTIVE day, never on calendar-weekday averages, so PTO and part-time '
     'schedules do not distort the ranking. A warehouse resolved from a virtual code is '
     'INFERRED from the technician\'s route, not stamped on the ticket.'),
    ('Dates have no time component',
     'Ticket dates carry no time, so an "active day" is a calendar day and a half-day counts '
     'as a full one. Ticket share is a proxy for time on site, not a measurement of it.'),
    ('No patient identifiers',
     'No names, addresses, birth dates, phone numbers, raw patient IDs, MRNs or account '
     'numbers appear in any output. Where a source table carries them they are not selected. '
     'The only permitted patient reference is the scrambled Patient_Token, and it is used '
     'only to pair a pickup with a delivery inside a single day.'),
    ('Read-only by construction',
     'Every query is a single SELECT or WITH, asserted in-process before it reaches the '
     'connection; the connecting account holds read access only.'),
]


# Family order, at module level: the closing summary reports it, and building the document
# must not be the only thing that knows what the families are.
METRIC_FAMILIES = []
for _f0, *_ in METRIC_REGISTRY:
    if _f0 not in METRIC_FAMILIES:
        METRIC_FAMILIES.append(_f0)


def build_dictionary_blocks():
    B = [h1('Metric Dictionary'),
         p(f'DME Express operations reporting — every published metric, the data elements '
           f'behind it, how it is calculated, and the assumptions required to interpret it. '
           f'Window {FILTER_START} to {FILTER_END}, generated {RUN_DATE} from '
           f'ops_dashboard v{NB_VERSION}.'),
         note('This document is generated from the notebook and validated against it on every '
              'run: a metric named here that does not exist in the output fails the build, and '
              'any published column with no definition here is listed as a gap. Edit it freely '
              '— it is a Word file — but expect a regenerated copy to reset your edits.')]

    B += [h2('How to read the pack'),
          bullets([
              'Every metric is weekly. The dashed black line on each panel is the trailing '
              '4-week average — read it for direction, and the weekly series for events.',
              'No panel has two vertical scales. Volumes and the rates derived from them sit '
              'in separate panels, because a twin-axis chart has no defensible crossing point.',
              'Hatched bars and hollow points are partial weeks at the window edge. They are '
              'excluded from the trailing average.',
              'Where a rate is missing, its denominator was too thin to carry one. The inputs '
              'are always in the Excel workbook, so every suppression can be audited.',
              'The company page is the reconciliation truth. Site rows deliberately sum to '
              'less than it — see "Site rows do not sum to the company row" below.',
          ])]

    B += [h2('Restatements in this version — read before comparing to an earlier pack'),
          numbers([
              'CENSUS PER TECHNICIAN is restated. The previous pack divided average daily '
              'census by the average number of technicians ON THE ROAD per weekday and '
              'labelled it "census per active technician" (~120 company-wide). Divided by '
              'technician HEADCOUNT it is ~85-93. The whole difference is weekday attendance '
              'of ~71-74%. Both figures are now published side by side with the attendance '
              'wedge between them; the headline is the headcount one.',
              'VP AND METRO LOST-EQUIPMENT RATES rise sharply. They were previously divided '
              'by the whole company\'s patient-days rather than their own.',
              'CENSUS SOURCE corrected: one customer-exclusion definition instead of two, '
              'average daily census pooled instead of summed from per-warehouse averages, and '
              'virtual-warehouse census no longer dropped from the numerator while its '
              'technicians stayed in the denominator.',
              'EVERY METRIC MOVED FROM MONTHLY TO WEEKLY, and trailing averages are now '
              'computed on a calendar spine and pooled for rates.',
          ])]

    B += [pagebreak(), h2('Metrics by family')]
    for fam in METRIC_FAMILIES:
        B.append(h3(fam))
        for (f, frame, col, label, formula, source, grain, direction, assumptions) \
                in METRIC_REGISTRY:
            if f != fam:
                continue
            B.append(kv([('Metric', label),
                         ('Output column', col),
                         ('Calculation', formula),
                         ('Data elements', source),
                         ('Grain / denominator', grain),
                         ('Direction', direction)]))
            B.append(bullets(assumptions))

    B += [h2('Cross-cutting assumptions — these apply to every metric above')]
    for k, v in _CROSS:
        B += [h3(k), p(v)]

    B += [pagebreak(), h2('Known limitations still open')]
    B += [bullets([
        'The denominator for any per-technician CASELOAD target is unsettled: ~300 payroll '
        'technicians, ~250 attributed to tickets, and ~180 on the road on an average weekday '
        'give figures ~40% apart end to end. Nothing here should be compared to a target '
        'without naming which denominator the target uses.',
        'Whether facility "(F)" and inpatient-unit "(IPU)" census should count toward '
        'technician caseload is a business question, not a data one. They are currently '
        'excluded, matching the APC snapshot definition.',
        'The lost-equipment feed lags the rest of the pack. Its final weeks are missing data.',
        'Warehouse-level comparisons spanning Jun-2026 remain provisional until the "Z CS" '
        'routing change and five sites that went silent after Jun-2026 are explained.',
        'The reason whitelist is a silent single point of failure: a reason renamed upstream '
        'drops its tickets with no error. Diagnostic D1 tracks the funnel for this.',
    ])]
    return B


# ── VALIDATION: the dictionary must match the code ───────────────────────────
_doc_undefined, _doc_documented = [], set()
for (_f, _frame, _col, *_rest) in METRIC_REGISTRY:
    _fr = globals().get(_frame)
    if _fr is None:
        _doc_undefined.append(f'{_frame} (frame missing)')
    elif len(_fr) and _col not in _fr.columns:
        _doc_undefined.append(f'{_frame}.{_col}')
    _doc_documented.add(_col)
if _doc_undefined:
    raise KeyError('Metric dictionary documents metrics the notebook does not produce: '
                   f'{_doc_undefined}. Fix METRIC_REGISTRY or the producing cell — a '
                   'dictionary that describes a column nobody outputs is worse than none.')

# The reverse direction is a warning, not a failure: plenty of columns are intermediate
# working values that do not need a public definition.
_PUBLISHED_FRAMES = {'dash_wk_prod_co': dash_wk_prod_co, 'dash_wk_census_co': dash_wk_census_co,
                     'dash_wk_ot_co': dash_wk_ot_co, 'dash_wk_lost_co': dash_wk_lost_co,
                     'dash_wk_redel_co': dash_wk_redel_co,
                     'dash_wk_stockout_co': dash_wk_stockout_co}
# Working values and reconciliation columns. These are deliberately NOT in the public
# dictionary: they exist so a number can be traced or a correction measured, and listing
# every one of them would bury the metrics a reader actually quotes.
_INTERNAL = {'week_start', 'week_end', 'week_label', 'period', 'is_partial_week',
             'weekday_days_in_week', 'calendar_days_in_week', 'weekdays_in_period',
             'calendar_days_in_period', 'scope', 'week_basis', 'source_through',
             'denominator_note', 'ratio_suppressed', 't4w_weeks_observed', 'census_grain',
             'stockout_events', 'day_basis_note', 'delivery_year', 'delivery_month',
             'so_year', 'so_month', 'lost_year', 'lost_month', 'weekday_days',
             # intermediate numerators/denominators, all defined via the metrics above
             'virtual_tickets', 'total_active_tech_days_whole', 'active_weekdays_whole',
             'tickets', 'tech_days', 'census_days', 'census_warehouses', 'employees',
             'technicians', 'employees_with_ot', 'reg_hours', 'total_hours',
             'recovered_assets', 'discarded_assets', 'unresolved_assets', 'mature_assets',
             'lost_product_types', 'redelivery_items', 'stockout_items', 'total_tickets',
             'fulfilled_orders', 'canceled_orders', 'open_orders', 'techs_distinct',
             'avg_active_techs', 'total_inventory_count',
             # reconciliation columns, retained to measure a restatement rather than to quote
             'pt_days_unfiltered', 'adc_unfiltered', 'restatement_vs_v140_pct',
             'weekday_daily_tickets_per_tech', 'rolling_4wk_avg', 'ot_overstatement_pct',
             'p75_days_to_fulfil', 'median_open_age', 'lost_assets_per_1k_pt_days'}
_undocumented = []
for _nm, _fr in _PUBLISHED_FRAMES.items():
    if _fr is None or not len(_fr):
        continue
    for _c in _fr.columns:
        if (_c.endswith(TRAILING_SUFFIX) or _c in _INTERNAL or _c in _doc_documented
                or _c.startswith('_')):
            continue
        _undocumented.append(f'{_nm}.{_c}')
if _undocumented:
    print(f'  NOTE: {len(_undocumented)} published column(s) have no dictionary entry: '
          f'{_undocumented[:12]}{" ..." if len(_undocumented) > 12 else ""}')
    print('  Intermediate working values are fine to leave; anything a reader would quote '
          'should get an entry in METRIC_REGISTRY.')

dict_blocks = build_dictionary_blocks()
_dict_path = os.path.join(OUT_DIR, REPORT_DOCX_DICTIONARY)
if render_docx(dict_blocks, _dict_path,
               title=None, subtitle=None):
    print(f'Metric dictionary (editable Word): {_dict_path}')
else:
    print('Metric dictionary SKIPPED — python-docx unavailable.')
print(f'  {len(METRIC_REGISTRY)} metrics documented across {len(METRIC_FAMILIES)} families; '
      f'{len(_CROSS)} cross-cutting assumptions.')

## Cell 22 — VP RECOMMENDATIONS → EDITABLE WORD *(new in v1.6.0)*

One document per VP: a scorecard against the company, then the top 5 actions ranked by adverse gap, each with the figure it rests on and concrete steps. Technicians are named for **efficiency** and **overtime** only; lost equipment, attendance and stock outs name **sites**, because attribution for those does not support naming a person.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VP RECOMMENDATIONS -> ONE EDITABLE WORD DOCUMENT PER VP            new v1.6.0
#
# Top RECS_TOP_N recommendations per VP, ranked by how far that VP sits from the company
# figure on each metric, in the adverse direction. Nothing is invented here: every number
# comes from vp_scorecard and tech_perf_vp (Cell 17.5), and every recommendation states the
# figure it rests on so a VP can check it.
#
# NAMING INDIVIDUALS — the rule, and why it is narrow. The request was for recommendations
# "up to and including specific technicians that are dragging down their metrics either in
# overtime or technician efficiencies", and those two are exactly where the data supports
# naming someone:
#   * tickets per ACTIVE day is denominator-honest (PTO and part-time do not distort it),
#     it carries a presence floor, and it is measured against the technician's OWN VP median
#     rather than a company one, so route density is not mistaken for effort;
#   * overtime is computed on the person, from de-duplicated payroll hours, before any site
#     allocation.
# Everywhere else the pack names SITES, not people:
#   * lost equipment — CLAUDE.md is explicit that attribution is proximity, not fault;
#   * attendance — the feed cannot separate approved PTO from an unfilled schedule;
#   * stock outs — supply-side, nothing to do with the technician on the route.
# Each named list carries the coaching caveat in the document AND on the underlying row.
# ─────────────────────────────────────────────────────────────────────────────
# metric -> (why it matters, actions, naming mode)
#   naming mode: 'efficiency' | 'overtime' | 'site' | None
_RECS_PLAYBOOK = {
    'tickets_per_active_day_per_tech': (
        'Throughput per day worked is the cleanest measure of route productivity, and it is '
        'the metric technicians are ranked on. A gap here is usually route density, '
        'scheduling, or a small number of technicians well below their peers — in that order '
        'of likelihood.',
        ['Review route construction at the lowest-rate sites before looking at individuals: '
         'a whole-site gap is a routing or geography problem, not a personnel one.',
         'Check the re-attribution share on the productivity panel first. If it is rising, '
         'part of the gap is a feed change rather than an operational one.',
         'For the named technicians below, start with a coaching conversation and a ride-along '
         'to establish whether the gap is skill, route, equipment or vehicle.',
         'Confirm each named technician\'s ticket mix — a technician carrying more complex '
         'set-ups will legitimately complete fewer stops per day.'],
        'efficiency'),
    'attendance_rate_pct': (
        'Attendance sets how many technicians are actually on the road, which drives both '
        'coverage and the overtime the remaining technicians absorb. It is also the wedge '
        'between the two census measures in this pack.',
        ['Compare attendance against paid leave on the same page. Low attendance with high '
         'leave is planned time off; low attendance with LOW leave is an unfilled schedule, '
         'and only the second is a scheduling problem you can fix.',
         'Check whether low attendance and high overtime are appearing at the same sites — '
         'that pattern is a headcount gap being covered with premium hours.',
         'Verify roster accuracy at the affected sites: a technician who left and was never '
         'removed depresses attendance without any operational cause.'],
        'site'),
    'ot_pct_of_worked': (
        'Overtime is the most immediately controllable labour cost in the pack, and a '
        'persistent share above the company figure usually reflects headcount or route '
        'balance rather than individual behaviour.',
        ['Check the overtime concentration below. Overtime spread thinly across a whole VP is '
         'a capacity problem; overtime concentrated in a few technicians is a scheduling or '
         'route-balance problem, and the two need opposite responses.',
         'Compare against attendance in the same weeks — covering absence with overtime is '
         'cheaper than a vacancy only up to a point.',
         'Rebalance routes at the sites carrying the most overtime hours before adding '
         'headcount.',
         'Confirm on-call arrangements at the affected sites; on-call hours count toward the '
         '40-hour threshold and can push a normal week into overtime.'],
        'overtime'),
    'ot_hours_per_tech': (
        'Overtime hours per technician shows whether the overtime share reflects a few long '
        'weeks or a structural shortfall in available hours.',
        ['Review the highest-hour technicians below for sustained rather than occasional '
         'overtime — sustained overtime is a capacity signal and also a retention risk.',
         'Check whether the same names recur week after week; if they do, the schedule, not '
         'the individual, is the thing to change.'],
        'overtime'),
    'lost_cost_per_1k_pt_days': (
        'Census-normalised equipment loss is the only way to compare a large site with a '
        'small one. This figure has been RESTATED UPWARD for VPs and metros in this version — '
        'the previous pack divided by the whole company\'s census.',
        ['Start with the highest-loss sites listed below, not with technicians. Attribution '
         'here is proximity, not fault.',
         'Review the recovery rate alongside the loss rate: a high loss rate with a high '
         'recovery rate is a tracking problem, while a high loss rate with a low recovery '
         'rate is a control problem.',
         'Check the resolution mix at the affected sites — assets discarded without a '
         'recovery attempt are a different failure from assets never found.',
         'Confirm the equipment mix. A site carrying more high-value assets will show a '
         'higher cost rate at the same physical loss rate.'],
        'site'),
    'recovery_rate_pct': (
        'Recovery rate on mature cohorts measures whether lost equipment is actually chased. '
        'It is censored for recent weeks by design, so only cohorts at least 90 days old '
        'count.',
        ['Review the retrieval process at the sites listed below, and the median days to '
         'resolve alongside — a long resolution time usually means the attempt starts late.',
         'Check whether unresolved assets are simply never worked rather than worked and not '
         'found; those need different interventions.'],
        'site'),
    'redel_per_100_tickets': (
        'Redeliveries are rework: a second visit paid for twice, and a patient who waited. '
        'The rate is keyed to the ORIGINATING week, so the most recent weeks read '
        'structurally low.',
        ['Review the redelivery reasons for the affected sites — wrong item, missing part and '
         'patient-not-home need three different fixes.',
         'Check the named technicians below for a pattern, but treat a high rate on low '
         'volume as noise; the eligibility floor applies here too.',
         'Confirm order accuracy upstream before treating this as a field problem. A '
         'redelivery caused by a picking error is not a technician issue.'],
        'efficiency'),
    'stockouts_per_100_tickets': (
        'Stock-out incidence per 100 tickets separates a genuine supply problem from simply '
        'doing more work.',
        ['Review the product mix driving incidence at the affected sites — a concentrated '
          'product list is a purchasing or par-level fix, not a field one.',
         'Check par levels and replenishment lead times at the highest-incidence sites.',
         'This metric is supply-side. It should not be raised with technicians.'],
        'site'),
    'open_pct': (
        'Orders with no completion evidence at all are the real backlog — a patient waiting '
        'with nothing recorded against their order.',
        ['Work the open backlog list in the StockOut_Backlog sheet, oldest first.',
         'Recent weeks are CENSORED and read structurally high; use the trailing figure, not '
         'the last week, to judge whether this is getting worse.'],
        'site'),
    'canceled_pct': (
        'Orders whose only completion evidence sits on a CANCELED ticket were ABANDONED, not '
        'delivered. Company-wide this is roughly 28% of stock-outs, and it is the most '
        'actionable group in the pack.',
        ['Sample the StockOut_Canceled sheet for this VP and establish WHY orders are being '
         'dropped — substituted, no longer needed, patient discharged, or simply lost track '
         'of. The four have entirely different remedies.',
         'Confirm whether cancellation is a deliberate, recorded decision or a default that '
         'happens when nothing else does.',
         'This is a process question, not a technician question.'],
        'site'),
}


def _fmt_val(v, unit):
    if v is None or pd.isna(v):
        return '—'
    if unit == '%':
        return f'{float(v):,.1f}%'
    if unit == '$':
        return f'${float(v):,.2f}'
    return f'{float(v):,.2f}'


def _ordinal(n):
    """1st, 2nd, 3rd, 11th... A document that goes to a VP should not say "2th"."""
    if n is None or pd.isna(n):
        return '—'
    n = int(round(float(n)))
    if 10 <= n % 100 <= 20:
        suf = 'th'
    else:
        suf = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f'{n}{suf}'


def _fmt_gap(v, unit):
    """A gap between two percentages is measured in PERCENTAGE POINTS, and writing it as
    "30.0%" invites a reader to hear "30% worse" when the relative gap is 41%. Both are
    given, each labelled."""
    if v is None or pd.isna(v):
        return '—'
    if unit == '%':
        return f'{float(v):,.1f} percentage points'
    return _fmt_val(v, unit)


def _named_efficiency(vp, n=None):
    """The eligible technicians furthest BELOW their own VP's median tickets per active day."""
    n = n or RECS_NAME_N
    d = tech_perf_vp[(tech_perf_vp['vp'] == vp) & tech_perf_vp['rank_eligible']
                     & tech_perf_vp['tickets_per_active_day'].notna()]
    d = d[d['eff_vs_vp_median_pct'] < -RECS_EFF_GAP_PCT]
    return d.sort_values('tickets_per_active_day').head(n)


def _named_overtime(vp, n=None):
    """The eligible technicians carrying the most overtime hours in the window."""
    n = n or RECS_NAME_N
    d = tech_perf_vp[(tech_perf_vp['vp'] == vp) & tech_perf_vp['rank_eligible']
                     & tech_perf_vp['ot_hours'].notna() & (tech_perf_vp['ot_hours'] > 0)]
    return d.sort_values('ot_hours', ascending=False).head(n)


def _named_sites(vp, metric, n=None):
    """Worst sites inside the VP for a metric, from the weekly warehouse frames."""
    n = n or RECS_NAME_N
    _src = {'lost_cost_per_1k_pt_days': 'dash_wk_lost_wh', 'recovery_rate_pct': 'dash_wk_lost_wh',
            'stockouts_per_100_tickets': 'dash_wk_stockout_wh', 'open_pct': 'dash_wk_stockout_wh',
            'canceled_pct': 'dash_wk_stockout_wh', 'attendance_rate_pct': 'dash_wk_census_wh',
            'redel_per_100_tickets': 'dash_wk_redel_wh'}.get(metric)
    fr = globals().get(_src)
    if fr is None or fr.empty or metric not in fr.columns or 'vp' not in fr.columns:
        return pd.DataFrame()
    d = fr[(fr['vp'] == vp) & (fr['week_start'] >= _recs_cut)].copy()
    if d.empty:
        return pd.DataFrame()
    d['_v'] = pd.to_numeric(d[metric], errors='coerce')
    g = (d.groupby('tech_warehouse', dropna=False, as_index=False)
         .agg(value=('_v', 'mean'), weeks=('week_start', 'nunique')))
    g = g[g['value'].notna() & (g['weeks'] >= RECS_MIN_WEEKS)]
    # Virtual and unresolved codes are not buildings anyone can visit. They stay in the VP
    # total — dropping them from the total would hide real volume — but a list headed "sites
    # driving this figure" has to name sites.
    g = g[~g['tech_warehouse'].astype(str).str.upper().str.startswith(VIRTUAL_WH_PREFIX.upper())]
    g = g[g['tech_warehouse'].astype(str).ne(VIRTUAL_UNRESOLVED_LABEL)]
    _dir = {m[0]: m[3] for m in _SCORE_METRICS}.get(metric, -1)
    return g.sort_values('value', ascending=(_dir == +1)).head(n)


def build_vp_rec_blocks(vp):
    """The recommendation document for one VP, as blocks."""
    sc = vp_scorecard[(vp_scorecard['vp'] == vp) & ~vp_scorecard['too_thin']]
    adverse = sc[sc['is_adverse'] & ~sc['immaterial'] & sc['metric'].isin(_RECS_PLAYBOOK)]
    adverse = adverse.sort_values('severity_pct', ascending=False).head(RECS_TOP_N)
    _n_imm = int((sc['is_adverse'] & sc['immaterial']).sum())

    B = [h1(f'Recommendations — {vp}'),
         p(f'Top {RECS_TOP_N} actions to improve this VP\'s metrics, ranked by how far the VP '
           f'sits from the company figure. Window {_recs_cut.date()} to {AS_OF_DATE} '
           f'({RECS_LOOKBACK_WEEKS} weeks); generated {RUN_DATE} from ops_dashboard '
           f'v{NB_VERSION}.'),
         note('Every figure here is a trailing 4-week value averaged over the window, so no '
              'single week drives a recommendation. Each item names the metric it rests on — '
              'the same number appears on this VP\'s dashboard page and in the Excel workbook.')]

    # ── scorecard ─────────────────────────────────────────────────────────────
    B.append(h2('Scorecard against the company'))
    if sc.empty:
        B.append(p('No metric for this VP has enough observed weeks in the window to report.',
                   style='note'))
    else:
        _rows = []
        for r in sc.sort_values(['family', 'metric_label']).itertuples(index=False):
            _rows.append([r.metric_label, _fmt_val(r.vp_value, r.unit),
                          _fmt_val(r.company_value, r.unit),
                          ('—' if r.direction == 0 else
                           ('adverse' if r.is_adverse else 'better than company')),
                          (f'{r.rank_in_metric} of {r.vps_in_metric}'
                           if pd.notna(r.rank_in_metric) else '—'),
                          r.weeks_observed])
        B.append(table(['Metric', 'This VP', 'Company', 'Direction of gap', 'Rank among VPs',
                        'Weeks'], _rows, widths=[30, 11, 11, 16, 13, 7],
                       caption='Rank 1 = best among VPs on that metric. "Patients on service '
                               'per technician" is a LOAD measure and is never ranked as good '
                               'or bad.'))

    # ── the recommendations ──────────────────────────────────────────────────
    B.append(h2(f'Top {RECS_TOP_N} recommendations'))
    if len(adverse) < RECS_TOP_N:
        B.append(note(f'{len(adverse)} recommendation(s) rather than {RECS_TOP_N}: this VP is '
                      f'adverse on {_n_imm} further metric(s) by less than '
                      f'{RECS_MIN_SEVERITY_PCT:.0f}% of the company figure, which is inside '
                      f'the noise of a 13-week window. Those stay in the scorecard above '
                      f'rather than being written up as actions — a list padded with rounding '
                      f'teaches the reader to ignore the list.'))
    if adverse.empty:
        B.append(p('This VP is at or better than the company figure on every metric with a '
                   'usable base in this window. No adverse recommendations are generated — '
                   'which is a result, not a gap in the report.', style='note'))
        B.append(p('Use the scorecard above to pick a stretch target rather than a remedy, and '
                   'check the load measure (patients on service per technician) before adding '
                   'volume.'))
    for i, r in enumerate(adverse.itertuples(index=False), 1):
        why, actions, naming = _RECS_PLAYBOOK[r.metric]
        _worse = 'above' if r.direction == -1 else 'below'
        B.append(h3(f'{i}. {r.metric_label}'))
        B.append(kv([
            ('This VP', _fmt_val(r.vp_value, r.unit)),
            ('Company', _fmt_val(r.company_value, r.unit)),
            ('Gap', f'{_fmt_gap(abs(r.gap_vs_company), r.unit)} {_worse} the company figure'
                    + (f'  — {abs(r.gap_pct_of_company):,.0f}% worse in relative terms'
                       if pd.notna(r.gap_pct_of_company) else '')),
            ('Rank among VPs', f'{r.rank_in_metric} of {r.vps_in_metric}'
             if pd.notna(r.rank_in_metric) else '—'),
            ('Base', f'{r.weeks_observed} observed weeks'),
        ]))
        B.append(p(f'Why it matters. {why}'))
        B.append(p('Recommended actions:'))
        B.append(bullets(actions))

        if naming == 'efficiency':
            d = _named_efficiency(vp)
            if len(d):
                B.append(p(f'Technicians furthest below this VP\'s median of '
                           f'{d["vp_median_tickets_per_active_day"].iloc[0]:,.2f} tickets per '
                           f'active day (eligible technicians only, at least '
                           f'{RECS_MIN_ACTIVE_DAYS} active weekdays in the window):'))
                B.append(table(
                    ['Technician', 'Main site', 'Tickets/active day', 'vs VP median',
                     'Percentile in VP', 'Tickets', 'Active days'],
                    [[f'{t.techfirstname} {t.techlastname}', t.main_warehouse or '—',
                      t.tickets_per_active_day, f'{t.eff_vs_vp_median_pct:,.0f}%',
                      _ordinal(t.eff_percentile_in_vp), t.total_tickets, t.active_weekdays]
                     for t in d.itertuples(index=False)],
                    widths=[20, 18, 12, 10, 12, 9, 9]))
                B.append(caveat(COACHING_CAVEAT))
            else:
                B.append(p(f'No eligible technician in this VP sits more than '
                           f'{RECS_EFF_GAP_PCT:.0f}% below the VP median, so the gap is '
                           f'spread across the whole VP rather than concentrated in '
                           f'individuals. Treat this as a routing or scheduling question.',
                           style='note'))

        elif naming == 'overtime':
            d = _named_overtime(vp)
            if len(d):
                _tot = tech_perf_vp[(tech_perf_vp['vp'] == vp)]['ot_hours'].sum()
                _share = d['ot_hours'].sum() / _tot * 100 if _tot else np.nan
                B.append(p(f'Technicians carrying the most overtime in the window. These '
                           f'{len(d)} account for '
                           f'{_share:,.0f}% of the VP\'s attributed overtime hours:'
                           if pd.notna(_share) else
                           'Technicians carrying the most overtime in the window:'))
                B.append(table(
                    ['Technician', 'Main site', 'OT hours', 'OT % of worked',
                     'vs VP median (pp)', 'Weeks with OT', 'Weeks observed'],
                    [[f'{t.techfirstname} {t.techlastname}', t.main_warehouse or '—',
                      t.ot_hours, f'{t.ot_pct_of_worked:,.1f}%'
                      if pd.notna(t.ot_pct_of_worked) else '—',
                      f'{t.ot_pct_vs_vp_median_pp:+,.1f}'
                      if pd.notna(t.ot_pct_vs_vp_median_pp) else '—',
                      t.weeks_with_ot, t.ot_weeks_observed]
                     for t in d.itertuples(index=False)],
                    widths=[20, 18, 10, 12, 13, 11, 11],
                    caption='Overtime is inferred from de-duplicated daily payroll hours over '
                            'the Thu-Wed payroll week, excluding PTO and holiday. It is '
                            'computed on the PERSON, before any site allocation.'))
                B.append(caveat('High overtime hours are a SCHEDULING and CAPACITY signal '
                                'first. A technician who is repeatedly available to cover '
                                'gaps will appear here, and that is not a performance '
                                'problem. Check the roster and route balance at the named '
                                'sites before raising this with any individual.'))
            else:
                B.append(p('No technician in this VP could be matched to payroll hours with '
                           'overtime in the window, so this recommendation cannot name '
                           'individuals. The VP-level figure still stands.', style='note'))

        elif naming == 'site':
            d = _named_sites(vp, r.metric)
            if len(d):
                B.append(p('Sites driving this figure inside the VP:'))
                B.append(table(['Site', r.metric_label, 'Weeks'],
                               [[t.tech_warehouse or '—', _fmt_val(t.value, r.unit), t.weeks]
                                for t in d.itertuples(index=False)],
                               widths=[45, 40, 15]))
                if r.family == 'lost equipment':
                    B.append(caveat('Lost-equipment attribution is PROXIMITY, NOT FAULT. It is '
                                    'suitable for pattern detection and coaching '
                                    'conversations, and never for discipline, without '
                                    'ticket-level review.'))
            else:
                B.append(p('No site inside this VP has enough observed weeks to be listed '
                           'individually for this metric.', style='note'))

    # ── what NOT to conclude ─────────────────────────────────────────────────
    B += [h2('How to use this document'),
          bullets([
              'These are prompts for a conversation, not findings. Every item states the '
              'figure it rests on so it can be checked before it is acted on.',
              'Technicians are named only where the metric attaches to a person — tickets per '
              'active day and overtime. Lost equipment, attendance and stock outs name SITES, '
              'because the data does not support naming individuals for those.',
              'Tickets per active day divides by days actually worked, so PTO and part-time '
              'schedules do not push a technician onto a list. A technician below the median '
              'is not necessarily underperforming: route density, ticket mix and vehicle '
              'reliability all move this number.',
              'Comparison is against this VP\'s OWN median, not the company\'s, so a VP '
              'running long rural routes is not measured against dense urban ones.',
              'Recent weeks are incomplete for redeliveries and stock outs by construction — '
              'read the trailing figures, not the last week.',
          ]),
          caveat(COACHING_CAVEAT)]
    return B


# ── generate one document per VP ─────────────────────────────────────────────
vp_rec_blocks = {}
vp_rec_paths = {}
_vps = ([v for v in sorted(vp_scorecard['vp'].dropna().unique())]
        if len(vp_scorecard) else [])
for _vp in _vps:
    vp_rec_blocks[_vp] = build_vp_rec_blocks(_vp)
    _safe = re.sub(r'[^A-Za-z0-9]+', '_', str(_vp)).strip('_')
    _pth = os.path.join(OUT_DIR, REPORT_DOCX_VP_RECS.format(vp=_safe))
    if render_docx(vp_rec_blocks[_vp], _pth):
        vp_rec_paths[_vp] = _pth
print(f'VP recommendation documents: {len(vp_rec_paths)} written for {len(_vps)} VP(s)'
      + (f' -> {OUT_DIR}' if vp_rec_paths else ' (python-docx unavailable — SKIPPED)'))
for _vp in _vps:
    _n = len([b for b in vp_rec_blocks[_vp] if b['t'] == 'h3'])
    _named = len(_named_efficiency(_vp)) + len(_named_overtime(_vp))
    print(f'  {_vp}: {_n} recommendation(s), {_named} technician(s) named')

## Cell 23 — CONSOLIDATED PUBLISHABLE PDF *(new in v1.6.0)*

Cover → executive summary → metric dictionary → company page → **each VP's page followed immediately by that VP's recommendations** → metros → warehouses → appendix. Bookmarked by section; one page size throughout. The restatement notice is on page 2, not in the appendix.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSOLIDATED PUBLISHABLE PDF                                       new v1.6.0
#
# One distributable document, assembled in reading order:
#   1  cover
#   2  executive summary — the company headlines, and the restatement notice, first
#   3  metric dictionary (same content as the editable Word file, from one source)
#   4  company dashboard page
#   5  per VP: dashboard page, then that VP's recommendations
#   6  metro dashboard pages
#   7  warehouse dashboard pages
#   8  appendix — limitations and the open questions this pack cannot answer
#
# ONE PAGE SIZE THROUGHOUT. Narrative pages are generated at the dashboard page size rather
# than at Letter, because a distributed PDF that alternates page size makes every reader
# fight their viewer, and scaling twelve-panel dashboard pages down to Letter makes the
# panels unreadable. The text column stays at a readable measure and the surplus page width
# becomes margin — see render_pdf_bytes.
#
# THE RESTATEMENT GOES ON PAGE 2, NOT IN AN APPENDIX. Two figures in this pack move against
# the last one, one of them by ~40%, and one of those has already been quoted at ~120. A
# reader who compares a number to last month's pack without seeing that notice will reach a
# wrong conclusion, so it sits where it cannot be missed.
# ─────────────────────────────────────────────────────────────────────────────
def _latest_trailing(df, col, gcols=(), key=None):
    """Most recent non-null trailing value, for the headline figures on the summary page."""
    if df is None or df.empty or col not in df.columns:
        return np.nan
    tcol = col + TRAILING_SUFFIX
    use = tcol if tcol in df.columns else col
    d = df
    if key is not None and gcols:
        for c, v in zip(gcols, key):
            d = d[d[c] == v]
    d = d[pd.to_numeric(d[use], errors='coerce').notna()]
    if d.empty:
        return np.nan
    return float(pd.to_numeric(d.sort_values('week_start')[use], errors='coerce').iloc[-1])


def build_cover_blocks():
    return [
        p(f'Weekly operations reporting for the company, each VP, each metro and each '
          f'warehouse, with a trailing {ROLL_WEEKS}-week average on every metric.'),
        kv([('Reporting window', f'{FILTER_START} to {FILTER_END}'),
            ('Generated', RUN_DATE),
            ('Source notebook', f'ops_dashboard v{NB_VERSION}'),
            ('Reporting week', f'{PROD_WEEK_LABEL} (overtime uses the {OT_WEEK_LABEL} '
                               f'payroll week)'),
            ('Entities covered', f'company, {len(_pub_vps)} VP(s), '
                                 f'{len(_pub_metros)} metro(s), {len(_pub_whs)} warehouse(s)'),
            ('Companion files', 'the editable Word metric dictionary and per-VP '
                                'recommendation documents, and the Excel workbook holding '
                                'every underlying figure')]),
        caveat('Internal operations reporting. Contains no patient-identifiable information. '
               'Technician names appear in the recommendation sections and are for coaching '
               'and scheduling conversations only — see the caveat printed with each list.'),
        note('Every chart in this document is weekly. The dashed black line is the trailing '
             f'{ROLL_WEEKS}-week average; hatched bars and hollow points are partial weeks at '
             'the window edge and are excluded from that average.'),
    ]


def build_exec_summary_blocks():
    B = [h1('Executive summary')]

    # ── the restatement notice, first ────────────────────────────────────────
    B.append(h2('Read this before comparing any figure to the previous pack'))
    _rec = (tbl_census_denominator_recon.iloc[-1]
            if 'tbl_census_denominator_recon' in globals()
            and len(tbl_census_denominator_recon) else None)
    if _rec is not None:
        B.append(p(
            f'CENSUS PER TECHNICIAN IS RESTATED. The previous pack divided average daily '
            f'census by the average number of technicians ON THE ROAD per weekday and '
            f'labelled it "census per active technician". Divided by technician HEADCOUNT — '
            f'the measure a caseload question actually asks for — the figure is materially '
            f'lower. For {_rec["period"]}: average daily census of '
            f'{_rec["adc"]:,.0f} was carried by {_rec["techs_distinct"]:,.0f} distinct '
            f'technicians ({_rec["avg_active_techs"]:,.0f} on the road on an average weekday, '
            f'i.e. {_rec["attendance_rate_pct"]:.0f}% weekday attendance), so the headline '
            f'reads {_rec["census_per_tech_headcount"]:,.1f} against '
            f'{_rec["census_per_active_tech_weekday"]:,.1f} on the old basis — a restatement '
            f'of {_rec["restatement_vs_v140_pct"]:+.0f}%.'))
        B.append(p(
            'The difference is ATTENDANCE, not census and not productivity. Average daily '
            'census is a stock — patients on service — and a patient does not leave service '
            'because their technician takes PTO, so the caseload is carried by the whole '
            'roster. Both figures are published side by side throughout this pack, with the '
            'attendance wedge shaded between them on every census panel. Any target quoted '
            'against the old number is approximately 40% out.'))
        B.append(table(['Basis', f'{_rec["period"]} value', 'What it answers'],
                       [['Per technician on the roster (HEADLINE)',
                         f'{_rec["census_per_tech_headcount"]:,.1f}',
                         'How many patients does each technician carry?'],
                        ['Per technician on the road per weekday (previous basis)',
                         f'{_rec["census_per_active_tech_weekday"]:,.1f}',
                         'How thin are we in the field on a working day?'],
                        ['Weekday attendance — the wedge between them',
                         f'{_rec["attendance_rate_pct"]:.0f}%',
                         'What share of available weekdays are actually covered?']],
                       widths=[38, 16, 46]))
    B.append(p(
        'VP AND METRO EQUIPMENT-LOSS RATES RISE SHARPLY. Lost equipment per 1,000 '
        'patient-days was previously divided by the WHOLE COMPANY\'s patient-days at VP and '
        'metro grain rather than by that entity\'s own, understating those rows by roughly the '
        'ratio of company census to entity census. Company and warehouse figures were '
        'correct and are unchanged.'))
    B.append(p(
        'PRODUCTIVITY, OVERTIME, REDELIVERIES AND STOCK OUTS ARE NOT AFFECTED by either '
        'restatement. Tickets per active day divides a flow by a flow and its denominator is '
        'reconciled on every run; none of the other three divides by census.'))
    B.append(p(
        f'ALL METRICS ARE NOW WEEKLY with a trailing {ROLL_WEEKS}-week average. Previously '
        'only productivity was weekly and the rest were monthly, so two panels on one page '
        'were two different time bases.'))

    # ── company headlines ────────────────────────────────────────────────────
    B.append(pagebreak())
    B.append(h2('Company position — latest trailing 4-week figures'))
    _rows = []
    for _col, _label, _fam, _dir, _vpf, _cof, _unit in _SCORE_METRICS:
        _fr = globals().get(_cof)
        _v = _latest_trailing(_fr, _col)
        if pd.isna(_v):
            continue
        _rows.append([_label, _fmt_val(_v, _unit),
                      {1: 'higher is better', -1: 'lower is better', 0: 'context'}[_dir], _fam])
    B.append(table(['Metric', 'Company (trailing 4wk)', 'Direction', 'Family'], _rows,
                   widths=[42, 18, 18, 22],
                   caption='The most recent week with a complete trailing window. Partial weeks '
                           'at the window edge are excluded.'))

    # ── where the VPs sit ────────────────────────────────────────────────────
    if len(vp_scorecard):
        B.append(h2('Where each VP sits'))
        _adv = (vp_scorecard[vp_scorecard['is_adverse'] & ~vp_scorecard['too_thin']]
                .groupby('vp', as_index=False)
                .agg(adverse_metrics=('metric', 'nunique'),
                     worst_metric=('metric_label', 'first'),
                     worst_gap_pct=('severity_pct', 'max')))
        _all_vps = pd.DataFrame({'vp': sorted(vp_scorecard['vp'].dropna().unique())})
        _adv = _all_vps.merge(_adv, on='vp', how='left')
        _adv['adverse_metrics'] = _adv['adverse_metrics'].fillna(0).astype(int)
        B.append(table(['VP', 'Metrics adverse vs company', 'Largest gap', 'Worst gap size'],
                       [[r.vp, r.adverse_metrics, r.worst_metric or '—',
                         f'{r.worst_gap_pct:,.0f}% worse' if pd.notna(r.worst_gap_pct) else '—']
                        for r in _adv.itertuples(index=False)],
                       widths=[28, 22, 34, 16],
                       caption='Each VP has a recommendations section immediately after their '
                               'dashboard page.'))
    return B


def build_appendix_blocks():
    B = [h1('Appendix — limitations and open questions')]
    B.append(h2('Limitations that apply to this whole pack'))
    B.append(bullets([v for _, v in _CROSS]))
    B.append(h2('What this pack cannot answer'))
    B.append(bullets([
        'Which denominator a per-technician CASELOAD target should use. Roughly 300 payroll '
        'technicians, roughly 250 attributed to tickets and roughly 180 on the road on an '
        'average weekday give figures about 40% apart end to end. No figure here should be '
        'compared to a target without naming which denominator the target uses.',
        'Whether facility "(F)" and inpatient-unit "(IPU)" census should count toward '
        'technician caseload. Currently excluded, matching the company APC snapshot '
        'definition; the alternative reading is defensible and would raise every census '
        'figure.',
        'Why roughly 300 technicians on payroll become roughly 250 attributed to tickets. '
        'Part is the payroll name match (about 75% ties) and part is genuine non-field roles; '
        'the remainder is unreconciled.',
        'What the virtual warehouse code "Z CS" is and who owns it. It began carrying real '
        'completed field work in Jun-2026, and the pack infers a physical site for those '
        'tickets from the technician\'s own route rather than reading one off the record.',
        'Whether five warehouses that went silent after Jun-2026 closed or were renamed. '
        'Warehouse-level comparisons spanning that month remain provisional.',
        'Whether census rows with a NULL customer value are real patients. They currently '
        'fall outside the filtered census figure because NOT LIKE is NULL-unsafe.',
    ]))
    B.append(h2('Verification status'))
    B.append(p(
        'Every internal reconciliation in the notebook is asserted on each run and printed: '
        'apportioned technician-days must equal distinct (technician, date); the census and '
        'productivity panels must rest on the same technician population; overtime '
        'apportionment must neither create nor destroy hours; and VP patient-days must not '
        'exceed the company\'s. Any failure prints a warning naming the metric that must not '
        'be circulated.'))
    B.append(caveat('Attribution caveats, repeated because they travel with the numbers: '
                    'lost-equipment attribution is PROXIMITY, NOT FAULT and is never grounds '
                    'for discipline without ticket-level review; technicians are ranked on '
                    'tickets per ACTIVE day so that PTO and part-time schedules do not '
                    'distort the ranking; and a warehouse resolved from a virtual code is '
                    'INFERRED from the technician\'s route, not stamped on the ticket.'))
    return B


# ── assemble ─────────────────────────────────────────────────────────────────
_pub_vps = sorted(dash_wk_prod_vp['vp'].dropna().unique()) if len(dash_wk_prod_vp) else []
_pub_metros = (sorted(dash_wk_prod_metro['metro'].dropna().unique())
               if len(dash_wk_prod_metro) else [])
_pub_whs = (sorted(dash_wk_prod_wh['tech_warehouse'].dropna().unique())
            if len(dash_wk_prod_wh) else [])

_publish_path = os.path.join(OUT_DIR, REPORT_PUBLISH_PDF)
if not (_RL_OK and _PYPDF_OK):
    print('*** Consolidated PDF SKIPPED — reportlab and/or pypdf unavailable. '
          'Run Cell 1 (%pip install) and re-run. The dashboard PDF, the Excel workbook and '
          'any Word documents produced above are unaffected.')
else:
    _writer = PdfWriter()
    _sections = []          # (title, first page index) -> becomes the PDF outline

    def _add_pdf(buf, label=None):
        """Append every page of a PDF (BytesIO or path) and record the bookmark."""
        if buf is None:
            return 0
        _start = len(_writer.pages)
        rd = PdfReader(buf)
        for pg in rd.pages:
            _writer.add_page(pg)
        if label:
            _sections.append((label, _start))
        return len(rd.pages)

    def _add_blocks(blocks, label=None, title=None, subtitle=None, big_title=False):
        return _add_pdf(render_pdf_bytes(blocks, title=title, subtitle=subtitle,
                                         big_title=big_title), label)

    _n = 0
    # The cover is a title page, so it stays single-column and centred.
    _n += _add_pdf(render_pdf_bytes(build_cover_blocks(),
                                    title='DME Express — Operations Report',
                                    subtitle=f'{FILTER_START} to {FILTER_END}  |  generated '
                                            f'{RUN_DATE}',
                                    big_title=True, columns=1), 'Cover')
    _n += _add_blocks(build_exec_summary_blocks(), 'Executive summary')
    _n += _add_blocks(dict_blocks, 'Metric dictionary')

    # company page
    _n += _add_pdf(dash_page_pdfs.get(('Company', 'DME Express — All Operations')),
                   'Company dashboard')

    # each VP: dashboard page, then their recommendations
    for _vp in _pub_vps:
        _n += _add_pdf(dash_page_pdfs.get(('VP', _vp)), f'VP: {_vp}')
        if _vp in vp_rec_blocks:
            _n += _add_blocks(vp_rec_blocks[_vp], f'VP: {_vp} — recommendations')

    for _m in _pub_metros:
        _n += _add_pdf(dash_page_pdfs.get(('Metro', _m)), f'Metro: {_m}')
    if _pub_whs:
        _first_wh = len(_writer.pages)
        for _w in _pub_whs:
            _add_pdf(dash_page_pdfs.get(('Warehouse', _w)))
        _sections.append(('Warehouse pages', _first_wh))

    _n += _add_blocks(build_appendix_blocks(), 'Appendix')

    # A bookmark per section, so an 80-page pack is navigable rather than scrollable.
    for _label, _idx in _sections:
        try:
            _writer.add_outline_item(_label, _idx)
        except Exception:
            pass
    _writer.add_metadata({'/Title': f'DME Express Operations Report {RUN_DATE}',
                          '/Author': 'DME Express Analytics',
                          '/Subject': f'Weekly operations reporting, {FILTER_START} to '
                                      f'{FILTER_END}',
                          '/Keywords': 'internal; no patient identifiers'})
    with open(_publish_path, 'wb') as _f:
        _writer.write(_f)
    print(f'Consolidated publishable PDF: {_publish_path}')
    print(f'  {len(_writer.pages)} pages | {len(_sections)} bookmarked sections | '
          f'cover + exec summary + dictionary + company + {len(_pub_vps)} VP page(s) each '
          f'followed by recommendations + {len(_pub_metros)} metro + {len(_pub_whs)} '
          f'warehouse + appendix')
    # Every entity Cell 18 renders should be here. A silently absent page is how a VP ends
    # up reading a pack that has no page for their region.
    _missing = [k for k in ([('Company', 'DME Express — All Operations')]
                            + [('VP', v) for v in _pub_vps]
                            + [('Metro', m) for m in _pub_metros]
                            + [('Warehouse', w) for w in _pub_whs])
                if k not in dash_page_pdfs]
    if _missing:
        print(f'  *** WARNING: {len(_missing)} expected dashboard page(s) were not captured '
              f'and are absent from the consolidated PDF: {_missing[:5]}. Re-run Cell 18.')

print('\nDeliverables written to ' + OUT_DIR + ':')
for _lbl, _pth in ([('Dashboard PDF (full size, per-entity pages)', DASH_PDF_NAME),
                    ('Excel workbook (every underlying figure)', DASH_XLSX_NAME),
                    ('Metric dictionary (editable Word)', REPORT_DOCX_DICTIONARY),
                    ('Consolidated publishable PDF', REPORT_PUBLISH_PDF)]):
    _ok = os.path.exists(os.path.join(OUT_DIR, _pth))
    print(f'  [{"x" if _ok else " "}] {_lbl}: {_pth}')
for _vp, _pth in sorted(vp_rec_paths.items()):
    print(f'  [x] Recommendations (editable Word) — {_vp}: {os.path.basename(_pth)}')